**MambaVision-T-1K — Causal Attribution Experiment**

**Hugging Face Check point**

In [ ]:
import os, pickle
from huggingface_hub import HfApi, hf_hub_download, create_repo
from kaggle_secrets import UserSecretsClient

HF_REPO_ID = "Toseef11/mambavision-Experiment-checkpoints"   # private dataset repo
LOCAL_CKPT_DIR = "/kaggle/working/mambavision_checkpoints"
os.makedirs(LOCAL_CKPT_DIR, exist_ok=True)

# Token comes from Kaggle Secret "TOKEN" (Add-ons -> Secrets). Never print it.
HF_TOKEN = UserSecretsClient().get_secret("TOKEN")
api = HfApi(token=HF_TOKEN)
create_repo(HF_REPO_ID, repo_type="dataset", private=True, exist_ok=True, token=HF_TOKEN)

def save_checkpoint(name, obj):
    """Pickle obj locally, then upload it to the HF dataset repo as <name>.pkl."""
    path = os.path.join(LOCAL_CKPT_DIR, f"{name}.pkl")
    with open(path, "wb") as f:
        pickle.dump(obj, f)
    api.upload_file(path_or_fileobj=path, path_in_repo=f"{name}.pkl",
                    repo_id=HF_REPO_ID, repo_type="dataset",
                    commit_message=f"save checkpoint {name}")
    print(f"saved checkpoint: {name}")

def load_checkpoint(name):
    """Load <name>.pkl from the local folder, or download it from HF if missing."""
    path = os.path.join(LOCAL_CKPT_DIR, f"{name}.pkl")
    if not os.path.exists(path):
        path = hf_hub_download(repo_id=HF_REPO_ID, filename=f"{name}.pkl",
                               repo_type="dataset", token=HF_TOKEN,
                               local_dir=LOCAL_CKPT_DIR)
    with open(path, "rb") as f:
        return pickle.load(f)

def list_checkpoints():
    """Names of all checkpoints saved in the HF repo."""
    files = api.list_repo_files(HF_REPO_ID, repo_type="dataset")
    return sorted(f[:-4] for f in files if f.endswith(".pkl"))

print("Checkpoints on HF:", list_checkpoints())

In [ ]:
# ============================================================
# STEP 0 — KAGGLE SETUP
# ============================================================
import os, sys, subprocess

# 1) Working folders
PROJECT_DIR = "/kaggle/working"
CKPT_DIR = os.path.join(PROJECT_DIR, "mambavision_checkpoints")
RESULTS_DIR = os.path.join(PROJECT_DIR, "final_results")
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# 2) GPU check
import torch
print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__, "| CUDA:", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("GPU off hai. Kaggle Settings -> Accelerator -> GPU T4 select karein.")

# 3) Hugging Face token (Kaggle Secret "TOKEN"). Never print it.
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, whoami, create_repo

HF_TOKEN = UserSecretsClient().get_secret("TOKEN")
HF_REPO_ID = "Toseef11/mambavision-Experiment-checkpoints"   # private dataset repo
print("HF user:", whoami(token=HF_TOKEN)["name"])
create_repo(HF_REPO_ID, repo_type="dataset", private=True, exist_ok=True, token=HF_TOKEN)
print("HF checkpoint repo ready:", f"https://huggingface.co/datasets/{HF_REPO_ID}")

print("\nSTEP 0 COMPLETE")

## Step 1 — Environment verification ##
Checks PyTorch, CUDA and the GPU (Tesla T4 expected) before loading the model.

In [ ]:
# ============================================================
# STEP 1 — VERIFY KAGGLE PYTORCH / CUDA ENVIRONMENT
# ============================================================

import torch
import torchvision

print("========== STEP 1: ENVIRONMENT CHECK ==========")

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())

    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))
        print(
            f"GPU {i} memory:",
            round(
                torch.cuda.get_device_properties(i).total_memory / 1024**3,
                2
            ),
            "GB"
        )

print("\nSTEP 1 CHECK COMPLETED.")

## Step 2 — Mamba dependencies + Transformers ##
**Installs or verifies `mamba-ssm`, `causal-conv1d` and Transformers, which the MambaVision Mamba mixers need.**

In [ ]:
import sys, glob, subprocess, torch
from importlib.metadata import version as pkg_version

WHEEL_DIR = "/kaggle/input/datasets/everyhing/mamba-deps-wheels-torch-2-11-0-cu128-cuda-12-8"
whls = glob.glob(f"{WHEEL_DIR}/**/*.whl", recursive=True)
print("wheels:", [w.split("/")[-1] for w in whls])
assert any("causal_conv1d" in w for w in whls) and any("mamba_ssm" in w for w in whls), "Wheels nahi mili"

CONSTRAINTS = "/kaggle/working/constraints.txt"
with open(CONSTRAINTS, "w") as f:
    f.write(f"torch=={torch.__version__}\n")
    f.write("transformers==4.51.3\nhuggingface_hub==0.36.2\n")
    f.write("tokenizers>=0.21,<0.22\nsafetensors>=0.4.3,<0.6\n")
    f.write("timm==1.0.29\nnumpy==2.1.3\n")

def pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-c", CONSTRAINTS, *args])
    if r.returncode != 0:
        raise RuntimeError(f"pip install fail hua: {args}")

pip("einops", "packaging", "ninja", "setuptools<82", "wheel")
pip("--no-deps", "--force-reinstall", *whls)          # build nahi, sirf wheels install
pip("transformers==4.51.3", "huggingface_hub==0.36.2",
    "tokenizers>=0.21,<0.22", "safetensors>=0.4.3,<0.6",
    "timm==1.0.29", "numpy==2.1.3")

# Verification
import transformers, huggingface_hub, causal_conv1d, mamba_ssm
print("transformers  :", transformers.__version__, "(want 4.51.3)")
print("huggingface_hub:", huggingface_hub.__version__, "(want 0.36.2)")
print("causal-conv1d :", pkg_version("causal-conv1d"))
print("mamba-ssm     :", pkg_version("mamba-ssm"))
assert transformers.__version__ == "4.51.3" and huggingface_hub.__version__ == "0.36.2"
print("STEP 2 COMPLETE")

## Step 3 — Load MambaVision-T-1K ##
**Downloads `nvidia/MambaVision-T-1K` and loads it**

In [ ]:
# ============================================================
# STEP 3 — LOAD MAMBAVISION-T-1K
# ============================================================
import torch
from transformers import AutoModelForImageClassification

print("========== STEP 3: LOAD MAMBAVISION-T-1K ==========")

model_id = "nvidia/MambaVision-T-1K"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("Device:", device)

print("\nLoading MambaVision-T-1K...")

# transformers downloads and caches the model itself, so we do not call
# snapshot_download (it fails when huggingface_hub versions are mismatched).
model = AutoModelForImageClassification.from_pretrained(
    model_id, trust_remote_code=True, low_cpu_mem_usage=False
)
model = model.to(device)
model.eval()

print("\n========== MODEL LOADED ==========")
print("Model type:", type(model).__name__)
print("Model device:", next(model.parameters()).device)
print("Output classes:", model.model.head.out_features)

meta_params = [name for name, param in model.named_parameters() if param.is_meta]
print("Meta parameters:", len(meta_params))
print("\nSTEP 3 CHECK COMPLETED.")

## Step 4 — Correct preprocessing ##
Builds the timm evaluation transform from the model config (224 center crop) so preprocessing matches the model.

In [ ]:
from timm.data.transforms_factory import create_transform

print("========== STEP 4: PREPROCESSING ==========")

input_resolution = (3, 224, 224)
transform = create_transform(
    input_size=input_resolution,
    is_training=False,
    mean=model.config.mean,
    std=model.config.std,
    crop_mode=model.config.crop_mode,
    crop_pct=model.config.crop_pct
)

# Locked copy: Step 6 restores it, so no later cell can silently change the baseline.
TIMM_TRANSFORM = transform
EXPECTED_BASELINE_CORRECT = 41094   # 82.19% of 50,000 with this transform

print("Using model's own preprocessing config:")
print("  mean =", model.config.mean)
print("  std  =", model.config.std)
print("  crop_mode =", model.config.crop_mode)
print("  crop_pct  =", model.config.crop_pct)
print("\nSTEP 4 CHECK COMPLETED.")

## Step 5 — ImageNet validation dataset and fixed sampling ##
Loads the ImageNet-1k validation split (50,000 images) used for every experiment.

In [ ]:
from datasets import load_dataset

print("========== STEP 5: LOAD IMAGENET VALIDATION DATASET ==========")
dataset = load_dataset("Tsomaros/Imagenet-1k_validation", split="validation")
print("Dataset size:", len(dataset))
print("\nSTEP 5 CHECK COMPLETED.")

Sets the global seed (42), builds the fixed 50,000-index probe order and defines `prepare_batch`.

In [ ]:
import numpy as np

print("========== STEP 5b: RANDOM SAMPLING ==========")
GLOBAL_SEED = 42
_rng = np.random.RandomState(GLOBAL_SEED)
GLOBAL_RANDOM_INDICES = _rng.permutation(len(dataset)).tolist()

print("Dataset size:", len(dataset))
print("Random seed:", GLOBAL_SEED)
print("First 10 shuffled indices:", GLOBAL_RANDOM_INDICES[:10])

def prepare_batch(dataset, indices, transform, device):
    images = []
    labels = []
    for i in indices:
        image = dataset[i]["image"].convert("RGB")
        images.append(transform(image))
        labels.append(dataset[i]["label"])
    images = torch.stack(images).to(device)
    labels = torch.tensor(labels, device=device)
    return images, labels

print("\nSTEP 5b CHECK COMPLETED.")

Defines the batched prediction helpers (`batched_predict`, `batched_predict_with_hook`). Prediction is batched at size 64.

In [ ]:
def batched_predict(model, dataset, indices, transform, device, batch_size=64):
    all_preds, all_labels = [], []
    with torch.no_grad():
        for i in range(0, len(indices), batch_size):
            batch_idx = indices[i:i+batch_size]
            imgs, labels = prepare_batch(dataset, batch_idx, transform, device)
            logits = model(imgs).logits
            preds = logits.argmax(dim=-1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
    return all_preds, all_labels

def batched_predict_with_hook(model, dataset, indices, transform, device, hook_target, hook_fn, batch_size=64):
    handle = hook_target.register_forward_hook(hook_fn)
    all_preds = []
    try:
        with torch.no_grad():
            for i in range(0, len(indices), batch_size):
                batch_idx = indices[i:i+batch_size]
                imgs, _ = prepare_batch(dataset, batch_idx, transform, device)
                logits = model(imgs).logits
                preds = logits.argmax(dim=-1)
                all_preds.extend(preds.cpu().tolist())
    finally:
        handle.remove()
    return all_preds

def mean_ablation_hook(mean_value):
    def hook(module, inp, out):
        is_tuple = isinstance(out, tuple)
        tensor = out[0] if is_tuple else out
        tensor = mean_value.expand_as(tensor).clone()
        return (tensor,) + out[1:] if is_tuple else tensor
    return hook

print("Batch prediction + hook functions ready.")

Collects all 12 mixer blocks (L2B0–L2B7 and L3B0–L3B3) into `mixer_blocks`, tagged as Mamba or Attention.


In [ ]:
import re
mixer_blocks = []
for name, module in model.named_modules():
    if hasattr(module, "mixer"):
        m = re.search(r"levels\.(\d+)\.blocks\.(\d+)$", name)
        if m:
            level_idx, block_idx = m.groups()
            block_name = f"L{level_idx}B{block_idx}"
            mixer_type = type(module.mixer).__name__
            mixer_blocks.append((block_name, mixer_type, module))

print(f"mixer_blocks rebuilt: {len(mixer_blocks)} blocks")

## Step 6 — Full 50,000-image baseline ##
Runs the full 50,000-image baseline. Saves predictions, logits and log-probabilities as `baseline_logits_probs_new`. Expected accuracy 82.19% (0.82188).

In [ ]:
# ============================================================
# STEP 6 — FULL-SCALE BASELINE EVALUATION (+ logits for KL / logit-drop)
# ============================================================

import torch.nn.functional as F

print("========== STEP 6: BASELINE EVALUATION ==========")

N_FULL = 50000
BATCH_SIZE = 64

if N_FULL > len(dataset):
    raise ValueError(f"N_FULL={N_FULL} exceeds dataset size {len(dataset)}.")

eval_indices = GLOBAL_RANDOM_INDICES[:N_FULL]


def batched_predict_with_logits(model, dataset, indices, transform, device, batch_size=64):

    all_preds = []
    all_labels = []
    all_target_logits = []   # logit of the TRUE class, per image
    all_log_probs = []       # full log-softmax, per image (float16, for KL)

    n = len(indices)

    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        batch_idx = indices[start:end]

        imgs, labs = prepare_batch(dataset, batch_idx, transform, device)

        with torch.no_grad():
            outputs = model(imgs)

        logits = outputs["logits"] if isinstance(outputs, dict) else outputs.logits

        log_probs = F.log_softmax(logits, dim=-1)
        target_logits = logits.gather(1, labs.to(logits.device).view(-1, 1)).squeeze(1)

        all_preds.append(logits.argmax(dim=-1).cpu())
        all_labels.append(labs.cpu())
        all_target_logits.append(target_logits.float().cpu())
        all_log_probs.append(log_probs.half().cpu())   # float16 to save memory

        if start % (batch_size * 10) == 0:
            print(f"{end}/{n} images processed")

    return (
        torch.cat(all_preds),
        torch.cat(all_labels),
        torch.cat(all_target_logits),
        torch.cat(all_log_probs),
    )


baseline_preds, eval_labels, baseline_target_logits, baseline_log_probs = batched_predict_with_logits(
    model, dataset, eval_indices, transform, device, BATCH_SIZE
)

baseline_accuracy = (baseline_preds == eval_labels).float().mean().item()
n_correct = int((baseline_preds == eval_labels).sum().item())

print(f"\nFinal baseline accuracy: {baseline_accuracy * 100:.2f}% ({n_correct}/{N_FULL})")
print(f"Baseline target-logit mean: {baseline_target_logits.mean().item():.4f}")

# Environment record (so the baseline can be traced to the software that produced it)
import transformers, huggingface_hub
print("PyTorch:", torch.__version__, "| transformers:", transformers.__version__,
      "| huggingface_hub:", huggingface_hub.__version__)

# persist under a NEW name so the old baseline_logits_probs.pkl is not overwritten
save_checkpoint("baseline_logits_probs_new", {
    "baseline_target_logits": baseline_target_logits,
    "baseline_log_probs": baseline_log_probs,
    "baseline_preds": baseline_preds,
    "eval_labels": eval_labels,
    "baseline_accuracy": baseline_accuracy,
})

print("\nSTEP 6 CHECK COMPLETED (with logits/log-probs saved as baseline_logits_probs_new).")

## Step 7 — Architecture map ##
Prints the architecture map of the 12 mixer blocks (levels, block types, names).

In [ ]:
# ============================================================
# STEP 7 — ARCHITECTURE MAP
# ============================================================

print("========== STEP 7: ARCHITECTURE MAP ==========")


for level_idx, level in enumerate(
    model.model.levels
):

    if not hasattr(level, "blocks"):
        continue

    for block_idx, block in enumerate(
        level.blocks
    ):

        if hasattr(block, "mixer"):

            print(
                f"level{level_idx}.blocks[{block_idx}] "
                f"-> {type(block.mixer).__name__}"
            )


def get_all_mixer_blocks(model):

    blocks = []

    for level_idx, level in enumerate(
        model.model.levels
    ):

        if not hasattr(level, "blocks"):
            continue

        for block_idx, block in enumerate(
            level.blocks
        ):

            if hasattr(block, "mixer"):

                blocks.append(
                    (
                        f"L{level_idx}B{block_idx}",
                        type(block.mixer).__name__,
                        block
                    )
                )

    return blocks


mixer_blocks = get_all_mixer_blocks(model)

print(
    "\nTotal mixer blocks:",
    len(mixer_blocks)
)

print("\nSTEP 7 CHECK COMPLETED.")

## Step 8 — Reference activations / mean ablation ##
Captures reference activations for every block and computes the mean-ablation references.

In [ ]:
# ============================================================
# STEP 8 — MEAN ABLATION REFERENCE ACTIVATIONS
# ============================================================

print("========== STEP 8: REFERENCE ACTIVATIONS ==========")


def capture_all_activations_batch(
    model,
    images,
    mixer_blocks
):

    captured = {}

    def make_hook(name):

        def hook(module, inp, out):

            is_tuple = isinstance(
                out,
                tuple
            )

            t = (
                out[0]
                if is_tuple
                else out
            )

            captured[name] = t.detach()

        return hook


    hooks = [
        block.mixer.register_forward_hook(
            make_hook(name)
        )
        for name, _, block in mixer_blocks
    ]


    with torch.no_grad():

        model(images)


    for h in hooks:
        h.remove()


    return captured


# Fixed random reference set
REF_N = 256

ref_indices = GLOBAL_RANDOM_INDICES[:REF_N]

ref_images, _ = prepare_batch(
    dataset,
    ref_indices,
    transform,
    device
)


ref_acts = capture_all_activations_batch(
    model,
    ref_images,
    mixer_blocks
)


block_means = {

    name:
    act.mean(
        dim=(0, 1),
        keepdim=True
    )

    for name, act in ref_acts.items()

}


def mean_ablation_hook(mean_value):

    def hook(module, inp, out):

        is_tuple = isinstance(
            out,
            tuple
        )

        t = (
            out[0]
            if is_tuple
            else out
        )

        replaced = (
            mean_value
            .expand_as(t)
            .to(t.dtype)
        )

        if is_tuple:

            return (
                (replaced,)
                + out[1:]
            )

        return replaced

    return hook


print(
    f"Computed reference means for "
    f"{len(block_means)} blocks from "
    f"{REF_N} randomly-sampled images."
)

print("\nSTEP 8 CHECK COMPLETED.")

## Step 9 — Block-level causal ablation ##
Mean-ablates each block in turn on the 50K probe and records accuracy change, flips and per-block results.

In [ ]:
# ============================================================
# STEP 9 — BLOCK-LEVEL CAUSAL ABLATION
# ============================================================

print("========== STEP 9: BLOCK-LEVEL CAUSAL ABLATION ==========")


def batched_predict_with_hook(
    model,
    dataset,
    indices,
    transform,
    device,
    hook_target,
    hook_fn,
    batch_size=64
):
    all_preds = []
    n = len(indices)

    h = hook_target.register_forward_hook(hook_fn)

    try:
        for start in range(0, n, batch_size):
            end = min(start + batch_size, n)
            batch_idx = indices[start:end]

            imgs, _ = prepare_batch(dataset, batch_idx, transform, device)

            with torch.no_grad():
                outputs = model(imgs)

            logits = (
                outputs["logits"]
                if isinstance(outputs, dict)
                else outputs.logits
            )
            all_preds.append(logits.argmax(dim=-1).cpu())

    finally:
        h.remove()

    return torch.cat(all_preds)


ablation_results = []
all_ablated_preds = {}

for name, mtype, block in mixer_blocks:
    print(f"\nRunning ablation: {name}")

    mean_val = block_means[name]

    preds = batched_predict_with_hook(
        model,
        dataset,
        eval_indices,
        transform,
        device,
        block.mixer,
        mean_ablation_hook(mean_val),
        BATCH_SIZE
    )

    acc = (preds == eval_labels).float().mean().item()
    delta_pp = (acc - baseline_accuracy) * 100

    all_ablated_preds[name] = preds

    ablation_results.append({
        "block": name,
        "type": mtype,
        "accuracy": acc,
        "delta_pp": delta_pp
    })

    print(f"{name} ({mtype}): {acc * 100:.2f}% (delta {delta_pp:+.2f} pp)")


# ---- Save Step 9 results to Hugging Face ----
save_checkpoint("step9_block_ablation", {"ablation_results": ablation_results})

print("\nSTEP 9 CHECK COMPLETED.")

## Step 10 — Paired CI, McNemar and Bonferroni for block ablation ##
Computes paired bootstrap CIs, McNemar tests and the Bonferroni threshold (0.05/12) for the Step 9 block ablations.

In [ ]:
# ============================================================
# STEP 10 — McNEMAR TEST + PAIRED BOOTSTRAP CI + FLIP COUNTS
# ============================================================
from statsmodels.stats.contingency_tables import mcnemar
import numpy as np
import torch

print("========== STEP 10: McNEMAR + PAIRED CI + FLIP COUNTS ==========")

n_tests = len(ablation_results)
bonferroni_alpha = 0.05 / n_tests
print(f"Bonferroni-corrected significance threshold: p < {bonferroni_alpha:.5f} (for {n_tests} tests)\n")

N_BOOTSTRAP = 2000
rng = np.random.RandomState(GLOBAL_SEED)

# Make sure baseline and labels are on CPU for the comparisons below
baseline_preds_cpu = torch.as_tensor(baseline_preds).cpu()
labels_cpu = torch.as_tensor(eval_labels).cpu()
b_correct = (baseline_preds_cpu == labels_cpu)

for r in ablation_results:
    ab_preds = all_ablated_preds[r["block"]].cpu()
    a_correct = (ab_preds == labels_cpu)

    b_only = int(((b_correct) & (~a_correct)).sum().item())   # correct -> wrong
    c_only = int(((~b_correct) & (a_correct)).sum().item())   # wrong -> correct

    n_flipped = b_only + c_only
    flip_rate = n_flipped / len(labels_cpu)

    table = [[0, b_only], [c_only, 0]]
    pval = mcnemar(table, exact=(n_flipped < 25)).pvalue

    # Paired bootstrap of the accuracy difference (ablated - baseline)
    diffs = a_correct.int().numpy() - b_correct.int().numpy()
    n = len(diffs)
    boot_means = np.empty(N_BOOTSTRAP)
    for i in range(N_BOOTSTRAP):
        idx = rng.randint(0, n, size=n)
        boot_means[i] = diffs[idx].mean()
    ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5]) * 100

    r.update({
        "b_correct_to_wrong": b_only,
        "c_wrong_to_correct": c_only,
        "n_flipped": n_flipped,
        "flip_rate": flip_rate,
        "paired_ci_low_pp": float(ci_low),
        "paired_ci_high_pp": float(ci_high),
        "mcnemar_p": float(pval),
    })

    sig = "SIGNIFICANT (survives Bonferroni)" if pval < bonferroni_alpha else "not significant"
    print(f"{r['block']}: delta={r['delta_pp']:+.2f}pp, b={b_only}, c={c_only}, "
          f"flipped={n_flipped} ({flip_rate*100:.2f}%), "
          f"paired 95% CI=[{ci_low:+.2f}, {ci_high:+.2f}]pp, p={pval:.5f} -- {sig}")


# ---- Save Step 10 results to Hugging Face ----
save_checkpoint("step10", {"ablation_results": ablation_results})

print("\n========== STEP 10 COMPLETE ==========")

In [ ]:
import torch

def batched_predict_with_hook(model, dataset, indices, transform, device,
                              hook_target=None, hook_fn=None, batch_size=64):
    """
    Predictions with an optional forward hook on hook_target.
    hook_target / hook_fn None ho to bina ablation ke predictions aati hain.
    Returns: torch.LongTensor (CPU), length = len(indices)
    """
    model.eval()
    all_preds = []

    handle = None
    if hook_target is not None and hook_fn is not None:
        handle = hook_target.register_forward_hook(hook_fn)

    try:
        with torch.no_grad():
            for i in range(0, len(indices), batch_size):
                batch_idx = indices[i:i + batch_size]
                imgs, _ = prepare_batch(dataset, batch_idx, transform, device)

                out = model(imgs)
                if isinstance(out, dict):
                    logits = out["logits"]
                elif hasattr(out, "logits"):
                    logits = out.logits
                else:
                    logits = out

                preds = logits.argmax(dim=-1)
                all_preds.extend(preds.cpu().tolist())
    finally:
        if handle is not None:
            handle.remove()   # hook hamesha remove ho, error ho ya na ho

    return torch.tensor(all_preds, dtype=torch.long)

## Step 11 — Null distribution rebuild at 50K (channel ablation) ##


Step 11: builds 40 random (block, channel) null trials on the full 50K probe and saves them as `step11_null_50k`.

In [ ]:
# ============================================================
# STEP 11 — NULL DISTRIBUTION (50K)
# One random channel mean-ablated per trial, full 50K probe.
# Tracks delta_pp and flip_rate. Resumable; saves to HF after every trial.
# Checkpoint name kept as "step11b_null_50k" so resume and later cells still work.
# ============================================================
import time
import numpy as np
import torch

print("========== STEP 11: NULL DISTRIBUTION (50K) ==========")

CHECKPOINT_NAME = "step11b_null_50k"
N_TRIALS = 40

try:
    prior = load_checkpoint(CHECKPOINT_NAME)
    trials = prior["trials"]
    print(f"Resuming: {len(trials)} trial(s) already completed.")
except Exception:
    trials = []
    print("No prior progress found — starting fresh.")

# Same seed gives the same trial plan on every run, so resuming is safe
rng = np.random.RandomState(GLOBAL_SEED)
block_names = [name for (name, _, _) in mixer_blocks]
plan, seen = [], set()
while len(plan) < N_TRIALS:
    bname = block_names[rng.randint(0, len(block_names))]
    n_channels = block_means[bname].shape[-1]
    c_idx = rng.randint(0, n_channels)
    key = (bname, c_idx)
    if key in seen:
        continue
    seen.add(key)
    plan.append(key)

print(f"Trial plan — first 5: {plan[:5]} ...")


def single_channel_mean_ablation_hook(mean_value, channel_idx):
    def hook(module, inp, out):
        is_tuple = isinstance(out, tuple)
        t = out[0] if is_tuple else out
        t = t.clone()
        t[..., channel_idx] = mean_value[..., channel_idx]
        return (t,) + out[1:] if is_tuple else t
    return hook


block_module_lookup = {name: block for (name, _, block) in mixer_blocks}
baseline_preds_cpu = torch.as_tensor(baseline_preds).cpu()
eval_labels_cpu = torch.as_tensor(eval_labels).cpu()

for trial_idx in range(len(trials), N_TRIALS):
    bname, c_idx = plan[trial_idx]
    block = block_module_lookup[bname]
    mean_val = block_means[bname]

    t0 = time.time()
    preds = batched_predict_with_hook(
        model, dataset, eval_indices, transform, device,
        block.mixer,
        single_channel_mean_ablation_hook(mean_val, c_idx),
        BATCH_SIZE
    ).cpu()
    elapsed = time.time() - t0

    acc = (preds == eval_labels_cpu).float().mean().item()
    delta_pp = (acc - baseline_accuracy) * 100.0
    n_flipped = int((preds != baseline_preds_cpu).sum().item())
    flip_rate = n_flipped / len(eval_labels_cpu) * 100.0

    trials.append({
        "trial_idx": trial_idx,
        "block": bname,
        "channel": int(c_idx),
        "accuracy": acc,
        "delta_pp": delta_pp,
        "n_flipped": n_flipped,
        "flip_rate": flip_rate,
        "elapsed_sec": elapsed,
    })

    # ---- Save progress to Hugging Face after every trial (resumable) ----
    save_checkpoint(CHECKPOINT_NAME, {
        "trials": trials,
        "plan": plan,
        "N_TRIALS": N_TRIALS,
        "baseline_accuracy": baseline_accuracy,
        "probe_size": len(eval_indices),
        "definition": (
            "Null distribution = accuracy-change (delta_pp) and flip-rate from "
            "mean-ablating ONE randomly selected channel in ONE randomly selected "
            "block (Mamba or Attention), evaluated on the full 50,000-image "
            "validation probe, across 40 independent (block, channel) draws "
            "(seed=42, sampled without replacement)."
        ),
    })

    print(f"[{trial_idx + 1}/{N_TRIALS}] block={bname} ch={c_idx} "
          f"delta_pp={delta_pp:+.4f}pp  flip_rate={flip_rate:.4f}%  "
          f"({elapsed:.1f}s) -> saved")

deltas = np.array([t["delta_pp"] for t in trials])
flips = np.array([t["flip_rate"] for t in trials])
print("\n=== STEP 11 NULL DISTRIBUTION (50K, n=40) SUMMARY ===")
print(f"delta_pp   mean={deltas.mean():+.4f}  std={deltas.std():.4f}")
print(f"flip_rate  mean={flips.mean():.4f}  std={flips.std():.4f}")
print("All trials saved to HF checkpoint:", CHECKPOINT_NAME)

## STEP 11b — LOAD SAVED RESULTS (session resume) + save summary to HF ##

In [ ]:
# ============================================================
# STEP 11b — LOAD SAVED RESULTS (session resume) + save summary to HF
# ============================================================
import numpy as np
import torch

N_FULL = 50000
BATCH_SIZE = 64
eval_indices = GLOBAL_RANDOM_INDICES[:N_FULL]

# ---- Step 6 baseline ----
_b = load_checkpoint("baseline_logits_probs_new")
baseline_preds         = torch.as_tensor(_b["baseline_preds"]).cpu()
eval_labels            = torch.as_tensor(_b["eval_labels"]).cpu()
baseline_target_logits = torch.as_tensor(_b["baseline_target_logits"]).float().cpu()
baseline_log_probs     = torch.as_tensor(_b["baseline_log_probs"]).cpu()      # float16 (50000, 1000)
baseline_accuracy      = float(_b["baseline_accuracy"])

eval_labels_np    = eval_labels.numpy()
baseline_preds_np = baseline_preds.numpy()
baseline_correct  = (baseline_preds_np == eval_labels_np)

# Image order must match the Step 5b permutation used now
_chk = np.random.RandomState(0).choice(N_FULL, 300, replace=False)
_ds_labels = np.array([dataset[eval_indices[i]]["label"] for i in _chk])
assert (_ds_labels == eval_labels_np[_chk]).all(), "Label order mismatch: Step 5b permutation differs."
assert abs(baseline_correct.mean() - baseline_accuracy) < 1e-6

# ---- Step 10 block ablation ----
step10 = load_checkpoint("step10")
ablation_results = step10["ablation_results"]
real_by_block = {r["block"]: r for r in ablation_results}

# ---- Step 11 null distribution (read only — never overwritten here) ----
_s11 = load_checkpoint("step11b_null_50k")
trials = _s11["trials"]
deltas = np.array([t["delta_pp"] for t in trials])
flips  = np.array([t["flip_rate"] for t in trials])          # already in %

# ---- Step 8 reference means (must be in memory from Step 8) ----
assert "block_means" in globals() and len(block_means) == 12, "Run Step 8 first."
block_means_cpu = {k: v.detach().float().cpu() for k, v in block_means.items()}
step8_max_diff = None
try:
    _s8 = load_checkpoint("step8")
    step8_max_diff = max((block_means_cpu[k] - torch.as_tensor(_s8["block_means"][k]).float()).abs().max().item()
                         for k in block_means_cpu)
    print(f"step8 checkpoint found; max |diff| vs recomputed Step 8 means = {step8_max_diff:.2e}")
except Exception:
    save_checkpoint("step8", {"block_means": block_means_cpu, "REF_N": REF_N})
    print("step8 checkpoint saved to HF (block_means, REF_N).")

mixer_by_name = {name: block.mixer for name, _, block in mixer_blocks}
type_by_name  = {name: mtype for name, mtype, _ in mixer_blocks}

# ---- Summary -> Hugging Face ----
step11b_summary = {
    "baseline_accuracy": baseline_accuracy,
    "baseline_n_correct": int(baseline_correct.sum()),
    "N_FULL": N_FULL,
    "label_order_check": "passed (300 random images)",
    "step8_max_abs_diff": step8_max_diff,
    "null_n_trials": len(trials),
    "null_delta_pp_mean": float(deltas.mean()),
    "null_delta_pp_sd": float(deltas.std(ddof=1)),
    "null_delta_pp_min": float(deltas.min()),
    "null_delta_pp_max": float(deltas.max()),
    "null_abs_delta_pp_mean": float(np.abs(deltas).mean()),
    "null_flip_pct_mean": float(flips.mean()),
    "null_flip_pct_sd": float(flips.std(ddof=1)),
    "real_blocks": [
        {"block": r["block"], "type": r["type"], "delta_pp": float(r["delta_pp"]),
         "flip_pct": float(r["flip_rate"] * 100), "mcnemar_p": float(r["mcnemar_p"])}
        for r in ablation_results
    ],
}
save_checkpoint("step11b_summary", step11b_summary)

print(f"\nBaseline: {baseline_accuracy*100:.3f}% ({int(baseline_correct.sum())}/{N_FULL})")
print(f"Step 10 rows: {len(ablation_results)} | Step 11 null trials: {len(trials)}")
print(f"Null delta_pp mean={deltas.mean():+.4f}  sd={deltas.std(ddof=1):.4f}  | null flip% mean={flips.mean():.4f}")
print("Expected: baseline 82.188%, null mean -0.0030, sd 0.0092, flip% 0.1307")
print("STEP 11b COMPLETE — summary saved to HF as 'step11b_summary'")

## Cell B (helpers) ##

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar
import torch.nn.functional as F

def clear_all_hooks():
    for _, _, blk in mixer_blocks:
        for m in (blk, blk.mixer):
            m._forward_hooks.clear()
            m._forward_pre_hooks.clear()

def eval_pass(indices=None, logit_kl=False):
    indices = eval_indices if indices is None else indices
    if logit_kl:
        assert list(indices[:20]) == list(eval_indices[:20]), "logit/KL needs the eval_indices order"
    preds, drops, kls = [], [], []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(indices), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(indices))
            imgs, labs = prepare_batch(dataset, indices[start:end], transform, device)
            out = model(imgs)
            logits = (out["logits"] if isinstance(out, dict) else out.logits).float()
            preds.append(logits.argmax(-1).cpu())
            if logit_kl:
                tgt = logits.gather(1, labs.view(-1, 1)).squeeze(1)
                drops.append((baseline_target_logits[start:end].to(device) - tgt).cpu())
                base_lp = baseline_log_probs[start:end].to(device).float()
                kls.append((base_lp.exp() * (base_lp - F.log_softmax(logits, -1))).sum(-1).cpu())
    preds = torch.cat(preds).numpy()
    if logit_kl:
        return preds, torch.cat(drops).numpy(), torch.cat(kls).numpy()
    return preds

def paired_stats(new_preds, n_boot=2000, seed=42):
    new_preds = np.asarray(new_preds)
    n = len(new_preds)
    ref_correct = baseline_correct[:n]
    new_correct = (new_preds == eval_labels_np[:n])
    b = int((ref_correct & ~new_correct).sum())
    c = int((~ref_correct & new_correct).sum())
    p = mcnemar([[0, b], [c, 0]], exact=(b + c) < 25).pvalue if (b + c) > 0 else 1.0
    diff = new_correct.astype(np.int8) - ref_correct.astype(np.int8)
    rng = np.random.RandomState(seed)
    boot = np.array([diff[rng.randint(0, n, n)].mean() for _ in range(n_boot)]) * 100
    lo, hi = np.percentile(boot, [2.5, 97.5])
    stats = {
        "accuracy": float(new_correct.mean()),
        "delta_pp": float((new_correct.mean() - ref_correct.mean()) * 100),
        "b_correct_to_wrong": b, "c_wrong_to_correct": c,
        "n_flipped": b + c, "flip_rate_pct": float((b + c) / n * 100),
        "pred_change_pct": float((new_preds != baseline_preds_np[:n]).mean() * 100),
        "ci_low_pp": float(lo), "ci_high_pp": float(hi), "mcnemar_p": float(p),
    }
    return stats, new_correct

def pack(correct):   return np.packbits(np.asarray(correct, dtype=bool))
def unpack(packed):  return np.unpackbits(packed)[:N_FULL].astype(bool)

BONF_12 = 0.05 / 12
print("Helpers ready. Bonferroni threshold for 12 tests:", BONF_12)

## Step 11c (Mann-Whitney) ##

In [ ]:
from scipy.stats import mannwhitneyu

real_deltas = np.array([r["delta_pp"] for r in ablation_results])
real_flips  = np.array([r["flip_rate"] * 100 for r in ablation_results])   # fraction -> %

u_acc,  p_acc  = mannwhitneyu(np.abs(real_deltas), np.abs(deltas), alternative="greater")
u_flip, p_flip = mannwhitneyu(real_flips, flips, alternative="greater")

mamba_real = [r["delta_pp"] for r in ablation_results if r["type"] == "MambaVisionMixer"]
attn_real  = [r["delta_pp"] for r in ablation_results if r["type"] == "Attention"]
null_mean_abs = np.mean(np.abs(deltas))

print(f"{'Block':6s} {'Type':17s} {'Real Δpp':>9s} {'Real flip%':>10s} {'% null |Δ| >= real':>19s}")
for r in ablation_results:
    frac = (np.abs(deltas) >= abs(r["delta_pp"])).mean() * 100
    print(f"{r['block']:6s} {r['type']:17s} {r['delta_pp']:+9.3f} {r['flip_rate']*100:10.3f} {frac:18.1f}%")

partA_50k = {
    "u_accuracy_change": float(u_acc), "p_accuracy_change": float(p_acc),
    "u_flip_rate": float(u_flip), "p_flip_rate": float(p_flip),
    "mamba_bc_ratio": float(np.mean(np.abs(mamba_real)) / null_mean_abs),
    "attn_bc_ratio":  float(np.mean(np.abs(attn_real))  / null_mean_abs),
    "null_probe_size": N_FULL, "null_n_trials": len(deltas),
    "flip_note": "real flip = b+c (correctness flips); null flip = any prediction change (>= b+c), so the flip test is conservative",
}
print()
for k, v in partA_50k.items():
    print(f"  {k}: {v}")
save_checkpoint("partA_mannwhitney_50k", partA_50k)

## Step 11d — Grouped ablation ##

In [ ]:
CKPT_PARTC = "partC_grouped_ablation"
try:
    partC = load_checkpoint(CKPT_PARTC)
    print("Resuming partC, keys:", list(partC.keys()))
except Exception:
    partC = {}

for group, key in [("MambaVisionMixer", "mamba"), ("Attention", "attn")]:
    if f"{key}_delta_pp" in partC:
        print(f"[skip] all-{group}-off already done: {partC[f'{key}_delta_pp']:+.3f} pp")
        continue
    clear_all_hooks()
    handles = [blk.mixer.register_forward_hook(mean_ablation_hook(block_means[name]))
               for name, mtype, blk in mixer_blocks if mtype == group]
    try:
        preds = eval_pass()
    finally:
        for h in handles:
            h.remove()
    st, corr = paired_stats(preds)
    partC.update({
        f"{key}_group_acc": st["accuracy"] * 100,
        f"{key}_delta_pp": st["delta_pp"],
        f"{key}_ci_low_pp": st["ci_low_pp"], f"{key}_ci_high_pp": st["ci_high_pp"],
        f"{key}_flips": st["n_flipped"], f"{key}_mcnemar_p": st["mcnemar_p"],
        f"{key}_correct_packed": pack(corr),
        "baseline_acc_pct": baseline_accuracy * 100,
    })
    save_checkpoint(CKPT_PARTC, partC)
    print(f"All-{group}-off: acc={st['accuracy']*100:.3f}%  Δ={st['delta_pp']:+.3f} pp  "
          f"CI[{st['ci_low_pp']:+.3f},{st['ci_high_pp']:+.3f}]  flips={st['n_flipped']}")

print(f"\nMamba off: {partC['mamba_delta_pp']:+.3f} pp | Attention off: {partC['attn_delta_pp']:+.3f} pp")

## Step 12 — Class-Difficulty Analysis of Block Ablation ##

### Step 12A — Per-Image Block Ablation Predictions ###

In [ ]:
# ============================================================
# STEP 12 — Rebuild per-image predictions of the Step 9 block ablations
# Saves to HF after every block as "step9_ablated_preds" (resumable)
# ============================================================
CKPT_PREDS = "step9_ablated_preds"

if "all_ablated_preds" in globals() and len(all_ablated_preds) == 12:
    print("all_ablated_preds already in memory — nothing to rebuild.")
else:
    try:
        _saved_preds = load_checkpoint(CKPT_PREDS)
    except Exception:
        _saved_preds = {}
    all_ablated_preds = {k: torch.as_tensor(v.astype(np.int64)) for k, v in _saved_preds.items()}
    print(f"Loaded {len(all_ablated_preds)}/12 blocks from checkpoint.")

    for name, mtype, blk in mixer_blocks:
        if name in all_ablated_preds:
            continue
        clear_all_hooks()
        h = blk.mixer.register_forward_hook(mean_ablation_hook(block_means[name]))
        try:
            preds = eval_pass()
        finally:
            h.remove()
        acc = (preds == eval_labels_np).mean()
        stored = real_by_block[name]["accuracy"]
        flag = "OK" if abs(acc - stored) < 5e-4 else "CHECK: differs from Step 10"
        print(f"{name}: acc={acc:.5f}  (Step 10: {stored:.5f})  {flag}")
        all_ablated_preds[name] = torch.as_tensor(preds)
        _saved_preds[name] = preds.astype(np.int16)
        save_checkpoint(CKPT_PREDS, _saved_preds)

print("all_ablated_preds ready:", sorted(all_ablated_preds))

### Step 12B — Stratified Ablation by Class Difficulty ###

In [ ]:
# ============================================================
# STEP 12B — Stratified Ablation by Class Difficulty
# Saves to HF as "step12_stratified"
# ============================================================
labels_np = eval_labels_np
classes_present = np.unique(labels_np)
per_class_acc = {int(c): float((baseline_preds_np[labels_np == c] == c).mean()) for c in classes_present}

sorted_classes = sorted(per_class_acc, key=lambda c: per_class_acc[c])
n_classes = len(sorted_classes)
hard_classes   = set(sorted_classes[: n_classes // 3])
medium_classes = set(sorted_classes[n_classes // 3 : 2 * n_classes // 3])
easy_classes   = set(sorted_classes[2 * n_classes // 3 :])
strata = {
    "Easy":   np.isin(labels_np, list(easy_classes)),
    "Medium": np.isin(labels_np, list(medium_classes)),
    "Hard":   np.isin(labels_np, list(hard_classes)),
}
for s, mask in strata.items():
    print(f"{s}: {mask.sum()} images, {len(set(labels_np[mask]))} classes, "
          f"baseline acc = {(baseline_preds_np[mask] == labels_np[mask]).mean():.4f}")

stratified_results = []
for block_name, mixer_type, _ in mixer_blocks:
    abl = all_ablated_preds[block_name].numpy()
    row = {"block": block_name, "type": mixer_type}
    for s, mask in strata.items():
        base_c = baseline_preds_np[mask] == labels_np[mask]
        abl_c  = abl[mask] == labels_np[mask]
        row[f"{s}_baseline_acc"]  = float(base_c.mean())
        row[f"{s}_ablated_acc"]   = float(abl_c.mean())
        row[f"{s}_delta_pp"]      = float((abl_c.mean() - base_c.mean()) * 100)
        row[f"{s}_flip_rate_pct"] = float((base_c & ~abl_c).sum() / mask.sum() * 100)   # correct -> wrong
    stratified_results.append(row)
    print(f"{block_name:6s} ({mixer_type:16s}) Easy={row['Easy_delta_pp']:+.3f}  "
          f"Medium={row['Medium_delta_pp']:+.3f}  Hard={row['Hard_delta_pp']:+.3f} pp")

save_checkpoint("step12_stratified", {
    "stratified_results": stratified_results,
    "easy_classes": sorted(easy_classes), "medium_classes": sorted(medium_classes),
    "hard_classes": sorted(hard_classes), "per_class_acc": per_class_acc,
})
print("\nSTEP 12B COMPLETE — saved to HF as 'step12_stratified'")

## Step 13 — Freezing of Δ, B, C in Mamba blocks ##
Step 13 part A: captures reference dt/B/C values (mean, global, position and shuffled modes) for each Mamba block.

In [ ]:
# ============================================================
# STEP 13A — Capture reference dt / B / C per Mamba block
# Saves to HF as "step13_freeze_refs"
# ============================================================
import sys

clear_all_hooks()
mamba_block_names   = [name for name, mtype, _ in mixer_blocks if mtype == "MambaVisionMixer"]
mamba_block_modules = {name: mod for name, mtype, mod in mixer_blocks if mtype == "MambaVisionMixer"}

target_module = sys.modules[type(mamba_block_modules[mamba_block_names[0]].mixer).__module__]
if "_original_selective_scan_fn" not in globals():
    _original_selective_scan_fn = target_module.selective_scan_fn   # keep the true original once
target_module.selective_scan_fn = _original_selective_scan_fn

_active_block_name = [None]
def _pre(name):
    def f(module, inp):
        _active_block_name[0] = name
    return f
def _post(module, inp, out):
    _active_block_name[0] = None

def install_block_name_hooks():
    hs = []
    for name, mod in mamba_block_modules.items():
        hs.append(mod.mixer.register_forward_pre_hook(_pre(name)))
        hs.append(mod.mixer.register_forward_hook(_post))
    return hs

_captured_by_block = {name: {} for name in mamba_block_names}
def _capturing_fn(x, dt, A, B, C, D, z=None, delta_bias=None, delta_softplus=None, return_last_state=None):
    name = _active_block_name[0]
    if name is not None:
        _captured_by_block[name].update({"dt": dt.detach().clone(), "B": B.detach().clone(), "C": C.detach().clone()})
    return _original_selective_scan_fn(x, dt, A, B, C, D, z=z, delta_bias=delta_bias,
                                       delta_softplus=delta_softplus, return_last_state=return_last_state)

_hs = install_block_name_hooks()
target_module.selective_scan_fn = _capturing_fn
try:
    with torch.no_grad():
        model(ref_images)
finally:
    target_module.selective_scan_fn = _original_selective_scan_fn
    for h in _hs:
        h.remove()

_freeze_refs = {}
for bi, name in enumerate(mamba_block_names):
    refs = {}
    for ti, tname in enumerate(("dt", "B", "C")):
        t = _captured_by_block[name][tname]                       # (N_ref, channels, L)
        mean_t = t.mean(dim=(0, 2), keepdim=True)                 # (1, channels, 1)
        pos_t  = t.mean(dim=0, keepdim=True)                      # (1, channels, L)
        gen = torch.Generator().manual_seed(GLOBAL_SEED + 10 * bi + ti)   # fixed seed -> reproducible shuffle
        perm = torch.randperm(mean_t.shape[1], generator=gen).to(mean_t.device)
        refs[tname] = {"mean": mean_t, "position": pos_t, "global": mean_t.mean(),
                       "shuffle": mean_t[:, perm, :]}
    _freeze_refs[name] = refs
    print(f"{name}: dt {tuple(_captured_by_block[name]['dt'].shape)}  "
          f"B {tuple(_captured_by_block[name]['B'].shape)}  C {tuple(_captured_by_block[name]['C'].shape)}")

del _captured_by_block   # free GPU memory (256-image tensors)
torch.cuda.empty_cache()

save_checkpoint("step13_freeze_refs", {
    name: {t: {m: v.detach().cpu() for m, v in modes.items()} for t, modes in refs.items()}
    for name, refs in _freeze_refs.items()
})
print("STEP 13A COMPLETE — references for:", list(_freeze_refs))

Step 13 part B: installs the freezing patch, which replaces Δ, B or C with the reference values when active.

In [ ]:
# ============================================================
# STEP 13B — Freezing patch (defined here, switched on only inside 13C / 13D / 14)
# ============================================================
_freeze_config = {"active": False, "block_names": set(), "tensor": None, "mode": None}

def _apply_mode(orig, ref, mode):
    if mode == "global":
        return torch.full_like(orig, ref["global"].item())
    return ref[mode].expand_as(orig).to(device=orig.device, dtype=orig.dtype)

def _patched_fn(x, dt, A, B, C, D, z=None, delta_bias=None, delta_softplus=None, return_last_state=None):
    name = _active_block_name[0]
    if _freeze_config["active"] and name in _freeze_config["block_names"]:
        ref, mode, tt = _freeze_refs[name], _freeze_config["mode"], _freeze_config["tensor"]
        if tt in ("delta", "all"): dt = _apply_mode(dt, ref["dt"], mode)
        if tt in ("B", "all"):     B  = _apply_mode(B,  ref["B"],  mode)
        if tt in ("C", "all"):     C  = _apply_mode(C,  ref["C"],  mode)
    return _original_selective_scan_fn(x, dt, A, B, C, D, z=z, delta_bias=delta_bias,
                                       delta_softplus=delta_softplus, return_last_state=return_last_state)

def install_freeze_patch():
    clear_all_hooks()
    hs = install_block_name_hooks()
    target_module.selective_scan_fn = _patched_fn
    return hs

def remove_freeze_patch(hs):
    _freeze_config["active"] = False
    target_module.selective_scan_fn = _original_selective_scan_fn
    for h in hs:
        h.remove()

print("STEP 13B COMPLETE — freezing patch defined (not active yet).")

**Step 13 part C: small sanity check on 500 images before the full run. Prints the expected range.**

In [ ]:
# ============================================================
# STEP 13C — Sanity check on 500 images before the full 50K grid
# ============================================================
sanity_idx = eval_indices[:500]
clear_all_hooks()
plain = eval_pass(sanity_idx)                          # no patch
_hs = install_freeze_patch()
try:
    ctrl = eval_pass(sanity_idx)                       # patch on, nothing frozen
    assert (ctrl == plain).all(), "Patch changes predictions while inactive — stop and check."
    _freeze_config.update({"active": True, "block_names": {"L2B0"}, "tensor": "all", "mode": "mean"})
    test = eval_pass(sanity_idx)
finally:
    remove_freeze_patch(_hs)

base500 = baseline_correct[:500].mean()
frozen500 = (test == eval_labels_np[:500]).mean()
print("Control (inactive patch) identical to unpatched model: OK")
print(f"L2B0 all-frozen (mean), 500 images: baseline={base500:.4f}  frozen={frozen500:.4f}  "
      f"delta_pp={(frozen500 - base500)*100:+.3f}  | predictions changed: {(test != plain).sum()}/500")
print("Expect roughly −1 to +1 pp; 'predictions changed' must be > 0 (proves freezing is active).")

**Step 13 part D: full 50K freezing grid (54 Configruations)**

In [ ]:
# ============================================================
# STEP 13D — Full 50K freezing grid (54 configurations)
# Resumable: saves to HF as "step13_freezing_50k" after every configuration
# ============================================================
CKPT_FREEZE = "step13_freezing_50k"

configs = []
for name in mamba_block_names:
    configs += [{"block": name, "tensor": "delta", "mode": m} for m in ["mean", "global", "position", "shuffle"]]
    configs += [{"block": name, "tensor": "B", "mode": m} for m in ["mean", "global"]]
    configs += [{"block": name, "tensor": "C", "mode": m} for m in ["mean", "global"]]
    configs += [{"block": name, "tensor": "all", "mode": "mean"}]
print(f"Total configurations: {len(configs)}")

try:
    results_freeze = load_checkpoint(CKPT_FREEZE)["results"]
    print(f"Resuming: {len(results_freeze)}/{len(configs)} done.")
except Exception:
    results_freeze = []
done_keys = {(r["block"], r["tensor"], r["mode"]) for r in results_freeze}

_hs = install_freeze_patch()
try:
    for cfg in configs:
        key = (cfg["block"], cfg["tensor"], cfg["mode"])
        if key in done_keys:
            continue
        _freeze_config.update({"active": True, "block_names": {cfg["block"]},
                               "tensor": cfg["tensor"], "mode": cfg["mode"]})
        preds, drop, kl = eval_pass(logit_kl=True)
        _freeze_config["active"] = False
        st, _ = paired_stats(preds)
        row = {**cfg, **st,
               "correct_class_logit_drop": float(drop.mean()),
               "kl_divergence": float(kl.mean())}
        results_freeze.append(row)
        save_checkpoint(CKPT_FREEZE, {"results": results_freeze, "configs_total": len(configs)})
        print(f"[{len(results_freeze)}/{len(configs)}] {cfg['block']:6s} {cfg['tensor']:5s} {cfg['mode']:8s} "
              f"Δ={st['delta_pp']:+.3f}pp CI[{st['ci_low_pp']:+.3f},{st['ci_high_pp']:+.3f}] "
              f"p={st['mcnemar_p']:.2e} flip={st['flip_rate_pct']:.3f}% "
              f"logit_drop={row['correct_class_logit_drop']:+.4f} KL={row['kl_divergence']:.5f}")
finally:
    remove_freeze_patch(_hs)

print("\nDONE — all configurations complete." if len(results_freeze) == len(configs)
      else f"\nStopped at {len(results_freeze)}/{len(configs)} — rerun this cell to resume.")

## Step 14 — Freeze all six Mamba blocks together ##
Step 14: freezes Δ, B and C in all six Mamba blocks at once (mean mode) on 50K. Saves `step16_freeze_all_mamba`. Used for the selectivity fraction.

In [ ]:
# ============================================================
# STEP 14 — Freeze all six Mamba blocks together (Δ+B+C, mean mode) + selectivity fraction
# Needs: Step 13A + 13B in this session, and Step 11d saved on HF
# Saves to HF as "step14_freeze_all_mamba"
# ============================================================
if "partC" not in globals():
    partC = load_checkpoint("partC_grouped_ablation")
assert "mamba_correct_packed" in partC, "Run Step 11d (grouped ablation) first."
assert "install_freeze_patch" in globals(), "Run Step 13A and 13B first."

_hs = install_freeze_patch()
try:
    _freeze_config.update({"active": True, "block_names": set(mamba_block_names), "tensor": "all", "mode": "mean"})
    preds_all_frozen = eval_pass()
finally:
    remove_freeze_patch(_hs)

st_frozen, frozen_correct = paired_stats(preds_all_frozen)
mamba_off_correct = unpack(partC["mamba_correct_packed"])

drop_all_frozen_pp = -st_frozen["delta_pp"]
drop_all_mamba_off_pp = -partC["mamba_delta_pp"]
selectivity_fraction = drop_all_frozen_pp / drop_all_mamba_off_pp

# paired bootstrap CI of the fraction (same resampled images in numerator and denominator)
bc = baseline_correct.astype(np.int8)
d_frozen = bc - frozen_correct.astype(np.int8)
d_off    = bc - mamba_off_correct.astype(np.int8)
rng = np.random.RandomState(42)
boot = []
for _ in range(2000):
    idx = rng.randint(0, N_FULL, N_FULL)
    boot.append(d_frozen[idx].mean() / d_off[idx].mean())
frac_lo, frac_hi = np.percentile(boot, [2.5, 97.5])

print(f"All 6 Mamba blocks frozen: acc={st_frozen['accuracy']:.5f}  Δ={st_frozen['delta_pp']:+.3f} pp "
      f"CI[{st_frozen['ci_low_pp']:+.3f},{st_frozen['ci_high_pp']:+.3f}]  p={st_frozen['mcnemar_p']:.2e}")
print(f"All Mamba off (Step 11d): Δ={partC['mamba_delta_pp']:+.3f} pp")
print(f"Selectivity fraction = {selectivity_fraction:.3f}  95% CI [{frac_lo:.3f}, {frac_hi:.3f}]")

save_checkpoint("step14_freeze_all_mamba", {
    **{f"frozen_{k}": v for k, v in st_frozen.items()},
    "acc_all_frozen": st_frozen["accuracy"], "delta_pp_all_frozen": st_frozen["delta_pp"],
    "drop_all_mamba_off_pp": drop_all_mamba_off_pp,
    "selectivity_fraction": float(selectivity_fraction),
    "selectivity_ci": [float(frac_lo), float(frac_hi)], "n_eval": N_FULL,
})
print("STEP 14 COMPLETE — saved to HF as 'step14_freeze_all_mamba'")

## Step 15 - Resampling Ablation at 50K + Spearman ##

### STEP 15A — Reference bank of real block outputs (256 images, seed 777)

In [ ]:
# ============================================================
# STEP 15A — Reference bank of real block outputs (256 images, seed 777)
# Kept in memory only (~650 MB, too large for HF); rebuild in ~1 min after a restart
# ============================================================
clear_all_hooks()
BANK_SIZE = 256
rng_bank = np.random.RandomState(777)
bank_idx = sorted(rng_bank.choice(eval_indices, size=BANK_SIZE, replace=False).tolist())

_bank_capture = {name: [] for name, _, _ in mixer_blocks}
def _bank_post(name):
    def f(module, inp, out):
        y = out[0] if isinstance(out, tuple) else out
        _bank_capture[name].append(y.detach().cpu())
    return f

_bank_hooks = [blk.mixer.register_forward_hook(_bank_post(name)) for name, _, blk in mixer_blocks]
try:
    with torch.no_grad():
        for start in range(0, BANK_SIZE, BATCH_SIZE):
            imgs, _ = prepare_batch(dataset, bank_idx[start:start + BATCH_SIZE], transform, device)
            model(imgs)
finally:
    for h in _bank_hooks:
        h.remove()

reference_bank = {name: torch.cat(v, dim=0) for name, v in _bank_capture.items()}
for name, t in reference_bank.items():
    print(f"{name}: bank shape {tuple(t.shape)}")
print("STEP 15A COMPLETE")

### STEP 15B — Resampling hooks (seeded) + quick test on 500 images

In [ ]:
# ============================================================
# STEP 15B — Resampling hooks (seeded) + quick test on 500 images
# ============================================================
clear_all_hooks()
_resample_config = {"active": False, "block_name": None}
_resample_gens = {}

def _make_resample_hook(name):
    def f(module, inp, out):
        if not _resample_config["active"] or _resample_config["block_name"] != name:
            return out
        is_tuple = isinstance(out, tuple)
        t = out[0] if is_tuple else out
        bank = reference_bank[name]
        idx = torch.randint(0, bank.shape[0], (t.shape[0],), generator=_resample_gens[name])
        replacement = bank[idx].to(device=t.device, dtype=t.dtype)
        return (replacement,) + out[1:] if is_tuple else replacement
    return f

def install_resample_hooks():
    clear_all_hooks()
    return [blk.mixer.register_forward_hook(_make_resample_hook(name)) for name, _, blk in mixer_blocks]

_hs = install_resample_hooks()
try:
    _resample_gens["L2B0"] = torch.Generator().manual_seed(777)
    _resample_config.update({"active": True, "block_name": "L2B0"})
    test = eval_pass(eval_indices[:500])
finally:
    _resample_config["active"] = False
    for h in _hs:
        h.remove()
acc_test = (test == eval_labels_np[:500]).mean()
print(f"Resampled L2B0, 500 images: acc={acc_test:.4f}  baseline={baseline_correct[:500].mean():.4f}  "
      f"Δ={(acc_test - baseline_correct[:500].mean())*100:+.2f} pp")

### STEP 15C — Resampling ablation at 50K (all 12 blocks)

In [ ]:
# ============================================================
# STEP 15C — Resampling ablation at 50K (all 12 blocks)
# Resumable: saves to HF as "resampling_50k_v2" after every block
# Needs 15A + 15B in this session
# ============================================================
CKPT_RS = "resampling_50k_v2"
RESET = False

try:
    results_rs = [] if RESET else load_checkpoint(CKPT_RS)["results"]
except Exception:
    results_rs = []
print(f"Resampling: {len(results_rs)}/12 already done.")
done_blocks = {r["block"] for r in results_rs}

_hs = install_resample_hooks()
try:
    for bi, (name, mtype, _) in enumerate(mixer_blocks):
        if name in done_blocks:
            continue
        _resample_gens[name] = torch.Generator().manual_seed(777 + bi)
        _resample_config.update({"active": True, "block_name": name})
        preds = eval_pass()
        _resample_config["active"] = False
        st, _ = paired_stats(preds)
        results_rs.append({"block": name, "type": mtype, **st})
        save_checkpoint(CKPT_RS, {"results": results_rs})
        print(f"[{len(results_rs)}/12] {name:6s} Δ={st['delta_pp']:+.3f}pp "
              f"CI[{st['ci_low_pp']:+.3f},{st['ci_high_pp']:+.3f}] flip={st['flip_rate_pct']:.3f}% p={st['mcnemar_p']:.2e}")
finally:
    _resample_config["active"] = False
    for h in _hs:
        h.remove()

print("\nDONE." if len(results_rs) == 12 else f"\nStopped at {len(results_rs)}/12 — rerun to resume.")

### STEP 15D — Spearman: mean-ablation vs resampling block ranking

In [ ]:
# ============================================================
# STEP 15D — Spearman: mean-ablation vs resampling block ranking
# Saves to HF as "resampling_vs_meanablation_spearman"
# ============================================================
from scipy.stats import spearmanr

if "results_rs" not in globals() or len(results_rs) < 12:
    results_rs = load_checkpoint("resampling_50k_v2")["results"]

rs_by_block = {r["block"]: r["delta_pp"] for r in results_rs}
blocks = [name for name, _, _ in mixer_blocks]
mean_vals = [real_by_block[b]["delta_pp"] for b in blocks]
rs_vals   = [rs_by_block[b] for b in blocks]

for b, m, r in zip(blocks, mean_vals, rs_vals):
    print(f"{b:6s} mean-ablation={m:+.3f}  resampling={r:+.3f}")
rho, p_rho = spearmanr(mean_vals, rs_vals)
print(f"\nSpearman rho={rho:.3f}, p={p_rho:.2e} (n=12 blocks, same 50K images)")

save_checkpoint("resampling_vs_meanablation_spearman", {
    "blocks": blocks, "mean_ablation_delta": mean_vals, "resampling_delta_pp": rs_vals,
    "spearman_rho": float(rho), "spearman_p": float(p_rho), "n_eval": N_FULL,
})
print("STEP 15D COMPLETE")

## Step 16 — same-size random perturbation (shuffled-channel null) ##

### STEP 16A — Shuffled-channel null hooks + quick test on 500 images

In [ ]:
# ============================================================
# STEP 16A — Shuffled-channel null hooks + quick test on 500 images
# Needs: Step 15A (reference_bank) in this session
# ============================================================
assert "reference_bank" in globals(), "Run Step 15A (bank) first."

torch.manual_seed(42)
channel_perms = {name: torch.randperm(block_means_cpu[name].shape[-1]) for name, _, _ in mixer_blocks}
_shufflenull_config = {"active": False, "block_name": None}

def _make_shufflenull_hook(name):
    mean_val = block_means_cpu[name]
    perm = channel_perms[name]
    def f(module, inp, out):
        if not _shufflenull_config["active"] or _shufflenull_config["block_name"] != name:
            return out
        is_tuple = isinstance(out, tuple)
        t = out[0] if is_tuple else out
        delta = mean_val.to(device=t.device, dtype=t.dtype).expand_as(t) - t
        replacement = t + delta[..., perm.to(t.device)]
        return (replacement,) + out[1:] if is_tuple else replacement
    return f

def install_shufflenull_hooks():
    clear_all_hooks()
    return [blk.mixer.register_forward_hook(_make_shufflenull_hook(name)) for name, _, blk in mixer_blocks]

_hs = install_shufflenull_hooks()
try:
    _shufflenull_config.update({"active": True, "block_name": "L2B0"})
    test = eval_pass(eval_indices[:500])
finally:
    _shufflenull_config["active"] = False
    for h in _hs:
        h.remove()
print(f"Shuffled null L2B0, 500 images: Δ={((test == eval_labels_np[:500]).mean() - baseline_correct[:500].mean())*100:+.2f} pp")

### STEP 16B — Shuffled-channel null, full 50K, all 12 blocks

In [ ]:
# ============================================================
# STEP 16B — Shuffled-channel null, full 50K, all 12 blocks
# Resumable: saves to HF as "shuffled_null_50k" after every block
# ============================================================
CKPT_SN = "shuffled_null_50k"
try:
    results_null = load_checkpoint(CKPT_SN)["results"]
except Exception:
    results_null = []
print(f"Shuffled null: {len(results_null)}/12 already done.")
done_blocks = {r["block"] for r in results_null}

_hs = install_shufflenull_hooks()
try:
    for name, mtype, _ in mixer_blocks:
        if name in done_blocks:
            continue
        _shufflenull_config.update({"active": True, "block_name": name})
        preds = eval_pass()
        _shufflenull_config["active"] = False
        st, corr = paired_stats(preds)
        results_null.append({"block": name, "type": mtype, **st, "correct_packed": pack(corr)})
        save_checkpoint(CKPT_SN, {"results": results_null, "baseline_acc": baseline_accuracy})
        print(f"[{len(results_null)}/12] {name:6s} [{mtype:16s}] Δ={st['delta_pp']:+.3f}pp "
              f"CI[{st['ci_low_pp']:+.3f},{st['ci_high_pp']:+.3f}] flip={st['flip_rate_pct']:.2f}%")
finally:
    _shufflenull_config["active"] = False
    for h in _hs:
        h.remove()
print("\nDONE." if len(results_null) == 12 else f"\nStopped at {len(results_null)}/12 — rerun to resume.")

### STEP 16C — Real ablation vs same-size shuffled perturbation (paired, per block)

In [ ]:
# ============================================================
# STEP 16C — Real ablation vs same-size shuffled perturbation (paired, per block)
# Saves to HF: "damage_vs_perturbation_table", "normalized_perturbation_sizes", "perturbation_sizes"
# ============================================================
if "all_ablated_preds" not in globals() or len(all_ablated_preds) < 12:
    _sp = load_checkpoint("step9_ablated_preds")
    all_ablated_preds = {k: torch.as_tensor(v.astype(np.int64)) for k, v in _sp.items()}
if "results_null" not in globals() or len(results_null) < 12:
    results_null = load_checkpoint("shuffled_null_50k")["results"]
null_by_block = {r["block"]: r for r in results_null}

raw_pert, norm_pert = {}, {}
for name, _, _ in mixer_blocks:
    x = reference_bank[name].float()
    d = block_means_cpu[name].expand_as(x) - x
    d_norm = d.reshape(d.shape[0], -1).norm(dim=1)
    x_norm = x.reshape(x.shape[0], -1).norm(dim=1)
    raw_pert[name] = float(d_norm.mean())
    norm_pert[name] = float((d_norm / x_norm).mean())

rows, n_as_much = [], 0
print(f"{'Block':6s} {'Type':16s} {'RealΔpp':>8s} {'NullΔpp':>8s} {'Real−Null':>9s} {'95% CI':>18s} {'NormPert':>8s}  Verdict")
for name, mtype, _ in mixer_blocks:
    real_c = (all_ablated_preds[name].numpy() == eval_labels_np)
    null_c = unpack(null_by_block[name]["correct_packed"])
    diff = real_c.astype(np.int8) - null_c.astype(np.int8)
    rng = np.random.RandomState(42)
    boot = np.array([diff[rng.randint(0, N_FULL, N_FULL)].mean() for _ in range(2000)]) * 100
    lo, hi = np.percentile(boot, [2.5, 97.5])
    as_much = hi >= 0        # real is NOT significantly worse than the null
    n_as_much += int(as_much)
    verdict = "null as damaging" if as_much else "real worse than null"
    r = {"block": name, "type": mtype,
         "real_dpp": real_by_block[name]["delta_pp"], "null_dpp": null_by_block[name]["delta_pp"],
         "real_minus_null_pp": float(diff.mean() * 100), "ci_low": float(lo), "ci_high": float(hi),
         "raw_pert": raw_pert[name], "norm_pert": norm_pert[name], "null_as_damaging": bool(as_much)}
    rows.append(r)
    print(f"{name:6s} {mtype:16s} {r['real_dpp']:+8.3f} {r['null_dpp']:+8.3f} {r['real_minus_null_pp']:+9.3f} "
          f"[{lo:+.3f},{hi:+.3f}] {norm_pert[name]:8.3f}  {verdict}")

print(f"\nSame-size random perturbation damages as much as real ablation in {n_as_much}/12 blocks.")
save_checkpoint("normalized_perturbation_sizes", norm_pert)
save_checkpoint("perturbation_sizes", raw_pert)
save_checkpoint("damage_vs_perturbation_table", {"rows": rows, "n_null_as_damaging": n_as_much})
print("STEP 16C COMPLETE")

### STEP 16D — Figure: block damage vs normalized perturbation size

In [ ]:
# ============================================================
# STEP 16D — Figure: block damage vs normalized perturbation size
# Saved locally and uploaded to HF as "damage_vs_perturbation_size.png"
# ============================================================
import os
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

os.makedirs(RESULTS_DIR, exist_ok=True)
fig, ax = plt.subplots(figsize=(7, 6))
for r in rows:
    is_attn = r["type"] == "Attention"
    ax.scatter(r["norm_pert"], abs(r["real_dpp"]), color="#d62728" if is_attn else "#1f77b4",
               marker="o" if is_attn else "^", s=90, edgecolor="black", linewidth=0.6, zorder=3)
    ax.annotate(r["block"], (r["norm_pert"], abs(r["real_dpp"])), textcoords="offset points",
                xytext=(6, 4), fontsize=8)
ax.legend(handles=[
    Line2D([0], [0], marker="^", color="w", markerfacecolor="#1f77b4", markeredgecolor="black", markersize=10, label="Mamba"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#d62728", markeredgecolor="black", markersize=10, label="Attention"),
], loc="upper left")
ax.set_xlabel("Normalized perturbation size (‖Δ‖ / ‖x‖)")
ax.set_ylabel("Mean-ablation damage |Δ accuracy| (pp)")
ax.set_title("Block damage vs. perturbation size (50K ImageNet-1k)")
ax.grid(alpha=0.3)
plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, "damage_vs_perturbation_size.png")
plt.savefig(fig_path, dpi=300)
plt.show()

api.upload_file(path_or_fileobj=fig_path, path_in_repo="figures/damage_vs_perturbation_size.png",
                repo_id=HF_REPO_ID, repo_type="dataset", commit_message="figure: damage vs perturbation")
print("STEP 16D COMPLETE — figure saved and uploaded to HF (figures/)")

## Step 17 — Gate vs SSM pathway ablation ##

### STEP 17A — Gate / SSM branch reference means (2,000 images) + control check

In [ ]:
# ============================================================
# STEP 17A — Gate / SSM branch reference means (2,000 images) + control check
# Saves to HF as "pathway_means_mamba_v2"
# ============================================================
import sys, types

clear_all_hooks()
mamba_blocks = [(name, mod) for name, mtype, mod in mixer_blocks if mtype == "MambaVisionMixer"]
orig_mod = sys.modules[type(mamba_blocks[0][1].mixer).__module__]
F_, rearrange_ = orig_mod.F, orig_mod.rearrange
selective_scan_fn_ = _original_selective_scan_fn if "_original_selective_scan_fn" in globals() else orig_mod.selective_scan_fn
orig_mod.selective_scan_fn = selective_scan_fn_          # make sure no freezing patch is left

_pathway_config = {"active_block": None, "mode": None, "capture": False}
_sum_z = {n: None for n, _ in mamba_blocks}; _sum_y = {n: None for n, _ in mamba_blocks}; _cnt = {n: 0 for n, _ in mamba_blocks}
gate_means, ssm_means = {}, {}

def make_pathway_forward(block_name):
    def patched_forward(self, hidden_states):
        _, seqlen, _ = hidden_states.shape
        xz = rearrange_(self.in_proj(hidden_states), "b l d -> b d l")
        x, z = xz.chunk(2, dim=1)
        A = -torch.exp(self.A_log.float())
        x = F_.silu(F_.conv1d(input=x, weight=self.conv1d_x.weight, bias=self.conv1d_x.bias, padding="same", groups=self.d_inner // 2))
        z = F_.silu(F_.conv1d(input=z, weight=self.conv1d_z.weight, bias=self.conv1d_z.bias, padding="same", groups=self.d_inner // 2))
        x_dbl = self.x_proj(rearrange_(x, "b d l -> (b l) d"))
        dt, B, C = torch.split(x_dbl, [self.dt_rank, self.d_state, self.d_state], dim=-1)
        dt = rearrange_(self.dt_proj(dt), "(b l) d -> b d l", l=seqlen)
        B = rearrange_(B, "(b l) dstate -> b dstate l", l=seqlen).contiguous()
        C = rearrange_(C, "(b l) dstate -> b dstate l", l=seqlen).contiguous()
        y = selective_scan_fn_(x, dt, A, B, C, self.D.float(), z=None,
                               delta_bias=self.dt_proj.bias.float(), delta_softplus=True, return_last_state=None)
        if _pathway_config["capture"]:
            sz, sy = z.detach().float().sum(dim=(0, 2)).cpu(), y.detach().float().sum(dim=(0, 2)).cpu()
            _sum_z[block_name] = sz if _sum_z[block_name] is None else _sum_z[block_name] + sz
            _sum_y[block_name] = sy if _sum_y[block_name] is None else _sum_y[block_name] + sy
            _cnt[block_name] += z.shape[0] * z.shape[2]
        if _pathway_config["active_block"] == block_name:
            if _pathway_config["mode"] == "gate_off":
                z = gate_means[block_name].to(device=z.device, dtype=z.dtype).expand_as(z)
            elif _pathway_config["mode"] == "ssm_off":
                y = ssm_means[block_name].to(device=y.device, dtype=y.dtype).expand_as(y)
        y = rearrange_(torch.cat([y, z], dim=1), "b d l -> b l d")
        return self.out_proj(y)
    return patched_forward

_orig_forwards = {name: mod.mixer.forward for name, mod in mamba_blocks}
def install_pathway_patch():
    for name, mod in mamba_blocks:
        mod.mixer.forward = types.MethodType(make_pathway_forward(name), mod.mixer)
def remove_pathway_patch():
    _pathway_config.update({"active_block": None, "mode": None, "capture": False})
    for name, mod in mamba_blocks:
        mod.mixer.forward = _orig_forwards[name]

GATE_PROBE_N = 2000
plain = eval_pass(eval_indices[:500])              # original forward
install_pathway_patch()
try:
    _pathway_config["capture"] = True
    eval_pass(GLOBAL_RANDOM_INDICES[:GATE_PROBE_N])
    _pathway_config["capture"] = False
    ctrl = eval_pass(eval_indices[:500])           # patch on, nothing ablated
finally:
    remove_pathway_patch()

assert (ctrl == plain).all(), "Patched forward differs from the original — stop and check."
print("Control: patched forward reproduces the original model on 500 images. OK")

for name, _ in mamba_blocks:
    gate_means[name] = (_sum_z[name] / _cnt[name]).view(1, -1, 1)
    ssm_means[name]  = (_sum_y[name] / _cnt[name]).view(1, -1, 1)
    print(f"{name}: gate_mean {tuple(gate_means[name].shape)}  ssm_mean {tuple(ssm_means[name].shape)}")
save_checkpoint("pathway_means_mamba_v2", {"gate_means": gate_means, "ssm_means": ssm_means, "probe_n": GATE_PROBE_N})
print("STEP 17A COMPLETE")

### STEP 17B — Gate-off / SSM-off ablation, full 50K (12 runs)

In [ ]:
# ============================================================
# STEP 17B — Gate-off / SSM-off ablation, full 50K (12 runs)
# Resumable: saves to HF as "pathway_ablation_50k_v3" after every run
# Needs 17A in this session
# ============================================================
CKPT_PW = "pathway_ablation_50k_v3"
try:
    results_pathway = load_checkpoint(CKPT_PW)["results"]
except Exception:
    results_pathway = []
done_pairs = {(r["block"], r["mode"]) for r in results_pathway}
print(f"Pathway sweep: {len(results_pathway)}/12 already done.")

rng_lk = np.random.RandomState(42)
install_pathway_patch()
try:
    for name, _ in mamba_blocks:
        for mode in ["gate_off", "ssm_off"]:
            if (name, mode) in done_pairs:
                continue
            _pathway_config.update({"active_block": name, "mode": mode})
            preds, drop, kl = eval_pass(logit_kl=True)
            _pathway_config.update({"active_block": None, "mode": None})
            st, corr = paired_stats(preds)
            boot_ld, boot_kl = np.empty(2000), np.empty(2000)
            for i in range(2000):
                idx = rng_lk.randint(0, N_FULL, N_FULL)
                boot_ld[i], boot_kl[i] = drop[idx].mean(), kl[idx].mean()
            row = {"block": name, "mode": mode, **st,
                   "mean_logit_drop": float(drop.mean()), "logit_drop_ci": np.percentile(boot_ld, [2.5, 97.5]).tolist(),
                   "mean_kl": float(kl.mean()), "kl_ci": np.percentile(boot_kl, [2.5, 97.5]).tolist(),
                   "correct_packed": pack(corr)}
            results_pathway.append(row)
            save_checkpoint(CKPT_PW, {"results": results_pathway, "baseline_acc": baseline_accuracy})
            print(f"[{len(results_pathway)}/12] {name:6s} [{mode:8s}] Δ={st['delta_pp']:+.3f}pp "
                  f"CI[{st['ci_low_pp']:+.3f},{st['ci_high_pp']:+.3f}] p={st['mcnemar_p']:.2e} "
                  f"ΔLogit={row['mean_logit_drop']:+.4f} KL={row['mean_kl']:.5f}")
            torch.cuda.empty_cache()
finally:
    remove_pathway_patch()
print("\nDONE." if len(results_pathway) == 12 else f"\nStopped at {len(results_pathway)}/12 — rerun to resume.")

### STEP 17C — Gate vs SSM summary

In [ ]:
# ============================================================
# STEP 17C — Gate vs SSM summary: McNemar, ±0.15 pp equivalence, paired difference, sign test
# Saves to HF as "tost_pathway_results"
# ============================================================
from scipy.stats import binomtest

if "results_pathway" not in globals() or len(results_pathway) < 12:
    results_pathway = load_checkpoint("pathway_ablation_50k_v3")["results"]
if "mamba_blocks" not in globals():
    mamba_blocks = [(name, mod) for name, mtype, mod in mixer_blocks if mtype == "MambaVisionMixer"]

EPS = 0.15
pw = {(r["block"], r["mode"]): r for r in results_pathway}

print(f"{'Block':6s} {'Mode':9s} {'Δacc pp':>8s} {'95% CI':>18s} {'McNemar p':>10s} {'Bonf':>5s} {'within±EPS':>10s} {'ΔLogit':>8s} {'KL':>8s}")
for name, _ in mamba_blocks:
    for mode in ["gate_off", "ssm_off"]:
        r = pw[(name, mode)]
        print(f"{name:6s} {mode:9s} {r['delta_pp']:+8.3f} [{r['ci_low_pp']:+.3f},{r['ci_high_pp']:+.3f}] "
              f"{r['mcnemar_p']:10.2e} {'SIG' if r['mcnemar_p'] < BONF_12 else '-':>5s} "
              f"{'yes' if (r['ci_low_pp'] > -EPS and r['ci_high_pp'] < EPS) else 'no':>10s} "
              f"{r['mean_logit_drop']:+8.4f} {r['mean_kl']:8.5f}")

print("\nSSM off vs gate off, paired per image (pp; negative = SSM off more damaging):")
tost_results = []
for name, _ in mamba_blocks:
    d = unpack(pw[(name, "ssm_off")]["correct_packed"]).astype(np.int8) - unpack(pw[(name, "gate_off")]["correct_packed"]).astype(np.int8)
    rng = np.random.RandomState(42)
    boot = np.array([d[rng.randint(0, N_FULL, N_FULL)].mean() for _ in range(2000)]) * 100
    lo, hi = np.percentile(boot, [2.5, 97.5])
    if -EPS < lo and hi < EPS:
        verdict = "EQUIVALENT (within ±EPS)"
    elif lo > 0 or hi < 0:
        verdict = "DIFFERENT (CI excludes 0)"
    else:
        verdict = "INCONCLUSIVE"
    tost_results.append({"block": name, "mean_diff_pp": float(d.mean() * 100),
                         "ci_low": float(lo), "ci_high": float(hi), "verdict": verdict})
    print(f"{name}: {d.mean()*100:+.3f}  CI[{lo:+.3f},{hi:+.3f}]  {verdict}")

n_ssm_worse = sum(r["mean_diff_pp"] < 0 for r in tost_results)
sign_p = binomtest(n_ssm_worse, len(tost_results), 0.5, alternative="greater").pvalue
print(f"\nSSM off more damaging than gate off in {n_ssm_worse}/{len(tost_results)} blocks; sign test p={sign_p:.4f}")
save_checkpoint("tost_pathway_results", {"eps": EPS, "results": tost_results,
                                         "n_ssm_worse": n_ssm_worse, "sign_test_p": float(sign_p)})
print("STEP 17C COMPLETE")

# STEP 18 — Result summary (export only, no new experiment)

In [ ]:
# ============================================================
# STEP 18 — Final results export (CSV + readable summary), uploaded to HF
# ============================================================
import csv, os

def _to_py(v):
    if isinstance(v, (np.floating, np.integer)):
        return v.item()
    if isinstance(v, (int, float, str, bool)):
        return v
    if isinstance(v, torch.Tensor) and v.numel() == 1:
        return v.item()
    return None

def _short(v):
    if isinstance(v, torch.Tensor):
        return f"<tensor shape={tuple(v.shape)}>"
    if isinstance(v, np.ndarray):
        return f"<ndarray shape={v.shape}>"
    if isinstance(v, (list, tuple)):
        return f"<list len={len(v)}>" if len(v) > 4 else str([round(x, 5) if isinstance(x, float) else x for x in v])
    if isinstance(v, dict):
        return f"<dict keys={list(v)[:12]}>"
    if isinstance(v, (float, np.floating)):
        return f"{float(v):.6g}"
    return str(v)[:120]

names = list_checkpoints()
rows_csv, lines = [], [f"Checkpoints in {HF_REPO_ID}: {len(names)}"]
for n in names:
    obj = load_checkpoint(n)
    lines.append(f"\n### {n}")
    items = None
    if isinstance(obj, dict):
        for k, v in obj.items():
            if isinstance(v, list) and v and isinstance(v[0], dict):
                items = v
                lines.append(f"  {k}: {len(v)} rows")
                for r in v:
                    lines.append("      - " + ", ".join(f"{kk}={_short(vv)}" for kk, vv in r.items()
                                                       if not isinstance(vv, np.ndarray)))
            else:
                lines.append(f"  {k}: {_short(v)}")
        if items is None:
            items = [obj]
    for it in items or []:
        flat = {"experiment": n}
        for k, v in it.items():
            if isinstance(v, (list, tuple)) and len(v) == 2 and all(isinstance(x, (int, float)) for x in v):
                flat[f"{k}_low"], flat[f"{k}_high"] = float(v[0]), float(v[1])
                continue
            pv = _to_py(v)
            if pv is not None:
                flat[k] = pv
        rows_csv.append(flat)

os.makedirs(RESULTS_DIR, exist_ok=True)
keys = []
for r in rows_csv:
    keys += [k for k in r if k not in keys]
csv_path = os.path.join(RESULTS_DIR, "all_experiments_results.csv")
txt_path = os.path.join(RESULTS_DIR, "ALL_RESULTS_SUMMARY.txt")
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=keys, restval="")
    w.writeheader()
    w.writerows(rows_csv)
with open(txt_path, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))
print("\n".join(lines))

for p in (csv_path, txt_path):
    api.upload_file(path_or_fileobj=p, path_in_repo=f"final_results/{os.path.basename(p)}",
                    repo_id=HF_REPO_ID, repo_type="dataset", commit_message="final results export")
print(f"\nSaved {len(rows_csv)} rows -> {csv_path}")
print("STEP 18 COMPLETE — CSV + summary saved locally and uploaded to HF (final_results/)")

## STEP 19 — Statistical tests from the old report that need no GPU ##

In [ ]:
# ============================================================
# STEP 19 — Statistical tests from the old report that need no GPU
# (Mamba vs Attention Mann-Whitney; pathway sign tests on accuracy, logit, KL, flip)
# Saves to HF as "step19_report_tests"
# ============================================================
from scipy.stats import mannwhitneyu, binomtest

# ---- Mann-Whitney: Attention vs Mamba, real block ablation (Step 10) ----
m_d = [abs(r["delta_pp"]) for r in ablation_results if r["type"] == "MambaVisionMixer"]
a_d = [abs(r["delta_pp"]) for r in ablation_results if r["type"] == "Attention"]
m_f = [r["flip_rate"] * 100 for r in ablation_results if r["type"] == "MambaVisionMixer"]
a_f = [r["flip_rate"] * 100 for r in ablation_results if r["type"] == "Attention"]
u_d, p_d = mannwhitneyu(a_d, m_d, alternative="two-sided")
u_f, p_f = mannwhitneyu(m_f, a_f, alternative="two-sided")

# ---- damage per unit normalized perturbation (old Figure 1) ----
norm_pert = load_checkpoint("normalized_perturbation_sizes")
per_unit = {r["block"]: abs(r["delta_pp"]) / norm_pert[r["block"]] for r in ablation_results}
m_pu = np.mean([per_unit[r["block"]] for r in ablation_results if r["type"] == "MambaVisionMixer"])
a_pu = np.mean([per_unit[r["block"]] for r in ablation_results if r["type"] == "Attention"])

# ---- pathway sign tests (SSM vs gate), two-sided ----
pw_rows = load_checkpoint("pathway_ablation_50k_v3")["results"]
pw = {(r["block"], r["mode"]): r for r in pw_rows}
blocks6 = sorted({r["block"] for r in pw_rows})
def _count(fn):
    return sum(fn(pw[(b, "ssm_off")]) > fn(pw[(b, "gate_off")]) for b in blocks6)
sign = {
    "accuracy_drop": _count(lambda r: -r["delta_pp"]),
    "abs_logit_drop": _count(lambda r: abs(r["mean_logit_drop"])),
    "kl": _count(lambda r: r["mean_kl"]),
    "flip_rate": _count(lambda r: r["flip_rate_pct"]),
}
sign_p = {k: float(binomtest(v, len(blocks6), 0.5).pvalue) for k, v in sign.items()}

print(f"Mann-Whitney |Δ| Attention vs Mamba: U={u_d:.0f}, p={p_d:.5f}")
print(f"Mann-Whitney flip Mamba vs Attention: U={u_f:.0f}, p={p_f:.5f}")
print(f"Damage per unit normalized perturbation: Mamba={m_pu:.3f}, Attention={a_pu:.3f}, ratio={a_pu/m_pu:.2f}x")
for k in sign:
    print(f"Sign test SSM > gate ({k}): {sign[k]}/6, two-sided p={sign_p[k]:.4f}")

save_checkpoint("step19_report_tests", {
    "mw_delta_U": float(u_d), "mw_delta_p": float(p_d),
    "mw_flip_U": float(u_f), "mw_flip_p": float(p_f),
    "damage_per_unit_pert": per_unit, "mamba_per_unit_mean": float(m_pu),
    "attn_per_unit_mean": float(a_pu), "attn_over_mamba_ratio": float(a_pu / m_pu),
    "sign_counts": sign, "sign_p_two_sided": sign_p,
})
print("STEP 19 COMPLETE")

## STEP 20B — Update norm, cosine and token concentration per block (2,000 images) ##

In [ ]:
# ============================================================
# STEP 20B — Update norm, cosine and token concentration per block (2,000 images)
#   update u  = gamma_1 * mixer(norm1(x))   (what the mixer adds to the residual stream)
#   ratio     = ||u|| / ||x||     per image (x = block input = residual stream)
#   cosine    = cos(u, x)         per image (flattened over tokens and channels)
#   token conc= max_t ||u_t|| / mean_t ||u_t||  per image
# Saves to HF as "update_norms_2k"   (~3 min)
# ============================================================
import torch.nn.functional as F

UN_PROBE_N = 2000
un_indices = GLOBAL_RANDOM_INDICES[:UN_PROBE_N]
_x_in = {}
_un = {name: {"ratio": [], "cos": [], "conc": []} for name, _, _ in mixer_blocks}

def _blk_pre(name):
    def f(module, args):
        _x_in[name] = args[0].detach()
    return f

def _mixer_post(name, blk):
    g = getattr(blk, "gamma_1", 1.0)
    def f(module, inp, out):
        y = out[0] if isinstance(out, tuple) else out
        gg = g.to(y.device, y.dtype) if isinstance(g, torch.Tensor) else g
        u = (y * gg).float()
        x = _x_in[name].float()
        uf, xf = u.flatten(1), x.flatten(1)
        _un[name]["ratio"].append((uf.norm(dim=1) / xf.norm(dim=1)).cpu())
        _un[name]["cos"].append(F.cosine_similarity(uf, xf, dim=1).cpu())
        tok = u.norm(dim=-1)                                   # (B, L)
        _un[name]["conc"].append((tok.max(dim=1).values / tok.mean(dim=1)).cpu())
    return f

clear_all_hooks()
_hs = []
for name, _, blk in mixer_blocks:
    _hs.append(blk.register_forward_pre_hook(_blk_pre(name)))
    _hs.append(blk.mixer.register_forward_hook(_mixer_post(name, blk)))
try:
    eval_pass(un_indices)
finally:
    for h in _hs:
        h.remove()

update_norm_rows = []
print(f"{'Block':6s} {'Type':17s} {'Ratio':>7s} {'±sd':>6s} {'Cosine':>7s} {'±sd':>6s} {'TokConc':>8s}")
for name, mtype, _ in mixer_blocks:
    r = torch.cat(_un[name]["ratio"]); c = torch.cat(_un[name]["cos"]); k = torch.cat(_un[name]["conc"])
    row = {"block": name, "type": mtype,
           "ratio_mean": float(r.mean()), "ratio_std": float(r.std()),
           "cosine_mean": float(c.mean()), "cosine_std": float(c.std()),
           "token_concentration_max_over_mean": float(k.mean()), "n_probe": int(len(r))}
    update_norm_rows.append(row)
    print(f"{name:6s} {mtype:17s} {row['ratio_mean']:7.4f} {row['ratio_std']:6.4f} "
          f"{row['cosine_mean']:+7.4f} {row['cosine_std']:6.4f} {row['token_concentration_max_over_mean']:8.2f}")

save_checkpoint("update_norms_2k", {"results": update_norm_rows,
    "definition": "u = gamma_1*mixer(norm1(x)); ratio=||u||/||x||; cosine=cos(u,x); conc=max/mean token ||u_t||; first 2000 images of the seed-42 permutation"})
print("STEP 20B COMPLETE — compare with old Table 11 (e.g. L3B1 ratio ≈ 0.99, L3B3 ≈ 0.22)")

## STEP 20C — Receptive-field comparison ##

In [ ]:
# ============================================================
# STEP 20C — Receptive-field comparison (old Figure 2)
#   Mamba block: exact implicit attention matrix of the selective scan
#                M_d[t,s] = sum_n C[t,n] * exp(sum_{k=s+1..t} Δ_k A_dn) * Δ_s B[s,n]   (s <= t)
#                averaged |M_d| over channels -> (L, L), rows normalised
#   Attention block: softmax(QK^T) recomputed from the module's own weights, averaged over heads
#   Metrics per row, averaged: mean 2-D grid distance, entropy, ERF (#tokens for 90% mass)
#   Also: attention restricted to the causal (lower-triangular) part, so the shapes match Mamba's
# 32 images, depth-matched pairs.  Saves "figure2_attention_distance" + PNG to HF  (~5 min)
# ============================================================
import sys, os
import torch.nn.functional as F
import matplotlib.pyplot as plt

RF_N_IMG = 32
rf_idx = GLOBAL_RANDOM_INDICES[:RF_N_IMG]
pairs = [("L2B0", "L2B4"), ("L2B1", "L2B5"), ("L2B2", "L2B6"), ("L2B3", "L2B7"), ("L3B0", "L3B2"), ("L3B1", "L3B3")]
blk_by_name = {name: blk for name, _, blk in mixer_blocks}
mamba_names = [m for m, _ in pairs]
attn_names  = [a for _, a in pairs]

# ---- capture: dt/B/C of Mamba blocks, input+output of Attention mixers (one forward) ----
_ssm_mod = sys.modules[type(blk_by_name["L2B0"].mixer).__module__]
_orig_scan = _original_selective_scan_fn if "_original_selective_scan_fn" in globals() else _ssm_mod.selective_scan_fn
_cur = [None]
_cap = {n: {} for n in mamba_names}
_att_io = {}

def _scan_capture(x, dt, A, B, C, D, z=None, delta_bias=None, delta_softplus=None, return_last_state=None):
    if _cur[0] is not None:
        _cap[_cur[0]].update({"dt": dt.detach(), "B": B.detach(), "C": C.detach()})
    return _orig_scan(x, dt, A, B, C, D, z=z, delta_bias=delta_bias,
                      delta_softplus=delta_softplus, return_last_state=return_last_state)

def _pre(n):
    def f(m, a): _cur[0] = n
    return f
def _post(m, a, o): _cur[0] = None
def _att_hook(n):
    def f(m, inp, out):
        _att_io[n] = (inp[0].detach(), (out[0] if isinstance(out, tuple) else out).detach())
    return f

clear_all_hooks()
_hs = []
for n in mamba_names:
    _hs += [blk_by_name[n].mixer.register_forward_pre_hook(_pre(n)), blk_by_name[n].mixer.register_forward_hook(_post)]
for n in attn_names:
    _hs.append(blk_by_name[n].mixer.register_forward_hook(_att_hook(n)))
imgs, _ = prepare_batch(dataset, rf_idx, transform, device)
_ssm_mod.selective_scan_fn = _scan_capture
try:
    with torch.no_grad():
        model(imgs)
finally:
    _ssm_mod.selective_scan_fn = _orig_scan
    for h in _hs:
        h.remove()

# ---- maps ----
def implicit_ssm_attention(dt, A, Bm, Cm, delta_bias):
    """dt (D,L) raw, A (D,N), Bm/Cm (N,L) -> (L,L) mean_d |M_d|"""
    Dd, L = dt.shape
    delta = F.softplus(dt.float() + delta_bias.float()[:, None])                 # (D, L)
    cum = torch.cumsum(delta[:, :, None] * A.float()[:, None, :], dim=1)         # (D, L, N)
    diff = cum[:, :, None, :] - cum[:, None, :, :]                                # (D, t, s, N)
    mask = torch.tril(torch.ones(L, L, dtype=torch.bool, device=dt.device))
    diff = diff.masked_fill(~mask[None, :, :, None], float("-inf"))
    Bd = delta[:, :, None] * Bm.float().T[None, :, :]                             # (D, s, N)
    M = torch.einsum("tn,dtsn,dsn->dts", Cm.float().T, torch.exp(diff), Bd)       # (D, t, s)
    return M.abs().mean(0)

def attention_from_module(mod, x):
    Bq, Nt, Cc = x.shape
    H = mod.num_heads
    qkv = mod.qkv(x).reshape(Bq, Nt, 3, H, Cc // H).permute(2, 0, 3, 1, 4)
    q, k, v = qkv.unbind(0)
    qn, kn = getattr(mod, "q_norm", None), getattr(mod, "k_norm", None)
    if qn is not None: q = qn(q)
    if kn is not None: k = kn(k)
    scale = getattr(mod, "scale", (Cc // H) ** -0.5)
    attn = ((q * scale) @ k.transpose(-2, -1)).softmax(dim=-1)                   # (B, H, N, N)
    out = mod.proj((attn @ v).transpose(1, 2).reshape(Bq, Nt, Cc))
    return attn, out

def spatial_metrics(Mx):
    L = Mx.shape[0]; g = int(round(L ** 0.5))
    coords = torch.stack([torch.arange(L, device=Mx.device) // g, torch.arange(L, device=Mx.device) % g], 1).float()
    P = Mx / Mx.sum(1, keepdim=True).clamp(min=1e-12)
    dist = (P * torch.cdist(coords, coords)).sum(1).mean().item()
    ent = -(P * torch.log(P + 1e-12)).sum(1).mean().item()
    srt = torch.sort(P, dim=1, descending=True).values
    erf = ((torch.cumsum(srt, 1) < 0.9).sum(1).float() + 1).mean().item()
    return dist, ent, erf

# ---- validate the attention recomputation against the real module output ----
with torch.no_grad():
    for n in attn_names:
        xin, yout = _att_io[n]
        _, yre = attention_from_module(blk_by_name[n].mixer, xin)
        err = (yre - yout).abs().max().item()
        assert err < 1e-2 * yout.abs().max().item(), f"{n}: recomputed attention output differs (max err {err:.3e})"
print("Attention recomputation matches module outputs. OK")

rf_rows = []
with torch.no_grad():
    for mn, an in pairs:
        mx = blk_by_name[mn].mixer
        A = -torch.exp(mx.A_log.float())
        dbias = mx.dt_proj.bias.detach()
        attn_all, _ = attention_from_module(blk_by_name[an].mixer, _att_io[an][0])
        L = attn_all.shape[-1]
        tril = torch.tril(torch.ones(L, L, device=attn_all.device))
        acc = {k: [] for k in ["m_dist", "m_ent", "m_erf", "a_dist", "a_ent", "a_erf", "ac_dist", "ac_ent", "ac_erf"]}
        for i in range(RF_N_IMG):
            Mm = implicit_ssm_attention(_cap[mn]["dt"][i], A, _cap[mn]["B"][i], _cap[mn]["C"][i], dbias)
            Aa = attn_all[i].mean(0).float()
            for key, mat in (("m", Mm), ("a", Aa), ("ac", Aa * tril)):
                d_, e_, r_ = spatial_metrics(mat)
                acc[f"{key}_dist"].append(d_); acc[f"{key}_ent"].append(e_); acc[f"{key}_erf"].append(r_)
        row = {"mamba_block": mn, "attn_block": an, "L": L, "n_images": RF_N_IMG}
        for k, v in acc.items():
            row[k] = float(np.mean(v)); row[k + "_sd"] = float(np.std(v))
        rf_rows.append(row)
        print(f"{mn} vs {an} (L={L}): distance Mamba={row['m_dist']:.2f}  Attn={row['a_dist']:.2f}  Attn-causal={row['ac_dist']:.2f} | "
              f"entropy {row['m_ent']:.2f}/{row['a_ent']:.2f}/{row['ac_ent']:.2f} | ERF {row['m_erf']:.1f}/{row['a_erf']:.1f}/{row['ac_erf']:.1f}")

save_checkpoint("figure2_attention_distance", {"rows": rf_rows, "pairs": pairs, "n_images": RF_N_IMG,
    "note": "m = Mamba implicit attention (causal), a = full attention, ac = attention restricted to causal part"})

# ---- figure ----
os.makedirs(RESULTS_DIR, exist_ok=True)
fig, ax = plt.subplots(figsize=(8, 4.5))
xs = np.arange(len(rf_rows)); w = 0.27
ax.bar(xs - w, [r["m_dist"] for r in rf_rows], w, yerr=[r["m_dist_sd"] for r in rf_rows], label="Mamba (implicit)", color="#1f77b4", capsize=3)
ax.bar(xs,     [r["a_dist"] for r in rf_rows], w, yerr=[r["a_dist_sd"] for r in rf_rows], label="Attention (full)", color="#d62728", capsize=3)
ax.bar(xs + w, [r["ac_dist"] for r in rf_rows], w, yerr=[r["ac_dist_sd"] for r in rf_rows], label="Attention (causal part)", color="#f4a6a6", capsize=3)
ax.set_xticks(xs); ax.set_xticklabels([f"{r['mamba_block']}\nvs {r['attn_block']}" for r in rf_rows])
ax.set_ylabel("Mean interaction distance (tokens)")
ax.set_title(f"Interaction distance, depth-matched block pairs ({RF_N_IMG} images, mean ± sd)")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
fig2_path = os.path.join(RESULTS_DIR, "figure2_attention_distance.png")
plt.savefig(fig2_path, dpi=300); plt.show()
api.upload_file(path_or_fileobj=fig2_path, path_in_repo="figures/figure2_attention_distance.png",
                repo_id=HF_REPO_ID, repo_type="dataset", commit_message="figure 2: attention distance")
del _cap, _att_io, imgs
torch.cuda.empty_cache()
print("STEP 20C COMPLETE")

## STEP 20A — External control: DeiT-Small on the same 50K probe ##

In [ ]:
# ============================================================
# STEP 20A — External control: DeiT-Small on the same 50K probe
# Paired comparison with MambaVision (same images, McNemar + paired CI)
# Saves to HF as "deit_control_50k"   (~10 min)
# ============================================================
from transformers import AutoModelForImageClassification, AutoImageProcessor

DEIT_ID = "facebook/deit-small-patch16-224"
deit = AutoModelForImageClassification.from_pretrained(DEIT_ID).to(device).eval()
deit_proc = AutoImageProcessor.from_pretrained(DEIT_ID)
print("DeiT loaded:", type(deit).__name__, "| params:", sum(p.numel() for p in deit.parameters()))

deit_preds = []
with torch.no_grad():
    for start in range(0, N_FULL, BATCH_SIZE):
        idx = eval_indices[start:start + BATCH_SIZE]
        imgs = [dataset[i]["image"].convert("RGB") for i in idx]
        pix = deit_proc(images=imgs, return_tensors="pt")["pixel_values"].to(device)
        deit_preds.append(deit(pixel_values=pix).logits.argmax(-1).cpu())
        if start % (BATCH_SIZE * 100) == 0:
            print(f"{start + len(idx)}/{N_FULL}")
deit_preds = torch.cat(deit_preds).numpy()

deit_acc = float((deit_preds == eval_labels_np).mean())
st_deit, deit_correct = paired_stats(deit_preds)       # DeiT vs MambaVision baseline, same images
print(f"\nDeiT-Small accuracy: {deit_acc*100:.3f}%   (MambaVision: {baseline_accuracy*100:.3f}%)")
print(f"DeiT − MambaVision: {st_deit['delta_pp']:+.3f} pp  CI[{st_deit['ci_low_pp']:+.3f},{st_deit['ci_high_pp']:+.3f}]  "
      f"McNemar p={st_deit['mcnemar_p']:.2e}")
print(f"Images MambaVision right & DeiT wrong: {st_deit['b_correct_to_wrong']} | DeiT right & MambaVision wrong: {st_deit['c_wrong_to_correct']}")

save_checkpoint("deit_control_50k", {
    "model_name": DEIT_ID, "deit_accuracy": deit_acc, "N_eval": N_FULL,
    "vs_mambavision": st_deit, "deit_preds": deit_preds.astype(np.int16),
    "preprocessing": "AutoImageProcessor of the DeiT checkpoint",
})
del deit
torch.cuda.empty_cache()
print("STEP 20A COMPLETE — expected ≈ 77.13% (old run)")

## STEP 21A — Review fixes that need no GPU (uses saved checkpoints only) ##

In [ ]:
# ============================================================
# STEP 21A — Review fixes that need no GPU (uses saved checkpoints only)
#  (1) RQ5: sum of the six per-block combined (Δ+B+C) freezes vs the joint freeze;
#           same additivity check for block ablation (sum of singles vs all-Mamba-off)
#  (2) RQ5: equivalence check (95% CI inside ±0.15 pp) for each per-block combined freeze
#  (3) Overlap audit: the Step 8 reference images (first 256) and the pathway means
#      (first 2,000) are part of the 50K probe. Recompute the main effects on the
#      48,000 images that were never used to build any reference (positions 2000..49999).
# Saves to HF as "step21a_review_checks"   (~1 min)
# ============================================================
EPS = 0.15
DISJOINT_START = 2000          # first 2000 eval positions contain every reference image
fr = {(r["block"], r["tensor"], r["mode"]): r for r in load_checkpoint("step13_freezing_50k")["results"]}
mamba6 = [n for n, t, _ in mixer_blocks if t == "MambaVisionMixer"]

# ---- (1) additivity ----
sum_combined = sum(fr[(b, "all", "mean")]["delta_pp"] for b in mamba6)
joint = load_checkpoint("step14_freeze_all_mamba")["delta_pp_all_frozen"]
sum_single_abl = sum(real_by_block[b]["delta_pp"] for b in mamba6)
partC = load_checkpoint("partC_grouped_ablation")
print("(1) Additivity")
print(f"  Freezing Δ+B+C: sum of 6 per-block = {sum_combined:+.3f} pp | joint = {joint:+.3f} pp | joint/sum = {joint/sum_combined:.2f}x")
print(f"  Block ablation: sum of 6 single Mamba = {sum_single_abl:+.3f} pp | all-Mamba-off = {partC['mamba_delta_pp']:+.3f} pp | ratio = {partC['mamba_delta_pp']/sum_single_abl:.2f}x")

# ---- (2) equivalence for per-block combined freeze ----
print("\n(2) Per-block combined (Δ+B+C, mean) freeze — equivalence to zero at ±0.15 pp")
equiv_rows = []
for b in mamba6:
    r = fr[(b, "all", "mean")]
    eq = (r["ci_low_pp"] > -EPS) and (r["ci_high_pp"] < EPS)
    equiv_rows.append({"block": b, "delta_pp": r["delta_pp"], "ci_low": r["ci_low_pp"], "ci_high": r["ci_high_pp"], "equivalent": bool(eq)})
    print(f"  {b}: {r['delta_pp']:+.3f}  CI[{r['ci_low_pp']:+.3f},{r['ci_high_pp']:+.3f}]  {'EQUIVALENT' if eq else 'not equivalent'}")

# ---- (3) overlap audit on the 48,000 disjoint images ----
sel = np.arange(DISJOINT_START, N_FULL)
bc_d = baseline_correct[sel]
def _paired(correct_full, seed=42):
    c = correct_full[sel]
    diff = c.astype(np.int8) - bc_d.astype(np.int8)
    rng = np.random.RandomState(seed)
    boot = np.array([diff[rng.randint(0, len(diff), len(diff))].mean() for _ in range(2000)]) * 100
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return float(diff.mean() * 100), float(lo), float(hi)

if "all_ablated_preds" not in globals() or len(all_ablated_preds) < 12:
    _sp = load_checkpoint("step9_ablated_preds")
    all_ablated_preds = {k: torch.as_tensor(v.astype(np.int64)) for k, v in _sp.items()}
sn = {r["block"]: r for r in load_checkpoint("shuffled_null_50k")["results"]}
pwr = {(r["block"], r["mode"]): r for r in load_checkpoint("pathway_ablation_50k_v3")["results"]}

print(f"\n(3) Effects on 50K vs on the {len(sel)} images never used for any reference")
audit = []
for name, mtype, _ in mixer_blocks:
    real_c = (all_ablated_preds[name].numpy() == eval_labels_np)
    d, lo, hi = _paired(real_c)
    nd, nlo, nhi = _paired(unpack(sn[name]["correct_packed"]))
    audit.append({"block": name, "type": mtype, "real_50k": real_by_block[name]["delta_pp"], "real_disjoint": d,
                  "real_disjoint_ci": [lo, hi], "null_50k": sn[name]["delta_pp"], "null_disjoint": nd, "null_disjoint_ci": [nlo, nhi]})
    print(f"  {name:6s} real {real_by_block[name]['delta_pp']:+.3f} -> {d:+.3f} [{lo:+.3f},{hi:+.3f}] | "
          f"null {sn[name]['delta_pp']:+.3f} -> {nd:+.3f} [{nlo:+.3f},{nhi:+.3f}]")
for key in ["mamba", "attn"]:
    d, lo, hi = _paired(unpack(partC[f"{key}_correct_packed"]))
    audit.append({"block": f"all_{key}_off", "real_50k": partC[f"{key}_delta_pp"], "real_disjoint": d, "real_disjoint_ci": [lo, hi]})
    print(f"  all-{key}-off {partC[f'{key}_delta_pp']:+.3f} -> {d:+.3f} [{lo:+.3f},{hi:+.3f}]")
for (b, m), r in sorted(pwr.items()):
    d, lo, hi = _paired(unpack(r["correct_packed"]))
    audit.append({"block": b, "mode": m, "real_50k": r["delta_pp"], "real_disjoint": d, "real_disjoint_ci": [lo, hi]})
    print(f"  {b} {m:8s} {r['delta_pp']:+.3f} -> {d:+.3f} [{lo:+.3f},{hi:+.3f}]")

save_checkpoint("step21a_review_checks", {
    "additivity": {"freeze_sum_per_block": sum_combined, "freeze_joint": joint,
                   "ablation_sum_single_mamba": sum_single_abl, "ablation_all_mamba_off": partC["mamba_delta_pp"]},
    "combined_freeze_equivalence": equiv_rows, "eps": EPS,
    "disjoint_audit": audit, "disjoint_positions": [DISJOINT_START, N_FULL],
})
print("\nSTEP 21A COMPLETE")

## Step 22 — Robustness Checks and Diagnostics ##

### Step 22A — KL / Logit No-op Floor ###
### Step 22B — Joint Freeze: Logit Drop and KL ###
### Step 22C — Block Ablation: Logit Drop and KL ###
### Step 22D — Receptive Field Analysis (v2) ###
### Step 22E — DeiT-Small External Control ###
### Step 22F — Stratified Ablation with External Difficulty Tiers ###
### Step 22G — High-Norm Token Locations ###
### Step 22H — L3B1 Gate Logit Diagnostic ###
### Step 22I — Report Figures from Frozen Data ###

### Step 22A — KL / logit no-op floor ###
The unmodified model is evaluated through exactly the same pipeline used for every KL and logit-drop number. Any non-zero KL here is numerical noise. Saves `kl_noop_floor`.

In [ ]:
# ============================================================
# STEP 22A — KL / logit-drop no-op floor (no intervention at all)
# ============================================================
clear_all_hooks()
preds0, drop0, kl0 = eval_pass(logit_kl=True)
agree = float((preds0 == baseline_preds_np).mean())
rng = np.random.RandomState(42)
boot = np.array([kl0[rng.randint(0, N_FULL, N_FULL)].mean() for _ in range(2000)])
floor = {"mean_kl": float(kl0.mean()), "kl_ci": np.percentile(boot, [2.5, 97.5]).tolist(),
         "max_kl": float(kl0.max()), "p99_kl": float(np.percentile(kl0, 99)),
         "mean_logit_drop": float(drop0.mean()), "max_abs_logit_drop": float(np.abs(drop0).max()),
         "prediction_agreement_with_step6": agree, "n": N_FULL,
         "note": "baseline log-probs stored in float16; forward in float32"}
for k, v in floor.items():
    print(f"  {k}: {v}")
save_checkpoint("kl_noop_floor", floor)
print("STEP 22A COMPLETE — compare mean_kl with pathway KLs (0.0014–0.021) and freezing KLs")

### Step 22B — Logit drop and KL for the joint freeze (RQ5) ###
One 50K pass with all six Mamba blocks frozen (Δ+B+C, mean). Saves `step14_joint_freeze_logit_kl`.

In [ ]:
# ============================================================
# STEP 22B — Joint freeze (all 6 Mamba blocks, Δ+B+C, mean): logit drop + KL with CIs
# ============================================================
assert "install_freeze_patch" in globals(), "Run Step 13A and 13B first."
_hs = install_freeze_patch()
try:
    _freeze_config.update({"active": True, "block_names": set(mamba_block_names), "tensor": "all", "mode": "mean"})
    pj, dj, kj = eval_pass(logit_kl=True)
finally:
    remove_freeze_patch(_hs)
st_j, _ = paired_stats(pj)
rng = np.random.RandomState(42)
bl, bk = np.empty(2000), np.empty(2000)
for i in range(2000):
    idx = rng.randint(0, N_FULL, N_FULL)
    bl[i], bk[i] = dj[idx].mean(), kj[idx].mean()
row = {**st_j, "mean_logit_drop": float(dj.mean()), "logit_drop_ci": np.percentile(bl, [2.5, 97.5]).tolist(),
       "mean_kl": float(kj.mean()), "kl_ci": np.percentile(bk, [2.5, 97.5]).tolist()}
print(f"Joint freeze: Δ={row['delta_pp']:+.3f} pp (check: −1.136) | logit drop={row['mean_logit_drop']:+.4f} "
      f"CI{[round(x,4) for x in row['logit_drop_ci']]} | KL={row['mean_kl']:.5f} CI{[round(x,5) for x in row['kl_ci']]}")
save_checkpoint("step14_joint_freeze_logit_kl", row)
print("STEP 22B COMPLETE")

### Step 22C — Block Ablation: Logit Drop and KL ###
Same mean-ablation hooks as Step 9; 12 passes, resumable, saves `step9_logit_kl` after every block. Accuracy is checked against Step 10.

In [ ]:
# ============================================================
# STEP 22C — Block mean-ablation: true-class logit drop + KL per block (resumable)
# ============================================================
CKPT_LK9 = "step9_logit_kl"
try:
    rows_lk9 = load_checkpoint(CKPT_LK9)["results"]
except Exception:
    rows_lk9 = []
done = {r["block"] for r in rows_lk9}
print(f"Done: {len(done)}/12")

for name, mtype, blk in mixer_blocks:
    if name in done:
        continue
    clear_all_hooks()
    h = blk.mixer.register_forward_hook(mean_ablation_hook(block_means[name]))
    try:
        p, d, k = eval_pass(logit_kl=True)
    finally:
        h.remove()
    acc = float((p == eval_labels_np).mean())
    rng = np.random.RandomState(42)
    bl, bk = np.empty(2000), np.empty(2000)
    for i in range(2000):
        idx = rng.randint(0, N_FULL, N_FULL)
        bl[i], bk[i] = d[idx].mean(), k[idx].mean()
    r = {"block": name, "type": mtype, "accuracy": acc, "step10_accuracy": real_by_block[name]["accuracy"],
         "mean_logit_drop": float(d.mean()), "logit_drop_ci": np.percentile(bl, [2.5, 97.5]).tolist(),
         "mean_kl": float(k.mean()), "kl_ci": np.percentile(bk, [2.5, 97.5]).tolist()}
    rows_lk9.append(r)
    save_checkpoint(CKPT_LK9, {"results": rows_lk9})
    flag = "OK" if abs(acc - r["step10_accuracy"]) < 5e-4 else "CHECK"
    print(f"[{len(rows_lk9)}/12] {name:6s} acc={acc:.5f} ({flag}) logit_drop={r['mean_logit_drop']:+.4f} KL={r['mean_kl']:.5f}")
print("STEP 22C COMPLETE" if len(rows_lk9) == 12 else "Stopped — rerun to resume")

### Step 22D — Receptive Field Analysis (v2) ###
1,000 images, all 12 blocks (no arbitrary pairing), uniform-mask baselines, horizontal/vertical split, spread across heads, 95% CIs. Mamba map = exact implicit attention of the selective scan (mean |M_d| over channels). Saves `figure2_attention_distance_v2`.

In [ ]:
# ============================================================
# STEP 22D — Receptive field v2: 1,000 images, all 12 blocks, uniform baselines, H/V split, head spread
# ============================================================
import sys, os
import torch.nn.functional as F
import matplotlib.pyplot as plt

RF2_N = 1000
RF2_BS = 32
rf2_idx = GLOBAL_RANDOM_INDICES[:RF2_N]
blk_by_name = {name: blk for name, _, blk in mixer_blocks}
m_names = [n for n, t, _ in mixer_blocks if t == "MambaVisionMixer"]
a_names = [n for n, t, _ in mixer_blocks if t == "Attention"]

_ssm_mod = sys.modules[type(blk_by_name[m_names[0]].mixer).__module__]
_orig_scan = _original_selective_scan_fn if "_original_selective_scan_fn" in globals() else _ssm_mod.selective_scan_fn

def implicit_ssm_attention(dt, A, Bm, Cm, delta_bias):
    Dd, L = dt.shape
    delta = F.softplus(dt.float() + delta_bias.float()[:, None])
    cum = torch.cumsum(delta[:, :, None] * A.float()[:, None, :], dim=1)
    diff = cum[:, :, None, :] - cum[:, None, :, :]
    mask = torch.tril(torch.ones(L, L, dtype=torch.bool, device=dt.device))
    diff = diff.masked_fill(~mask[None, :, :, None], float("-inf"))
    Bd = delta[:, :, None] * Bm.float().T[None, :, :]
    M = torch.einsum("tn,dtsn,dsn->dts", Cm.float().T, torch.exp(diff), Bd)
    return M.abs().mean(0)

def attention_from_module(mod, x):
    Bq, Nt, Cc = x.shape
    H = mod.num_heads
    qkv = mod.qkv(x).reshape(Bq, Nt, 3, H, Cc // H).permute(2, 0, 3, 1, 4)
    q, k, v = qkv.unbind(0)
    qn, kn = getattr(mod, "q_norm", None), getattr(mod, "k_norm", None)
    if qn is not None: q = qn(q)
    if kn is not None: k = kn(k)
    scale = getattr(mod, "scale", (Cc // H) ** -0.5)
    attn = ((q * scale) @ k.transpose(-2, -1)).softmax(dim=-1)
    out = mod.proj((attn @ v).transpose(1, 2).reshape(Bq, Nt, Cc))
    return attn, out

_geom = {}
def geometry(L, device):
    key = (L, str(device))
    if key not in _geom:
        g = int(round(L ** 0.5))
        r = (torch.arange(L, device=device) // g).float(); c = (torch.arange(L, device=device) % g).float()
        dr = (r[:, None] - r[None, :]).abs(); dc = (c[:, None] - c[None, :]).abs()
        _geom[key] = (torch.sqrt(dr ** 2 + dc ** 2), dr, dc)
    return _geom[key]

def metrics(Mx):
    """Mx (..., L, L) non-negative -> dict of per-map scalars (averaged over rows)."""
    L = Mx.shape[-1]
    D, dr, dc = geometry(L, Mx.device)
    P = Mx / Mx.sum(-1, keepdim=True).clamp(min=1e-12)
    srt = torch.sort(P, dim=-1, descending=True).values
    return {"dist": (P * D).sum(-1).mean(-1), "dist_v": (P * dr).sum(-1).mean(-1), "dist_h": (P * dc).sum(-1).mean(-1),
            "entropy": -(P * torch.log(P + 1e-12)).sum(-1).mean(-1),
            "erf90": ((torch.cumsum(srt, -1) < 0.9).sum(-1).float() + 1).mean(-1)}

def uniform_baselines(L, device):
    full = torch.ones(L, L, device=device); causal = torch.tril(full)
    return {k: float(v) for k, v in metrics(full).items()}, {k: float(v) for k, v in metrics(causal).items()}

# ---- accumulate per-image metrics over 1,000 images ----
acc = {n: {} for n in m_names + a_names}
def _add(n, key, vals):
    acc[n].setdefault(key, []).append(vals.detach().float().cpu())

_cur = [None]; _cap = {}; _att_in = {}
def _scan_capture(x, dt, A, B, C, D, z=None, delta_bias=None, delta_softplus=None, return_last_state=None):
    if _cur[0] is not None:
        _cap[_cur[0]] = (dt.detach(), B.detach(), C.detach())
    return _orig_scan(x, dt, A, B, C, D, z=z, delta_bias=delta_bias, delta_softplus=delta_softplus, return_last_state=return_last_state)
def _rf_pre(n):
    def f(m, a): _cur[0] = n
    return f
def _rf_post(m, a, o): _cur[0] = None
def _att_hook(n):
    def f(m, inp, out): _att_in[n] = (inp[0].detach(), (out[0] if isinstance(out, tuple) else out).detach())
    return f

clear_all_hooks()
_hs = []
for n in m_names:
    _hs += [blk_by_name[n].mixer.register_forward_pre_hook(_rf_pre(n)), blk_by_name[n].mixer.register_forward_hook(_rf_post)]
for n in a_names:
    _hs.append(blk_by_name[n].mixer.register_forward_hook(_att_hook(n)))
_ssm_mod.selective_scan_fn = _scan_capture
checked = False
try:
    with torch.no_grad():
        for start in range(0, RF2_N, RF2_BS):
            imgs, _ = prepare_batch(dataset, rf2_idx[start:start + RF2_BS], transform, device)
            _cap.clear(); _att_in.clear()
            model(imgs)
            for n in a_names:
                xin, yout = _att_in[n]
                attn, yre = attention_from_module(blk_by_name[n].mixer, xin)       # (B, H, L, L)
                if not checked:
                    err = (yre - yout).abs().max().item()
                    assert err < 1e-2 * yout.abs().max().item(), f"{n}: attention recomputation differs ({err:.3e})"
                L = attn.shape[-1]
                head = metrics(attn.float())                                       # (B, H)
                _add(n, "head_dist", head["dist"])
                full = metrics(attn.float().mean(1))                                # head-averaged map
                causal = metrics(attn.float().mean(1) * torch.tril(torch.ones(L, L, device=attn.device)))
                for k, v in full.items(): _add(n, k, v)
                for k, v in causal.items(): _add(n, "causal_" + k, v)
            checked = True
            for n in m_names:
                dt, Bm, Cm = _cap[n]
                mx = blk_by_name[n].mixer
                A = -torch.exp(mx.A_log.float()); db = mx.dt_proj.bias.detach()
                for i in range(dt.shape[0]):
                    for k, v in metrics(implicit_ssm_attention(dt[i], A, Bm[i], Cm[i], db)).items():
                        _add(n, k, v.reshape(1))
            if start % (RF2_BS * 8) == 0:
                print(f"{min(start + RF2_BS, RF2_N)}/{RF2_N}")
finally:
    _ssm_mod.selective_scan_fn = _orig_scan
    for h in _hs:
        h.remove()
print("Attention recomputation check passed. OK")

def _summ(t):
    t = torch.cat(t).numpy().astype(np.float64)
    return float(t.mean()), float(1.96 * t.std(ddof=1) / np.sqrt(len(t)))

rf2_rows = []
for name, mtype, _ in mixer_blocks:
    a = acc[name]
    L = 196 if name.startswith("L2") else 49
    u_full, u_causal = uniform_baselines(L, "cpu")
    row = {"block": name, "type": mtype, "L": L, "n_images": RF2_N}
    for k in ["dist", "dist_v", "dist_h", "entropy", "erf90"]:
        row[k], row[k + "_ci95"] = _summ(a[k])
    base = u_causal if mtype == "MambaVisionMixer" else u_full
    row["uniform_dist"] = base["dist"]; row["dist_over_uniform"] = row["dist"] / base["dist"]
    row["uniform_causal_dist"] = u_causal["dist"]; row["uniform_full_dist"] = u_full["dist"]
    if mtype == "Attention":
        for k in ["dist", "entropy", "erf90"]:
            row["causal_" + k], row["causal_" + k + "_ci95"] = _summ(a["causal_" + k])
        row["causal_dist_over_uniform"] = row["causal_dist"] / u_causal["dist"]
        hd = torch.cat(a["head_dist"]).mean(0).numpy()                     # mean distance per head
        row["head_dist_mean"] = hd.tolist(); row["head_dist_sd"] = float(hd.std()); row["head_dist_range"] = [float(hd.min()), float(hd.max())]
    rf2_rows.append(row)
    extra = (f" | causal part {row['causal_dist']:.2f} (/unif {row['causal_dist_over_uniform']:.2f}) | heads {row['head_dist_range'][0]:.2f}–{row['head_dist_range'][1]:.2f}"
             if mtype == "Attention" else "")
    print(f"{name:6s} {mtype:16s} dist={row['dist']:.2f}±{row['dist_ci95']:.2f} (/uniform {row['dist_over_uniform']:.2f}) "
          f"V={row['dist_v']:.2f} H={row['dist_h']:.2f} ent={row['entropy']:.2f} ERF90={row['erf90']:.1f}{extra}")

save_checkpoint("figure2_attention_distance_v2", {"rows": rf2_rows, "n_images": RF2_N,
    "definitions": "P=row-normalised map; dist=sum_s P*euclid grid distance, averaged over rows; dist_v/dist_h=vertical/horizontal parts; "
                   "ERF90=#tokens for 90% row mass; Mamba map=mean_d |implicit selective-scan matrix| (causal); "
                   "Mamba normalised by uniform-causal map, Attention by uniform-full map; CI = 1.96*sd/sqrt(n) over images"})

os.makedirs(RESULTS_DIR, exist_ok=True)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, key, ttl in [(axes[0], "dist", "Mean interaction distance (tokens)"), (axes[1], "dist_over_uniform", "Distance / uniform-mask distance")]:
    xs = np.arange(len(rf2_rows))
    cols = ["#1f77b4" if r["type"] == "MambaVisionMixer" else "#d62728" for r in rf2_rows]
    vals = [r[key] for r in rf2_rows]
    errs = [r["dist_ci95"] if key == "dist" else r["dist_ci95"] / r["uniform_dist"] for r in rf2_rows]
    ax.bar(xs, vals, yerr=errs, color=cols, capsize=3)
    if key == "dist":
        ax.scatter([i for i, r in enumerate(rf2_rows) if r["type"] == "Attention"],
                   [r["causal_dist"] for r in rf2_rows if r["type"] == "Attention"], marker="_", s=300, color="black",
                   label="Attention, causal part only", zorder=3)
        ax.legend()
    else:
        ax.axhline(1.0, color="gray", ls="--", lw=1)
    ax.set_xticks(xs); ax.set_xticklabels([r["block"] for r in rf2_rows], rotation=45)
    ax.set_title(ttl); ax.grid(axis="y", alpha=0.3)
fig.suptitle(f"Blue = Mamba (implicit attention), red = Attention; {RF2_N} images, 95% CI")
plt.tight_layout()
p = os.path.join(RESULTS_DIR, "figure2_attention_distance_v2.png"); plt.savefig(p, dpi=300); plt.show()
api.upload_file(path_or_fileobj=p, path_in_repo="figures/figure2_attention_distance_v2.png", repo_id=HF_REPO_ID, repo_type="dataset")
del acc, _cap, _att_in
torch.cuda.empty_cache()
print("STEP 22D COMPLETE")

### Step 22E — DeiT-Small External Control ###
DeiT's own evaluation preprocessing: bicubic resize to 248 (crop ratio 0.9), centre crop 224, ImageNet mean/std. Saves `deit_control_50k_v2`.

In [ ]:
# ============================================================
# STEP 22E — DeiT-Small, timm-style eval preprocessing (bicubic 248 -> centre crop 224)
# ============================================================
from transformers import AutoModelForImageClassification
from torchvision import transforms as T

DEIT_ID = "facebook/deit-small-patch16-224"
deit = AutoModelForImageClassification.from_pretrained(DEIT_ID).to(device).eval()
deit_tf = T.Compose([T.Resize(248, interpolation=T.InterpolationMode.BICUBIC), T.CenterCrop(224), T.ToTensor(),
                     T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

deit_preds = []
with torch.no_grad():
    for start in range(0, N_FULL, BATCH_SIZE):
        idx = eval_indices[start:start + BATCH_SIZE]
        pix = torch.stack([deit_tf(dataset[i]["image"].convert("RGB")) for i in idx]).to(device)
        deit_preds.append(deit(pixel_values=pix).logits.argmax(-1).cpu())
        if start % (BATCH_SIZE * 100) == 0:
            print(f"{start + len(idx)}/{N_FULL}")
deit_preds = torch.cat(deit_preds).numpy()
deit_acc = float((deit_preds == eval_labels_np).mean())
st_deit, _ = paired_stats(deit_preds)
print(f"\nDeiT-Small: {deit_acc*100:.3f}%   (published ≈ 79.8%; old run 77.13%) | MambaVision {baseline_accuracy*100:.3f}% (published ≈ 82.3%)")
print(f"DeiT − MambaVision: {st_deit['delta_pp']:+.3f} pp CI[{st_deit['ci_low_pp']:+.3f},{st_deit['ci_high_pp']:+.3f}]")
save_checkpoint("deit_control_50k_v2", {"model_name": DEIT_ID, "deit_accuracy": deit_acc, "N_eval": N_FULL,
    "preprocessing": "Resize 248 bicubic, CenterCrop 224, ImageNet mean/std", "vs_mambavision": st_deit,
    "deit_preds": deit_preds.astype(np.int16)})
del deit
torch.cuda.empty_cache()
print("STEP 22E COMPLETE")

### Step 22F — Stratified Ablation with External Difficulty Tiers ###
Tiers come from DeiT-Small's per-class accuracy (22E), not MambaVision's own, so the strata are not selected on the outcome. Reports n per tier and paired bootstrap 95% CIs. Saves `step12_stratified_external`.

In [ ]:
# ============================================================
# STEP 22F — Stratified block ablation with DeiT-defined difficulty tiers + CIs
# ============================================================
_dc = load_checkpoint("deit_control_50k_v2")
dp = _dc["deit_preds"].astype(np.int64)
if "all_ablated_preds" not in globals() or len(all_ablated_preds) < 12:
    _sp = load_checkpoint("step9_ablated_preds")
    all_ablated_preds = {k: torch.as_tensor(v.astype(np.int64)) for k, v in _sp.items()}

classes = np.unique(eval_labels_np)
deit_cls_acc = {int(c): float((dp[eval_labels_np == c] == c).mean()) for c in classes}
order = sorted(deit_cls_acc, key=lambda c: deit_cls_acc[c])
k3 = len(order) // 3
tiers = {"Hard": set(order[:k3]), "Medium": set(order[k3:2 * k3]), "Easy": set(order[2 * k3:])}
masks = {t: np.isin(eval_labels_np, list(s)) for t, s in tiers.items()}
for t, m in masks.items():
    print(f"{t}: n={m.sum()} images, {len(tiers[t])} classes, MambaVision baseline acc={baseline_correct[m].mean():.4f}, DeiT acc={(dp[m]==eval_labels_np[m]).mean():.4f}")

strat2 = []
print(f"\n{'Block':6s} {'Type':16s} " + "  ".join(f"{t:>22s}" for t in ["Easy", "Medium", "Hard"]))
for name, mtype, _ in mixer_blocks:
    abl_c = (all_ablated_preds[name].numpy() == eval_labels_np)
    row = {"block": name, "type": mtype}; cells = []
    for t in ["Easy", "Medium", "Hard"]:
        m = masks[t]
        diff = abl_c[m].astype(np.int8) - baseline_correct[m].astype(np.int8)
        rng = np.random.RandomState(42)
        boot = np.array([diff[rng.randint(0, len(diff), len(diff))].mean() for _ in range(2000)]) * 100
        lo, hi = np.percentile(boot, [2.5, 97.5])
        row.update({f"{t}_n": int(m.sum()), f"{t}_delta_pp": float(diff.mean() * 100), f"{t}_ci": [float(lo), float(hi)]})
        cells.append(f"{diff.mean()*100:+.3f} [{lo:+.2f},{hi:+.2f}]")
    strat2.append(row)
    print(f"{name:6s} {mtype:16s} " + "  ".join(f"{c:>22s}" for c in cells))

save_checkpoint("step12_stratified_external", {"tier_source": "DeiT-Small per-class accuracy (22E)",
    "rows": strat2, "tiers": {t: sorted(s) for t, s in tiers.items()}, "deit_per_class_acc": deit_cls_acc})
print("STEP 22F COMPLETE")

### Step 22G — High-Norm Token Locations ###
For every block, records which token carries the largest update norm (‖γ·mixer(norm(x))‖ per token) in each of 2,000 images. If the same few positions win in most images (corners or background, as in Darcet et al.), the high token concentration in L3B1 is a positional artefact. Saves `high_norm_tokens` + heatmap.

In [ ]:
# ============================================================
# STEP 22G — Location of the highest-update-norm token, per block (2,000 images)
# ============================================================
import os
import matplotlib.pyplot as plt

HN_N = 2000
_top = {name: [] for name, _, _ in mixer_blocks}
def _mp(name, blk):
    g = getattr(blk, "gamma_1", 1.0)
    def f(m, inp, out):
        y = out[0] if isinstance(out, tuple) else out
        gg = g.to(y.device, y.dtype) if isinstance(g, torch.Tensor) else g
        tok = (y * gg).float().norm(dim=-1)                     # (B, L)
        _top[name].append(torch.topk(tok, 3, dim=1).indices.cpu())
    return f
clear_all_hooks()
_hs = []
for name, _, blk in mixer_blocks:
    _hs.append(blk.mixer.register_forward_hook(_mp(name, blk)))
try:
    eval_pass(GLOBAL_RANDOM_INDICES[:HN_N])
finally:
    for h in _hs:
        h.remove()

hn_rows = []
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for ax, (name, mtype, _) in zip(axes.flat, mixer_blocks):
    top = torch.cat(_top[name])                                 # (N, 3)
    L = 196 if name.startswith("L2") else 49; g = int(L ** 0.5)
    counts = np.bincount(top[:, 0].numpy(), minlength=L)
    mode_pos = int(counts.argmax()); mode_frac = float(counts.max() / len(top))
    in_top3 = float((top == mode_pos).any(1).float().mean())
    r, c = divmod(mode_pos, g)
    border = r in (0, g - 1) or c in (0, g - 1)
    top5 = [(int(p), float(counts[p] / len(top))) for p in counts.argsort()[::-1][:5]]
    hn_rows.append({"block": name, "type": mtype, "mode_position": mode_pos, "mode_rc": [r, c], "on_border": bool(border),
                    "mode_fraction_top1": mode_frac, "mode_in_top3_fraction": in_top3, "top5_positions": top5,
                    "chance_fraction": 1.0 / L})
    print(f"{name:6s} {mtype:16s} most frequent top token = pos {mode_pos} (row {r}, col {c}{', border' if border else ''}) "
          f"in {mode_frac*100:.1f}% of images (chance {100/L:.1f}%), in top-3 of {in_top3*100:.1f}%")
    ax.imshow(counts.reshape(g, g) / len(top), cmap="viridis"); ax.set_title(f"{name}", fontsize=9); ax.axis("off")
fig.suptitle(f"Where the highest-update-norm token sits ({HN_N} images; colour = fraction of images)")
plt.tight_layout()
p = os.path.join(RESULTS_DIR, "high_norm_token_positions.png"); plt.savefig(p, dpi=200); plt.show()
api.upload_file(path_or_fileobj=p, path_in_repo="figures/high_norm_token_positions.png", repo_id=HF_REPO_ID, repo_type="dataset")
save_checkpoint("high_norm_tokens", {"rows": hn_rows, "n_images": HN_N})
print("STEP 22G COMPLETE")

### Step 22H — L3B1 Gate Logit Diagnostic ###
On 5,000 images, compares baseline vs L3B1 gate_off vs L3B1 ssm_off: true-class logit, log-sum-exp, p(true), logit-vector norm, top-1 margin, entropy, accuracy. If the true logit and log-sum-exp both rise, gate-off rescales the logits rather than removing class information. Saves `l3b1_gate_logit_diagnostic`.

In [ ]:
# ============================================================
# STEP 22H — L3B1 gate_off logit diagnostic (5,000 images)
# ============================================================
assert "install_pathway_patch" in globals(), "Run Step 17A first."
DG_N = 5000
dg_idx = eval_indices[:DG_N]

def logit_stats(mode):
    out = {k: [] for k in ["true", "lse", "p_true", "norm", "margin", "entropy", "correct"]}
    install_pathway_patch()
    try:
        _pathway_config.update({"active_block": "L3B1" if mode else None, "mode": mode})
        with torch.no_grad():
            for s in range(0, DG_N, BATCH_SIZE):
                imgs, labs = prepare_batch(dataset, dg_idx[s:s + BATCH_SIZE], transform, device)
                _o = model(imgs)
                lg = (_o["logits"] if isinstance(_o, dict) else _o.logits).float()
                lse = torch.logsumexp(lg, -1); tl = lg.gather(1, labs.view(-1, 1)).squeeze(1)
                top2 = lg.topk(2, -1).values; p = lg.softmax(-1)
                out["true"].append(tl); out["lse"].append(lse); out["p_true"].append((tl - lse).exp())
                out["norm"].append(lg.norm(dim=-1)); out["margin"].append(top2[:, 0] - top2[:, 1])
                out["entropy"].append(-(p * (p + 1e-12).log()).sum(-1)); out["correct"].append((lg.argmax(-1) == labs).float())
    finally:
        remove_pathway_patch()
    return {k: float(torch.cat(v).mean()) for k, v in out.items()}

diag = {"baseline": logit_stats(None), "gate_off": logit_stats("gate_off"), "ssm_off": logit_stats("ssm_off")}
print(f"{'':10s} " + " ".join(f"{k:>9s}" for k in diag["baseline"]))
for m, d in diag.items():
    print(f"{m:10s} " + " ".join(f"{v:9.4f}" for v in d.values()))
for m in ["gate_off", "ssm_off"]:
    print(f"{m}: Δ true logit {diag[m]['true']-diag['baseline']['true']:+.4f} | Δ logsumexp {diag[m]['lse']-diag['baseline']['lse']:+.4f} | "
          f"Δ logit norm {diag[m]['norm']-diag['baseline']['norm']:+.4f} | Δ p(true) {diag[m]['p_true']-diag['baseline']['p_true']:+.5f}")
save_checkpoint("l3b1_gate_logit_diagnostic", {"n": DG_N, **diag})
print("STEP 22H COMPLETE")

### Step 22I — Report Figures from Frozen Data ###
Loads every checkpoint, writes `figures_data.json` (the single frozen source), and draws every figure from that file only, so figures and tables cannot drift apart again. Uploads JSON + PNGs to HF `figures/`.

In [ ]:
# ============================================================
# STEP 22I — Figures 1, 3, 4, 5, 6, 7 regenerated from one frozen JSON
# ============================================================
import json, os
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

def _ld(n):
    try: return load_checkpoint(n)
    except Exception as e: print("missing:", n, e); return None
s10 = load_checkpoint("step10")["ablation_results"]
sn  = load_checkpoint("shuffled_null_50k")["results"]
dvp = load_checkpoint("damage_vs_perturbation_table")["rows"]
rs  = load_checkpoint("resampling_50k_v2")["results"]
sp  = load_checkpoint("resampling_vs_meanablation_spearman")
pw  = load_checkpoint("pathway_ablation_50k_v3")["results"]
fz  = load_checkpoint("step13_freezing_50k")["results"]
j14 = load_checkpoint("step14_freeze_all_mamba")
pc  = load_checkpoint("partC_grouped_ablation")
floor = _ld("kl_noop_floor")

def _clean(x):
    if isinstance(x, dict): return {k: _clean(v) for k, v in x.items() if not isinstance(v, np.ndarray)}
    if isinstance(x, list): return [_clean(v) for v in x]
    if isinstance(x, tuple): return [_clean(v) for v in x]
    if isinstance(x, (np.floating, np.integer, np.bool_)): return x.item()
    return x
FD = _clean({"baseline_accuracy": baseline_accuracy, "block_ablation": s10, "shuffled_null": sn, "real_vs_null": dvp,
             "resampling": rs, "spearman": {k: sp[k] for k in ["spearman_rho", "spearman_p"]}, "pathway": pw,
             "freezing": fz, "joint_freeze": j14, "grouped": {k: v for k, v in pc.items() if not k.endswith("packed")},
             "kl_floor": floor,
             "block_ablation_logit_kl": (_ld("step9_logit_kl") or {}).get("results"),
             "joint_freeze_logit_kl": _ld("step14_joint_freeze_logit_kl"),
             "stratified_external": (_ld("step12_stratified_external") or {}).get("rows"),
             "receptive_field_v2": (_ld("figure2_attention_distance_v2") or {}).get("rows"),
             "high_norm_tokens": (_ld("high_norm_tokens") or {}).get("rows"),
             "deit_control": {k: v for k, v in (_ld("deit_control_50k_v2") or {}).items() if k != "deit_preds"},
             "review_checks": _ld("step21a_review_checks"),
             "l3b1_logit_diagnostic": _ld("l3b1_gate_logit_diagnostic")})
os.makedirs(RESULTS_DIR, exist_ok=True)
fd_path = os.path.join(RESULTS_DIR, "figures_data.json")
with open(fd_path, "w") as f: json.dump(FD, f, indent=1)
with open(fd_path) as f: FD = json.load(f)          # draw ONLY from the frozen file
saved = []
def _save(fig, name):
    p = os.path.join(RESULTS_DIR, name); fig.savefig(p, dpi=300, bbox_inches="tight"); plt.show(); saved.append(p)
COL = {"MambaVisionMixer": "#1f77b4", "Attention": "#d62728"}

# ---- Figure 1: damage vs normalised perturbation, real and null ----
fig, ax = plt.subplots(figsize=(7.5, 6))
for r in FD["real_vs_null"]:
    c = COL[r["type"]]
    ax.plot([r["norm_pert"]] * 2, [abs(r["real_dpp"]), abs(r["null_dpp"])], color=c, alpha=0.4, lw=1)
    ax.scatter(r["norm_pert"], abs(r["real_dpp"]), color=c, marker="o", s=70, edgecolor="k", zorder=3)
    ax.scatter(r["norm_pert"], abs(r["null_dpp"]), facecolor="white", edgecolor=c, marker="s", s=60, lw=1.5, zorder=3)
    ax.annotate(r["block"], (r["norm_pert"], abs(r["real_dpp"])), xytext=(5, 3), textcoords="offset points", fontsize=8)
ax.legend(handles=[Line2D([0], [0], marker="o", color="w", markerfacecolor="gray", markeredgecolor="k", ms=9, label="real mean ablation"),
                   Line2D([0], [0], marker="s", color="w", markerfacecolor="white", markeredgecolor="gray", ms=9, label="shuffled null (same size)"),
                   Line2D([0], [0], color=COL["MambaVisionMixer"], lw=4, label="Mamba"),
                   Line2D([0], [0], color=COL["Attention"], lw=4, label="Attention")])
ax.set_xlabel("Normalised perturbation  ‖x̃ − x‖ / ‖x‖  (mixer output, mean over 256 images)")
ax.set_ylabel("|Δ accuracy| (pp), 50K images"); ax.grid(alpha=0.3)
ax.set_title("Figure 1. Damage vs perturbation size: real ablation and size-matched null")
_save(fig, "fig1_damage_vs_size_real_and_null.png")

# ---- Figure 3: mean ablation vs resampling ----
mb = {r["block"]: r for r in FD["block_ablation"]}; rb = {r["block"]: r for r in FD["resampling"]}
fig, ax = plt.subplots(figsize=(6, 5.5))
for b in mb:
    ax.scatter(mb[b]["delta_pp"], rb[b]["delta_pp"], color=COL[mb[b]["type"]], s=60, edgecolor="k")
    ax.annotate(b, (mb[b]["delta_pp"], rb[b]["delta_pp"]), xytext=(4, 3), textcoords="offset points", fontsize=8)
lim = [min(min(r["delta_pp"] for r in FD["resampling"]), -1.2) - 0.1, 0.05]
ax.plot(lim, lim, ls="--", color="gray", lw=1); ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel("Mean-ablation Δ (pp)"); ax.set_ylabel("Resampling Δ (pp)"); ax.grid(alpha=0.3)
ax.set_title(f"Figure 3. Spearman ρ = {FD['spearman']['spearman_rho']:.3f}, p = {FD['spearman']['spearman_p']:.1e} (n = 12 blocks)")
_save(fig, "fig3_mean_vs_resampling.png")

# ---- Figure 4: SSM pathway vs gate pathway (v3), four metrics ----
pwd = {(r["block"], r["mode"]): r for r in FD["pathway"]}
B6 = sorted({r["block"] for r in FD["pathway"]}); xs = np.arange(len(B6)); w = 0.38
fl = FD["kl_floor"]["mean_kl"] if FD.get("kl_floor") else None
fig, axes = plt.subplots(2, 2, figsize=(11, 7.5))
panels = [("delta_pp", "Δ accuracy (pp), 95% CI", True), ("mean_logit_drop", "True-class logit drop", False),
          ("mean_kl", "KL(baseline ‖ intervened)", False), ("flip_rate_pct", "Flip rate (%), b+c", False)]
for ax, (key, ttl, ci) in zip(axes.flat, panels):
    for j, (mode, col) in enumerate([("gate_off", "#9467bd"), ("ssm_off", "#2ca02c")]):
        v = [pwd[(b, mode)][key] for b in B6]
        err = None
        if ci:
            err = np.array([[pwd[(b, mode)][key] - pwd[(b, mode)]["ci_low_pp"],
                             pwd[(b, mode)]["ci_high_pp"] - pwd[(b, mode)][key]] for b in B6]).T
        ax.bar(xs + (j - 0.5) * w, v, w, yerr=err, color=col, capsize=3, label=mode)
    if key == "mean_kl" and fl is not None:
        ax.axhline(fl, color="k", ls=":", lw=1, label=f"no-op floor {fl:.1e}")
    ax.axhline(0, color="k", lw=0.6); ax.set_xticks(xs); ax.set_xticklabels(B6); ax.set_title(ttl); ax.grid(axis="y", alpha=0.3)
axes[0, 0].legend(fontsize=8); axes[1, 0].legend(fontsize=8)
fig.suptitle("Figure 4. SSM pathway vs gate pathway (v3 run, 50K images)")
plt.tight_layout(); _save(fig, "fig4_ssm_vs_gate_pathway.png")

# ---- Figure 5: baseline / joint freeze / all Mamba off ----
j = FD["joint_freeze"]; g = FD["grouped"]
vals = [FD["baseline_accuracy"] * 100, j["acc_all_frozen"] * 100, g["mamba_group_acc"]]
errs = [[0, j["delta_pp_all_frozen"] - j["frozen_ci_low_pp"], g["mamba_delta_pp"] - g["mamba_ci_low_pp"]],
        [0, j["frozen_ci_high_pp"] - j["delta_pp_all_frozen"], g["mamba_ci_high_pp"] - g["mamba_delta_pp"]]]
fig, ax = plt.subplots(figsize=(6.5, 5))
ax.bar(["Baseline", "All Mamba Δ,B,C frozen", "All Mamba blocks off"], vals, yerr=errs, capsize=5,
       color=["#7f7f7f", "#1f77b4", "#aec7e8"])
for i, v in enumerate(vals): ax.text(i + 0.22, v + 0.08, f"{v:.2f}%", ha="left", fontsize=10)
ax.set_ylim(min(vals) - 2, max(vals) + 1.2); ax.set_ylabel("Top-1 accuracy (%)"); ax.grid(axis="y", alpha=0.3)
sf = j["selectivity_fraction"]; sfc = j.get("selectivity_ci", [np.nan, np.nan])
ax.set_title(f"Figure 5. Selectivity fraction = {sf*100:.1f}% (95% CI {sfc[0]*100:.1f}–{sfc[1]*100:.1f}%)")
_save(fig, "fig5_selectivity_fraction.png")

# ---- Figure 6: per-block component freeze (mean mode) with CIs ----
fzd = {(r["block"], r["tensor"], r["mode"]): r for r in FD["freezing"]}
comps = [("delta", "Δ"), ("B", "B"), ("C", "C"), ("all", "Δ+B+C")]
fig, ax = plt.subplots(figsize=(10, 4.8)); w = 0.2
for j_, (t, lab) in enumerate(comps):
    v = [fzd[(b, t, "mean")]["delta_pp"] for b in B6]
    e = np.array([[fzd[(b, t, "mean")]["delta_pp"] - fzd[(b, t, "mean")]["ci_low_pp"],
                   fzd[(b, t, "mean")]["ci_high_pp"] - fzd[(b, t, "mean")]["delta_pp"]] for b in B6]).T
    ax.bar(xs + (j_ - 1.5) * w, v, w, yerr=e, capsize=2, label=lab)
ax.axhspan(-0.15, 0.15, color="gray", alpha=0.12, label="±0.15 pp")
ax.axhline(0, color="k", lw=0.6); ax.set_xticks(xs); ax.set_xticklabels(B6); ax.set_ylabel("Δ accuracy (pp), 95% CI")
ax.legend(ncol=5, fontsize=8); ax.grid(axis="y", alpha=0.3)
ax.set_title("Figure 6. Freezing selectivity per block (mean replacement, 50K images)")
_save(fig, "fig6_component_freeze_ci.png")

# ---- Figure 7: real − null per block with paired CI ----
fig, ax = plt.subplots(figsize=(9, 4.5))
rv = FD["real_vs_null"]; xs12 = np.arange(len(rv))
v = [r["real_minus_null_pp"] for r in rv]
e = np.array([[r["real_minus_null_pp"] - r["ci_low"], r["ci_high"] - r["real_minus_null_pp"]] for r in rv]).T
ax.bar(xs12, v, yerr=e, capsize=3, color=[COL[r["type"]] for r in rv])
ax.axhline(0, color="k", lw=0.8)
ax.set_xticks(xs12); ax.set_xticklabels([r["block"] for r in rv], rotation=45)
ax.set_ylabel("acc(real ablation) − acc(shuffled null), pp")
ax.set_title("Figure 7. Real vs size-matched null, paired 95% CI (below 0 = real ablation worse)")
ax.grid(axis="y", alpha=0.3)
_save(fig, "fig7_real_minus_null.png")

# ---- upload ----
for p in saved + [fd_path]:
    api.upload_file(path_or_fileobj=p, path_in_repo=f"figures/{os.path.basename(p)}", repo_id=HF_REPO_ID, repo_type="dataset")
print(f"Uploaded {len(saved)} figures + figures_data.json")
print("STEP 22I COMPLETE — figures + figures_data.json uploaded to HF figures/")

## Step 23 — Result summary (export only, no new experiment) ##
Writes every saved checkpoint (up to Step 22) into `report_data.json` for the consolidated report
and uploads it to HF (`final_results/report_data.json`). Small arrays are kept; large arrays are
listed by shape only. Runs on CPU; no GPU needed.

In [ ]:
# ============================================================
# STEP 23 — Full results dump (every checkpoint -> report_data.json)
# ============================================================
import json, os, time
import numpy as np
import torch

def to_jsonable(x):
    if isinstance(x, torch.Tensor):
        return x.tolist() if x.numel() <= 20 else f"<tensor shape={tuple(x.shape)}>"
    if isinstance(x, np.ndarray):
        return x.tolist() if x.size <= 20 else f"<array shape={x.shape}>"
    if isinstance(x, (np.floating, np.integer, np.bool_)):
        return x.item()
    if isinstance(x, dict):
        if len(x) > 200:
            return f"<dict with {len(x)} keys>"
        return {str(k): to_jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        if len(x) > 60 and not (len(x) and isinstance(x[0], dict)):
            return f"<list len={len(x)}>"
        return [to_jsonable(v) for v in x]
    if x is None or isinstance(x, (int, float, str, bool)):
        return x
    return str(x)[:200]

names = list_checkpoints()
report = {"repo": HF_REPO_ID, "created": time.strftime("%Y-%m-%d %H:%M"), "n_checkpoints": len(names), "checkpoints": {}}
for n in names:
    try:
        report["checkpoints"][n] = to_jsonable(load_checkpoint(n))
        print(f"  ok   {n}")
    except Exception as e:
        report["checkpoints"][n] = f"<load error: {e}>"
        print(f"  FAIL {n}: {e}")

os.makedirs(RESULTS_DIR, exist_ok=True)
out_path = os.path.join(RESULTS_DIR, "report_data.json")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=1, ensure_ascii=False)
api.upload_file(path_or_fileobj=out_path, path_in_repo="final_results/report_data.json",
                repo_id=HF_REPO_ID, repo_type="dataset", commit_message="full results dump for report")

print(f"\n{len(names)} checkpoints -> {out_path} ({os.path.getsize(out_path)/1024:.0f} KB), also uploaded to HF final_results/")
expected = ["update_norms_2k", "kl_noop_floor", "step9_logit_kl", "figure2_attention_distance_v2",
            "deit_control_50k_v2", "step12_stratified_external", "high_norm_tokens", "l3b1_gate_logit_diagnostic"]
missing = [e for e in expected if e not in names]
print("Missing (run these steps if needed):", missing if missing else "NONE")
print("STEP 23 COMPLETE")

## Step 24 — Speed test and image cache ##
Measures where the time of one 50K pass goes, then stores the resized and cropped images once
so later steps do not decode JPEGs again. Results must stay identical to the Step 6 baseline.

### Step 24A — Speed test (data loading vs model) ###
Times 5 batches: image loading, the fp32 forward pass and an fp16 forward pass (for information only).

In [ ]:
# ============================================================
# STEP 24A — SPEED TEST: where does the time of one 50K pass go?
# ============================================================
import time, math

assert "eval_indices" in globals() and "baseline_preds_np" in globals(), "Run Step 11b and its helper cell first."
clear_all_hooks()
model.eval()

def _sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def _logits(out):
    return out["logits"] if isinstance(out, dict) else out.logits

N_TEST_BATCHES = 6
_idx = [eval_indices[i * BATCH_SIZE:(i + 1) * BATCH_SIZE] for i in range(N_TEST_BATCHES)]

# warm-up batch (not timed)
imgs, _ = prepare_batch(dataset, _idx[0], transform, device)
with torch.no_grad():
    model(imgs)
_sync()

t_load, t_fwd32, t_fwd16, agree16 = [], [], [], []
fp16_ok = True
for bi in _idx[1:]:
    t0 = time.perf_counter()
    imgs, _ = prepare_batch(dataset, bi, transform, device)
    _sync(); t_load.append(time.perf_counter() - t0)

    t0 = time.perf_counter()
    with torch.no_grad():
        p32 = _logits(model(imgs)).argmax(-1)
    _sync(); t_fwd32.append(time.perf_counter() - t0)

    if fp16_ok:
        try:
            t0 = time.perf_counter()
            with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
                p16 = _logits(model(imgs)).argmax(-1)
            _sync(); t_fwd16.append(time.perf_counter() - t0)
            agree16.append((p32 == p16).float().mean().item())
        except Exception as e:
            fp16_ok = False
            print("fp16 forward failed (ignored):", type(e).__name__, str(e)[:120])

n_batches = math.ceil(N_FULL / BATCH_SIZE)
L, F32 = np.mean(t_load), np.mean(t_fwd32)
print(f"Per batch of {BATCH_SIZE}:  loading {L:.3f} s | fp32 forward {F32:.3f} s"
      + (f" | fp16 forward {np.mean(t_fwd16):.3f} s (argmax agreement {np.mean(agree16)*100:.1f}%)" if t_fwd16 else ""))
print(f"Projected 50K pass:  loading {L*n_batches/60:.1f} min + model {F32*n_batches/60:.1f} min "
      f"= {(L+F32)*n_batches/60:.1f} min")
print(f"Share of time spent loading images: {L/(L+F32)*100:.0f}%")
print("\nSTEP 24A COMPLETE")

### Step 24B — Image cache (resize + centre crop once, stored as uint8) ###
Applies the timm resize and centre crop to all 50,000 images in the Step 5b order and stores them
as uint8 (about 7.5 GB, temporary disk). Normalisation is applied later on the GPU. Uint8 storage
is lossless, because the crop is uint8 before ToTensor. Rebuild once per session.

In [ ]:
# ============================================================
# STEP 24B — IMAGE CACHE (uint8, Step 5b order)
# ============================================================
import os, shutil, time
from torch.utils.data import Dataset as _TorchDataset, DataLoader
from torchvision import transforms as T

# ---- labels in the fixed order (fix for the earlier NameError; LAB comes from Step 45A) ----
if "eval_labels_np" not in globals():
    assert "LAB" in globals(), "Run Step 45A first."
    eval_labels_np = LAB.copy()

# ---- split the locked timm transform into: PIL part | ToTensor | Normalize ----
_tf = list(TIMM_TRANSFORM.transforms)
print("Timm transform:", [type(t).__name__ for t in _tf])
_k = next(i for i, t in enumerate(_tf) if "ToTensor" in type(t).__name__)
assert len(_tf) == _k + 2 and type(_tf[_k + 1]).__name__ == "Normalize", "Unexpected transform layout - send me this output."
PRE_TRANSFORM = T.Compose(_tf[:_k])                      # resize + centre crop, returns a uint8 PIL image
NORM_MEAN = torch.as_tensor(_tf[_k + 1].mean, dtype=torch.float32, device=device).view(1, 3, 1, 1)
NORM_STD  = torch.as_tensor(_tf[_k + 1].std,  dtype=torch.float32, device=device).view(1, 3, 1, 1)

# ---- pick a folder with enough free disk ----
need_gb = N_FULL * 224 * 224 * 3 / 1e9
CACHE_DIR = None
for d in ["/kaggle/temp", "/tmp", "/kaggle/working"]:
    try:
        os.makedirs(d, exist_ok=True)
        free_gb = shutil.disk_usage(d).free / 1e9
        print(f"{d}: {free_gb:.1f} GB free")
        if free_gb > need_gb + 1.5 and CACHE_DIR is None:
            CACHE_DIR = d
    except Exception:
        pass
assert CACHE_DIR is not None, f"Need about {need_gb:.1f} GB of free disk."
CACHE_IMG  = os.path.join(CACHE_DIR, f"mv_cache_images_u8_{N_FULL}.npy")
CACHE_LAB  = os.path.join(CACHE_DIR, f"mv_cache_labels_{N_FULL}.npy")
CACHE_DONE = os.path.join(CACHE_DIR, f"mv_cache_{N_FULL}.done")
print("Cache folder:", CACHE_DIR)

class _CacheSource(_TorchDataset):
    def __len__(self):
        return N_FULL
    def __getitem__(self, pos):
        ex = dataset[eval_indices[pos]]
        img = np.asarray(PRE_TRANSFORM(ex["image"].convert("RGB")), dtype=np.uint8)
        assert img.shape == (224, 224, 3), img.shape
        return img.copy(), ex["label"]

if not os.path.exists(CACHE_DONE):
    n_workers = os.cpu_count() or 2
    loader = DataLoader(_CacheSource(), batch_size=256, shuffle=False, num_workers=n_workers)
    mm = np.lib.format.open_memmap(CACHE_IMG, mode="w+", dtype=np.uint8, shape=(N_FULL, 224, 224, 3))
    labels_out = np.empty(N_FULL, dtype=np.int64)
    pos, t0 = 0, time.time()
    for bi, (x, y) in enumerate(loader):
        n = x.shape[0]
        mm[pos:pos + n] = x.numpy()
        labels_out[pos:pos + n] = y.numpy()
        pos += n
        if bi % 20 == 0:
            print(f"  {pos:,}/{N_FULL:,} images  ({(time.time()-t0)/60:.1f} min)")
    mm.flush(); del mm
    np.save(CACHE_LAB, labels_out)
    open(CACHE_DONE, "w").write("ok")
    print(f"Cache built in {(time.time()-t0)/60:.1f} min with {n_workers} workers")
else:
    print("Cache already exists in this session - reusing it.")

IMG_CACHE = np.load(CACHE_IMG, mmap_mode="r")
LAB_CACHE = np.load(CACHE_LAB)
assert (LAB_CACHE == eval_labels_np).all(), "Cached labels do not match the Step 6 order."

def get_cached_batch(start, end):
    """Normalised float32 batch on the GPU, identical to prepare_batch(...) for the same positions."""
    x = torch.from_numpy(np.ascontiguousarray(IMG_CACHE[start:end])).to(device, non_blocking=True)
    x = x.permute(0, 3, 1, 2).float().div(255)
    return x.sub(NORM_MEAN).div(NORM_STD)

# ---- equality check against the original loader (first 64 images) ----
ref_imgs, _ = prepare_batch(dataset, eval_indices[:64], transform, device)
max_diff = (get_cached_batch(0, 64) - ref_imgs).abs().max().item()
print(f"Max pixel difference vs prepare_batch (64 images): {max_diff:.2e}  (must be ~0)")
assert max_diff < 1e-5, "Cached images differ from prepare_batch - do not use the cache."
print("\nSTEP 24B COMPLETE")

### Step 24C — Fast evaluation pass + baseline reproduction check ###
Defines `eval_pass_fast` (reads the cache and also returns the true-class log-probability drop and KL).
Runs one full 50K baseline pass with it: predictions must match the Step 6 baseline on 100% of images.
Saves the timings as `step24_speed`.

In [ ]:
# ============================================================
# STEP 24C — FAST EVAL PASS + BASELINE REPRODUCTION CHECK
# ============================================================
import time
assert "get_cached_batch" in globals(), "Run Step 24B first."
_LAB_GPU = torch.as_tensor(LAB_CACHE, device=device)

def eval_pass_fast(n=None, logprob_kl=False, bs=None):
    """One forward pass over cache positions 0..n-1 (same order as eval_indices).
    logprob_kl=True also returns per-image true-class log-prob drop and KL(baseline || intervened)."""
    n = N_FULL if n is None else n
    bs = BATCH_SIZE if bs is None else bs
    preds, lp_drops, kls = [], [], []
    model.eval()
    with torch.no_grad():
        for s in range(0, n, bs):
            e = min(s + bs, n)
            out = model(get_cached_batch(s, e))
            logits = (out["logits"] if isinstance(out, dict) else out.logits).float()
            preds.append(logits.argmax(-1).cpu())
            if logprob_kl:
                lp = F.log_softmax(logits, -1)
                y = _LAB_GPU[s:e].view(-1, 1)
                base_lp = baseline_log_probs[s:e].to(device).float()
                lp_drops.append((base_lp.gather(1, y) - lp.gather(1, y)).squeeze(1).cpu())
                kls.append((base_lp.exp() * (base_lp - lp)).sum(-1).cpu())
    preds = torch.cat(preds).numpy()
    if logprob_kl:
        return preds, torch.cat(lp_drops).numpy(), torch.cat(kls).numpy()
    return preds

clear_all_hooks()
torch.cuda.synchronize(); t0 = time.time()
fast_preds, fast_lp_drop, fast_kl = eval_pass_fast(logprob_kl=True)
torch.cuda.synchronize(); fast_min = (time.time() - t0) / 60

agree = (fast_preds == baseline_preds_np).mean()
n_correct = int((fast_preds == eval_labels_np).sum())
print(f"Fast 50K pass: {fast_min:.1f} min")
print(f"Agreement with Step 6 baseline predictions: {agree*100:.4f}%  (must be 100%)")
print(f"Correct: {n_correct:,} (expected {EXPECTED_BASELINE_CORRECT:,})")
print(f"Log-prob drop vs baseline: mean {fast_lp_drop.mean():+.2e}, max |.| {np.abs(fast_lp_drop).max():.2e}  (float16 storage noise)")
print(f"KL vs baseline: mean {fast_kl.mean():.2e}  (no-op floor was 8.0e-07)")

FAST_OK = bool(agree == 1.0 and n_correct == EXPECTED_BASELINE_CORRECT)
print("\nFAST PATH OK - use eval_pass_fast in Steps 25-29." if FAST_OK
      else "\nFAST PATH NOT IDENTICAL - keep using eval_pass and send me this output.")

save_checkpoint("step24_speed", {
    "per_batch_load_s": float(np.mean(t_load)), "per_batch_fwd32_s": float(np.mean(t_fwd32)),
    "per_batch_fwd16_s": float(np.mean(t_fwd16)) if t_fwd16 else None,
    "fp16_argmax_agreement": float(np.mean(agree16)) if agree16 else None,
    "fast_pass_min": float(fast_min), "fast_agreement": float(agree), "fast_n_correct": n_correct,
    "fast_ok": FAST_OK, "batch_size": BATCH_SIZE, "cache_dir": CACHE_DIR,
})
print("\nSTEP 24C COMPLETE")

## Step 25 — Joint position-mode freeze ##
Mean-mode freezing (Step 14) removes two things: image-dependence (selectivity) and
position-dependence. Here Δ, B and C are fixed to their per-position, per-channel means
(over the 256 Step 8 reference images) in all six Mamba blocks at once. Only image-dependence
is removed. Mean mode is re-run on the same fast path, as a reproduction check and to obtain
log-probability drops. If position mode costs ≈ 0 pp, image-dependent selectivity is inert;
if it costs ≈ 1 pp, selectivity is real.

### Step 25A — Joint freeze of Δ, B, C: mean mode vs position mode ###
Two 50K runs (about 7 min). Reports Δ accuracy, log-probability drop, KL, the ratio to
all-Mamba-off with a paired bootstrap CI, and the paired contrast position − mean.
Saves `step25_joint_freeze_modes` after each run (resumable).

In [ ]:
# ============================================================
# STEP 25A — JOINT FREEZE OF Δ, B, C IN ALL 6 MAMBA BLOCKS: MEAN vs POSITION MODE
# ============================================================
assert "FAST_OK" in globals() and FAST_OK, "Run Step 24 first (fast path must be OK)."
assert "install_freeze_patch" in globals() and "_freeze_refs" in globals(), "Run Step 13 Part A and Part B first."

CKPT_25 = "step25_joint_freeze_modes"
partC = load_checkpoint("partC_grouped_ablation")
mamba_off_correct = unpack(partC["mamba_correct_packed"])
step14 = load_checkpoint("step14_freeze_all_mamba")

def mean_ci(x, n_boot=2000, seed=42):
    x = np.asarray(x, dtype=np.float64); n = len(x)
    rng = np.random.RandomState(seed)
    boot = np.array([x[rng.randint(0, n, n)].mean() for _ in range(n_boot)])
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return float(x.mean()), [float(lo), float(hi)]

def ratio_ci(correct_a, correct_b, n_boot=2000, seed=42):
    """drop(a) / drop(b) vs baseline, same resampled images for both."""
    bc = baseline_correct.astype(np.int8)
    da, db = bc - correct_a.astype(np.int8), bc - correct_b.astype(np.int8)
    rng = np.random.RandomState(seed); n = len(bc)
    boot = []
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        boot.append(da[idx].mean() / db[idx].mean())
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return float(da.mean() / db.mean()), [float(lo), float(hi)]

def run_freeze(tensor, mode, blocks=None):
    blocks = set(mamba_block_names) if blocks is None else set(blocks)
    hs = install_freeze_patch()
    try:
        _freeze_config.update({"active": True, "block_names": blocks, "tensor": tensor, "mode": mode})
        preds, lp_drop, kl = eval_pass_fast(logprob_kl=True)
    finally:
        remove_freeze_patch(hs)
    st, corr = paired_stats(preds)
    lp_m, lp_ci = mean_ci(lp_drop)
    kl_m, kl_ci = mean_ci(kl)
    return {**st, "logprob_drop": lp_m, "logprob_drop_ci": lp_ci, "kl": kl_m, "kl_ci": kl_ci,
            "correct_packed": pack(corr)}, corr

try:
    res25 = load_checkpoint(CKPT_25)
    print("Resuming; already done:", [k for k in res25 if k.startswith("all|")])
except Exception:
    res25 = {}

for mode in ["mean", "position"]:
    key = f"all|{mode}"
    if key in res25:
        continue
    t0 = time.time()
    r, corr = run_freeze("all", mode)
    r["sf_vs_all_mamba_off"], r["sf_ci"] = ratio_ci(corr, mamba_off_correct)
    res25[key] = r
    save_checkpoint(CKPT_25, res25)
    print(f"{mode:8s} done in {(time.time()-t0)/60:.1f} min")

# ---- paired contrast: position − mean (positive = position mode less damaging) ----
c_mean = unpack(res25["all|mean"]["correct_packed"])
c_pos  = unpack(res25["all|position"]["correct_packed"])
diff = (c_pos.astype(np.int8) - c_mean.astype(np.int8)) * 100.0
d_m, d_ci = mean_ci(diff)
res25["contrast_position_minus_mean_pp"] = d_m
res25["contrast_ci"] = d_ci
save_checkpoint(CKPT_25, res25)

# ---- report ----
print(f"\nReproduction check: mean mode Δ = {res25['all|mean']['delta_pp']:+.3f} pp "
      f"(Step 14: {step14['delta_pp_all_frozen']:+.3f} pp)")
print(f"\n{'Mode':9s} {'Δ acc (pp) [95% CI]':28s} {'McNemar p':>10s} {'log-p drop [CI]':26s} {'KL':>8s} {'÷ all-Mamba-off [CI]':>24s}")
for mode in ["mean", "position"]:
    r = res25[f"all|{mode}"]
    print(f"{mode:9s} {r['delta_pp']:+.3f} [{r['ci_low_pp']:+.3f},{r['ci_high_pp']:+.3f}]      "
          f"{r['mcnemar_p']:10.2e} {r['logprob_drop']:+.4f} [{r['logprob_drop_ci'][0]:+.4f},{r['logprob_drop_ci'][1]:+.4f}]  "
          f"{r['kl']:8.4f}  {r['sf_vs_all_mamba_off']*100:5.1f}% [{r['sf_ci'][0]*100:.1f}, {r['sf_ci'][1]*100:.1f}]")
print(f"\nPosition − mean (paired): {d_m:+.3f} pp  95% CI [{d_ci[0]:+.3f}, {d_ci[1]:+.3f}]")
print("\nSTEP 25A COMPLETE")

### Step 25B — Which tensor carries the effect? (Δ, B, C separately, all six blocks) ###
Joint freeze of one tensor at a time, in mean and position mode (6 runs, about 20 min).
Shows whether the position-mode result comes from Δ, B or C. Adds to `step25_joint_freeze_modes`.

In [ ]:
# ============================================================
# STEP 25B — JOINT FREEZE PER TENSOR (Δ, B, C) × (mean, position)
# ============================================================
for tensor in ["delta", "B", "C"]:
    for mode in ["mean", "position"]:
        key = f"{tensor}|{mode}"
        if key in res25:
            continue
        t0 = time.time()
        r, corr = run_freeze(tensor, mode)
        r["sf_vs_all_mamba_off"], r["sf_ci"] = ratio_ci(corr, mamba_off_correct)
        res25[key] = r
        save_checkpoint(CKPT_25, res25)
        print(f"{tensor:5s} {mode:8s} done in {(time.time()-t0)/60:.1f} min")

print(f"\n{'Tensor':6s} {'Mode':9s} {'Δ acc (pp) [95% CI]':28s} {'McNemar p':>10s} {'log-p drop':>11s} {'KL':>8s}")
for tensor in ["all", "delta", "B", "C"]:
    for mode in ["mean", "position"]:
        r = res25[f"{tensor}|{mode}"]
        print(f"{tensor:6s} {mode:9s} {r['delta_pp']:+.3f} [{r['ci_low_pp']:+.3f},{r['ci_high_pp']:+.3f}]      "
              f"{r['mcnemar_p']:10.2e} {r['logprob_drop']:+11.4f} {r['kl']:8.4f}")
print("\nSTEP 25B COMPLETE")

## Step 26 — Group shuffled nulls ##
The shuffled-channel null of Step 16, applied to all six Mamba blocks at once and to all six
Attention blocks at once (same permutations as Step 16). Each block's perturbation is computed
from its actual output in the perturbed run, as in the grouped mean ablation (Step 11d).
Compares each group null with the real grouped ablation, paired per image.
Saves `step26_group_nulls`.

In [ ]:
# ============================================================
# STEP 26 — GROUP SHUFFLED NULLS: all Mamba at once, all Attention at once
# ============================================================
assert "FAST_OK" in globals() and FAST_OK, "Run Step 24 first."
assert "block_means_cpu" in globals(), "Run Step 8 first."
CKPT_26 = "step26_group_nulls"

# ---- same permutations as Step 16 ----
torch.manual_seed(42)
channel_perms = {name: torch.randperm(block_means_cpu[name].shape[-1]) for name, _, _ in mixer_blocks}
_gnull = {"active": False, "blocks": set()}

def _make_gnull_hook(name):
    mean_val = block_means_cpu[name]
    perm = channel_perms[name]
    def f(module, inp, out):
        if not _gnull["active"] or name not in _gnull["blocks"]:
            return out
        is_tuple = isinstance(out, tuple)
        t = out[0] if is_tuple else out
        delta = mean_val.to(device=t.device, dtype=t.dtype).expand_as(t) - t
        rep = t + delta[..., perm.to(t.device)]
        return (rep,) + out[1:] if is_tuple else rep
    return f

def install_gnull_hooks():
    clear_all_hooks()
    return [blk.mixer.register_forward_hook(_make_gnull_hook(name)) for name, _, blk in mixer_blocks]

def run_null(blocks, n=None, logprob_kl=False):
    hs = install_gnull_hooks()
    try:
        _gnull.update({"active": True, "blocks": set(blocks)})
        return eval_pass_fast(n=n, logprob_kl=logprob_kl)
    finally:
        _gnull["active"] = False
        for h in hs:
            h.remove()

# ---- check: single-block L2B0 null must reproduce Step 16 (first 5,000 images) ----
sn16 = {r["block"]: r for r in load_checkpoint("shuffled_null_50k")["results"]}
p_chk = run_null(["L2B0"], n=5000)
agree = ((p_chk == eval_labels_np[:5000]) == unpack(sn16["L2B0"]["correct_packed"])[:5000]).mean()
print(f"Reproduction of Step 16 L2B0 null on 5,000 images: {agree*100:.2f}% per-image agreement (must be ~100%)")
assert agree > 0.999, "Permutations differ from Step 16 - send me this output."

# ---- group runs ----
groups = {"mamba": [n for n, t, _ in mixer_blocks if t == "MambaVisionMixer"],
          "attn":  [n for n, t, _ in mixer_blocks if t == "Attention"]}
try:
    res26 = load_checkpoint(CKPT_26)
    print("Resuming; already done:", list(res26))
except Exception:
    res26 = {}

for key, blocks in groups.items():
    if key in res26:
        continue
    t0 = time.time()
    preds, lp_drop, kl = run_null(blocks, logprob_kl=True)
    st, null_corr = paired_stats(preds)
    real_corr = unpack(partC[f"{key}_correct_packed"])
    diff = (real_corr.astype(np.int8) - null_corr.astype(np.int8)) * 100.0     # real − null, pp
    d_m, d_ci = mean_ci(diff)
    lp_m, lp_ci = mean_ci(lp_drop); kl_m, kl_ci = mean_ci(kl)
    res26[key] = {**st, "blocks": blocks, "logprob_drop": lp_m, "logprob_drop_ci": lp_ci, "kl": kl_m, "kl_ci": kl_ci,
                  "real_delta_pp": float(partC[f"{key}_delta_pp"]),
                  "real_minus_null_pp": d_m, "real_minus_null_ci": d_ci,
                  "correct_packed": pack(null_corr)}
    save_checkpoint(CKPT_26, res26)
    print(f"{key} group null done in {(time.time()-t0)/60:.1f} min")

print(f"\n{'Group':10s} {'Real Δ (pp)':>12s} {'Null Δ (pp) [95% CI]':>30s} {'Real − null [paired CI]':>30s} {'Null/real':>10s}")
for key, label in [("mamba", "All Mamba"), ("attn", "All Attn")]:
    r = res26[key]
    print(f"{label:10s} {r['real_delta_pp']:+12.3f} {r['delta_pp']:+11.3f} [{r['ci_low_pp']:+.3f},{r['ci_high_pp']:+.3f}]"
          f"   {r['real_minus_null_pp']:+9.3f} [{r['real_minus_null_ci'][0]:+.3f},{r['real_minus_null_ci'][1]:+.3f}]"
          f"   {r['delta_pp']/r['real_delta_pp']:9.2f}")
print("\nSTEP 26 COMPLETE")

## Step 27 — Shuffled null with 10 permutations per block ##

### Step 27A — 10 permutations × 12 blocks, one shared prefix per block ###
For each block, the network runs once up to that block; the batch is then copied 10 times and
each copy receives a different channel permutation of the same Δx = mean − x. Permutation 0 is
the Step 16 permutation (reproduction check). Saves `step27_multi_perm_null` after every block.

In [ ]:
# ============================================================
# STEP 27A — MULTI-PERMUTATION NULL: 10 permutations × 12 blocks (shared prefix)
# ============================================================
assert "FAST_OK" in globals() and FAST_OK, "Run Step 24 first."
assert "channel_perms" in globals(), "Run Step 26 first (it defines the Step 16 permutations)."
CKPT_27 = "step27_multi_perm_null"
K_PERM = 10           # permutation 0 = Step 16 permutation
K_CHUNK = 10          # if CUDA runs out of memory, set to 5 (the prefix is then computed twice)

def perms_for_block(bi, name):
    C = block_means_cpu[name].shape[-1]
    ps = [channel_perms[name]]
    for j in range(1, K_PERM):
        g = torch.Generator().manual_seed(1000 + 100 * bi + j)
        ps.append(torch.randperm(C, generator=g))
    return ps

_fork = {"k": 1, "perms": None, "name": None}

def _fork_pre_hook(module, args):
    x = args[0]
    return (x.repeat(_fork["k"], *([1] * (x.dim() - 1))),) + tuple(args[1:])

def _fork_mixer_hook(module, inp, out):
    is_tuple = isinstance(out, tuple)
    t = out[0] if is_tuple else out
    k = _fork["k"]; B = t.shape[0] // k
    mean_val = block_means_cpu[_fork["name"]].to(device=t.device, dtype=t.dtype)
    pieces = []
    for j in range(k):
        tj = t[j * B:(j + 1) * B]
        delta = mean_val.expand_as(tj) - tj
        pieces.append(tj + delta[..., _fork["perms"][j].to(t.device)])
    rep = torch.cat(pieces, 0)
    return (rep,) + out[1:] if is_tuple else rep

def eval_fork(name, blk, perms):
    """Returns correct (k, N) bool, log-prob drop mean (k,), KL mean (k,)."""
    k = len(perms)
    clear_all_hooks()
    _fork.update({"k": k, "perms": perms, "name": name})
    h1 = blk.register_forward_pre_hook(_fork_pre_hook)
    h2 = blk.mixer.register_forward_hook(_fork_mixer_hook)
    corr = np.zeros((k, N_FULL), dtype=bool)
    lp_sum = torch.zeros(k, dtype=torch.float64); kl_sum = torch.zeros(k, dtype=torch.float64)
    try:
        with torch.no_grad():
            for s in range(0, N_FULL, BATCH_SIZE):
                e = min(s + BATCH_SIZE, N_FULL); B = e - s
                out = model(get_cached_batch(s, e))
                logits = (out["logits"] if isinstance(out, dict) else out.logits).float().view(k, B, -1)
                lp = F.log_softmax(logits, -1)
                y = _LAB_GPU[s:e]
                corr[:, s:e] = (logits.argmax(-1) == y.view(1, B)).cpu().numpy()
                base_lp = baseline_log_probs[s:e].to(device).float()
                lp_true = lp.gather(2, y.view(1, B, 1).expand(k, B, 1)).squeeze(2)
                lp_sum += (base_lp.gather(1, y.view(-1, 1)).view(1, B) - lp_true).sum(1).double().cpu()
                kl_sum += (base_lp.exp().unsqueeze(0) * (base_lp.unsqueeze(0) - lp)).sum((1, 2)).double().cpu()
    finally:
        h1.remove(); h2.remove()
    return corr, (lp_sum / N_FULL).numpy(), (kl_sum / N_FULL).numpy()

try:
    res27 = load_checkpoint(CKPT_27)
    print("Resuming; already done:", list(res27))
except Exception:
    res27 = {}

for bi, (name, mtype, blk) in enumerate(mixer_blocks):
    if name in res27:
        continue
    t0 = time.time()
    perms = perms_for_block(bi, name)
    corr_all, lp_all, kl_all = [], [], []
    for c0 in range(0, K_PERM, K_CHUNK):
        c, lpd, kl = eval_fork(name, blk, perms[c0:c0 + K_CHUNK])
        corr_all.append(c); lp_all.append(lpd); kl_all.append(kl)
    corr = np.concatenate(corr_all, 0); lpd = np.concatenate(lp_all); kl = np.concatenate(kl_all)
    deltas = (corr.mean(1) - baseline_correct.mean()) * 100
    agree0 = (corr[0] == unpack(sn16[name]["correct_packed"])).mean()
    res27[name] = {"type": mtype, "delta_pp": deltas.tolist(), "logprob_drop": lpd.tolist(), "kl": kl.tolist(),
                   "agree_perm0_vs_step16": float(agree0),
                   "correct_packed": [pack(c) for c in corr]}
    save_checkpoint(CKPT_27, res27)
    print(f"{name:5s} {(time.time()-t0)/60:4.1f} min | null Δ over {K_PERM} perms: mean {deltas.mean():+.3f}, "
          f"sd {deltas.std(ddof=1):.3f}, range [{deltas.min():+.3f}, {deltas.max():+.3f}] | perm0 vs Step 16: {agree0*100:.3f}%")
print("\nSTEP 27A COMPLETE")

### Step 27B — Real ablation vs mean of 10 nulls, per block and pooled (bound) ###
Content effect per image = acc(mean of 10 nulls) − acc(real ablation); positive = real ablation more
damaging. Per block: paired bootstrap CI and Holm correction over 12 blocks. Pooled over all 12
blocks, and over Mamba and Attention separately: mean with a 95% CI and a one-sided 95% upper bound,
compared with a bound X fixed in advance. CPU only. Saves `step27_null_summary`.

In [ ]:
# ============================================================
# STEP 27B — REAL vs MEAN-OF-10 NULLS, POOLED BOUND (CPU only)
# ============================================================
from scipy.stats import norm, spearmanr
BOUND_X = 0.10        # pp — fixed in advance: "content-specific damage averages less than X pp per block"

res27 = load_checkpoint(CKPT_27)
_sp = load_checkpoint("step9_ablated_preds")
real_corr = {b: (np.asarray(_sp[b]).astype(np.int64) == eval_labels_np) for b in res27}

def boot_mean(x, n_boot=2000, seed=42):
    x = np.asarray(x, dtype=np.float64); n = len(x); rng = np.random.RandomState(seed)
    bm = np.array([x[rng.randint(0, n, n)].mean() for _ in range(n_boot)])
    lo, hi = np.percentile(bm, [2.5, 97.5]); up = np.percentile(bm, 95)
    se = bm.std(ddof=1)
    p = 2 * norm.sf(abs(x.mean()) / se) if se > 0 else 1.0
    return float(x.mean()), [float(lo), float(hi)], float(up), float(p)

content = {}          # per image, pp: mean over perms of null_correct − real_correct
rows = []
for b, r in res27.items():
    null_c = np.stack([unpack(p) for p in r["correct_packed"]]).astype(np.float64)
    content[b] = (null_c.mean(0) - real_corr[b].astype(np.float64)) * 100
    m, ci, up, p = boot_mean(content[b])
    d = np.array(r["delta_pp"])
    rows.append({"block": b, "type": r["type"], "null_mean_pp": float(d.mean()), "null_sd_pp": float(d.std(ddof=1)),
                 "null_min_pp": float(d.min()), "null_max_pp": float(d.max()),
                 "real_pp": float((real_corr[b].mean() - baseline_correct.mean()) * 100),
                 "content_pp": m, "content_ci": ci, "p": p})

# Holm over 12 blocks
order = np.argsort([x["p"] for x in rows]); m_tests = len(rows); running = 0
for rank, i in enumerate(order):
    running = max(running, min(1.0, (m_tests - rank) * rows[i]["p"]))
    rows[i]["p_holm"] = float(running)

print(f"{'Block':6s} {'Real Δ':>8s} {'Null mean ± sd (10 perms)':>27s} {'Content effect [95% CI]':>28s} {'p':>8s} {'Holm p':>8s}")
for x in rows:
    print(f"{x['block']:6s} {x['real_pp']:+8.3f}   {x['null_mean_pp']:+.3f} ± {x['null_sd_pp']:.3f} "
          f"[{x['null_min_pp']:+.2f},{x['null_max_pp']:+.2f}]   {x['content_pp']:+.3f} [{x['content_ci'][0]:+.3f},{x['content_ci'][1]:+.3f}]"
          f" {x['p']:8.4f} {x['p_holm']:8.4f}")

pooled = {}
for label, sel in [("all12", list(content)), ("mamba", [b for b in content if res27[b]["type"] == "MambaVisionMixer"]),
                   ("attn", [b for b in content if res27[b]["type"] == "Attention"])]:
    per_img = np.mean([content[b] for b in sel], axis=0)
    m, ci, up, p = boot_mean(per_img)
    pooled[label] = {"blocks": sel, "mean_pp": m, "ci": ci, "upper95_one_sided": up, "p": p,
                     "below_bound_X": bool(up < BOUND_X)}
    print(f"\nPooled {label:6s}: content effect {m:+.3f} pp  95% CI [{ci[0]:+.3f}, {ci[1]:+.3f}]  "
          f"one-sided 95% upper bound {up:+.3f} pp  -> {'below' if up < BOUND_X else 'NOT below'} X = {BOUND_X} pp")

rho = spearmanr([x["real_pp"] for x in rows], [x["null_mean_pp"] for x in rows])
print(f"\nSpearman (real Δ vs mean null Δ, 12 blocks): ρ = {rho.correlation:.3f}, p = {rho.pvalue:.2e}")

save_checkpoint("step27_null_summary", {"bound_X_pp": BOUND_X, "per_block": rows, "pooled": pooled,
                                        "spearman_real_vs_meannull": [float(rho.correlation), float(rho.pvalue)]})
print("\nSTEP 27B COMPLETE")

## Step 28 — Positive control: copying in mamba-130m ##
A pretrained language Mamba (state-spaces/mamba-130m-hf) is given random token sequences
repeated twice. A working model copies from context, so its loss on the second copy drops sharply.
Copying requires content-dependent (selective) dynamics. The same freezing method as in
MambaVision (Δ, B, C replaced by mean-mode or position-mode references from separate reference
sequences, all layers at once) should make copying collapse. If it does, the method can detect
selectivity when it is present. All variables use the prefix pc_ so the MambaVision model is untouched.

### Step 28A — Load mamba-130m, build the copy task, baseline ###
256 test sequences and 64 separate reference sequences, each 256 random tokens repeated twice
(seed 42). The pure-PyTorch scan path is used, so Δ (dt_proj output, pre-softplus) and B, C
(x_proj output) can be replaced with hooks.

In [ ]:
# ============================================================
# STEP 28A — mamba-130m copy task: model, data, baseline   (prefix pc_ = positive control)
# ============================================================
import torch, numpy as np, time
import torch.nn.functional as F
from transformers import MambaForCausalLM

PC_MODEL_ID = "state-spaces/mamba-130m-hf"
pc_model = MambaForCausalLM.from_pretrained(PC_MODEL_ID, torch_dtype=torch.float32).to(device).eval()
pc_mixers = [layer.mixer for layer in pc_model.backbone.layers]
assert all(hasattr(m, "slow_forward") for m in pc_mixers), "This transformers version has no slow_forward - send me the output."
for m in pc_mixers:
    m.forward = m.slow_forward          # pure-PyTorch scan: dt_proj and x_proj are called as modules
PC_R, PC_N = pc_mixers[0].time_step_rank, pc_mixers[0].ssm_state_size
print(f"{len(pc_mixers)} layers | d_inner {pc_mixers[0].intermediate_size} | d_state {PC_N} | dt_rank {PC_R}")

PC_SEED, PC_L, PC_N_TEST, PC_N_REF, PC_BS = 42, 256, 256, 64, 16
PC_VOCAB_LO, PC_VOCAB_HI = 1000, 50000
_pc_rng = np.random.RandomState(PC_SEED)
def pc_make_seqs(n):
    first = _pc_rng.randint(PC_VOCAB_LO, PC_VOCAB_HI, size=(n, PC_L))
    return torch.tensor(np.concatenate([first, first], 1), dtype=torch.long)
pc_test_ids = pc_make_seqs(PC_N_TEST)
pc_ref_ids  = pc_make_seqs(PC_N_REF)    # separate sequences, used only for the freezing references

def pc_evaluate(ids):
    """Per sequence: copy accuracy and mean loss on the second copy, mean loss on the first copy."""
    acc2, loss2, loss1 = [], [], []
    with torch.no_grad():
        for s in range(0, len(ids), PC_BS):
            x = ids[s:s + PC_BS].to(device)
            logits = pc_model(x, use_cache=False).logits.float()[:, :-1]
            tgt = x[:, 1:]
            nll = -F.log_softmax(logits, -1).gather(2, tgt.unsqueeze(-1)).squeeze(-1)   # col j predicts token j+1
            pred = logits.argmax(-1)
            loss1.append(nll[:, :PC_L - 1].mean(1).cpu())          # first copy, tokens 1..L-1 (unpredictable)
            loss2.append(nll[:, PC_L:].mean(1).cpu())              # second copy, tokens L+1..2L-1 (copyable)
            acc2.append((pred[:, PC_L:] == tgt[:, PC_L:]).float().mean(1).cpu())
    return {"copy_acc": torch.cat(acc2).numpy(), "loss_second": torch.cat(loss2).numpy(),
            "loss_first": torch.cat(loss1).numpy()}

t0 = time.time()
pc_base = pc_evaluate(pc_test_ids)
print(f"Baseline ({(time.time()-t0)/60:.1f} min): copy accuracy {pc_base['copy_acc'].mean()*100:.1f}% | "
      f"loss first copy {pc_base['loss_first'].mean():.2f} | loss second copy {pc_base['loss_second'].mean():.2f} "
      f"(chance loss ≈ {np.log(PC_VOCAB_HI - PC_VOCAB_LO):.2f})")
print("\nSTEP 28A COMPLETE")

### Step 28B — Freeze Δ, B, C in all layers (mean and position mode) ###
References are computed from the 64 reference sequences: mean mode averages over sequences and
positions; position mode averages over sequences only. Each condition is compared with the
baseline per sequence (paired bootstrap). Saves `step28_positive_control`.

In [ ]:
# ============================================================
# STEP 28B — FREEZING Δ, B, C IN ALL LAYERS: does copying collapse?
# ============================================================
# ---- references (running sums, to save GPU memory) ----
_pc_acc = [{"dt": None, "xp": None, "n": 0} for _ in pc_mixers]
def _pc_cap_dt(i):
    def f(mod, inp, out):
        s = out.detach().sum(0); _pc_acc[i]["dt"] = s if _pc_acc[i]["dt"] is None else _pc_acc[i]["dt"] + s
        _pc_acc[i]["n"] += out.shape[0]
    return f
def _pc_cap_xp(i):
    def f(mod, inp, out):
        s = out.detach().sum(0); _pc_acc[i]["xp"] = s if _pc_acc[i]["xp"] is None else _pc_acc[i]["xp"] + s
    return f
_hs = [m.dt_proj.register_forward_hook(_pc_cap_dt(i)) for i, m in enumerate(pc_mixers)] + \
      [m.x_proj.register_forward_hook(_pc_cap_xp(i)) for i, m in enumerate(pc_mixers)]
with torch.no_grad():
    for s in range(0, PC_N_REF, PC_BS):
        pc_model(pc_ref_ids[s:s + PC_BS].to(device), use_cache=False)
for h in _hs: h.remove()
pc_refs = []
for a in _pc_acc:
    dt_pos, xp_pos = (a["dt"] / a["n"]).unsqueeze(0), (a["xp"] / a["n"]).unsqueeze(0)     # (1, 2L, dim)
    pc_refs.append({"dt": {"position": dt_pos, "mean": dt_pos.mean(1, keepdim=True)},
                    "xp": {"position": xp_pos, "mean": xp_pos.mean(1, keepdim=True)}})

# ---- freezing hooks (same logic as MambaVision Step 13: replace Δ, B, C by a reference) ----
_pc_fz = {"active": False, "tensor": None, "mode": None}
def _pc_dt_hook(i):
    def f(mod, inp, out):
        if _pc_fz["active"] and _pc_fz["tensor"] in ("delta", "all"):
            return pc_refs[i]["dt"][_pc_fz["mode"]].expand_as(out).to(out.dtype)
        return out
    return f
def _pc_xp_hook(i):
    def f(mod, inp, out):
        if not _pc_fz["active"] or _pc_fz["tensor"] == "delta":
            return out
        ref = pc_refs[i]["xp"][_pc_fz["mode"]].expand_as(out)
        out = out.clone()
        if _pc_fz["tensor"] in ("B", "all"): out[..., PC_R:PC_R + PC_N] = ref[..., PC_R:PC_R + PC_N]
        if _pc_fz["tensor"] in ("C", "all"): out[..., PC_R + PC_N:] = ref[..., PC_R + PC_N:]
        return out
    return f
pc_hooks = [m.dt_proj.register_forward_hook(_pc_dt_hook(i)) for i, m in enumerate(pc_mixers)] + \
           [m.x_proj.register_forward_hook(_pc_xp_hook(i)) for i, m in enumerate(pc_mixers)]

# sanity: hooks installed but inactive must reproduce the baseline exactly
_chk = pc_evaluate(pc_test_ids[:PC_BS])
assert np.allclose(_chk["copy_acc"], pc_base["copy_acc"][:PC_BS]), "Inactive hooks changed the output."

def pc_paired_ci(a, b, n_boot=2000, seed=42):
    d = np.asarray(a) - np.asarray(b); rng = np.random.RandomState(seed); n = len(d)
    bm = [d[rng.randint(0, n, n)].mean() for _ in range(n_boot)]
    return float(d.mean()), [float(np.percentile(bm, 2.5)), float(np.percentile(bm, 97.5))]

pc_results = {"baseline": {k: float(v.mean()) for k, v in pc_base.items()}}
pc_conditions = [("all", "mean"), ("all", "position"), ("delta", "mean"), ("delta", "position"),
                 ("B", "mean"), ("B", "position"), ("C", "mean"), ("C", "position")]
try:
    for tensor, mode in pc_conditions:
        t0 = time.time()
        _pc_fz.update({"active": True, "tensor": tensor, "mode": mode})
        r = pc_evaluate(pc_test_ids)
        _pc_fz["active"] = False
        d_acc, ci_acc = pc_paired_ci(r["copy_acc"] * 100, pc_base["copy_acc"] * 100)
        d_l2, ci_l2 = pc_paired_ci(r["loss_second"], pc_base["loss_second"])
        pc_results[f"{tensor}|{mode}"] = {"copy_acc": float(r["copy_acc"].mean()), "loss_second": float(r["loss_second"].mean()),
                                          "loss_first": float(r["loss_first"].mean()),
                                          "delta_copy_acc_pp": d_acc, "delta_copy_acc_ci": ci_acc,
                                          "delta_loss_second": d_l2, "delta_loss_second_ci": ci_l2}
        print(f"{tensor:5s} {mode:8s} ({(time.time()-t0)/60:.1f} min): copy acc {r['copy_acc'].mean()*100:5.1f}% "
              f"(Δ {d_acc:+.1f} pp [{ci_acc[0]:+.1f}, {ci_acc[1]:+.1f}]) | loss 2nd copy {r['loss_second'].mean():.2f}")
finally:
    _pc_fz["active"] = False
    for h in pc_hooks: h.remove()

save_checkpoint("step28_positive_control", {
    "model": PC_MODEL_ID, "seed": PC_SEED, "half_length": PC_L, "n_test": PC_N_TEST, "n_ref": PC_N_REF,
    "vocab_range": [PC_VOCAB_LO, PC_VOCAB_HI], "layers": len(pc_mixers), "results": pc_results,
    "note": "Δ = dt_proj output (pre-softplus, incl. bias); B, C = x_proj output slices; all layers frozen at once"})

# free GPU memory before Step 29
del pc_model, pc_refs, _pc_acc
torch.cuda.empty_cache()
print("\nSTEP 28B COMPLETE")

## Step 29 — Replication on MambaVision-S ##
Repeats E2 (single-block mean ablation), E3 (grouped ablation), E5 (shuffled-channel null) and E9
(joint freeze, mean and position mode) on nvidia/MambaVision-S-1K, with the same images (Step 5b order),
the same 256 reference images and the same statistics. All variables use the prefix s_ so the
MambaVision-T model and results are untouched. Saves `step29_mambavision_s` after every run.

### Step 29A — Load MambaVision-S, preprocessing, baseline ###

In [ ]:
# ============================================================
# STEP 29A — MAMBAVISION-S: load, preprocessing, baseline
# ============================================================
import re, sys, time
from transformers import AutoModelForImageClassification
from timm.data.transforms_factory import create_transform
from torchvision import transforms as T
from torch.utils.data import Dataset as _TorchDataset, DataLoader

S_MODEL_ID = "nvidia/MambaVision-S-1K"
CKPT_29 = "step29_mambavision_s"
s_model = AutoModelForImageClassification.from_pretrained(S_MODEL_ID, trust_remote_code=True,
                                                          low_cpu_mem_usage=False).to(device).eval()
print(f"MambaVision-S parameters: {sum(p.numel() for p in s_model.parameters()):,}")

s_transform = create_transform(input_size=(3, 224, 224), is_training=False, mean=s_model.config.mean,
                               std=s_model.config.std, crop_mode=s_model.config.crop_mode,
                               crop_pct=s_model.config.crop_pct)
SAME_TF = repr(s_transform) == repr(TIMM_TRANSFORM)
print("S preprocessing:", s_transform, "\nSame as T (cache reusable):", SAME_TF)

if SAME_TF:
    s_get_batch = get_cached_batch
else:
    _tf = list(s_transform.transforms)
    _k = next(i for i, t in enumerate(_tf) if "ToTensor" in type(t).__name__)
    S_PRE = T.Compose(_tf[:_k])
    S_MEAN = torch.as_tensor(_tf[_k + 1].mean, dtype=torch.float32, device=device).view(1, 3, 1, 1)
    S_STD  = torch.as_tensor(_tf[_k + 1].std,  dtype=torch.float32, device=device).view(1, 3, 1, 1)
    S_CACHE = os.path.join(CACHE_DIR, f"mv_cache_S_images_u8_{N_FULL}.npy")
    class _SCacheSource(_TorchDataset):
        def __len__(self): return N_FULL
        def __getitem__(self, pos):
            ex = dataset[eval_indices[pos]]
            return np.asarray(S_PRE(ex["image"].convert("RGB")), dtype=np.uint8).copy(), ex["label"]
    if not os.path.exists(S_CACHE + ".done"):
        mm = np.lib.format.open_memmap(S_CACHE, mode="w+", dtype=np.uint8, shape=(N_FULL, 224, 224, 3))
        pos = 0
        for x, _ in DataLoader(_SCacheSource(), batch_size=256, num_workers=os.cpu_count() or 2):
            mm[pos:pos + len(x)] = x.numpy(); pos += len(x)
        mm.flush(); del mm; open(S_CACHE + ".done", "w").write("ok")
    S_IMG_CACHE = np.load(S_CACHE, mmap_mode="r")
    def s_get_batch(start, end):
        x = torch.from_numpy(np.array(S_IMG_CACHE[start:end])).to(device, non_blocking=True)
        return x.permute(0, 3, 1, 2).float().div(255).sub(S_MEAN).div(S_STD)
    _ref, _ = prepare_batch(dataset, eval_indices[:64], s_transform, device)
    assert (s_get_batch(0, 64) - _ref).abs().max().item() < 1e-5, "S cache differs from prepare_batch."
    print("S cache built and checked.")

# ---- 12 mixer blocks of S ----
s_mixer_blocks = []
for name, module in s_model.named_modules():
    if hasattr(module, "mixer"):
        m = re.search(r"levels\.(\d+)\.blocks\.(\d+)$", name)
        if m:
            s_mixer_blocks.append((f"L{m.group(1)}B{m.group(2)}", type(module.mixer).__name__, module))
print("S mixer blocks:", [(n, "Mamba" if t == "MambaVisionMixer" else "Attn") for n, t, _ in s_mixer_blocks])

def s_clear_hooks():
    for _, _, blk in s_mixer_blocks:
        for m in (blk, blk.mixer):
            m._forward_hooks.clear(); m._forward_pre_hooks.clear()

def s_eval(n=None, mode=None):
    """mode=None: preds | 'store': preds + log-probs (fp16, CPU) | 'lpkl': preds + log-prob drop + KL vs S baseline."""
    n = N_FULL if n is None else n
    preds, extra1, extra2 = [], [], []
    with torch.no_grad():
        for s in range(0, n, BATCH_SIZE):
            e = min(s + BATCH_SIZE, n)
            out = s_model(s_get_batch(s, e))
            logits = (out["logits"] if isinstance(out, dict) else out.logits).float()
            preds.append(logits.argmax(-1).cpu())
            if mode == "store":
                extra1.append(F.log_softmax(logits, -1).half().cpu())
            elif mode == "lpkl":
                lp = F.log_softmax(logits, -1); y = _LAB_GPU[s:e].view(-1, 1)
                base = S_BASE_LP[s:e].to(device).float()
                extra1.append((base.gather(1, y) - lp.gather(1, y)).squeeze(1).cpu())
                extra2.append((base.exp() * (base - lp)).sum(-1).cpu())
    preds = torch.cat(preds).numpy()
    if mode == "store":
        return preds, torch.cat(extra1)
    if mode == "lpkl":
        return preds, torch.cat(extra1).numpy(), torch.cat(extra2).numpy()
    return preds

def s_paired(new_preds):
    new_c = (np.asarray(new_preds) == eval_labels_np); ref = S_BASE_CORRECT
    b = int((ref & ~new_c).sum()); c = int((~ref & new_c).sum())
    p = mcnemar([[0, b], [c, 0]], exact=(b + c) < 25).pvalue if (b + c) > 0 else 1.0
    m, ci = mean_ci((new_c.astype(np.int8) - ref.astype(np.int8)) * 100.0)
    return {"accuracy": float(new_c.mean()), "delta_pp": m, "ci_low_pp": ci[0], "ci_high_pp": ci[1],
            "b_correct_to_wrong": b, "c_wrong_to_correct": c, "flip_rate_pct": (b + c) / N_FULL * 100,
            "mcnemar_p": float(p)}, new_c

# ---- baseline (always recomputed: the log-probs are needed in memory for KL) ----
s_clear_hooks()
t0 = time.time()
s_base_preds, S_BASE_LP = s_eval(mode="store")
S_BASE_CORRECT = (s_base_preds == eval_labels_np)
S_PASS_MIN = (time.time() - t0) / 60
print(f"MambaVision-S baseline: {S_BASE_CORRECT.mean()*100:.3f}% ({S_BASE_CORRECT.sum():,}/50,000) "
      f"| one pass = {S_PASS_MIN:.1f} min | published top-1 83.3%")

try:
    res29 = load_checkpoint(CKPT_29)
    assert abs(res29["baseline"]["accuracy"] - S_BASE_CORRECT.mean()) < 1e-9, "Baseline differs from the saved one!"
    print("Resuming; saved parts:", [k for k in res29 if k != "baseline"])
except Exception as e:
    if "differs" in str(e): raise
    res29 = {"baseline": {"accuracy": float(S_BASE_CORRECT.mean()), "correct_packed": pack(S_BASE_CORRECT),
                          "model": S_MODEL_ID}}
    save_checkpoint(CKPT_29, res29)
print("\nSTEP 29A COMPLETE")

### Step 29B — Reference means and Δ, B, C references (256 Step 8 images) ###

In [ ]:
# ============================================================
# STEP 29B — S references: mixer-output means and Δ/B/C (mean + position mode)
# ============================================================
REF_N = 256
s_clear_hooks()

# ---- per-channel mixer-output means (same definition as Step 8) ----
_cap = {name: [] for name, _, _ in s_mixer_blocks}
_hs = [blk.mixer.register_forward_hook(
           lambda mod, i, o, name=name: _cap[name].append((o[0] if isinstance(o, tuple) else o).detach()))
       for name, _, blk in s_mixer_blocks]
with torch.no_grad():
    for s in range(0, REF_N, 64):
        s_model(s_get_batch(s, s + 64))
for h in _hs: h.remove()
s_block_means = {name: torch.cat(v).mean(dim=(0, 1), keepdim=True) for name, v in _cap.items()}
del _cap

# ---- Δ (pre-softplus), B, C references for the S Mamba blocks ----
s_mamba = {name: blk for name, t, blk in s_mixer_blocks if t == "MambaVisionMixer"}
s_mamba_names = list(s_mamba)
s_target_module = sys.modules[type(s_mamba[s_mamba_names[0]].mixer).__module__]
if "_s_orig_scan" not in globals():
    _s_orig_scan = s_target_module.selective_scan_fn
if "_patched_fn" in globals():
    assert _s_orig_scan is not _patched_fn, "T freeze patch is active - run remove_freeze_patch first."
s_target_module.selective_scan_fn = _s_orig_scan

_s_active = [None]
_s_capture = {"on": False, "data": {n: {"dt": [], "B": [], "C": []} for n in s_mamba_names}}
_s_fz = {"active": False, "blocks": set(), "tensor": "all", "mode": "mean"}
s_freeze_refs = {}

def _s_scan(x, dt, A, B, C, D, z=None, delta_bias=None, delta_softplus=None, return_last_state=None):
    name = _s_active[0]
    if name is not None and _s_capture["on"]:
        d = _s_capture["data"][name]; d["dt"].append(dt.detach()); d["B"].append(B.detach()); d["C"].append(C.detach())
    if name is not None and _s_fz["active"] and name in _s_fz["blocks"]:
        ref, mode, tt = s_freeze_refs[name], _s_fz["mode"], _s_fz["tensor"]
        if tt in ("delta", "all"): dt = ref["dt"][mode].expand_as(dt).to(device=dt.device, dtype=dt.dtype)
        if tt in ("B", "all"):     B  = ref["B"][mode].expand_as(B).to(device=B.device, dtype=B.dtype)
        if tt in ("C", "all"):     C  = ref["C"][mode].expand_as(C).to(device=C.device, dtype=C.dtype)
    return _s_orig_scan(x, dt, A, B, C, D, z=z, delta_bias=delta_bias,
                        delta_softplus=delta_softplus, return_last_state=return_last_state)

def s_install_scan_patch():
    s_clear_hooks()
    hs = []
    for name, blk in s_mamba.items():
        hs.append(blk.mixer.register_forward_pre_hook(lambda mod, inp, name=name: _s_active.__setitem__(0, name)))
        hs.append(blk.mixer.register_forward_hook(lambda mod, inp, out: _s_active.__setitem__(0, None)))
    s_target_module.selective_scan_fn = _s_scan
    return hs

def s_remove_scan_patch(hs):
    _s_fz["active"] = False; _s_capture["on"] = False
    s_target_module.selective_scan_fn = _s_orig_scan
    for h in hs: h.remove()

_hs = s_install_scan_patch()
try:
    _s_capture["on"] = True
    with torch.no_grad():
        for s in range(0, REF_N, 64):
            s_model(s_get_batch(s, s + 64))
finally:
    s_remove_scan_patch(_hs)
for name in s_mamba_names:
    refs = {}
    for tname in ("dt", "B", "C"):
        t = torch.cat(_s_capture["data"][name][tname])          # (N_ref, C, L)
        refs[tname] = {"mean": t.mean(dim=(0, 2), keepdim=True), "position": t.mean(dim=0, keepdim=True)}
    s_freeze_refs[name] = refs
_s_capture["data"] = None
print("Means for", len(s_block_means), "blocks; Δ/B/C references for", s_mamba_names)
print("\nSTEP 29B COMPLETE")

### Step 29C — E3 grouped ablation and E9 joint freeze (mean and position mode) ###

In [ ]:
# ============================================================
# STEP 29C — S: grouped ablation (E3) and joint Δ/B/C freeze (E9)
# ============================================================
s_groups = {"mamba": [n for n, t, _ in s_mixer_blocks if t == "MambaVisionMixer"],
            "attn":  [n for n, t, _ in s_mixer_blocks if t == "Attention"]}

# ---- E3: grouped mean ablation ----
res29.setdefault("E3", {})
for key, names in s_groups.items():
    if key in res29["E3"]:
        continue
    s_clear_hooks()
    hs = [blk.mixer.register_forward_hook(mean_ablation_hook(s_block_means[name]))
          for name, _, blk in s_mixer_blocks if name in names]
    try:
        preds = s_eval()
    finally:
        for h in hs: h.remove()
    st, corr = s_paired(preds)
    res29["E3"][key] = {**st, "blocks": names, "correct_packed": pack(corr)}
    save_checkpoint(CKPT_29, res29)
    print(f"All {key} off: Δ = {st['delta_pp']:+.3f} pp [{st['ci_low_pp']:+.3f}, {st['ci_high_pp']:+.3f}]")

# ---- E9: joint freeze of Δ, B, C in all S Mamba blocks ----
def s_ratio_ci(c_a, c_b, n_boot=2000, seed=42):
    bc = S_BASE_CORRECT.astype(np.int8)
    da, db = bc - c_a.astype(np.int8), bc - c_b.astype(np.int8)
    rng = np.random.RandomState(seed); n = len(bc)
    boot = [da[idx].mean() / db[idx].mean() for idx in (rng.randint(0, n, n) for _ in range(n_boot))]
    return float(da.mean() / db.mean()), [float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))]

mamba_off_s = unpack(res29["E3"]["mamba"]["correct_packed"])
res29.setdefault("E9", {})
for mode in ["mean", "position"]:
    if mode in res29["E9"]:
        continue
    hs = s_install_scan_patch()
    try:
        _s_fz.update({"active": True, "blocks": set(s_mamba_names), "tensor": "all", "mode": mode})
        preds, lpd, kl = s_eval(mode="lpkl")
    finally:
        s_remove_scan_patch(hs)
    st, corr = s_paired(preds)
    sf, sf_ci = s_ratio_ci(corr, mamba_off_s)
    lp_m, lp_ci = mean_ci(lpd); kl_m, kl_ci = mean_ci(kl)
    res29["E9"][mode] = {**st, "logprob_drop": lp_m, "logprob_drop_ci": lp_ci, "kl": kl_m, "kl_ci": kl_ci,
                         "ratio_to_all_mamba_off": sf, "ratio_ci": sf_ci, "correct_packed": pack(corr)}
    save_checkpoint(CKPT_29, res29)
    print(f"Joint freeze ({mode:8s}): Δ = {st['delta_pp']:+.3f} pp [{st['ci_low_pp']:+.3f}, {st['ci_high_pp']:+.3f}] "
          f"p = {st['mcnemar_p']:.2e} | ÷ all-Mamba-off = {sf*100:.1f}% [{sf_ci[0]*100:.1f}, {sf_ci[1]*100:.1f}]")
print("\nSTEP 29C COMPLETE")

### Step 29D — E2 single-block mean ablation and E5 shuffled null, per block ###
For each of the 12 S blocks: real mean ablation (with log-prob drop and KL) and a shuffled-channel null
with the same Δx norm (one permutation per block, seed 42 + block index), compared per image.
Saved after every block (resumable).

In [ ]:
# ============================================================
# STEP 29D — S: per-block real ablation (E2) and shuffled null (E5)
# ============================================================
def _s_null_hook(name, perm):
    mean_val = s_block_means[name]
    def f(module, inp, out):
        is_tuple = isinstance(out, tuple)
        t = out[0] if is_tuple else out
        delta = mean_val.to(dtype=t.dtype).expand_as(t) - t
        rep = t + delta[..., perm.to(t.device)]
        return (rep,) + out[1:] if is_tuple else rep
    return f

res29.setdefault("E2E5", {})
for bi, (name, mtype, blk) in enumerate(s_mixer_blocks):
    if name in res29["E2E5"]:
        continue
    t0 = time.time()
    # real mean ablation
    s_clear_hooks()
    h = blk.mixer.register_forward_hook(mean_ablation_hook(s_block_means[name]))
    try:
        preds_r, lpd, kl = s_eval(mode="lpkl")
    finally:
        h.remove()
    st_r, c_r = s_paired(preds_r)
    # shuffled null
    g = torch.Generator().manual_seed(42 + bi)
    perm = torch.randperm(s_block_means[name].shape[-1], generator=g)
    s_clear_hooks()
    h = blk.mixer.register_forward_hook(_s_null_hook(name, perm))
    try:
        preds_n = s_eval()
    finally:
        h.remove()
    st_n, c_n = s_paired(preds_n)
    d_m, d_ci = mean_ci((c_r.astype(np.int8) - c_n.astype(np.int8)) * 100.0)     # real − null
    lp_m, lp_ci = mean_ci(lpd); kl_m, kl_ci = mean_ci(kl)
    res29["E2E5"][name] = {"type": mtype, "real": {**st_r, "logprob_drop": lp_m, "logprob_drop_ci": lp_ci,
                                                   "kl": kl_m, "kl_ci": kl_ci, "correct_packed": pack(c_r)},
                           "null": {**st_n, "correct_packed": pack(c_n)},
                           "real_minus_null_pp": d_m, "real_minus_null_ci": d_ci}
    save_checkpoint(CKPT_29, res29)
    print(f"{name:5s} {'Mamba' if mtype == 'MambaVisionMixer' else 'Attn ':5s} ({(time.time()-t0)/60:.1f} min) "
          f"real {st_r['delta_pp']:+.3f} | null {st_n['delta_pp']:+.3f} | real − null {d_m:+.3f} [{d_ci[0]:+.3f}, {d_ci[1]:+.3f}]")
print("\nSTEP 29D COMPLETE")

### Step 29E — S summary and comparison with T (CPU only) ###

In [ ]:
# ============================================================
# STEP 29E — S SUMMARY vs T
# ============================================================
from scipy.stats import spearmanr, norm
res29 = load_checkpoint(CKPT_29)
rows = res29["E2E5"]
real = np.array([r["real"]["delta_pp"] for r in rows.values()])
null = np.array([r["null"]["delta_pp"] for r in rows.values()])
types = [r["type"] for r in rows.values()]
pvals = []
for r in rows.values():
    se = (r["real_minus_null_ci"][1] - r["real_minus_null_ci"][0]) / 3.92
    pvals.append(2 * norm.sf(abs(r["real_minus_null_pp"]) / se) if se > 0 else 1.0)
n_bonf = sum(p < 0.05 / len(rows) for p in pvals)
real_worse = sum(r["real_minus_null_ci"][1] < 0 for r in rows.values())
null_worse = sum(r["real_minus_null_ci"][0] > 0 for r in rows.values())
rho = spearmanr(np.abs(real), np.abs(null))
mam = [i for i, t in enumerate(types) if t == "MambaVisionMixer"]; att = [i for i, t in enumerate(types) if t == "Attention"]

print(f"MambaVision-S baseline: {res29['baseline']['accuracy']*100:.3f}%")
print(f"E2: mean |Δ| Mamba {np.abs(real[mam]).mean():.3f} pp vs Attention {np.abs(real[att]).mean():.3f} pp")
print(f"E5: real worse than null in {real_worse}/{len(rows)}, null worse in {null_worse}/{len(rows)} (nominal); "
      f"{n_bonf}/{len(rows)} contrasts survive Bonferroni")
print(f"E5: Spearman |real| vs |null| across blocks: ρ = {rho.correlation:.2f} (p = {rho.pvalue:.3f})   [T: ρ = 0.83]")
for key in ["mamba", "attn"]:
    g = res29["E3"][key]; s_sum = real[mam if key == "mamba" else att].sum()
    print(f"E3: all {key} off {g['delta_pp']:+.3f} pp = {g['delta_pp']/s_sum:.2f}× sum of singles ({s_sum:+.3f})   "
          f"[T: {'3.67' if key == 'mamba' else '6.96'}×]")
for mode in ["mean", "position"]:
    e = res29["E9"][mode]
    print(f"E9: joint freeze {mode:8s} {e['delta_pp']:+.3f} pp [{e['ci_low_pp']:+.3f}, {e['ci_high_pp']:+.3f}] | "
          f"÷ all-Mamba-off {e['ratio_to_all_mamba_off']*100:.1f}% [{e['ratio_ci'][0]*100:.1f}, {e['ratio_ci'][1]*100:.1f}]   "
          f"[T: {'29.1' if mode == 'mean' else '20.3'}%]")
print("\nSTEP 29E COMPLETE")

## Step 30 — Remaining items from Review 4 (log-prob metric, token-0 test, Figure 8 data) ##
30A: positions of the highest-update token for all 12 blocks (2,000 images), for a Figure 8 with a
shared colour scale. 30B: causal test of the scan-start token: only the token-0 update of L2B3 and
L3B1 is replaced by its position-specific mean. 30C: true-class log-probability drop for the 12
single-block mean ablations (replaces the logit drop in the E2 table). 

In [ ]:
# ============================================================
# STEP 30 — setup: fast eval pass without re-running the 24C baseline (already verified: step24_speed)
# ============================================================
assert "get_cached_batch" in globals(), "Run Step 24B first."
assert load_checkpoint("step24_speed")["fast_ok"], "Fast path was not verified."
import time
_LAB_GPU = torch.as_tensor(LAB_CACHE, device=device)

def eval_pass_fast(n=None, logprob_kl=False, bs=None):
    n = N_FULL if n is None else n
    bs = BATCH_SIZE if bs is None else bs
    preds, lp_drops, kls = [], [], []
    with torch.no_grad():
        for s in range(0, n, bs):
            e = min(s + bs, n)
            out = model(get_cached_batch(s, e))
            logits = (out["logits"] if isinstance(out, dict) else out.logits).float()
            preds.append(logits.argmax(-1).cpu())
            if logprob_kl:
                lp = F.log_softmax(logits, -1); y = _LAB_GPU[s:e].view(-1, 1)
                base_lp = baseline_log_probs[s:e].to(device).float()
                lp_drops.append((base_lp.gather(1, y) - lp.gather(1, y)).squeeze(1).cpu())
                kls.append((base_lp.exp() * (base_lp - lp)).sum(-1).cpu())
    preds = torch.cat(preds).numpy()
    if logprob_kl:
        return preds, torch.cat(lp_drops).numpy(), torch.cat(kls).numpy()
    return preds

def mean_ci(x, n_boot=2000, seed=42):
    x = np.asarray(x, dtype=np.float64); n = len(x); rng = np.random.RandomState(seed)
    boot = np.array([x[rng.randint(0, n, n)].mean() for _ in range(n_boot)])
    return float(x.mean()), [float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))]

assert "ref_acts" in globals(), "Run Step 8 first (ref_acts holds the 256 reference activations)."
model.eval(); clear_all_hooks()
print("Step 30 setup ready.")

### Step 30A — Figure 8 data: position of the highest-update token, all blocks (2,000 images) ###

In [ ]:
# ============================================================
# STEP 30A — HIGH-NORM TOKEN POSITION COUNTS (for Figure 8 with a shared colour scale)
# ============================================================
N_HN = 2000
counts = {name: None for name, _, _ in mixer_blocks}
def _hn_hook(name):
    def f(m, i, o):
        t = o[0] if isinstance(o, tuple) else o                     # (B, L, C); update u = mixer output (γ1 = 1)
        top = t.float().norm(dim=-1).argmax(dim=1)                   # (B,)
        c = torch.bincount(top, minlength=t.shape[1]).cpu()
        counts[name] = c if counts[name] is None else counts[name] + c
    return f
clear_all_hooks()
hs = [blk.mixer.register_forward_hook(_hn_hook(name)) for name, _, blk in mixer_blocks]
try:
    eval_pass_fast(n=N_HN)
finally:
    for h in hs: h.remove()
hn_old = {r["block"]: r for r in load_checkpoint("high_norm_tokens")["rows"]}
for name, c in counts.items():
    print(f"{name}: most frequent position {int(c.argmax())} ({c.max().item()/N_HN*100:.1f}%)   "
          f"[saved table: {hn_old[name]['mode_position']} ({hn_old[name]['mode_fraction_top1']*100:.1f}%)]")
save_checkpoint("step30_highnorm_heatmaps", {"n_images": N_HN, "counts": {k: v.numpy() for k, v in counts.items()}})
print("\nSTEP 30A COMPLETE")

### Step 30B — Causal test of the scan-start token (L2B3, L3B1) ###
Only the token-0 row of the mixer output is replaced by its position-specific mean over the 256
reference images; all other tokens are unchanged.

In [ ]:
# ============================================================
# STEP 30B (and 30D) — PATCH ONE TOKEN OF THE MIXER UPDATE
# ============================================================
CKPT_30T = "step30_token_patch"
def _tok_hook(row, tok):
    def f(m, i, o):
        is_t = isinstance(o, tuple); t = (o[0] if is_t else o).clone()
        t[:, tok, :] = row.to(device=t.device, dtype=t.dtype)
        return (t,) + o[1:] if is_t else t
    return f

def run_token_patch(name, tok):
    blk = dict((n, b) for n, _, b in mixer_blocks)[name]
    row = ref_acts[name][:, tok, :].mean(0)                          # position-specific mean of this token
    clear_all_hooks()
    h = blk.mixer.register_forward_hook(_tok_hook(row, tok))
    try:
        preds, lpd, kl = eval_pass_fast(logprob_kl=True)
    finally:
        h.remove()
    st, corr = paired_stats(preds)
    lp_m, lp_ci = mean_ci(lpd); kl_m, kl_ci = mean_ci(kl)
    return {**st, "token": tok, "logprob_drop": lp_m, "logprob_drop_ci": lp_ci, "kl": kl_m, "kl_ci": kl_ci,
            "correct_packed": pack(corr)}

try:
    res30t = load_checkpoint(CKPT_30T)
except Exception:
    res30t = {}
for name in ["L2B3", "L3B1"]:
    key = f"{name}|token0"
    if key in res30t: continue
    t0 = time.time(); res30t[key] = run_token_patch(name, 0); save_checkpoint(CKPT_30T, res30t)
    r = res30t[key]
    print(f"{key}: Δ = {r['delta_pp']:+.3f} pp [{r['ci_low_pp']:+.3f}, {r['ci_high_pp']:+.3f}] p = {r['mcnemar_p']:.2e} | "
          f"log-p drop {r['logprob_drop']:+.4f} | KL {r['kl']:.5f} ({(time.time()-t0)/60:.1f} min)")
print("\nSTEP 30B COMPLETE")

### Step 30C — E2 with the true-class log-probability drop (12 blocks) ###
Re-runs the single-block mean ablations on the fast path and records the log-probability drop and
KL. Δ accuracy and KL must reproduce Step 10 / Step 9. Resumable.

In [ ]:
# ============================================================
# STEP 30C — SINGLE-BLOCK MEAN ABLATION WITH LOG-PROB DROP
# ============================================================
CKPT_30E2 = "step30_e2_logprob"
s10 = {r["block"]: r for r in load_checkpoint("step10")["ablation_results"]}
try:
    res30e2 = load_checkpoint(CKPT_30E2)
    print("Resuming; done:", list(res30e2))
except Exception:
    res30e2 = {}
for name, mtype, blk in mixer_blocks:
    if name in res30e2: continue
    t0 = time.time()
    clear_all_hooks()
    h = blk.mixer.register_forward_hook(mean_ablation_hook(block_means[name]))
    try:
        preds, lpd, kl = eval_pass_fast(logprob_kl=True)
    finally:
        h.remove()
    st, _ = paired_stats(preds)
    lp_m, lp_ci = mean_ci(lpd); kl_m, kl_ci = mean_ci(kl)
    res30e2[name] = {"type": mtype, "delta_pp": st["delta_pp"], "logprob_drop": lp_m, "logprob_drop_ci": lp_ci,
                     "kl": kl_m, "kl_ci": kl_ci}
    save_checkpoint(CKPT_30E2, res30e2)
    print(f"{name}: Δ {st['delta_pp']:+.3f} (Step 10: {s10[name]['delta_pp']:+.3f}) | log-p drop {lp_m:+.4f} "
          f"[{lp_ci[0]:+.4f}, {lp_ci[1]:+.4f}] | KL {kl_m:.4f} ({(time.time()-t0)/60:.1f} min)")
print("\nSTEP 30C COMPLETE")

## Step 31 — Pathway ablation with the true-class log-probability drop ##
Re-runs the 12 gate-off / SSM-off conditions of Step 17B on the fast path and records the
true-class log-probability drop and KL. Uses the saved Step 17A reference means
(`pathway_means_mamba_v2`), so Step 17A is not re-run. Δ accuracy must reproduce Step 17B.
Resumable; saves `step31_pathway_logprob` after every run.

In [ ]:
# ============================================================
# STEP 31 — PATHWAY ABLATION WITH LOG-PROB DROP (gate_off / ssm_off, 6 Mamba blocks)
# ============================================================
import sys, types, time
assert "eval_pass_fast" in globals() and "mean_ci" in globals(), "Run Step 24B and the Step 30 setup cell first."
CKPT_31 = "step31_pathway_logprob"

clear_all_hooks()
mamba_blocks = [(name, mod) for name, mtype, mod in mixer_blocks if mtype == "MambaVisionMixer"]
orig_mod = sys.modules[type(mamba_blocks[0][1].mixer).__module__]
F_, rearrange_ = orig_mod.F, orig_mod.rearrange
selective_scan_fn_ = orig_mod.selective_scan_fn          # fresh session: no freezing patch is active

_pw_saved = load_checkpoint("pathway_means_mamba_v2")
gate_means, ssm_means = _pw_saved["gate_means"], _pw_saved["ssm_means"]
_pathway_config = {"active_block": None, "mode": None}

def make_pathway_forward(block_name):            # same forward as Step 17A
    def patched_forward(self, hidden_states):
        _, seqlen, _ = hidden_states.shape
        xz = rearrange_(self.in_proj(hidden_states), "b l d -> b d l")
        x, z = xz.chunk(2, dim=1)
        A = -torch.exp(self.A_log.float())
        x = F_.silu(F_.conv1d(input=x, weight=self.conv1d_x.weight, bias=self.conv1d_x.bias, padding="same", groups=self.d_inner // 2))
        z = F_.silu(F_.conv1d(input=z, weight=self.conv1d_z.weight, bias=self.conv1d_z.bias, padding="same", groups=self.d_inner // 2))
        x_dbl = self.x_proj(rearrange_(x, "b d l -> (b l) d"))
        dt, B, C = torch.split(x_dbl, [self.dt_rank, self.d_state, self.d_state], dim=-1)
        dt = rearrange_(self.dt_proj(dt), "(b l) d -> b d l", l=seqlen)
        B = rearrange_(B, "(b l) dstate -> b dstate l", l=seqlen).contiguous()
        C = rearrange_(C, "(b l) dstate -> b dstate l", l=seqlen).contiguous()
        y = selective_scan_fn_(x, dt, A, B, C, self.D.float(), z=None,
                               delta_bias=self.dt_proj.bias.float(), delta_softplus=True, return_last_state=None)
        if _pathway_config["active_block"] == block_name:
            if _pathway_config["mode"] == "gate_off":
                z = gate_means[block_name].to(device=z.device, dtype=z.dtype).expand_as(z)
            elif _pathway_config["mode"] == "ssm_off":
                y = ssm_means[block_name].to(device=y.device, dtype=y.dtype).expand_as(y)
        y = rearrange_(torch.cat([y, z], dim=1), "b d l -> b l d")
        return self.out_proj(y)
    return patched_forward

_orig_forwards = {name: mod.mixer.forward for name, mod in mamba_blocks}
def install_pathway_patch():
    for name, mod in mamba_blocks:
        mod.mixer.forward = types.MethodType(make_pathway_forward(name), mod.mixer)
def remove_pathway_patch():
    _pathway_config.update({"active_block": None, "mode": None})
    for name, mod in mamba_blocks:
        mod.mixer.forward = _orig_forwards[name]

# ---- control: patch installed but inactive must reproduce the original model (500 images) ----
plain = eval_pass_fast(n=500)
install_pathway_patch()
try:
    ctrl = eval_pass_fast(n=500)
finally:
    remove_pathway_patch()
assert (ctrl == plain).all(), "Patched forward differs from the original - send me this output."
print("Control OK: inactive patch reproduces the original model on 500 images.")

ref17 = {(r["block"], r["mode"]): r for r in load_checkpoint("pathway_ablation_50k_v3")["results"]}
try:
    res31 = load_checkpoint(CKPT_31)
    print("Resuming; done:", list(res31))
except Exception:
    res31 = {}

install_pathway_patch()
try:
    for name, _ in mamba_blocks:
        for mode in ["gate_off", "ssm_off"]:
            key = f"{name}|{mode}"
            if key in res31:
                continue
            t0 = time.time()
            _pathway_config.update({"active_block": name, "mode": mode})
            preds, lpd, kl = eval_pass_fast(logprob_kl=True)
            _pathway_config.update({"active_block": None, "mode": None})
            st, _ = paired_stats(preds)
            lp_m, lp_ci = mean_ci(lpd); kl_m, kl_ci = mean_ci(kl)
            res31[key] = {"block": name, "mode": mode, "delta_pp": st["delta_pp"],
                          "delta_pp_step17": float(ref17[(name, mode)]["delta_pp"]),
                          "logprob_drop": lp_m, "logprob_drop_ci": lp_ci, "kl": kl_m, "kl_ci": kl_ci}
            save_checkpoint(CKPT_31, res31)
            print(f"{key:15s} Δ {st['delta_pp']:+.3f} (Step 17: {ref17[(name, mode)]['delta_pp']:+.3f}) | "
                  f"log-p drop {lp_m:+.4f} [{lp_ci[0]:+.4f}, {lp_ci[1]:+.4f}] | KL {kl_m:.5f} ({(time.time()-t0)/60:.1f} min)")
finally:
    remove_pathway_patch()
print("\nSTEP 31 COMPLETE" if len(res31) == 12 else f"\nStopped at {len(res31)}/12 - rerun to resume.")

## Step 32 — Group shuffled nulls on MambaVision-S ##
The Step 26 group null repeated on MambaVision-S: the shuffled-channel null applied to all seven Mamba
blocks at once and to all five Attention blocks at once, with the same per-block permutations as the
Step 29D single-block nulls (seed 42 + block index). Compared per image with the real grouped ablation
(Step 29C). Needs Steps 29A and 29B in this session. Saves `step32_s_group_nulls`.

In [ ]:
# ============================================================
# STEP 32 — MAMBAVISION-S GROUP SHUFFLED NULLS
# ============================================================
assert "s_block_means" in globals() and "s_eval" in globals(), "Run Steps 29A and 29B first."
CKPT_32 = "step32_s_group_nulls"
res29 = load_checkpoint("step29_mambavision_s")

s_perms = {}
for bi, (name, _, _) in enumerate(s_mixer_blocks):          # same permutations as Step 29D
    g = torch.Generator().manual_seed(42 + bi)
    s_perms[name] = torch.randperm(s_block_means[name].shape[-1], generator=g)

_s_gnull = {"active": False, "blocks": set()}
def _s_gnull_hook(name):
    mean_val, perm = s_block_means[name], s_perms[name]
    def f(module, inp, out):
        if not _s_gnull["active"] or name not in _s_gnull["blocks"]:
            return out
        is_tuple = isinstance(out, tuple)
        t = out[0] if is_tuple else out
        delta = mean_val.to(dtype=t.dtype).expand_as(t) - t
        rep = t + delta[..., perm.to(t.device)]
        return (rep,) + out[1:] if is_tuple else rep
    return f

def s_run_null(blocks, n=None):
    s_clear_hooks()
    hs = [blk.mixer.register_forward_hook(_s_gnull_hook(name)) for name, _, blk in s_mixer_blocks]
    try:
        _s_gnull.update({"active": True, "blocks": set(blocks)})
        return s_eval(n=n)
    finally:
        _s_gnull["active"] = False
        for h in hs: h.remove()

# ---- check: single-block L2B0 null must reproduce Step 29D (first 5,000 images) ----
p_chk = s_run_null(["L2B0"], n=5000)
agree = ((p_chk == eval_labels_np[:5000]) == unpack(res29["E2E5"]["L2B0"]["null"]["correct_packed"])[:5000]).mean()
print(f"Reproduction of Step 29D L2B0 null on 5,000 images: {agree*100:.2f}% (must be ~100%)")
assert agree > 0.999, "Permutations differ from Step 29D - send me this output."

s_groups = {"mamba": [n for n, t, _ in s_mixer_blocks if t == "MambaVisionMixer"],
            "attn":  [n for n, t, _ in s_mixer_blocks if t == "Attention"]}
try:
    res32 = load_checkpoint(CKPT_32); print("Resuming; done:", list(res32))
except Exception:
    res32 = {}
for key, blocks in s_groups.items():
    if key in res32: continue
    t0 = time.time()
    st, null_c = s_paired(s_run_null(blocks))
    real_c = unpack(res29["E3"][key]["correct_packed"])
    d_m, d_ci = mean_ci((real_c.astype(np.int8) - null_c.astype(np.int8)) * 100.0)       # real − null
    singles = sum(res29["E2E5"][b]["null"]["delta_pp"] for b in blocks)
    res32[key] = {**st, "blocks": blocks, "real_delta_pp": float(res29["E3"][key]["delta_pp"]),
                  "real_minus_null_pp": d_m, "real_minus_null_ci": d_ci,
                  "sum_single_nulls_pp": float(singles), "correct_packed": pack(null_c)}
    save_checkpoint(CKPT_32, res32)
    print(f"{key} group null done in {(time.time()-t0)/60:.1f} min")

print(f"\n{'Group':10s} {'Real Δ':>8s} {'Null Δ [95% CI]':>26s} {'Real − null [CI]':>28s} {'Null ÷ sum of single nulls':>27s}")
for key, label in [("mamba", "All Mamba"), ("attn", "All Attn")]:
    r = res32[key]
    print(f"{label:10s} {r['real_delta_pp']:+8.3f} {r['delta_pp']:+9.3f} [{r['ci_low_pp']:+.3f},{r['ci_high_pp']:+.3f}]"
          f"   {r['real_minus_null_pp']:+9.3f} [{r['real_minus_null_ci'][0]:+.3f},{r['real_minus_null_ci'][1]:+.3f}]"
          f"   {r['delta_pp']/r['sum_single_nulls_pp']:8.2f}×")
print("[T: real − null  Mamba −2.70, Attention −16.37 pp; group null ÷ single nulls 1.02× / 1.28×]")
print("\nSTEP 32 COMPLETE")

## Step 33 — Frozen SSM as an explicit convolution kernel; truncation; throughput ##
With Δ, B and C frozen in mean mode, the selective scan becomes a linear time-invariant filter:
y_t = Σ_k K_d[k] x_{t−k} + D x_t, with K_d[k] = Σ_n C̄_n · exp(k Δ̄_d A_dn) · Δ̄_d B̄_n.
33A replaces the scan in all six Mamba blocks by this explicit causal convolution and checks that it
reproduces the Step 14 joint mean-mode freeze. 33B truncates the kernel to its first k taps
(k = 1 … 64) to measure how much memory the frozen filter needs. 33C measures throughput.
Needs Step 13 Part A and Part B in this session. Saves `step33_conv_kernel`.

### Step 33A — Explicit convolution kernel vs frozen scan (reproduction) ###

In [ ]:
# ============================================================
# STEP 33A — FROZEN SCAN → EXPLICIT CAUSAL CONVOLUTION (all six Mamba blocks)
# ============================================================
assert "_freeze_refs" in globals() and "install_freeze_patch" in globals(), "Run Step 13 Part A and Part B first."
assert "eval_pass_fast" in globals(), "Run Step 24B and the Step 30 setup cell first."
CKPT_33 = "step33_conv_kernel"
_conv_cfg = {"active": False, "kmax": None, "blocks": set()}

def frozen_kernel(name, A, delta_bias, delta_softplus, L, device):
    """K (D, L) of the LTI filter obtained by freezing Δ, B, C to their mean-mode references."""
    ref = _freeze_refs[name]
    dt = ref["dt"]["mean"].reshape(-1).float().to(device)                 # (D,)  same input the scan would get
    if delta_bias is not None:
        dt = dt + delta_bias.float().to(device)
    delta = F.softplus(dt) if delta_softplus else dt                     # (D,)
    Bm = ref["B"]["mean"].reshape(-1).float().to(device)                 # (N,)
    Cm = ref["C"]["mean"].reshape(-1).float().to(device)                 # (N,)
    dA = delta[:, None] * A.float()                                      # (D, N)
    k = torch.arange(L, device=device, dtype=torch.float32)
    decay = torch.exp(k[:, None, None] * dA[None])                       # (L, D, N)
    K = (decay * (Cm * Bm)[None, None, :]).sum(-1).T * delta[:, None]    # (D, L)
    return K

def _conv_scan(x, dt, A, B, C, D, z=None, delta_bias=None, delta_softplus=None, return_last_state=None):
    name = _active_block_name[0]
    if not (_conv_cfg["active"] and name in _conv_cfg["blocks"]):
        return _original_selective_scan_fn(x, dt, A, B, C, D, z=z, delta_bias=delta_bias,
                                           delta_softplus=delta_softplus, return_last_state=return_last_state)
    assert z is None and not return_last_state
    Bsz, Dch, L = x.shape
    K = frozen_kernel(name, A, delta_bias, delta_softplus, L, x.device)
    if _conv_cfg["kmax"] is not None:
        K[:, _conv_cfg["kmax"]:] = 0.0
    w = K.flip(-1).unsqueeze(1)                                          # (D, 1, L): causal, tap k = lag k
    y = F.conv1d(F.pad(x.float(), (L - 1, 0)), w, groups=Dch)
    y = y + D.float().view(1, -1, 1) * x.float()
    return y.to(x.dtype)

def run_conv(kmax=None, n=None, logprob_kl=True):
    clear_all_hooks()
    hs = install_block_name_hooks()
    target_module.selective_scan_fn = _conv_scan
    try:
        _conv_cfg.update({"active": True, "kmax": kmax, "blocks": set(mamba_block_names)})
        return eval_pass_fast(n=n, logprob_kl=logprob_kl)
    finally:
        _conv_cfg["active"] = False
        target_module.selective_scan_fn = _original_selective_scan_fn
        for h in hs: h.remove()

try:
    res33 = load_checkpoint(CKPT_33); print("Resuming; done:", list(res33))
except Exception:
    res33 = {}

if "full" not in res33:
    t0 = time.time()
    preds, lpd, kl = run_conv(kmax=None)
    st, corr = paired_stats(preds)
    frozen_c = unpack(load_checkpoint("step25_joint_freeze_modes")["all|mean"]["correct_packed"])
    agree = (corr == frozen_c).mean()
    lp_m, lp_ci = mean_ci(lpd); kl_m, kl_ci = mean_ci(kl)
    res33["full"] = {**st, "agreement_with_frozen_scan": float(agree), "logprob_drop": lp_m, "logprob_drop_ci": lp_ci,
                     "kl": kl_m, "kl_ci": kl_ci, "correct_packed": pack(corr)}
    save_checkpoint(CKPT_33, res33)
    print(f"({(time.time()-t0)/60:.1f} min)")
r = res33["full"]
print(f"Explicit conv kernel, all 6 blocks: Δ = {r['delta_pp']:+.3f} pp [{r['ci_low_pp']:+.3f}, {r['ci_high_pp']:+.3f}] "
      f"| frozen scan (Step 14): −1.136 pp | per-image agreement with frozen scan: {r['agreement_with_frozen_scan']*100:.3f}%")
print("\nSTEP 33A COMPLETE")

### Step 33B — Kernel truncation: how many taps does the frozen filter need? ###
Keeps only the first k taps of every frozen kernel (k = 1, 2, 4, 8, 16, 32, 64) in all six blocks.
Level 3 has only 49 tokens, so k = 64 equals the full kernel there.

In [ ]:
# ============================================================
# STEP 33B — TRUNCATED KERNELS (all six Mamba blocks at once)
# ============================================================
for kmax in [1, 2, 4, 8, 16, 32, 64]:
    key = f"k{kmax}"
    if key in res33: continue
    t0 = time.time()
    preds, lpd, kl = run_conv(kmax=kmax)
    st, corr = paired_stats(preds)
    full_c = unpack(res33["full"]["correct_packed"])
    d_m, d_ci = mean_ci((corr.astype(np.int8) - full_c.astype(np.int8)) * 100.0)        # truncated − full kernel
    lp_m, lp_ci = mean_ci(lpd); kl_m, kl_ci = mean_ci(kl)
    res33[key] = {**st, "kmax": kmax, "vs_full_kernel_pp": d_m, "vs_full_kernel_ci": d_ci,
                  "logprob_drop": lp_m, "kl": kl_m, "correct_packed": pack(corr)}
    save_checkpoint(CKPT_33, res33)
    print(f"k = {kmax:3d} ({(time.time()-t0)/60:.1f} min): Δ vs baseline {st['delta_pp']:+.3f} pp | "
          f"vs full kernel {d_m:+.3f} [{d_ci[0]:+.3f}, {d_ci[1]:+.3f}] | KL {kl_m:.4f}")
print("\nSTEP 33B COMPLETE")

### Step 33C — Throughput: original vs frozen scan vs explicit convolution ###
Forward time on 30 cached batches of 64 images (after warm-up), whole model.

In [ ]:
# ============================================================
# STEP 33C — THROUGHPUT
# ============================================================
def _time_batches(n_batches=30):
    with torch.no_grad():
        model(get_cached_batch(0, BATCH_SIZE)); torch.cuda.synchronize()
        imgs = [get_cached_batch(i * BATCH_SIZE, (i + 1) * BATCH_SIZE) for i in range(n_batches)]
        torch.cuda.synchronize(); t0 = time.perf_counter()
        for x in imgs: model(x)
        torch.cuda.synchronize()
    return (time.perf_counter() - t0) / n_batches

thr = {}
clear_all_hooks(); thr["original"] = _time_batches()
hs = install_freeze_patch()
try:
    _freeze_config.update({"active": True, "block_names": set(mamba_block_names), "tensor": "all", "mode": "mean"})
    thr["frozen_scan"] = _time_batches()
finally:
    remove_freeze_patch(hs)
for label, kmax in [("conv_full", None), ("conv_k16", 16)]:
    clear_all_hooks(); hs = install_block_name_hooks(); target_module.selective_scan_fn = _conv_scan
    try:
        _conv_cfg.update({"active": True, "kmax": kmax, "blocks": set(mamba_block_names)})
        thr[label] = _time_batches()
    finally:
        _conv_cfg["active"] = False; target_module.selective_scan_fn = _original_selective_scan_fn
        for h in hs: h.remove()

for k, v in thr.items():
    print(f"{k:12s}: {v*1000:7.1f} ms per batch of {BATCH_SIZE}  ({BATCH_SIZE/v:7.0f} images/s)")
res33["throughput_s_per_batch"] = {k: float(v) for k, v in thr.items()}
save_checkpoint(CKPT_33, res33)
print("\nSTEP 33C COMPLETE")

## Step 34 — DeiT-Small control: the same ablation and null protocol on a pure Transformer ##
Applies the MambaVision protocol to facebook/deit-small-patch16-224: per-layer mean ablation of the
attention output (per-channel mean over the 256 Step 8 images and all tokens), a shuffled-channel null
with the same Δx norm (seed 42 + layer index), and both for all 12 layers at once. Same 50K images
(Step 5b order), DeiT reference preprocessing (bicubic 248 → centre crop 224). Prefix d_.
Saves `step34_deit_protocol` after every run.

### Step 34A — DeiT-Small: cache, baseline and reference means ###

In [ ]:
# ============================================================
# STEP 34A — DEIT-SMALL: cache, baseline (with log-probs), reference means
# ============================================================
import os, time
from transformers import AutoModelForImageClassification
from torchvision import transforms as T
from torch.utils.data import Dataset as _TorchDataset, DataLoader
assert "CACHE_DIR" in globals() and "mean_ci" in globals(), "Run Step 24B and the Step 30 setup cell first."

DEIT_ID = "facebook/deit-small-patch16-224"
CKPT_34 = "step34_deit_protocol"
d_model = AutoModelForImageClassification.from_pretrained(DEIT_ID).to(device).eval()
_d_base = getattr(d_model, "vit", None) or getattr(d_model, "deit")      # this checkpoint loads as ViT
d_layers = list(_d_base.encoder.layer)
print(f"DeiT-Small ({type(d_model).__name__}): {len(d_layers)} layers, {sum(p.numel() for p in d_model.parameters()):,} parameters")

# ---- DeiT cache (bicubic 248 -> centre crop 224, uint8) ----
D_PRE = T.Compose([T.Resize(248, interpolation=T.InterpolationMode.BICUBIC), T.CenterCrop(224)])
D_MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
D_STD  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
D_CACHE = os.path.join(CACHE_DIR, f"mv_cache_DEIT_images_u8_{N_FULL}.npy")
class _DCacheSource(_TorchDataset):
    def __len__(self): return N_FULL
    def __getitem__(self, pos):
        ex = dataset[eval_indices[pos]]
        return np.asarray(D_PRE(ex["image"].convert("RGB")), dtype=np.uint8).copy(), ex["label"]
if not os.path.exists(D_CACHE + ".done"):
    t0 = time.time()
    mm = np.lib.format.open_memmap(D_CACHE, mode="w+", dtype=np.uint8, shape=(N_FULL, 224, 224, 3)); pos = 0
    for x, _ in DataLoader(_DCacheSource(), batch_size=256, num_workers=os.cpu_count() or 2):
        mm[pos:pos + len(x)] = x.numpy(); pos += len(x)
    mm.flush(); del mm; open(D_CACHE + ".done", "w").write("ok")
    print(f"DeiT cache built in {(time.time()-t0)/60:.1f} min")
D_IMG_CACHE = np.load(D_CACHE, mmap_mode="r")
def d_get_batch(start, end):
    x = torch.from_numpy(np.array(D_IMG_CACHE[start:end])).to(device, non_blocking=True)
    return x.permute(0, 3, 1, 2).float().div(255).sub(D_MEAN).div(D_STD)
_ref_tf = T.Compose([D_PRE, T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
_ref = torch.stack([_ref_tf(dataset[i]["image"].convert("RGB")) for i in eval_indices[:64]]).to(device)
assert (d_get_batch(0, 64) - _ref).abs().max().item() < 1e-5, "DeiT cache differs from the reference transform."

def d_clear_hooks():
    for layer in d_layers:
        for m in (layer, layer.attention):
            m._forward_hooks.clear(); m._forward_pre_hooks.clear()

def d_eval(n=None, mode=None):
    """mode=None: preds | 'store': preds + log-probs | 'lpkl': preds + log-prob drop + KL vs DeiT baseline."""
    n = N_FULL if n is None else n
    preds, e1, e2 = [], [], []
    with torch.no_grad():
        for s in range(0, n, BATCH_SIZE):
            e = min(s + BATCH_SIZE, n)
            logits = d_model(pixel_values=d_get_batch(s, e)).logits.float()
            preds.append(logits.argmax(-1).cpu())
            if mode == "store":
                e1.append(F.log_softmax(logits, -1).half().cpu())
            elif mode == "lpkl":
                lp = F.log_softmax(logits, -1); y = _LAB_GPU[s:e].view(-1, 1)
                base = D_BASE_LP[s:e].to(device).float()
                e1.append((base.gather(1, y) - lp.gather(1, y)).squeeze(1).cpu())
                e2.append((base.exp() * (base - lp)).sum(-1).cpu())
    preds = torch.cat(preds).numpy()
    if mode == "store": return preds, torch.cat(e1)
    if mode == "lpkl": return preds, torch.cat(e1).numpy(), torch.cat(e2).numpy()
    return preds

def d_paired(new_preds):
    new_c = (np.asarray(new_preds) == eval_labels_np); ref = D_BASE_CORRECT
    b = int((ref & ~new_c).sum()); c = int((~ref & new_c).sum())
    p = mcnemar([[0, b], [c, 0]], exact=(b + c) < 25).pvalue if (b + c) > 0 else 1.0
    m, ci = mean_ci((new_c.astype(np.int8) - ref.astype(np.int8)) * 100.0)
    return {"accuracy": float(new_c.mean()), "delta_pp": m, "ci_low_pp": ci[0], "ci_high_pp": ci[1],
            "b_correct_to_wrong": b, "c_wrong_to_correct": c, "mcnemar_p": float(p)}, new_c

# ---- baseline ----
d_clear_hooks(); t0 = time.time()
d_base_preds, D_BASE_LP = d_eval(mode="store")
D_BASE_CORRECT = (d_base_preds == eval_labels_np)
saved = np.asarray(load_checkpoint("deit_control_50k_v2")["deit_preds"]).astype(np.int64)
print(f"DeiT baseline: {D_BASE_CORRECT.mean()*100:.3f}% | one pass {(time.time()-t0)/60:.1f} min | "
      f"agreement with Step 22E predictions: {(d_base_preds == saved).mean()*100:.3f}%   (22E: 79.848%)")

# ---- reference means of the attention output (256 Step 8 images, all tokens) ----
_cap = {i: [] for i in range(len(d_layers))}
hs = [layer.attention.register_forward_hook(lambda m, inp, out, i=i: _cap[i].append((out[0] if isinstance(out, tuple) else out).detach()))
      for i, layer in enumerate(d_layers)]
with torch.no_grad():
    for s in range(0, 256, 64):
        d_model(pixel_values=d_get_batch(s, s + 64))
for h in hs: h.remove()
d_means = {i: torch.cat(v).mean(dim=(0, 1), keepdim=True) for i, v in _cap.items()}
del _cap
print("Reference means:", len(d_means), "layers, shape", tuple(d_means[0].shape))
try:
    res34 = load_checkpoint(CKPT_34); print("Resuming; done:", [k for k in res34 if k != "baseline"])
except Exception:
    res34 = {"baseline": {"accuracy": float(D_BASE_CORRECT.mean()), "correct_packed": pack(D_BASE_CORRECT)}}
    save_checkpoint(CKPT_34, res34)
print("\nSTEP 34A COMPLETE")

### Step 34B — Per-layer real ablation and shuffled null; all-layers group ablation and group null ###

In [ ]:
# ============================================================
# STEP 34B — DEIT: per-layer real vs null (24 runs) + group real vs null (2 runs)
# ============================================================
d_perms = {i: torch.randperm(d_means[i].shape[-1], generator=torch.Generator().manual_seed(42 + i))
           for i in range(len(d_layers))}
_d_cfg = {"layers": set(), "kind": None}          # kind: "real" or "null"
def _d_hook(i):
    mean_val, perm = d_means[i], d_perms[i]
    def f(m, inp, out):
        if i not in _d_cfg["layers"]:
            return out
        is_tuple = isinstance(out, tuple)
        t = out[0] if is_tuple else out
        mv = mean_val.to(dtype=t.dtype).expand_as(t)
        rep = mv.clone() if _d_cfg["kind"] == "real" else t + (mv - t)[..., perm.to(t.device)]
        return (rep,) + tuple(out[1:]) if is_tuple else rep
    return f
def d_run(layers, kind, mode=None):
    d_clear_hooks()
    hs = [layer.attention.register_forward_hook(_d_hook(i)) for i, layer in enumerate(d_layers)]
    try:
        _d_cfg.update({"layers": set(layers), "kind": kind})
        return d_eval(mode=mode)
    finally:
        _d_cfg["layers"] = set()
        for h in hs: h.remove()

res34.setdefault("layers", {})
for i in range(len(d_layers)):
    key = f"L{i}"
    if key in res34["layers"]: continue
    t0 = time.time()
    p_r, lpd, kl = d_run([i], "real", mode="lpkl"); st_r, c_r = d_paired(p_r)
    p_n = d_run([i], "null");                       st_n, c_n = d_paired(p_n)
    d_m, d_ci = mean_ci((c_r.astype(np.int8) - c_n.astype(np.int8)) * 100.0)      # real − null
    res34["layers"][key] = {"real": {**st_r, "logprob_drop": float(lpd.mean()), "kl": float(kl.mean()), "correct_packed": pack(c_r)},
                            "null": {**st_n, "correct_packed": pack(c_n)},
                            "real_minus_null_pp": d_m, "real_minus_null_ci": d_ci}
    save_checkpoint(CKPT_34, res34)
    print(f"Layer {i:2d} ({(time.time()-t0)/60:.1f} min): real {st_r['delta_pp']:+.3f} | null {st_n['delta_pp']:+.3f} | "
          f"real − null {d_m:+.3f} [{d_ci[0]:+.3f}, {d_ci[1]:+.3f}] | KL {kl.mean():.4f}")

res34.setdefault("group", {})
for kind in ["real", "null"]:
    if kind in res34["group"]: continue
    st, c = d_paired(d_run(range(len(d_layers)), kind))
    res34["group"][kind] = {**st, "correct_packed": pack(c)}
    save_checkpoint(CKPT_34, res34)
    print(f"All 12 layers, {kind}: Δ = {st['delta_pp']:+.3f} pp [{st['ci_low_pp']:+.3f}, {st['ci_high_pp']:+.3f}]")
print("\nSTEP 34B COMPLETE")

### Step 34C — DeiT summary and comparison with MambaVision (CPU only) ###

In [ ]:
# ============================================================
# STEP 34C — DEIT SUMMARY vs MAMBAVISION-T
# ============================================================
from scipy.stats import spearmanr, norm
res34 = load_checkpoint(CKPT_34)
L = res34["layers"]
real = np.array([L[k]["real"]["delta_pp"] for k in L]); null = np.array([L[k]["null"]["delta_pp"] for k in L])
pv = []
for k in L:
    lo, hi = L[k]["real_minus_null_ci"]; se = (hi - lo) / 3.92
    pv.append(2 * norm.sf(abs(L[k]["real_minus_null_pp"]) / se) if se > 0 else 1.0)
order = np.argsort(pv); holm = np.empty(len(pv)); run = 0
for r_, i in enumerate(order):
    run = max(run, min(1.0, (len(pv) - r_) * pv[i])); holm[i] = run
real_worse = [k for k, h in zip(L, holm) if h < 0.05 and L[k]["real_minus_null_pp"] < 0]
null_worse = [k for k, h in zip(L, holm) if h < 0.05 and L[k]["real_minus_null_pp"] > 0]
rho = spearmanr(np.abs(real), np.abs(null))

# pooled content effect (null − real) over the 12 layers, with a one-sided 95% upper bound
per_img = np.mean([(unpack(L[k]["null"]["correct_packed"]).astype(float) - unpack(L[k]["real"]["correct_packed"]).astype(float)) * 100
                   for k in L], axis=0)
rng = np.random.RandomState(42); n = len(per_img)
bm = np.array([per_img[rng.randint(0, n, n)].mean() for _ in range(2000)])

g = res34["group"]
print(f"DeiT-Small baseline: {res34['baseline']['accuracy']*100:.3f}%")
print(f"Per-layer real Δ: {real.min():+.3f} … {real.max():+.3f} pp | null Δ: {null.min():+.3f} … {null.max():+.3f} pp")
print(f"Spearman |real| vs |null| (12 layers): ρ = {rho.correlation:.2f} (p = {rho.pvalue:.3f})      [MambaVision-T: 0.83]")
print(f"After Holm: real worse than null in {real_worse or 'none'}; null worse in {null_worse or 'none'}")
print(f"Pooled content effect: {per_img.mean():+.3f} pp [{np.percentile(bm, 2.5):+.3f}, {np.percentile(bm, 97.5):+.3f}], "
      f"one-sided 95% upper bound {np.percentile(bm, 95):+.3f} pp      [MambaVision-T: +0.016]")
print(f"All 12 layers: real {g['real']['delta_pp']:+.2f} pp | null {g['null']['delta_pp']:+.2f} pp | "
      f"real ÷ sum of single reals {g['real']['delta_pp']/real.sum():.2f}×")
res34["summary"] = {"spearman": [float(rho.correlation), float(rho.pvalue)], "holm_p": holm.tolist(),
                    "pooled_content_pp": float(per_img.mean()), "pooled_ci": [float(np.percentile(bm, 2.5)), float(np.percentile(bm, 97.5))],
                    "pooled_upper95": float(np.percentile(bm, 95))}
save_checkpoint(CKPT_34, res34)
print("\nSTEP 34C COMPLETE")

## Step 35 — Higher-resolution evaluation (MambaVision-T at 384 × 384) ##
At 224 the Level-2/3 windows cover the whole feature map (the training regime). At 384 they are set to
the full map again ("global": 24 × 24 and 12 × 12 tokens); the default windowed setting is reported as a
sensitivity check. All references (mixer means, Δ/B/C references) are recomputed at 384 from the same
256 Step 8 images. Repeats the joint freeze (mean and position mode), all-Mamba-off, and the per-block
real ablation vs shuffled null. Prefix h_. Saves `step35_highres_384` after every run.

### Step 35A — 384 cache, baselines (default and global windows), references ###

In [ ]:
# ============================================================
# STEP 35A — MAMBAVISION-T AT 384: cache, window setting, baselines, references
# (resumable: if the global baseline and joint freezes are saved, the baseline is loaded, not re-run)
# ============================================================
import os, re, time
from timm.data.transforms_factory import create_transform
from torchvision import transforms as T
from torch.utils.data import Dataset as _TorchDataset, DataLoader
assert "install_freeze_patch" in globals() and "_freeze_refs" in globals(), "Run Step 13 Part A and Part B first."
assert "CACHE_DIR" in globals() and "mean_ci" in globals(), "Run Step 24B and the Step 30 setup cell first."
RES, CKPT_35 = 384, "step35_highres_384"
clear_all_hooks(); target_module.selective_scan_fn = _original_selective_scan_fn

# ---- preprocessing at 384 (model config, same as Step 4 but larger input) ----
h_tf = create_transform(input_size=(3, RES, RES), is_training=False, mean=model.config.mean, std=model.config.std,
                        crop_mode=model.config.crop_mode, crop_pct=model.config.crop_pct)
print(h_tf)
_tf = list(h_tf.transforms); _k = next(i for i, t in enumerate(_tf) if "ToTensor" in type(t).__name__)
H_PRE = T.Compose(_tf[:_k])
H_MEAN = torch.as_tensor(_tf[_k + 1].mean, dtype=torch.float32, device=device).view(1, 3, 1, 1)
H_STD  = torch.as_tensor(_tf[_k + 1].std,  dtype=torch.float32, device=device).view(1, 3, 1, 1)
H_CACHE = os.path.join(CACHE_DIR, f"mv_cache_T{RES}_images_u8_{N_FULL}.npy")
class _HCacheSource(_TorchDataset):
    def __len__(self): return N_FULL
    def __getitem__(self, pos):
        ex = dataset[eval_indices[pos]]
        return np.asarray(H_PRE(ex["image"].convert("RGB")), dtype=np.uint8).copy(), ex["label"]
if not os.path.exists(H_CACHE + ".done"):
    t0 = time.time()
    mm = np.lib.format.open_memmap(H_CACHE, mode="w+", dtype=np.uint8, shape=(N_FULL, RES, RES, 3)); pos = 0
    for x, _ in DataLoader(_HCacheSource(), batch_size=128, num_workers=os.cpu_count() or 2):
        mm[pos:pos + len(x)] = x.numpy(); pos += len(x)
    mm.flush(); del mm; open(H_CACHE + ".done", "w").write("ok")
    print(f"{RES} cache built in {(time.time()-t0)/60:.1f} min")
H_IMG_CACHE = np.load(H_CACHE, mmap_mode="r")
def h_get_batch(start, end):
    x = torch.from_numpy(np.array(H_IMG_CACHE[start:end])).to(device, non_blocking=True)
    return x.permute(0, 3, 1, 2).float().div(255).sub(H_MEAN).div(H_STD)
_ref, _ = prepare_batch(dataset, eval_indices[:16], h_tf, device)
assert (h_get_batch(0, 16) - _ref).abs().max().item() < 1e-5, "384 cache differs from prepare_batch."

# ---- window sizes of the hybrid levels ----
h_levels = {}
for name, mod in model.named_modules():
    m = re.search(r"levels\.(\d+)$", name)
    if m and hasattr(mod, "window_size"):
        h_levels[int(m.group(1))] = mod
if "ORIG_WS" not in globals():
    ORIG_WS = {i: lv.window_size for i, lv in h_levels.items()}
GRID = {2: RES // 16, 3: RES // 32}
print("Default window sizes:", ORIG_WS, "| token grid at", RES, ":", GRID)
def set_windows(mode):
    for i, lv in h_levels.items():
        lv.window_size = GRID[i] if (mode == "global" and i in GRID) else ORIG_WS[i]

def h_eval(n=None, mode=None):
    n = N_FULL if n is None else n
    preds, e1, e2 = [], [], []
    with torch.no_grad():
        for s in range(0, n, BATCH_SIZE):
            e = min(s + BATCH_SIZE, n)
            out = model(h_get_batch(s, e))
            logits = (out["logits"] if isinstance(out, dict) else out.logits).float()
            preds.append(logits.argmax(-1).cpu())
            if mode == "store":
                e1.append(F.log_softmax(logits, -1).half().cpu())
            elif mode == "lpkl":
                lp = F.log_softmax(logits, -1); y = _LAB_GPU[s:e].view(-1, 1)
                base = H_BASE_LP[s:e].to(device).float()
                e1.append((base.gather(1, y) - lp.gather(1, y)).squeeze(1).cpu())
                e2.append((base.exp() * (base - lp)).sum(-1).cpu())
    preds = torch.cat(preds).numpy()
    if mode == "store": return preds, torch.cat(e1)
    if mode == "lpkl": return preds, torch.cat(e1).numpy(), torch.cat(e2).numpy()
    return preds

def h_paired(new_preds):
    new_c = (np.asarray(new_preds) == eval_labels_np); ref = H_BASE_CORRECT
    b = int((ref & ~new_c).sum()); c = int((~ref & new_c).sum())
    p = mcnemar([[0, b], [c, 0]], exact=(b + c) < 25).pvalue if (b + c) > 0 else 1.0
    m, ci = mean_ci((new_c.astype(np.int8) - ref.astype(np.int8)) * 100.0)
    return {"accuracy": float(new_c.mean()), "delta_pp": m, "ci_low_pp": ci[0], "ci_high_pp": ci[1],
            "b_correct_to_wrong": b, "c_wrong_to_correct": c, "mcnemar_p": float(p)}, new_c

try:
    res35 = load_checkpoint(CKPT_35); print("Resuming; saved:", list(res35))
except Exception:
    res35 = {}

# ---- baselines: default windows (sensitivity) and global windows (main) ----
if "baseline_default" not in res35:
    set_windows("default"); t0 = time.time()
    p = h_eval(); res35["baseline_default"] = {"accuracy": float((p == eval_labels_np).mean())}
    save_checkpoint(CKPT_35, res35)
    print(f"384, default windows: {res35['baseline_default']['accuracy']*100:.3f}%  ({(time.time()-t0)/60:.1f} min)")
else:
    print(f"384, default windows (saved): {res35['baseline_default']['accuracy']*100:.3f}%")

set_windows("global")
if "baseline_global" in res35 and "joint_position" in res35:
    # resume: baseline and joint freezes are already saved -> load the baseline instead of re-running it
    H_BASE_CORRECT = unpack(res35["baseline_global"]["correct_packed"]); H_BASE_LP = None
    print(f"384 global baseline loaded from checkpoint: {H_BASE_CORRECT.mean()*100:.3f}%")
else:
    t0 = time.time()
    h_base_preds, H_BASE_LP = h_eval(mode="store")
    H_BASE_CORRECT = (h_base_preds == eval_labels_np)
    H_PASS_MIN = (time.time() - t0) / 60
    if "baseline_global" not in res35:
        res35["baseline_global"] = {"accuracy": float(H_BASE_CORRECT.mean()), "correct_packed": pack(H_BASE_CORRECT)}
        save_checkpoint(CKPT_35, res35)
    assert abs(res35["baseline_global"]["accuracy"] - H_BASE_CORRECT.mean()) < 1e-9
    print(f"384, global windows (main): {H_BASE_CORRECT.mean()*100:.3f}% | one pass {H_PASS_MIN:.1f} min | 224 baseline: 82.188%")

# ---- references at 384 (global windows): mixer means and Δ/B/C (mean + position mode) ----
_cap = {name: [] for name, _, _ in mixer_blocks}
hs = [blk.mixer.register_forward_hook(lambda m, i, o, name=name: _cap[name].append((o[0] if isinstance(o, tuple) else o).detach()))
      for name, _, blk in mixer_blocks]
with torch.no_grad():
    for s in range(0, 256, 32): model(h_get_batch(s, s + 32))
for h in hs: h.remove()
h_means = {name: torch.cat(v).mean(dim=(0, 1), keepdim=True) for name, v in _cap.items()}
del _cap

_scan_cap = {n: {"dt": [], "B": [], "C": []} for n in mamba_block_names}
def _h_capture_scan(x, dt, A, B, C, D, z=None, delta_bias=None, delta_softplus=None, return_last_state=None):
    n = _active_block_name[0]
    if n is not None:
        _scan_cap[n]["dt"].append(dt.detach()); _scan_cap[n]["B"].append(B.detach()); _scan_cap[n]["C"].append(C.detach())
    return _original_selective_scan_fn(x, dt, A, B, C, D, z=z, delta_bias=delta_bias,
                                       delta_softplus=delta_softplus, return_last_state=return_last_state)
clear_all_hooks(); hs = install_block_name_hooks(); target_module.selective_scan_fn = _h_capture_scan
try:
    with torch.no_grad():
        for s in range(0, 256, 32): model(h_get_batch(s, s + 32))
finally:
    target_module.selective_scan_fn = _original_selective_scan_fn
    for h in hs: h.remove()
refs384 = {}
for n in mamba_block_names:
    refs384[n] = {}
    for t in ("dt", "B", "C"):
        x = torch.cat(_scan_cap[n][t])                                   # (256, C, L)
        refs384[n][t] = {"mean": x.mean(dim=(0, 2), keepdim=True), "position": x.mean(dim=0, keepdim=True)}
del _scan_cap
_freeze_refs_224 = _freeze_refs                                          # keep the 224 references
print("References at 384 ready:", len(h_means), "means;", list(refs384), "Δ/B/C")
print("\nSTEP 35A COMPLETE")

### Step 35B — Joint freeze, all-Mamba-off and per-block real vs null at 384 ###

In [ ]:
# ============================================================
# STEP 35B — EXPERIMENTS AT 384 (global windows)
# ============================================================
set_windows("global")
torch.manual_seed(42)                                                    # same permutations as Step 16
h_perms = {name: torch.randperm(h_means[name].shape[-1]) for name, _, _ in mixer_blocks}
def _h_null_hook(name):
    mean_val, perm = h_means[name], h_perms[name]
    def f(m, inp, out):
        is_t = isinstance(out, tuple); t = out[0] if is_t else out
        rep = t + (mean_val.to(dtype=t.dtype).expand_as(t) - t)[..., perm.to(t.device)]
        return (rep,) + out[1:] if is_t else rep
    return f

def h_freeze(mode):
    global _freeze_refs
    _freeze_refs = refs384
    hs = install_freeze_patch()
    try:
        _freeze_config.update({"active": True, "block_names": set(mamba_block_names), "tensor": "all", "mode": mode})
        return h_eval(mode="lpkl")
    finally:
        remove_freeze_patch(hs); _freeze_refs = _freeze_refs_224

def h_hooks(blocks, kind):
    clear_all_hooks()
    return [blk.mixer.register_forward_hook(mean_ablation_hook(h_means[name]) if kind == "real" else _h_null_hook(name))
            for name, _, blk in mixer_blocks if name in blocks]

# ---- all-Mamba-off, then joint freeze (mean, position) ----
if "all_mamba_off" not in res35:
    hs = h_hooks(set(mamba_block_names), "real")
    try: p = h_eval()
    finally:
        for h in hs: h.remove()
    st, c = h_paired(p); res35["all_mamba_off"] = {**st, "correct_packed": pack(c)}; save_checkpoint(CKPT_35, res35)
mamba_off_h = unpack(res35["all_mamba_off"]["correct_packed"])
print(f"All Mamba off @384: {res35['all_mamba_off']['delta_pp']:+.3f} pp   [224: −3.900]")
for mode in ["mean", "position"]:
    key = f"joint_{mode}"
    if key not in res35:
        t0 = time.time(); p, lpd, kl = h_freeze(mode); st, c = h_paired(p)
        bc = H_BASE_CORRECT.astype(np.int8); da, db = bc - c.astype(np.int8), bc - mamba_off_h.astype(np.int8)
        rng = np.random.RandomState(42)
        boot = [da[i].mean() / db[i].mean() for i in (rng.randint(0, N_FULL, N_FULL) for _ in range(2000))]
        res35[key] = {**st, "ratio_to_all_mamba_off": float(da.mean() / db.mean()),
                      "ratio_ci": [float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))],
                      "logprob_drop": float(lpd.mean()), "kl": float(kl.mean()), "correct_packed": pack(c)}
        save_checkpoint(CKPT_35, res35); print(f"({(time.time()-t0)/60:.1f} min)")
    r = res35[key]
    print(f"Joint freeze {mode:8s} @384: {r['delta_pp']:+.3f} pp [{r['ci_low_pp']:+.3f}, {r['ci_high_pp']:+.3f}] | "
          f"÷ all-Mamba-off {r['ratio_to_all_mamba_off']*100:.1f}% [{r['ratio_ci'][0]*100:.1f}, {r['ratio_ci'][1]*100:.1f}]"
          f"   [224: {'29.1' if mode == 'mean' else '20.3'}%]")

# ---- per block: real ablation vs shuffled null ----
res35.setdefault("blocks", {})
for name, mtype, _ in mixer_blocks:
    if name in res35["blocks"]: continue
    t0 = time.time(); out = {}
    for kind in ["real", "null"]:
        hs = h_hooks({name}, kind)
        try: p = h_eval()
        finally:
            for h in hs: h.remove()
        out[kind] = h_paired(p)
    (st_r, c_r), (st_n, c_n) = out["real"], out["null"]
    d_m, d_ci = mean_ci((c_r.astype(np.int8) - c_n.astype(np.int8)) * 100.0)
    res35["blocks"][name] = {"type": mtype, "real": {**st_r, "correct_packed": pack(c_r)},
                             "null": {**st_n, "correct_packed": pack(c_n)},
                             "real_minus_null_pp": d_m, "real_minus_null_ci": d_ci}
    save_checkpoint(CKPT_35, res35)
    print(f"{name:5s} ({(time.time()-t0)/60:.1f} min): real {st_r['delta_pp']:+.3f} | null {st_n['delta_pp']:+.3f} | "
          f"real − null {d_m:+.3f} [{d_ci[0]:+.3f}, {d_ci[1]:+.3f}]")
print("\nSTEP 35B COMPLETE")

### Step 35C — 384 summary and comparison with 224 (CPU only) ###

In [ ]:
# ============================================================
# STEP 35C — 384 SUMMARY vs 224
# ============================================================
from scipy.stats import spearmanr, norm
res35 = load_checkpoint(CKPT_35)
Bk = res35["blocks"]; names = list(Bk)
real = np.array([Bk[n]["real"]["delta_pp"] for n in names]); null = np.array([Bk[n]["null"]["delta_pp"] for n in names])
types = ["Mamba" if Bk[n]["type"] == "MambaVisionMixer" else "Attn" for n in names]
pv = []
for n in names:
    lo, hi = Bk[n]["real_minus_null_ci"]; se = (hi - lo) / 3.92
    pv.append(2 * norm.sf(abs(Bk[n]["real_minus_null_pp"]) / se) if se > 0 else 1.0)
order = np.argsort(pv); holm = np.empty(len(pv)); run = 0
for r_, i in enumerate(order):
    run = max(run, min(1.0, (len(pv) - r_) * pv[i])); holm[i] = run
dvp224 = {r["block"]: r for r in load_checkpoint("damage_vs_perturbation_table")["rows"]}

print(f"{'Block':6s} {'Type':6s} {'224 real':>9s} {'384 real':>9s} {'384 null':>9s} {'384 real − null [CI]':>26s} {'Holm p':>8s} {'224 real − null':>16s}")
for i, n in enumerate(names):
    r = Bk[n]; lo, hi = r["real_minus_null_ci"]
    print(f"{n:6s} {types[i]:6s} {dvp224[n]['real_dpp']:+9.3f} {real[i]:+9.3f} {null[i]:+9.3f} "
          f"{r['real_minus_null_pp']:+9.3f} [{lo:+.3f},{hi:+.3f}] {holm[i]:8.4f} {dvp224[n]['real_minus_null_pp']:+16.3f}")

rho_rn = spearmanr(np.abs(real), np.abs(null))
rho_res = spearmanr([dvp224[n]["real_dpp"] for n in names], real)
mam = [i for i, t in enumerate(types) if t == "Mamba"]; att = [i for i, t in enumerate(types) if t == "Attn"]
per_img = np.mean([(unpack(Bk[n]["null"]["correct_packed"]).astype(float) - unpack(Bk[n]["real"]["correct_packed"]).astype(float)) * 100
                   for n in names], axis=0)
rng = np.random.RandomState(42); bm = np.array([per_img[rng.randint(0, N_FULL, N_FULL)].mean() for _ in range(2000)])

print(f"\n384 baseline (global windows): {res35['baseline_global']['accuracy']*100:.3f}% | default windows: {res35['baseline_default']['accuracy']*100:.3f}%")
print(f"Mean |Δ| per block: Mamba {np.abs(real[mam]).mean():.3f} pp vs Attention {np.abs(real[att]).mean():.3f} pp   [224: 0.18 vs 0.49]")
print(f"Spearman |real| vs |null| at 384: ρ = {rho_rn.correlation:.2f} (p = {rho_rn.pvalue:.3f})   [224: 0.83]")
print(f"Spearman real Δ 224 vs 384 (ranking stable across resolution?): ρ = {rho_res.correlation:.2f} (p = {rho_res.pvalue:.3f})")
print(f"After Holm: real worse in {[n for i, n in enumerate(names) if holm[i] < 0.05 and Bk[n]['real_minus_null_pp'] < 0]}, "
      f"null worse in {[n for i, n in enumerate(names) if holm[i] < 0.05 and Bk[n]['real_minus_null_pp'] > 0]}")
print(f"Pooled content effect: {per_img.mean():+.3f} pp [{np.percentile(bm, 2.5):+.3f}, {np.percentile(bm, 97.5):+.3f}], "
      f"one-sided 95% upper bound {np.percentile(bm, 95):+.3f}   [224: −0.016, bound +0.016]")
s_m = real[mam].sum()
print(f"All Mamba off {res35['all_mamba_off']['delta_pp']:+.3f} pp = {res35['all_mamba_off']['delta_pp']/s_m:.2f}× sum of single Mamba ({s_m:+.3f})   [224: 3.67×]")
for mode in ["mean", "position"]:
    r = res35[f"joint_{mode}"]
    print(f"Joint freeze {mode:8s}: {r['delta_pp']:+.3f} pp | ÷ all-Mamba-off {r['ratio_to_all_mamba_off']*100:.1f}% "
          f"[{r['ratio_ci'][0]*100:.1f}, {r['ratio_ci'][1]*100:.1f}]")
res35["summary"] = {"holm_p": {n: float(h) for n, h in zip(names, holm)}, "spearman_real_null": [float(rho_rn.correlation), float(rho_rn.pvalue)],
                    "spearman_224_vs_384": [float(rho_res.correlation), float(rho_res.pvalue)],
                    "pooled_content_pp": float(per_img.mean()), "pooled_ci": [float(np.percentile(bm, 2.5)), float(np.percentile(bm, 97.5))],
                    "pooled_upper95": float(np.percentile(bm, 95))}
save_checkpoint(CKPT_35, res35)
print("\nSTEP 35C COMPLETE")

## Step 36 — Pairwise block-interaction matrix (66 pairs, MambaVision-T, 224) ##
Mean-ablates every pair of the 12 mixer blocks at once (66 runs). Interaction per image =
acc(pair) − acc(block a) − acc(block b) + acc(baseline); negative = the pair costs more than the two
single ablations together (super-additive, consistent with compensation). Single-block results from
Step 12A. Holm over 66 pairs. Saves `step36_pairwise` after every pair.

### Step 36A — Pairwise ablation (66 pairs) ###

In [ ]:
# ============================================================
# STEP 36A — PAIRWISE ABLATION (66 pairs)
# ============================================================
import itertools, time
assert "eval_pass_fast" in globals() and "block_means" in globals(), "Run Step 8, Step 24B and the Step 30 setup cell first."

# ---- undo the 384 window change from Step 35A, then check that 224 is exactly the baseline again ----
if "set_windows" in globals():
    set_windows("default")
clear_all_hooks()
if "target_module" in globals():
    target_module.selective_scan_fn = _original_selective_scan_fn
_chk = eval_pass_fast(n=2000)
assert (_chk == baseline_preds_np[:2000]).all(), "224 model differs from the baseline - send me this output."
print("224 model restored: matches baseline on 2,000 images.")

CKPT_36 = "step36_pairwise"
names = [n for n, _, _ in mixer_blocks]
btype = {n: t for n, t, _ in mixer_blocks}
blk_of = {n: b for n, _, b in mixer_blocks}
_sp = load_checkpoint("step9_ablated_preds")
single_c = {b: (np.asarray(_sp[b]).astype(np.int64) == eval_labels_np) for b in names}
base_i8 = baseline_correct.astype(np.int8)

try:
    res36 = load_checkpoint(CKPT_36); print("Resuming;", len(res36), "pairs done")
except Exception:
    res36 = {}

pairs = list(itertools.combinations(names, 2))
for k, (a, b) in enumerate(pairs, 1):
    key = f"{a}+{b}"
    if key in res36: continue
    t0 = time.time()
    clear_all_hooks()
    hs = [blk_of[x].mixer.register_forward_hook(mean_ablation_hook(block_means[x])) for x in (a, b)]
    try:
        preds = eval_pass_fast()
    finally:
        for h in hs: h.remove()
    st, c = paired_stats(preds)
    inter = (c.astype(np.int8) - single_c[a].astype(np.int8) - single_c[b].astype(np.int8) + base_i8) * 100.0
    i_m, i_ci = mean_ci(inter)
    res36[key] = {"a": a, "b": b,
                  "types": f"{'M' if btype[a]=='MambaVisionMixer' else 'A'}{'M' if btype[b]=='MambaVisionMixer' else 'A'}",
                  "delta_pp": st["delta_pp"], "ci": [st["ci_low_pp"], st["ci_high_pp"]],
                  "interaction_pp": i_m, "interaction_ci": i_ci, "correct_packed": pack(c)}
    save_checkpoint(CKPT_36, res36)
    print(f"[{k:2d}/66] {key:10s} ({(time.time()-t0)/60:.1f} min): pair Δ {st['delta_pp']:+.3f} | "
          f"interaction {i_m:+.3f} [{i_ci[0]:+.3f}, {i_ci[1]:+.3f}]")
print("\nSTEP 36A COMPLETE" if len(res36) == 66 else f"\nStopped at {len(res36)}/66 - rerun to resume.")

### Step 36B — Interaction summary (CPU only) ###

In [ ]:
# ============================================================
# STEP 36B — PAIRWISE SUMMARY
# ============================================================
from scipy.stats import norm
res36 = load_checkpoint(CKPT_36)
rows = list(res36.values())
pv = []
for r in rows:
    lo, hi = r["interaction_ci"]; se = (hi - lo) / 3.92
    pv.append(2 * norm.sf(abs(r["interaction_pp"]) / se) if se > 0 else 1.0)
order = np.argsort(pv); holm = np.empty(len(pv)); run = 0
for i_, j in enumerate(order):
    run = max(run, min(1.0, (len(pv) - i_) * pv[j])); holm[j] = run
for grp in ["MM", "AA", "MA", "AM"]:
    sel = [i for i, r in enumerate(rows) if r["types"] == grp]
    if sel:
        v = np.array([rows[i]["interaction_pp"] for i in sel])
        print(f"{grp}: {len(sel):2d} pairs | mean interaction {v.mean():+.3f} pp | "
              f"super-additive (Holm) {sum(holm[i] < 0.05 and rows[i]['interaction_pp'] < 0 for i in sel)} | "
              f"sub-additive (Holm) {sum(holm[i] < 0.05 and rows[i]['interaction_pp'] > 0 for i in sel)}")
print("\nStrongest interactions:")
for j in np.argsort([r["interaction_pp"] for r in rows])[:8]:
    r = rows[j]
    print(f"  {r['a']}+{r['b']} ({r['types']}): {r['interaction_pp']:+.3f} [{r['interaction_ci'][0]:+.3f}, {r['interaction_ci'][1]:+.3f}]  Holm p = {holm[j]:.4f}")
save_checkpoint("step36_pairwise_summary", {"holm_p": {f"{r['a']}+{r['b']}": float(h) for r, h in zip(rows, holm)}})
print("\nSTEP 36B COMPLETE")

## Step 37 — Replication on MambaVision-B ##
Repeats E2, E3, E5 and E9 on nvidia/MambaVision-B-1K with the same images (Step 5b order), the same 256
reference images and the same statistics; the group nulls of Step 26 are included. MambaVision-B uses a
learned per-channel layer scale γ₁, so the shuffled null is permuted in the residual stream
(γ₁⊙Δx permuted, then divided by γ₁): real and null then have exactly the same norm in the residual
stream. Prefix b_. Saves `step37_mambavision_b` after every run.

### Step 37A — Load MambaVision-B, preprocessing, baseline (speed check) ###

In [ ]:
# ============================================================
# STEP 37A — MAMBAVISION-B: load, preprocessing, baseline
# ============================================================
import re, sys, time, os
from transformers import AutoModelForImageClassification
from timm.data.transforms_factory import create_transform
from torchvision import transforms as T
from torch.utils.data import Dataset as _TorchDataset, DataLoader
assert "CACHE_DIR" in globals() and "mean_ci" in globals(), "Run Step 24B and the Step 30 setup cell first."

B_MODEL_ID, CKPT_37 = "nvidia/MambaVision-B-1K", "step37_mambavision_b"
b_model = AutoModelForImageClassification.from_pretrained(B_MODEL_ID, trust_remote_code=True,
                                                          low_cpu_mem_usage=False).to(device).eval()
print(f"MambaVision-B parameters: {sum(p.numel() for p in b_model.parameters()):,} | revision {b_model.config._commit_hash}")

b_transform = create_transform(input_size=(3, 224, 224), is_training=False, mean=b_model.config.mean,
                               std=b_model.config.std, crop_mode=b_model.config.crop_mode,
                               crop_pct=b_model.config.crop_pct)
print("B preprocessing:", b_transform)
_tf = list(b_transform.transforms); _k = next(i for i, t in enumerate(_tf) if "ToTensor" in type(t).__name__)
B_PRE = T.Compose(_tf[:_k])
B_MEAN = torch.as_tensor(_tf[_k + 1].mean, dtype=torch.float32, device=device).view(1, 3, 1, 1)
B_STD  = torch.as_tensor(_tf[_k + 1].std,  dtype=torch.float32, device=device).view(1, 3, 1, 1)
B_CACHE = os.path.join(CACHE_DIR, f"mv_cache_B_images_u8_{N_FULL}.npy")
class _BCacheSource(_TorchDataset):
    def __len__(self): return N_FULL
    def __getitem__(self, pos):
        ex = dataset[eval_indices[pos]]
        return np.asarray(B_PRE(ex["image"].convert("RGB")), dtype=np.uint8).copy(), ex["label"]
if not os.path.exists(B_CACHE + ".done"):
    t0 = time.time()
    mm = np.lib.format.open_memmap(B_CACHE, mode="w+", dtype=np.uint8, shape=(N_FULL, 224, 224, 3)); pos = 0
    for x, _ in DataLoader(_BCacheSource(), batch_size=256, num_workers=os.cpu_count() or 2):
        mm[pos:pos + len(x)] = x.numpy(); pos += len(x)
    mm.flush(); del mm; open(B_CACHE + ".done", "w").write("ok")
    print(f"B cache built in {(time.time()-t0)/60:.1f} min")
B_IMG_CACHE = np.load(B_CACHE, mmap_mode="r")
def b_get_batch(start, end):
    x = torch.from_numpy(np.array(B_IMG_CACHE[start:end])).to(device, non_blocking=True)
    return x.permute(0, 3, 1, 2).float().div(255).sub(B_MEAN).div(B_STD)
_ref, _ = prepare_batch(dataset, eval_indices[:64], b_transform, device)
assert (b_get_batch(0, 64) - _ref).abs().max().item() < 1e-5, "B cache differs from prepare_batch."

# ---- mixer blocks of B, and their layer scale γ1 ----
b_mixer_blocks = []
for name, module in b_model.named_modules():
    if hasattr(module, "mixer"):
        m = re.search(r"levels\.(\d+)\.blocks\.(\d+)$", name)
        if m:
            b_mixer_blocks.append((f"L{m.group(1)}B{m.group(2)}", type(module.mixer).__name__, module))
b_gamma = {n: (blk.gamma_1.detach().float() if isinstance(getattr(blk, "gamma_1", None), torch.Tensor) else None)
           for n, _, blk in b_mixer_blocks}
print("B mixer blocks:", [(n, "Mamba" if t == "MambaVisionMixer" else "Attn") for n, t, _ in b_mixer_blocks])
for n, g in b_gamma.items():
    if g is not None:
        assert g.abs().min().item() > 1e-8, f"{n}: a γ1 entry is ~0; residual-space null cannot be used."
print("γ1:", "per-channel vectors, |γ| range "
      f"{min(g.abs().min().item() for g in b_gamma.values() if g is not None):.2e} – "
      f"{max(g.abs().max().item() for g in b_gamma.values() if g is not None):.2e}" if any(g is not None for g in b_gamma.values()) else "1 (none)")

def b_clear_hooks():
    for _, _, blk in b_mixer_blocks:
        for m in (blk, blk.mixer):
            m._forward_hooks.clear(); m._forward_pre_hooks.clear()

def b_eval(n=None, mode=None):
    n = N_FULL if n is None else n
    preds, e1, e2 = [], [], []
    with torch.no_grad():
        for s in range(0, n, BATCH_SIZE):
            e = min(s + BATCH_SIZE, n)
            out = b_model(b_get_batch(s, e))
            logits = (out["logits"] if isinstance(out, dict) else out.logits).float()
            preds.append(logits.argmax(-1).cpu())
            if mode == "store":
                e1.append(F.log_softmax(logits, -1).half().cpu())
            elif mode == "lpkl":
                lp = F.log_softmax(logits, -1); y = _LAB_GPU[s:e].view(-1, 1)
                base = B_BASE_LP[s:e].to(device).float()
                e1.append((base.gather(1, y) - lp.gather(1, y)).squeeze(1).cpu())
                e2.append((base.exp() * (base - lp)).sum(-1).cpu())
    preds = torch.cat(preds).numpy()
    if mode == "store": return preds, torch.cat(e1)
    if mode == "lpkl": return preds, torch.cat(e1).numpy(), torch.cat(e2).numpy()
    return preds

def b_paired(new_preds):
    new_c = (np.asarray(new_preds) == eval_labels_np); ref = B_BASE_CORRECT
    b = int((ref & ~new_c).sum()); c = int((~ref & new_c).sum())
    p = mcnemar([[0, b], [c, 0]], exact=(b + c) < 25).pvalue if (b + c) > 0 else 1.0
    m, ci = mean_ci((new_c.astype(np.int8) - ref.astype(np.int8)) * 100.0)
    return {"accuracy": float(new_c.mean()), "delta_pp": m, "ci_low_pp": ci[0], "ci_high_pp": ci[1],
            "b_correct_to_wrong": b, "c_wrong_to_correct": c, "mcnemar_p": float(p)}, new_c

b_clear_hooks(); t0 = time.time()
b_base_preds, B_BASE_LP = b_eval(mode="store")
B_BASE_CORRECT = (b_base_preds == eval_labels_np)
B_PASS_MIN = (time.time() - t0) / 60
n_blocks = len(b_mixer_blocks); n_mamba = sum(t == "MambaVisionMixer" for _, t, _ in b_mixer_blocks)
print(f"MambaVision-B baseline: {B_BASE_CORRECT.mean()*100:.3f}% | one pass = {B_PASS_MIN:.1f} min | published top-1 84.2%")
print(f"Estimated time for the rest of Step 37: ~{(4 + 2 + 2 * n_blocks) * B_PASS_MIN / 60:.1f} h "
      f"({4 + 2 + 2 * n_blocks} passes: E3 + group nulls + E9 + {n_blocks} blocks × real/null)")
try:
    res37 = load_checkpoint(CKPT_37)
    assert abs(res37["baseline"]["accuracy"] - B_BASE_CORRECT.mean()) < 1e-9, "Baseline differs from the saved one!"
    print("Resuming; saved parts:", [k for k in res37 if k != "baseline"])
except Exception as e:
    if "differs" in str(e): raise
    res37 = {"baseline": {"accuracy": float(B_BASE_CORRECT.mean()), "correct_packed": pack(B_BASE_CORRECT), "model": B_MODEL_ID}}
    save_checkpoint(CKPT_37, res37)
print("\nSTEP 37A COMPLETE")

### Step 37B — Reference means and Δ, B, C references (256 Step 8 images) ###

In [ ]:
# ============================================================
# STEP 37B — B references: mixer-output means and Δ/B/C (mean + position mode)
# ============================================================
b_clear_hooks()
_cap = {name: [] for name, _, _ in b_mixer_blocks}
_hs = [blk.mixer.register_forward_hook(lambda mod, i, o, name=name: _cap[name].append((o[0] if isinstance(o, tuple) else o).detach()))
       for name, _, blk in b_mixer_blocks]
with torch.no_grad():
    for s in range(0, 256, 32): b_model(b_get_batch(s, s + 32))
for h in _hs: h.remove()
b_block_means = {name: torch.cat(v).mean(dim=(0, 1), keepdim=True) for name, v in _cap.items()}
del _cap

b_mamba = {name: blk for name, t, blk in b_mixer_blocks if t == "MambaVisionMixer"}
b_mamba_names = list(b_mamba)
b_target_module = sys.modules[type(b_mamba[b_mamba_names[0]].mixer).__module__]
if "_b_orig_scan" not in globals():
    _b_orig_scan = b_target_module.selective_scan_fn
b_target_module.selective_scan_fn = _b_orig_scan
_b_active = [None]
_b_capture = {"on": False, "data": {n: {"dt": [], "B": [], "C": []} for n in b_mamba_names}}
_b_fz = {"active": False, "blocks": set(), "tensor": "all", "mode": "mean"}
b_freeze_refs = {}

def _b_scan(x, dt, A, B, C, D, z=None, delta_bias=None, delta_softplus=None, return_last_state=None):
    name = _b_active[0]
    if name is not None and _b_capture["on"]:
        d = _b_capture["data"][name]; d["dt"].append(dt.detach()); d["B"].append(B.detach()); d["C"].append(C.detach())
    if name is not None and _b_fz["active"] and name in _b_fz["blocks"]:
        ref, mode = b_freeze_refs[name], _b_fz["mode"]
        dt = ref["dt"][mode].expand_as(dt).to(device=dt.device, dtype=dt.dtype)
        B  = ref["B"][mode].expand_as(B).to(device=B.device, dtype=B.dtype)
        C  = ref["C"][mode].expand_as(C).to(device=C.device, dtype=C.dtype)
    return _b_orig_scan(x, dt, A, B, C, D, z=z, delta_bias=delta_bias,
                        delta_softplus=delta_softplus, return_last_state=return_last_state)

def b_install_scan_patch():
    b_clear_hooks(); hs = []
    for name, blk in b_mamba.items():
        hs.append(blk.mixer.register_forward_pre_hook(lambda mod, inp, name=name: _b_active.__setitem__(0, name)))
        hs.append(blk.mixer.register_forward_hook(lambda mod, inp, out: _b_active.__setitem__(0, None)))
    b_target_module.selective_scan_fn = _b_scan
    return hs
def b_remove_scan_patch(hs):
    _b_fz["active"] = False; _b_capture["on"] = False
    b_target_module.selective_scan_fn = _b_orig_scan
    for h in hs: h.remove()

_hs = b_install_scan_patch()
try:
    _b_capture["on"] = True
    with torch.no_grad():
        for s in range(0, 256, 32): b_model(b_get_batch(s, s + 32))
finally:
    b_remove_scan_patch(_hs)
for name in b_mamba_names:
    b_freeze_refs[name] = {}
    for tname in ("dt", "B", "C"):
        t = torch.cat(_b_capture["data"][name][tname])
        b_freeze_refs[name][tname] = {"mean": t.mean(dim=(0, 2), keepdim=True), "position": t.mean(dim=0, keepdim=True)}
_b_capture["data"] = None

# ---- shuffled null in the residual stream: permute γ1⊙Δx, divide by γ1 ----
b_perms = {name: torch.randperm(b_block_means[name].shape[-1], generator=torch.Generator().manual_seed(42 + bi))
           for bi, (name, _, _) in enumerate(b_mixer_blocks)}
_b_null = {"active": False, "blocks": set()}
def _b_null_hook(name):
    mean_val, perm, g = b_block_means[name], b_perms[name], b_gamma[name]
    def f(module, inp, out):
        if not _b_null["active"] or name not in _b_null["blocks"]:
            return out
        is_t = isinstance(out, tuple); t = out[0] if is_t else out
        delta = mean_val.to(dtype=t.dtype).expand_as(t) - t
        if g is None:
            rep = t + delta[..., perm.to(t.device)]
        else:
            gg = g.to(device=t.device, dtype=t.dtype)
            rep = t + (gg * delta)[..., perm.to(t.device)] / gg
        return (rep,) + out[1:] if is_t else rep
    return f
def b_run_null(blocks, n=None):
    b_clear_hooks()
    hs = [blk.mixer.register_forward_hook(_b_null_hook(name)) for name, _, blk in b_mixer_blocks]
    try:
        _b_null.update({"active": True, "blocks": set(blocks)})
        return b_eval(n=n)
    finally:
        _b_null["active"] = False
        for h in hs: h.remove()

# ---- check: residual-stream norms of real and null perturbations are equal ----
_n0, _, _blk0 = b_mixer_blocks[0]
with torch.no_grad():
    _o = []; _h = _blk0.mixer.register_forward_hook(lambda m, i, o: _o.append(o[0] if isinstance(o, tuple) else o))
    b_model(b_get_batch(0, 8)); _h.remove()
    t = _o[0]; delta = b_block_means[_n0].expand_as(t) - t
    gg = b_gamma[_n0].to(t.device) if b_gamma[_n0] is not None else torch.ones(t.shape[-1], device=t.device)
    real_res = (gg * delta).norm(); null_res = (gg * ((gg * delta)[..., b_perms[_n0].to(t.device)] / gg)).norm()
print(f"Residual-stream norm check ({_n0}): real {real_res.item():.4f} vs null {null_res.item():.4f}")
assert abs(real_res.item() - null_res.item()) / real_res.item() < 1e-4
print("Means for", len(b_block_means), "blocks; Δ/B/C references for", b_mamba_names)
print("\nSTEP 37B COMPLETE")

### Step 37C — E3 grouped ablation, group nulls and E9 joint freeze (mean and position mode) ###

In [ ]:
# ============================================================
# STEP 37C — B: grouped ablation (E3), group nulls, joint Δ/B/C freeze (E9)
# ============================================================
b_groups = {"mamba": [n for n, t, _ in b_mixer_blocks if t == "MambaVisionMixer"],
            "attn":  [n for n, t, _ in b_mixer_blocks if t != "MambaVisionMixer"]}
assert len(b_groups["mamba"]) == 8 and len(b_groups["attn"]) == 7, b_groups
print("Mamba group:", b_groups["mamba"]); print("Attention group:", b_groups["attn"])

res37.setdefault("E3", {}); res37.setdefault("group_null", {}); res37.setdefault("E9", {})

# ---- E3 grouped ablation + group shuffled null ----
for key, names_ in b_groups.items():
    if key not in res37["E3"]:
        t0 = time.time(); b_clear_hooks()
        hs = [blk.mixer.register_forward_hook(mean_ablation_hook(b_block_means[n]))
              for n, _, blk in b_mixer_blocks if n in names_]
        try: p = b_eval()
        finally:
            for h in hs: h.remove()
        st, c = b_paired(p)
        res37["E3"][key] = {**st, "blocks": names_, "correct_packed": pack(c)}
        save_checkpoint(CKPT_37, res37)
        print(f"  E3 {key} done ({(time.time()-t0)/60:.1f} min)")
    if key not in res37["group_null"]:
        t0 = time.time()
        st, c = b_paired(b_run_null(names_))
        real_c = unpack(res37["E3"][key]["correct_packed"])
        d_m, d_ci = mean_ci((real_c.astype(np.int8) - c.astype(np.int8)) * 100.0)
        res37["group_null"][key] = {**st, "real_minus_null_pp": d_m, "real_minus_null_ci": d_ci,
                                    "correct_packed": pack(c)}
        save_checkpoint(CKPT_37, res37)
        print(f"  group null {key} done ({(time.time()-t0)/60:.1f} min)")
    e3, gn = res37["E3"][key], res37["group_null"][key]
    print(f"All {key} off: real {e3['delta_pp']:+.3f} | null {gn['delta_pp']:+.3f} | real − null "
          f"{gn['real_minus_null_pp']:+.3f} [{gn['real_minus_null_ci'][0]:+.3f}, {gn['real_minus_null_ci'][1]:+.3f}]")

# ---- E9 joint freeze of Δ, B, C in all Mamba blocks: mean and position mode ----
mamba_off_b = unpack(res37["E3"]["mamba"]["correct_packed"])
for mode in ["mean", "position"]:
    if mode not in res37["E9"]:
        t0 = time.time()
        hs = b_install_scan_patch()
        try:
            _b_fz.update({"active": True, "blocks": set(b_mamba_names), "tensor": "all", "mode": mode})
            p, lpd, kl = b_eval(mode="lpkl")
        finally:
            b_remove_scan_patch(hs)
        st, c = b_paired(p)
        bc = B_BASE_CORRECT.astype(np.int8)
        da, db = bc - c.astype(np.int8), bc - mamba_off_b.astype(np.int8)
        rng = np.random.RandomState(42)
        boot = [da[i].mean() / db[i].mean() for i in (rng.randint(0, N_FULL, N_FULL) for _ in range(2000))]
        res37["E9"][mode] = {**st, "ratio_to_all_mamba_off": float(da.mean() / db.mean()),
                             "ratio_ci": [float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))],
                             "logprob_drop": float(lpd.mean()), "kl": float(kl.mean()),
                             "correct_packed": pack(c)}
        save_checkpoint(CKPT_37, res37)
        print(f"  E9 {mode} done ({(time.time()-t0)/60:.1f} min)")
    r = res37["E9"][mode]
    print(f"Joint freeze ({mode:8s}): {r['delta_pp']:+.3f} pp [{r['ci_low_pp']:+.3f}, {r['ci_high_pp']:+.3f}] | "
          f"÷ all-Mamba-off {r['ratio_to_all_mamba_off']*100:.1f}% [{r['ratio_ci'][0]*100:.1f}, {r['ratio_ci'][1]*100:.1f}] | "
          f"log-prob drop {r['logprob_drop']:+.4f} | KL {r['kl']:.4f}"
          f"   [T: {'29.1' if mode == 'mean' else '20.3'}%, S: {'40.9' if mode == 'mean' else '32.1'}%]")

b_target_module.selective_scan_fn = _b_orig_scan
b_clear_hooks()
print("\nSTEP 37C COMPLETE")

### Step 37D — E2 single-block mean ablation and E5 residual-matched shuffled null, per block ###

In [ ]:
# ============================================================
# STEP 37D — B: per-block real ablation (E2) and shuffled null (E5)
# ============================================================
res37.setdefault("E2E5", {})
for name, mtype, blk in b_mixer_blocks:
    if name in res37["E2E5"]: continue
    t0 = time.time()
    b_clear_hooks()
    h = blk.mixer.register_forward_hook(mean_ablation_hook(b_block_means[name]))
    try: p_r, lpd, kl = b_eval(mode="lpkl")
    finally: h.remove()
    st_r, c_r = b_paired(p_r)
    st_n, c_n = b_paired(b_run_null([name]))
    d_m, d_ci = mean_ci((c_r.astype(np.int8) - c_n.astype(np.int8)) * 100.0)
    res37["E2E5"][name] = {"type": mtype, "real": {**st_r, "logprob_drop": float(lpd.mean()), "kl": float(kl.mean()),
                                                   "correct_packed": pack(c_r)},
                           "null": {**st_n, "correct_packed": pack(c_n)},
                           "real_minus_null_pp": d_m, "real_minus_null_ci": d_ci}
    save_checkpoint(CKPT_37, res37)
    print(f"{name:5s} {'Mamba' if mtype == 'MambaVisionMixer' else 'Attn ':5s} ({(time.time()-t0)/60:.1f} min) "
          f"real {st_r['delta_pp']:+.3f} | null {st_n['delta_pp']:+.3f} | real − null {d_m:+.3f} [{d_ci[0]:+.3f}, {d_ci[1]:+.3f}]")
print("\nSTEP 37D COMPLETE")

### Step 37E — B summary and comparison with T and S (CPU only) ###

In [ ]:
# ============================================================
# STEP 37E — B SUMMARY vs T and S
# ============================================================
from scipy.stats import spearmanr, norm
res37 = load_checkpoint(CKPT_37)
R = res37["E2E5"]; names_ = list(R)
real = np.array([R[n]["real"]["delta_pp"] for n in names_]); null = np.array([R[n]["null"]["delta_pp"] for n in names_])
types = ["M" if R[n]["type"] == "MambaVisionMixer" else "A" for n in names_]
pv = []
for n in names_:
    lo, hi = R[n]["real_minus_null_ci"]; se = (hi - lo) / 3.92
    pv.append(2 * norm.sf(abs(R[n]["real_minus_null_pp"]) / se) if se > 0 else 1.0)
order = np.argsort(pv); holm = np.empty(len(pv)); run = 0
for r_, i in enumerate(order):
    run = max(run, min(1.0, (len(pv) - r_) * pv[i])); holm[i] = run
mam = [i for i, t in enumerate(types) if t == "M"]; att = [i for i, t in enumerate(types) if t == "A"]
rho = spearmanr(np.abs(real), np.abs(null))
per_img = np.mean([(unpack(R[n]["null"]["correct_packed"]).astype(float) - unpack(R[n]["real"]["correct_packed"]).astype(float)) * 100
                   for n in names_], axis=0)
rng = np.random.RandomState(42); bm = np.array([per_img[rng.randint(0, N_FULL, N_FULL)].mean() for _ in range(2000)])

print(f"MambaVision-B baseline: {res37['baseline']['accuracy']*100:.3f}%  ({len(names_)} blocks: {len(mam)} Mamba, {len(att)} Attention)")
print(f"E2: mean |Δ| Mamba {np.abs(real[mam]).mean():.3f} pp vs Attention {np.abs(real[att]).mean():.3f} pp   [T: 0.18 vs 0.49; S: 0.11 vs 0.36]")
print(f"E5: Spearman |real| vs |null| = {rho.correlation:.2f} (p = {rho.pvalue:.3f})   [T: 0.83, S: 0.68]")
print(f"E5 after Holm: real worse in {[n for i, n in enumerate(names_) if holm[i] < 0.05 and R[n]['real_minus_null_pp'] < 0]}, "
      f"null worse in {[n for i, n in enumerate(names_) if holm[i] < 0.05 and R[n]['real_minus_null_pp'] > 0]}")
print(f"E5 pooled content effect: {per_img.mean():+.3f} pp [{np.percentile(bm, 2.5):+.3f}, {np.percentile(bm, 97.5):+.3f}], "
      f"one-sided 95% upper bound {np.percentile(bm, 95):+.3f}   [T: −0.016 (bound 0.016); S: −0.006 (bound 0.029)]")
for key in ["mamba", "attn"]:
    e3, gn = res37["E3"][key], res37["group_null"][key]; s_ = real[mam if key == "mamba" else att].sum()
    print(f"E3: all {key} off {e3['delta_pp']:+.3f} pp = {e3['delta_pp']/s_:.2f}× sum of singles | group null {gn['delta_pp']:+.3f} | "
          f"real − null {gn['real_minus_null_pp']:+.3f}")
for mode in ["mean", "position"]:
    e = res37["E9"][mode]
    print(f"E9: joint freeze {mode:8s} {e['delta_pp']:+.3f} pp | ÷ all-Mamba-off {e['ratio_to_all_mamba_off']*100:.1f}% "
          f"[{e['ratio_ci'][0]*100:.1f}, {e['ratio_ci'][1]*100:.1f}]")
res37["summary"] = {"holm_p": {n: float(h) for n, h in zip(names_, holm)}, "spearman": [float(rho.correlation), float(rho.pvalue)],
                    "pooled_content_pp": float(per_img.mean()), "pooled_upper95": float(np.percentile(bm, 95))}
save_checkpoint(CKPT_37, res37)
print("\nSTEP 37E COMPLETE")

## Step 38 — Gain-matched truncation of the frozen-SSM kernel ##
Step 33 truncated the explicit kernel K_d[k] after k_max taps. That changes two things at once: the memory length
and the kernel gain Σ_k K_d[k]. Here the truncated kernel is gain-matched to the full kernel in every channel, so
only the memory length changes. Two variants: "scale" (multiply the kept taps so that the gain matches; channels
whose truncated gain is near zero fall back to "fold") and "fold" (add the removed tail to the last kept tap).
k_max ∈ {1, 2, 4, 8, 16, 32, 64}, all six Mamba blocks of MambaVision-T, 50,000 images. Saves `step38_renorm_truncation`.

### Step 38A — Gain-matched truncation (scale and fold) ###

In [ ]:
# ============================================================
# STEP 38A — GAIN-MATCHED TRUNCATION OF THE FROZEN-SSM KERNEL
# ============================================================
assert "frozen_kernel" in globals() and "_conv_cfg" in globals(), "Run Step 33A first."
CKPT_38 = "step38_renorm_truncation"
K_LIST = [1, 2, 4, 8, 16, 32, 64]
_rn = {"active": False, "kmax": None, "mode": "scale", "diag": None}

def _conv_scan_rn(x, dt, A, B, C, D, z=None, delta_bias=None, delta_softplus=None, return_last_state=None):
    name = _active_block_name[0]
    if not (_rn["active"] and name in mamba_block_names):
        return _original_selective_scan_fn(x, dt, A, B, C, D, z=z, delta_bias=delta_bias,
                                           delta_softplus=delta_softplus, return_last_state=return_last_state)
    assert z is None and not return_last_state
    Bsz, Dch, L = x.shape
    K = frozen_kernel(name, A, delta_bias, delta_softplus, L, x.device)
    km = _rn["kmax"]
    n_fallback = 0
    if km is not None and km < L:
        full_gain = K.sum(-1)
        Kt = K.clone(); Kt[:, km:] = 0.0
        tr_gain = Kt.sum(-1)
        Kf = Kt.clone(); Kf[:, km - 1] += full_gain - tr_gain           # fold: tail mass into the last kept tap
        if _rn["mode"] == "fold":
            K = Kf
        else:                                                           # scale, with fold as fallback
            s = full_gain / torch.where(tr_gain.abs() > 1e-12, tr_gain, torch.ones_like(tr_gain))
            safe = (tr_gain.abs() > 1e-12) & (s.abs() <= 10.0)
            K = torch.where(safe[:, None], Kt * s[:, None], Kf)
            n_fallback = int((~safe).sum().item())
        if _rn["diag"] is not None and name not in _rn["diag"]:
            ratio = (tr_gain / full_gain.where(full_gain.abs() > 1e-12, torch.ones_like(full_gain))).abs()
            _rn["diag"][name] = {"L": int(L), "median_gain_kept": float(ratio.median()),
                                 "fallback_channels": n_fallback, "channels": int(Dch)}
    w = K.flip(-1).unsqueeze(1)
    y = F.conv1d(F.pad(x.float(), (L - 1, 0)), w, groups=Dch)
    y = y + D.float().view(1, -1, 1) * x.float()
    return y.to(x.dtype)

def run_rn(kmax, mode):
    clear_all_hooks()
    hs = install_block_name_hooks()
    target_module.selective_scan_fn = _conv_scan_rn
    try:
        _rn.update({"active": True, "kmax": kmax, "mode": mode, "diag": {}})
        out = eval_pass_fast(logprob_kl=True)
        return out, _rn["diag"]
    finally:
        _rn["active"] = False
        target_module.selective_scan_fn = _original_selective_scan_fn
        for h in hs: h.remove()

try:
    res38 = load_checkpoint(CKPT_38); print("Resuming; done:", list(res38))
except Exception:
    res38 = {}

full_c = unpack(load_checkpoint("step33_conv_kernel")["full"]["correct_packed"])
OLD_33 = {1: -3.788, 2: -3.520, 4: -3.466, 8: -3.288, 16: -3.022, 32: -2.560, 64: -1.806}

for mode in ["scale", "fold"]:
    for k in K_LIST:
        key = f"{mode}|k{k}"
        if key not in res38:
            t0 = time.time()
            (preds, lpd, kl), diag = run_rn(k, mode)
            st, corr = paired_stats(preds)
            d_m, d_ci = mean_ci((corr.astype(np.int8) - full_c.astype(np.int8)) * 100.0)
            lp_m, lp_ci = mean_ci(lpd); kl_m, kl_ci = mean_ci(kl)
            res38[key] = {**st, "minus_full_pp": d_m, "minus_full_ci": d_ci, "logprob_drop": lp_m,
                          "logprob_drop_ci": lp_ci, "kl": kl_m, "kl_ci": kl_ci, "diag": diag,
                          "correct_packed": pack(corr)}
            save_checkpoint(CKPT_38, res38)
            print(f"  {key} done ({(time.time()-t0)/60:.1f} min)")
        r = res38[key]
        g = np.median([v["median_gain_kept"] for v in r["diag"].values()]) if r["diag"] else 1.0
        fb = sum(v["fallback_channels"] for v in r["diag"].values())
        print(f"{mode:5s} k={k:2d}: Δ {r['delta_pp']:+.3f} pp [{r['ci_low_pp']:+.3f}, {r['ci_high_pp']:+.3f}] | "
              f"vs full conv {r['minus_full_pp']:+.3f} [{r['minus_full_ci'][0]:+.3f}, {r['minus_full_ci'][1]:+.3f}] | "
              f"raw-truncation gain kept {g*100:.0f}% | fallback ch {fb} | Step 33 raw {OLD_33[k]:+.3f}")
print("\nSTEP 38A COMPLETE")

### Step 38B — Summary: raw vs gain-matched truncation (CPU only) ###

In [ ]:
# ============================================================
# STEP 38B — SUMMARY: RAW (STEP 33) vs GAIN-MATCHED TRUNCATION
# ============================================================
res38 = load_checkpoint(CKPT_38); FULL = -1.136
print(f"{'k':>3s} | {'raw (Step 33)':>13s} | {'scale':>8s} | {'fold':>8s} | share of raw gap closed (scale / fold)")
for k in K_LIST:
    raw = OLD_33[k]; s_ = res38[f"scale|k{k}"]["delta_pp"]; f_ = res38[f"fold|k{k}"]["delta_pp"]
    gap = raw - FULL
    cl = lambda v: (raw - v) / gap * 100 if abs(gap) > 1e-9 else float("nan")
    print(f"{k:3d} | {raw:+13.3f} | {s_:+8.3f} | {f_:+8.3f} | {cl(s_):5.0f}% / {cl(f_):5.0f}%")
print(f"Full conv kernel (no truncation): {FULL:+.3f} pp")
res38["summary"] = {"full_pp": FULL, "raw_step33": OLD_33,
                    "scale": {k: res38[f"scale|k{k}"]["delta_pp"] for k in K_LIST},
                    "fold": {k: res38[f"fold|k{k}"]["delta_pp"] for k in K_LIST}}
save_checkpoint(CKPT_38, res38)
print("\nSTEP 38B COMPLETE")

## Step 39 — Distribution shift: ImageNet-C corruptions on a 10K subset ##
Step 35 (384 px) showed that per-block content effects become visible off the training distribution. This step
tests the same idea with four ImageNet-C corruptions (Hendrycks & Dietterich formulas, severity 3), applied to the
cached 224×224 centre crops of the first 10,000 images in the Step 5b order. For every corruption: corrupted
baseline, single-block mean ablation (E2) and the Step 16 shuffled-channel null (E5, same permutations) for all
12 blocks, with accuracy, true-class log-prob drop and KL. The clean comparison uses the Step 9 and Step 16
checkpoints on the same 10,000 images. Saves `step39_imagenet_c` after every block.

### Step 39A — Corruption functions, evaluation path check, null hooks ###

In [ ]:
# ============================================================
# STEP 39A — IMAGENET-C CORRUPTIONS, EVAL PATH CHECK, NULL HOOKS
# ============================================================
import cv2, io, time
from PIL import Image
assert "get_cached_batch" in globals() and "eval_pass_fast" in globals(), "Run Step 24B and the Step 30 setup cell first."
assert "block_means" in globals() and "mean_ablation_hook" in globals(), "Run Step 8 first."

CKPT_39, N_SUB, SEVERITY = "step39_imagenet_c", 10_000, 3
CORRUPTIONS = ["gaussian_noise", "defocus_blur", "contrast", "jpeg_compression"]
SUB_LAB = LAB_CACHE[:N_SUB]
CLEAN_SUB = np.ascontiguousarray(IMG_CACHE[:N_SUB])                 # (10000, 224, 224, 3) uint8

# ---- ImageNet-C corruption functions (Hendrycks & Dietterich), severity 3 ----
def _disk(radius, alias_blur):
    if radius <= 8: L, ksize = np.arange(-8, 9), (3, 3)
    else:           L, ksize = np.arange(-radius, radius + 1), (5, 5)
    X, Y = np.meshgrid(L, L)
    k = np.array((X ** 2 + Y ** 2) <= radius ** 2, dtype=np.float32); k /= k.sum()
    return cv2.GaussianBlur(k, ksize=ksize, sigmaX=alias_blur)

def make_corrupted(name, sev=SEVERITY, chunk=500):
    out = np.empty_like(CLEAN_SUB)
    if name == "gaussian_noise":
        c = [.08, .12, .18, .26, .38][sev - 1]; rng = np.random.RandomState(39)
        for s in range(0, N_SUB, chunk):
            x = CLEAN_SUB[s:s + chunk].astype(np.float32) / 255.
            out[s:s + chunk] = np.uint8(np.clip(x + rng.normal(size=x.shape, scale=c).astype(np.float32), 0, 1) * 255)
    elif name == "contrast":
        c = [.4, .3, .2, .1, .05][sev - 1]
        for s in range(0, N_SUB, chunk):
            x = CLEAN_SUB[s:s + chunk].astype(np.float32) / 255.
            m = x.mean(axis=(1, 2, 3), keepdims=True)
            out[s:s + chunk] = np.uint8(np.clip((x - m) * c + m, 0, 1) * 255)
    elif name == "defocus_blur":
        r, a = [(3, .1), (4, .5), (6, .5), (8, .5), (10, .5)][sev - 1]; k = _disk(r, a)
        for i in range(N_SUB):
            x = CLEAN_SUB[i].astype(np.float32) / 255.
            y = np.stack([cv2.filter2D(x[:, :, d], -1, k) for d in range(3)], -1)
            out[i] = np.uint8(np.clip(y, 0, 1) * 255)
    elif name == "jpeg_compression":
        q = [25, 18, 15, 10, 7][sev - 1]
        for i in range(N_SUB):
            buf = io.BytesIO(); Image.fromarray(CLEAN_SUB[i]).save(buf, "JPEG", quality=q)
            out[i] = np.asarray(Image.open(buf).convert("RGB"), dtype=np.uint8)
    else:
        raise ValueError(name)
    return out

# ---- evaluation on an in-RAM uint8 array (same normalisation as get_cached_batch) ----
def c_eval(arr, base_lp=None, store=False):
    preds, lps, e1, e2 = [], [], [], []
    with torch.no_grad():
        for s in range(0, N_SUB, BATCH_SIZE):
            e = min(s + BATCH_SIZE, N_SUB)
            x = torch.from_numpy(arr[s:e]).to(device, non_blocking=True)
            x = x.permute(0, 3, 1, 2).float().div(255).sub(NORM_MEAN).div(NORM_STD)
            out = model(x)
            logits = (out["logits"] if isinstance(out, dict) else out.logits).float()
            preds.append(logits.argmax(-1).cpu())
            lp = F.log_softmax(logits, -1)
            if store: lps.append(lp.half().cpu())
            if base_lp is not None:
                y = _LAB_GPU[s:e].view(-1, 1); b = base_lp[s:e].to(device).float()
                e1.append((b.gather(1, y) - lp.gather(1, y)).squeeze(1).cpu())
                e2.append((b.exp() * (b - lp)).sum(-1).cpu())
    preds = torch.cat(preds).numpy()
    if store: return preds, torch.cat(lps)
    if base_lp is not None: return preds, torch.cat(e1).numpy(), torch.cat(e2).numpy()
    return preds

# ---- check: this path on clean images reproduces the Step 6 baseline exactly ----
clear_all_hooks(); model.eval()
t0 = time.time(); _cp = c_eval(CLEAN_SUB); C39_PASS_MIN = (time.time() - t0) / 60
assert (_cp == baseline_preds_np[:N_SUB]).all(), "Clean path differs from the baseline - send me this output."
print(f"Clean 10K: {(_cp == SUB_LAB).mean()*100:.3f}% (identical to baseline) | one 10K pass = {C39_PASS_MIN:.2f} min")

# ---- Step 16 shuffled-channel null: same seed, same order, same permutations ----
torch.manual_seed(42)
c_perms = {name: torch.randperm(block_means[name].shape[-1]) for name, _, _ in mixer_blocks}
_c_null = {"name": None}
def _c_null_hook(name):
    mv, perm = block_means[name], c_perms[name]
    def f(module, inp, out):
        if _c_null["name"] != name: return out
        is_t = isinstance(out, tuple); t = out[0] if is_t else out
        delta = mv.to(device=t.device, dtype=t.dtype).expand_as(t) - t
        rep = t + delta[..., perm.to(t.device)]
        return (rep,) + out[1:] if is_t else rep
    return f

def run_block(arr, name, blk, kind, base_lp):
    clear_all_hooks()
    if kind == "real":
        hs = [blk.mixer.register_forward_hook(mean_ablation_hook(block_means[name]))]
    else:
        hs = [blk.mixer.register_forward_hook(_c_null_hook(name))]; _c_null["name"] = name
    try:
        return c_eval(arr, base_lp=base_lp)
    finally:
        _c_null["name"] = None
        for h in hs: h.remove()

t0 = time.time(); _tmp = make_corrupted("defocus_blur"); gen_min = (time.time() - t0) / 60; del _tmp
print(f"Corruption generation (defocus, slowest kind) = {gen_min:.1f} min")
print(f"Estimated time for Step 39B: ~{len(CORRUPTIONS) * (gen_min + 25 * C39_PASS_MIN):.0f} min "
      f"({len(CORRUPTIONS)} corruptions × 25 passes of 10K images)")
print("\nSTEP 39A COMPLETE")

### Step 39B — Per corruption: baseline, real ablation and shuffled null for all 12 blocks ###

In [ ]:
# ============================================================
# STEP 39B — CORRUPTED BASELINE + E2 REAL + E5 NULL, ALL 12 BLOCKS
# ============================================================
try:
    res39 = load_checkpoint(CKPT_39); print("Resuming; saved:", {k: len(v["blocks"]) for k, v in res39.items() if "blocks" in v})
except Exception:
    res39 = {}

for cname in CORRUPTIONS:
    R = res39.setdefault(cname, {"blocks": {}})
    if "baseline" in R and len(R["blocks"]) == len(mixer_blocks):
        print(f"{cname}: already complete"); continue
    t0 = time.time()
    arr = make_corrupted(cname)
    clear_all_hooks()
    base_preds, base_lp = c_eval(arr, store=True)
    base_c = base_preds == SUB_LAB
    if "baseline" in R:
        assert abs(R["baseline"]["accuracy"] - base_c.mean()) < 1e-12, f"{cname}: baseline differs from the saved one!"
    else:
        R["baseline"] = {"accuracy": float(base_c.mean()), "correct_packed": np.packbits(base_c), "severity": SEVERITY}
        save_checkpoint(CKPT_39, res39)
    print(f"\n=== {cname} (severity {SEVERITY}): baseline {base_c.mean()*100:.2f}% "
          f"(clean {(baseline_preds_np[:N_SUB] == SUB_LAB).mean()*100:.2f}%) ===")
    for name, mtype, blk in mixer_blocks:
        if name in R["blocks"]: continue
        tb = time.time()
        pr, lr, kr = run_block(arr, name, blk, "real", base_lp)
        pn, ln, kn = run_block(arr, name, blk, "null", base_lp)
        cr, cn = pr == SUB_LAB, pn == SUB_LAB
        b8 = base_c.astype(np.int8)
        dr, dr_ci = mean_ci((cr.astype(np.int8) - b8) * 100.0)
        dn, dn_ci = mean_ci((cn.astype(np.int8) - b8) * 100.0)
        d, d_ci = mean_ci((cr.astype(np.int8) - cn.astype(np.int8)) * 100.0)
        l, l_ci = mean_ci(lr - ln)
        R["blocks"][name] = {"type": mtype,
            "real_pp": dr, "real_ci": dr_ci, "null_pp": dn, "null_ci": dn_ci,
            "real_minus_null_pp": d, "real_minus_null_ci": d_ci,
            "real_logprob_drop": float(lr.mean()), "null_logprob_drop": float(ln.mean()),
            "real_minus_null_logprob": l, "real_minus_null_logprob_ci": l_ci,
            "real_kl": float(kr.mean()), "null_kl": float(kn.mean()),
            "real_correct_packed": np.packbits(cr), "null_correct_packed": np.packbits(cn),
            "real_lpd_f16": lr.astype(np.float16), "null_lpd_f16": ln.astype(np.float16)}
        save_checkpoint(CKPT_39, res39)
        print(f"  {name:5s} {'Mamba' if mtype == 'MambaVisionMixer' else 'Attn ':5s} ({(time.time()-tb)/60:.1f} min) "
              f"real {dr:+.2f} | null {dn:+.2f} | real − null {d:+.2f} [{d_ci[0]:+.2f}, {d_ci[1]:+.2f}] | "
              f"log-prob real − null {l:+.4f} [{l_ci[0]:+.4f}, {l_ci[1]:+.4f}]")
    del arr, base_lp
    print(f"{cname} done in {(time.time()-t0)/60:.1f} min")
print("\nSTEP 39B COMPLETE")

### Step 39C — Summary: clean vs corrupted (CPU only) ###

In [ ]:
# ============================================================
# STEP 39C — SUMMARY: CLEAN (STEP 9 / STEP 16 ON THE SAME 10K) vs EACH CORRUPTION
# ============================================================
from scipy.stats import spearmanr, norm
res39 = load_checkpoint(CKPT_39)
names_ = [n for n, _, _ in mixer_blocks]
types_ = {n: ("M" if t == "MambaVisionMixer" else "A") for n, t, _ in mixer_blocks}
upk = lambda p: np.unpackbits(p)[:N_SUB].astype(bool)

def holm(pv):
    pv = np.asarray(pv); order = np.argsort(pv); out = np.empty(len(pv)); run = 0
    for r_, i in enumerate(order):
        run = max(run, min(1.0, (len(pv) - r_) * pv[i])); out[i] = run
    return out

def analyse(label, base_c, real_c, null_c):
    real = np.array([(real_c[n].astype(float) - base_c).mean() * 100 for n in names_])
    null = np.array([(null_c[n].astype(float) - base_c).mean() * 100 for n in names_])
    pv = []
    for n in names_:
        d = (real_c[n].astype(float) - null_c[n].astype(float)) * 100
        se = d.std(ddof=1) / np.sqrt(len(d)); pv.append(2 * norm.sf(abs(d.mean()) / se) if se > 0 else 1.0)
    h = holm(pv)
    per_img = np.mean([(null_c[n].astype(float) - real_c[n].astype(float)) * 100 for n in names_], axis=0)
    rng = np.random.RandomState(42); bm = np.array([per_img[rng.randint(0, N_SUB, N_SUB)].mean() for _ in range(2000)])
    rho = spearmanr(np.abs(real), np.abs(null))
    mam = [i for i, n in enumerate(names_) if types_[n] == "M"]; att = [i for i, n in enumerate(names_) if types_[n] == "A"]
    rw = [n for i, n in enumerate(names_) if h[i] < 0.05 and real[i] < null[i]]
    nw = [n for i, n in enumerate(names_) if h[i] < 0.05 and real[i] > null[i]]
    print(f"{label:17s} base {base_c.mean()*100:6.2f}% | |real| M {np.abs(real[mam]).mean():.2f} A {np.abs(real[att]).mean():.2f} | "
          f"ρ {rho.correlation:+.2f} (p {rho.pvalue:.3f}) | pooled content {per_img.mean():+.3f} "
          f"[{np.percentile(bm, 2.5):+.3f}, {np.percentile(bm, 97.5):+.3f}] | real worse {rw} | null worse {nw}")
    return {"baseline": float(base_c.mean()), "real_pp": dict(zip(names_, real.tolist())), "null_pp": dict(zip(names_, null.tolist())),
            "holm_p": dict(zip(names_, h.tolist())), "spearman": [float(rho.correlation), float(rho.pvalue)],
            "pooled_content_pp": float(per_img.mean()), "pooled_ci": [float(np.percentile(bm, 2.5)), float(np.percentile(bm, 97.5))],
            "real_worse_holm": rw, "null_worse_holm": nw}

summary = {}
_sp = load_checkpoint("step9_ablated_preds")
_n16 = {r["block"]: r for r in load_checkpoint("shuffled_null_50k")["results"]}
clean_base = (baseline_preds_np[:N_SUB] == SUB_LAB).astype(float)
summary["clean"] = analyse("clean (same 10K)", clean_base,
                           {n: np.asarray(_sp[n]).astype(np.int64)[:N_SUB] == SUB_LAB for n in names_},
                           {n: unpack(_n16[n]["correct_packed"])[:N_SUB] for n in names_})
for cname in CORRUPTIONS:
    R = res39[cname]
    summary[cname] = analyse(cname, upk(R["baseline"]["correct_packed"]).astype(float),
                             {n: upk(R["blocks"][n]["real_correct_packed"]) for n in names_},
                             {n: upk(R["blocks"][n]["null_correct_packed"]) for n in names_})
    lp = np.array([R["blocks"][n]["real_minus_null_logprob"] for n in names_])
    print(f"{'':17s} log-prob real − null: mean {lp.mean():+.4f} | blocks with CI > 0: "
          f"{[n for n in names_ if R['blocks'][n]['real_minus_null_logprob_ci'][0] > 0]}")
res39["summary"] = summary
save_checkpoint(CKPT_39, res39)
print("\nSTEP 39C COMPLETE")

### Step 39D — Clean 10K: real vs null log-prob (reference for the corruption runs) ###
Same 10,000 images and the same hooks as Step 39B, without corruption. Checks that the real ablation reproduces
Step 9 and the null reproduces Step 16 on these images, then compares the real − null log-prob of each block
with the four corruptions. Saved under "clean" in `step39_imagenet_c`.

In [ ]:
# ============================================================
# STEP 39D — CLEAN 10K: REAL vs NULL LOG-PROB + CHECK AGAINST STEP 9 / STEP 16
# ============================================================
assert "c_eval" in globals() and "run_block" in globals(), "Run Step 39A first."
res39 = load_checkpoint(CKPT_39)
R = res39.setdefault("clean", {"blocks": {}})
_sp9 = load_checkpoint("step9_ablated_preds")
_n16 = {r["block"]: r for r in load_checkpoint("shuffled_null_50k")["results"]}

clear_all_hooks()
base_preds, base_lp = c_eval(CLEAN_SUB, store=True)
base_c = base_preds == SUB_LAB
assert (base_preds == baseline_preds_np[:N_SUB]).all()
R["baseline"] = {"accuracy": float(base_c.mean()), "correct_packed": np.packbits(base_c)}
print(f"Clean 10K baseline: {base_c.mean()*100:.3f}%")

for name, mtype, blk in mixer_blocks:
    if name in R["blocks"]: continue
    tb = time.time()
    pr, lr, kr = run_block(CLEAN_SUB, name, blk, "real", base_lp)
    pn, ln, kn = run_block(CLEAN_SUB, name, blk, "null", base_lp)
    cr, cn = pr == SUB_LAB, pn == SUB_LAB
    agree9  = (pr == np.asarray(_sp9[name]).astype(np.int64)[:N_SUB]).mean()
    agree16 = (cn == unpack(_n16[name]["correct_packed"])[:N_SUB]).mean()
    b8 = base_c.astype(np.int8)
    dr, dr_ci = mean_ci((cr.astype(np.int8) - b8) * 100.0)
    dn, dn_ci = mean_ci((cn.astype(np.int8) - b8) * 100.0)
    d, d_ci = mean_ci((cr.astype(np.int8) - cn.astype(np.int8)) * 100.0)
    l, l_ci = mean_ci(lr - ln)
    R["blocks"][name] = {"type": mtype,
        "real_pp": dr, "real_ci": dr_ci, "null_pp": dn, "null_ci": dn_ci,
        "real_minus_null_pp": d, "real_minus_null_ci": d_ci,
        "real_logprob_drop": float(lr.mean()), "null_logprob_drop": float(ln.mean()),
        "real_minus_null_logprob": l, "real_minus_null_logprob_ci": l_ci,
        "real_kl": float(kr.mean()), "null_kl": float(kn.mean()),
        "agree_step9_preds": float(agree9), "agree_step16_correct": float(agree16),
        "real_correct_packed": np.packbits(cr), "null_correct_packed": np.packbits(cn),
        "real_lpd_f16": lr.astype(np.float16), "null_lpd_f16": ln.astype(np.float16)}
    save_checkpoint(CKPT_39, res39)
    print(f"  {name:5s} {'Mamba' if mtype == 'MambaVisionMixer' else 'Attn ':5s} ({(time.time()-tb)/60:.1f} min) "
          f"real {dr:+.2f} | null {dn:+.2f} | log-prob real − null {l:+.4f} [{l_ci[0]:+.4f}, {l_ci[1]:+.4f}] | "
          f"KL real {kr.mean():.4f} null {kn.mean():.4f} | agree Step 9 {agree9*100:.2f}% Step 16 {agree16*100:.2f}%")
del base_lp

# ---- comparison table: real − null log-prob, clean vs each corruption ----
conds = ["clean"] + CORRUPTIONS
print(f"\n{'block':6s}" + "".join(f"{c[:14]:>16s}" for c in conds))
for name, mtype, _ in mixer_blocks:
    row = f"{name:5s}{'M' if mtype == 'MambaVisionMixer' else 'A'}"
    for c in conds:
        b = res39[c]["blocks"][name]; lo, hi = b["real_minus_null_logprob_ci"]
        row += f"{b['real_minus_null_logprob']:+11.4f}{'*' if lo > 0 else ('-' if hi < 0 else ' ')}    "
    print(row)
print("* = CI above 0 (real lowers the true-class log-prob more than the null);  - = CI below 0")
print("\nSTEP 39D COMPLETE")

## Step 40 — 384 px: shuffled null with several permutations per block (10K subset) ##
Step 35 found many per-block real-vs-null differences at 384 px with global windows, using one permutation per
block. This step repeats the real ablation and four shuffled-null permutations (permutation 0 = Step 16, 1–3 =
Step 27 seed scheme) for all 12 blocks on the first 10,000 images (Step 5b order), so that the per-block
contrasts can be judged against the spread between permutations. Saves `step40_384_multiperm` after every pass.

### Step 40A — 384 cache, global windows, baseline check against Step 35 ###

In [ ]:
# ============================================================
# STEP 40A — 384 PX: CACHE, GLOBAL WINDOWS, BASELINE CHECK
# ============================================================
import re, os, gc, shutil, time
from timm.data.transforms_factory import create_transform
from torchvision import transforms as T
from torch.utils.data import Dataset as _TorchDataset, DataLoader
assert "block_means" in globals() and "mean_ci" in globals() and "CACHE_DIR" in globals(), "Run the setup cells first."

CKPT_40, N40, RES = "step40_384_multiperm", 10_000, 384
N_PERM = 4                                            # perm 0 = Step 16, perms 1..3 = Step 27 seed scheme
BLOCKS_40 = [n for n, _, _ in mixer_blocks]           # all 12 blocks
for _v in ["CLEAN_SUB", "IMG384"]:
    if _v in globals(): del globals()[_v]
gc.collect(); torch.cuda.empty_cache()

# ---- window sizes: full grid at 384 for Levels 2 and 3 ----
_levels = {}
for _n, _m in model.named_modules():
    _mm = re.search(r"levels\.(\d+)$", _n)
    if _mm and hasattr(_m, "window_size"):
        _levels[int(_mm.group(1))] = _m
if "ORIG_WS40" not in globals():
    ORIG_WS40 = {i: m.window_size for i, m in _levels.items()}
def set_ws(kind):
    for i, m in _levels.items():
        m.window_size = RES // (4 * 2 ** i) if (kind == "global" and i >= 2) else ORIG_WS40[i]
set_ws("default")
print("Default window sizes:", ORIG_WS40, "| global at 384:", {i: RES // (4 * 2 ** i) for i in _levels if i >= 2})

# ---- 384 preprocessing and uint8 cache (first 10K images, Step 5b order) ----
t384 = create_transform(input_size=(3, RES, RES), is_training=False, mean=model.config.mean, std=model.config.std,
                        crop_mode=model.config.crop_mode, crop_pct=model.config.crop_pct)
print("384 preprocessing:", t384)
_tf = list(t384.transforms); _k = next(i for i, t in enumerate(_tf) if "ToTensor" in type(t).__name__)
PRE384 = T.Compose(_tf[:_k])
M384 = torch.as_tensor(_tf[_k + 1].mean, dtype=torch.float32, device=device).view(1, 3, 1, 1)
S384 = torch.as_tensor(_tf[_k + 1].std,  dtype=torch.float32, device=device).view(1, 3, 1, 1)
C384 = os.path.join(CACHE_DIR, f"mv_cache_384_u8_{N40}.npy")
need_gb = N40 * RES * RES * 3 / 1e9
if not os.path.exists(C384 + ".done"):
    free_gb = shutil.disk_usage(CACHE_DIR).free / 1e9
    assert free_gb > need_gb + 1, f"Need {need_gb:.1f} GB free in {CACHE_DIR}, have {free_gb:.1f} GB."
    class _Src384(_TorchDataset):
        def __len__(self): return N40
        def __getitem__(self, pos):
            ex = dataset[eval_indices[pos]]
            return np.asarray(PRE384(ex["image"].convert("RGB")), dtype=np.uint8).copy(), ex["label"]
    t0 = time.time()
    mm = np.lib.format.open_memmap(C384, mode="w+", dtype=np.uint8, shape=(N40, RES, RES, 3)); pos = 0
    for x, _ in DataLoader(_Src384(), batch_size=128, num_workers=os.cpu_count() or 2):
        mm[pos:pos + len(x)] = x.numpy(); pos += len(x)
    mm.flush(); del mm; open(C384 + ".done", "w").write("ok")
    print(f"384 cache built in {(time.time()-t0)/60:.1f} min ({need_gb:.1f} GB)")
IMG384 = np.load(C384, mmap_mode="r")
def get384(s, e):
    x = torch.from_numpy(np.ascontiguousarray(IMG384[s:e])).to(device, non_blocking=True)
    return x.permute(0, 3, 1, 2).float().div(255).sub(M384).div(S384)
_ref, _ = prepare_batch(dataset, eval_indices[:16], t384, device)
assert (get384(0, 16) - _ref).abs().max().item() < 1e-5, "384 cache differs from prepare_batch."
SUB40_LAB = LAB_CACHE[:N40]

def e384(base_lp=None, store=False):
    preds, lps, e1, e2 = [], [], [], []
    with torch.no_grad():
        for s in range(0, N40, BATCH_SIZE):
            e = min(s + BATCH_SIZE, N40)
            out = model(get384(s, e))
            logits = (out["logits"] if isinstance(out, dict) else out.logits).float()
            preds.append(logits.argmax(-1).cpu()); lp = F.log_softmax(logits, -1)
            if store: lps.append(lp.half().cpu())
            if base_lp is not None:
                y = _LAB_GPU[s:e].view(-1, 1); b = base_lp[s:e].to(device).float()
                e1.append((b.gather(1, y) - lp.gather(1, y)).squeeze(1).cpu())
                e2.append((b.exp() * (b - lp)).sum(-1).cpu())
    preds = torch.cat(preds).numpy()
    if store: return preds, torch.cat(lps)
    if base_lp is not None: return preds, torch.cat(e1).numpy(), torch.cat(e2).numpy()
    return preds

# ---- baseline with global windows + check against Step 35 ----
clear_all_hooks(); model.eval(); set_ws("global")
t0 = time.time(); b40_preds, BASE40_LP = e384(store=True); P40_MIN = (time.time() - t0) / 60
B40_CORRECT = b40_preds == SUB40_LAB
print(f"384 global-window baseline, first 10K: {B40_CORRECT.mean()*100:.3f}% | one 10K pass = {P40_MIN:.2f} min")
ck35 = load_checkpoint("step35_highres_384")
print("Step 35 checkpoint keys:", list(ck35))
bg = ck35.get("baseline_global", {})
if isinstance(bg, dict) and "correct_packed" in bg:
    ag = (unpack(bg["correct_packed"])[:N40] == B40_CORRECT).mean()
    print(f"Agreement with the Step 35 global baseline on these 10K images: {ag*100:.3f}%")
    assert ag > 0.999, "Preprocessing/windows differ from Step 35 - stop here and send me the Step 35A code."
else:
    print("Step 35 baseline has no per-image record - compare by hand: Step 35 50K global baseline 81.892%.")

# ---- null permutations ----
torch.manual_seed(42)
_P0 = {name: torch.randperm(block_means[name].shape[-1]) for name, _, _ in mixer_blocks}
def perm_for(bi, name, j):
    if j == 0: return _P0[name]
    return torch.randperm(block_means[name].shape[-1], generator=torch.Generator().manual_seed(1000 + 100 * bi + j))
def _null_hook40(name, perm):
    mv = block_means[name]
    def f(module, inp, out):
        is_t = isinstance(out, tuple); t = out[0] if is_t else out
        delta = mv.to(device=t.device, dtype=t.dtype).expand_as(t) - t
        rep = t + delta[..., perm.to(t.device)]
        return (rep,) + out[1:] if is_t else rep
    return f
def run40(name, blk, perm=None):
    clear_all_hooks()
    hook = mean_ablation_hook(block_means[name]) if perm is None else _null_hook40(name, perm)
    h = blk.mixer.register_forward_hook(hook)
    try: return e384(base_lp=BASE40_LP)
    finally: h.remove()

try:
    res40 = load_checkpoint(CKPT_40)
    assert abs(res40["baseline"]["accuracy"] - B40_CORRECT.mean()) < 1e-12, "Baseline differs from the saved one!"
    print("Resuming; done:", {n: list(v) for n, v in res40["blocks"].items()})
except Exception as e:
    if "differs" in str(e): raise
    res40 = {"baseline": {"accuracy": float(B40_CORRECT.mean()), "correct_packed": np.packbits(B40_CORRECT),
                          "res": RES, "windows": "global"}, "blocks": {}}
    save_checkpoint(CKPT_40, res40)
n_left = sum((0 if "real" in res40["blocks"].get(n, {}) else 1) +
             sum(f"null{j}" not in res40["blocks"].get(n, {}) for j in range(N_PERM)) for n in BLOCKS_40)
print(f"Passes left: {n_left} → estimated {n_left * P40_MIN / 60:.1f} h")
print("\nSTEP 40A COMPLETE")

### Step 40B — Real ablation and four null permutations per block ###

In [ ]:
# ============================================================
# STEP 40B — 384 PX: REAL + N_PERM NULLS PER BLOCK (resumable)
# ============================================================
set_ws("global")
try:
    for bi, (name, mtype, blk) in enumerate(mixer_blocks):
        if name not in BLOCKS_40: continue
        E = res40["blocks"].setdefault(name, {"type": mtype})
        tb = time.time()
        if "real" not in E:
            p, lpd, kl = run40(name, blk)
            E["real"] = {"correct_packed": np.packbits(p == SUB40_LAB), "lpd_f16": lpd.astype(np.float16), "kl": float(kl.mean())}
            save_checkpoint(CKPT_40, res40)
        for j in range(N_PERM):
            if f"null{j}" in E: continue
            p, lpd, kl = run40(name, blk, perm_for(bi, name, j))
            E[f"null{j}"] = {"correct_packed": np.packbits(p == SUB40_LAB), "lpd_f16": lpd.astype(np.float16), "kl": float(kl.mean())}
            save_checkpoint(CKPT_40, res40)
        upk = lambda q: np.unpackbits(q)[:N40].astype(bool)
        base = B40_CORRECT.astype(float)
        rc = upk(E["real"]["correct_packed"]).astype(float)
        nd = [(upk(E[f"null{j}"]["correct_packed"]).astype(float) - base).mean() * 100 for j in range(N_PERM)]
        nmean = np.mean([upk(E[f"null{j}"]["correct_packed"]).astype(float) for j in range(N_PERM)], axis=0)
        d, d_ci = mean_ci((rc - nmean) * 100.0)
        print(f"{name:5s} {'Mamba' if mtype == 'MambaVisionMixer' else 'Attn ':5s} ({(time.time()-tb)/60:.1f} min) "
              f"real {(rc - base).mean()*100:+.2f} | nulls {' '.join(f'{x:+.2f}' for x in nd)} (SD {np.std(nd, ddof=1):.2f}) | "
              f"real − mean null {d:+.2f} [{d_ci[0]:+.2f}, {d_ci[1]:+.2f}]")
finally:
    clear_all_hooks(); set_ws("default")
print("\nSTEP 40B COMPLETE")

In [ ]:
# ---- minimal setup for Step 40C (no model, no GPU) ----
import numpy as np
CKPT_40, N40, N_PERM = "step40_384_multiperm", 10_000, 4
BLOCKS_40 = ["L2B0", "L2B1", "L2B2", "L2B3", "L2B4", "L2B5", "L2B6", "L2B7", "L3B0", "L3B1", "L3B2", "L3B3"]

def mean_ci(x, n_boot=2000, seed=42):
    x = np.asarray(x, dtype=np.float64); n = len(x); rng = np.random.RandomState(seed)
    boot = np.array([x[rng.randint(0, n, n)].mean() for _ in range(n_boot)])
    return float(x.mean()), [float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))]

_r = load_checkpoint(CKPT_40)
missing = [f"{n}:{k}" for n in BLOCKS_40 for k in ["real"] + [f"null{j}" for j in range(N_PERM)]
           if k not in _r["blocks"].get(n, {})]
print("Step 40B complete." if not missing else f"Step 40B NOT complete — {len(missing)} passes missing: {missing[:6]} ...")

### Step 40C — Summary: per-block contrasts against the permutation spread (CPU only) ###

In [ ]:
# ============================================================
# STEP 40C — 384 PX MULTI-PERMUTATION SUMMARY
# ============================================================
from scipy.stats import spearmanr, norm
res40 = load_checkpoint(CKPT_40)
upk = lambda q: np.unpackbits(q)[:N40].astype(bool)
base = upk(res40["baseline"]["correct_packed"]).astype(float)
names_ = [n for n in BLOCKS_40 if n in res40["blocks"]]
P = N_PERM
real = {n: upk(res40["blocks"][n]["real"]["correct_packed"]).astype(float) for n in names_}
nul = {n: [upk(res40["blocks"][n][f"null{j}"]["correct_packed"]).astype(float) for j in range(P)] for n in names_}

rows, pv = [], []
for n in names_:
    r_pp = (real[n] - base).mean() * 100
    n_pp = [(x - base).mean() * 100 for x in nul[n]]
    nm = np.mean(nul[n], axis=0)
    dd = (real[n] - nm) * 100
    m, ci = mean_ci(dd); se = dd.std(ddof=1) / np.sqrt(len(dd))
    pv.append(2 * norm.sf(abs(m) / se) if se > 0 else 1.0)
    per_perm = [(real[n] - x).mean() * 100 for x in nul[n]]
    lp_r = res40["blocks"][n]["real"]["lpd_f16"].astype(np.float64)
    lp_n = np.mean([res40["blocks"][n][f"null{j}"]["lpd_f16"].astype(np.float64) for j in range(P)], axis=0)
    l, l_ci = mean_ci(lp_r - lp_n)
    rows.append((n, r_pp, n_pp, m, ci, per_perm, l, l_ci))
order = np.argsort(pv); holm = np.empty(len(pv)); run = 0
for r_, i in enumerate(order):
    run = max(run, min(1.0, (len(pv) - r_) * pv[i])); holm[i] = run

print(f"384 px, global windows, first {N40:,} images | baseline {base.mean()*100:.2f}% | {P} null permutations per block\n")
print(f"{'block':5s} {'real':>7s} {'null mean':>9s} {'null SD':>7s} {'real−null [95% CI]':>24s} {'Holm p':>8s} "
      f"{'same sign in perms':>18s} {'log-prob real−null':>20s}")
for (n, r_pp, n_pp, m, ci, pp, l, l_ci), h in zip(rows, holm):
    same = sum(np.sign(x) == np.sign(m) for x in pp)
    print(f"{n:5s} {r_pp:+7.2f} {np.mean(n_pp):+9.2f} {np.std(n_pp, ddof=1):7.2f} {m:+8.2f} [{ci[0]:+.2f}, {ci[1]:+.2f}] "
          f"{h:8.4f} {same:>12d}/{P}     {l:+.4f} [{l_ci[0]:+.4f}, {l_ci[1]:+.4f}]")

rw = [r[0] for r, h in zip(rows, holm) if h < 0.05 and r[3] < 0]
nw = [r[0] for r, h in zip(rows, holm) if h < 0.05 and r[3] > 0]
print(f"\nHolm-significant, real worse: {rw}")
print(f"Holm-significant, null worse: {nw}")
print("Step 35 (one permutation, 50K): real worse L2B0 L2B1 L2B2 L2B4 L2B5 L2B6 L2B7 L3B2 L3B3; null worse L3B0 L3B1")

abs_real = np.array([abs(r[1]) for r in rows])
for j in range(P):
    rho = spearmanr(abs_real, [abs(r[2][j]) for r in rows])
    print(f"Spearman |real| vs |null perm {j}|: {rho.correlation:+.2f} (p {rho.pvalue:.3f})")
rho_m = spearmanr(abs_real, [abs(np.mean(r[2])) for r in rows])
print(f"Spearman |real| vs |mean null|: {rho_m.correlation:+.2f} (p {rho_m.pvalue:.3f})   [Step 35: 0.50]")

per_img = np.mean([(np.mean(nul[n], axis=0) - real[n]) * 100 for n in names_], axis=0)
rng = np.random.RandomState(42); bm = np.array([per_img[rng.randint(0, N40, N40)].mean() for _ in range(2000)])
pooled_j = [np.mean([(nul[n][j] - real[n]).mean() * 100 for n in names_]) for j in range(P)]
print(f"Pooled content effect (real worse = +): {per_img.mean():+.3f} pp [{np.percentile(bm, 2.5):+.3f}, "
      f"{np.percentile(bm, 97.5):+.3f}] | per permutation: {' '.join(f'{x:+.3f}' for x in pooled_j)}   [Step 35: +0.252]")

res40["summary"] = {"holm_p": dict(zip(names_, holm.tolist())), "real_worse": rw, "null_worse": nw,
                    "spearman_mean_null": [float(rho_m.correlation), float(rho_m.pvalue)],
                    "pooled_content_pp": float(per_img.mean()),
                    "pooled_ci": [float(np.percentile(bm, 2.5)), float(np.percentile(bm, 97.5))],
                    "pooled_per_perm": [float(x) for x in pooled_j]}
save_checkpoint(CKPT_40, res40)
print("\nSTEP 40C COMPLETE")

In [ ]:
# Check: Step 40 real / perm-0 vs Step 35 per-block records (first 10K images)
ck35 = load_checkpoint("step35_highres_384"); res40 = load_checkpoint(CKPT_40)
upk = lambda q: np.unpackbits(q)[:N40].astype(bool)
b35 = ck35["blocks"]
print("Step 35 'blocks' type:", type(b35).__name__, "| first keys:", list(b35)[:4] if isinstance(b35, dict) else len(b35))
_first = b35[list(b35)[0]] if isinstance(b35, dict) else b35[0]
print("Structure of one Step 35 block record:", {k: type(v).__name__ for k, v in _first.items()} if isinstance(_first, dict) else type(_first).__name__)
def _find_packed(obj, path=""):
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k == "correct_packed": yield path or "/", v
            elif isinstance(v, (dict, list)): yield from _find_packed(v, f"{path}/{k}")
    elif isinstance(obj, list):
        for i, v in enumerate(obj): yield from _find_packed(v, f"{path}[{i}]")
items = b35.items() if isinstance(b35, dict) else ((r.get("block"), r) for r in b35)
for name, rec in items:
    if name not in res40["blocks"]: continue
    r40 = upk(res40["blocks"][name]["real"]["correct_packed"]); n40 = upk(res40["blocks"][name]["null0"]["correct_packed"])
    for path, pk in _find_packed(rec):
        c35 = np.unpackbits(pk)[:N40].astype(bool)
        print(f"{name} Step35{path}: agree with Step40 real {(c35 == r40).mean()*100:.2f}% | with Step40 null0 {(c35 == n40).mean()*100:.2f}%")

## Step 40-R — 384 px multi-permutation null with the Step 35 reference means ##
Step 40 used the 224-px Step 8 means; Step 35 used means recomputed at 384 px with global windows (first 256
cached images). This step repeats Step 40 with the Step 35 means: real ablation (checked against Step 35) and
three new shuffled-null permutations per block on the first 10,000 images; the Step 35 null is the fourth
permutation. Needs Step 40A in the session. Saves `step40R_384_multiperm_384means` after every pass.

### Step 40R-A — Step 35 means, real ablation check, three extra permutations ###

In [ ]:
# ============================================================
# STEP 40R-A — 384 PX WITH STEP 35 MEANS: REAL + 3 NEW NULLS PER BLOCK (resumable)
# ============================================================
assert "e384" in globals() and "set_ws" in globals() and "BASE40_LP" in globals(), "Run Step 40A first."
CKPT_40R = "step40R_384_multiperm_384means"
NEW_PERMS_40R = [1, 2, 3]
ck35 = load_checkpoint("step35_highres_384")
upk = lambda q: np.unpackbits(q)[:N40].astype(bool)

# ---- Step 35 reference means: 384, global windows, first 256 cached images, batches of 32 ----
clear_all_hooks(); set_ws("global")
_cap = {name: [] for name, _, _ in mixer_blocks}
_hs = [blk.mixer.register_forward_hook(lambda m, i, o, name=name: _cap[name].append((o[0] if isinstance(o, tuple) else o).detach()))
       for name, _, blk in mixer_blocks]
with torch.no_grad():
    for s in range(0, 256, 32): model(get384(s, s + 32))
for h in _hs: h.remove()
H40_MEANS = {name: torch.cat(v).mean(dim=(0, 1), keepdim=True) for name, v in _cap.items()}
del _cap
print("Step 35 means rebuilt for", len(H40_MEANS), "blocks")

def _null_hook40r(name, perm):
    mv = H40_MEANS[name]
    def f(module, inp, out):
        is_t = isinstance(out, tuple); t = out[0] if is_t else out
        delta = mv.to(device=t.device, dtype=t.dtype).expand_as(t) - t
        rep = t + delta[..., perm.to(t.device)]
        return (rep,) + out[1:] if is_t else rep
    return f
def run40r(name, blk, perm=None):
    clear_all_hooks()
    h = blk.mixer.register_forward_hook(mean_ablation_hook(H40_MEANS[name]) if perm is None else _null_hook40r(name, perm))
    try: return e384(base_lp=BASE40_LP)
    finally: h.remove()
def perm40r(bi, name, j):
    return torch.randperm(H40_MEANS[name].shape[-1], generator=torch.Generator().manual_seed(1000 + 100 * bi + j))

try:
    res40r = load_checkpoint(CKPT_40R); print("Resuming; done:", {n: list(v) for n, v in res40r["blocks"].items()})
except Exception:
    res40r = {"baseline_accuracy": float(B40_CORRECT.mean()), "blocks": {}}

try:
    for bi, (name, mtype, blk) in enumerate(mixer_blocks):
        E = res40r["blocks"].setdefault(name, {"type": mtype})
        tb = time.time()
        if "real" not in E:
            p, lpd, kl = run40r(name, blk)
            c = p == SUB40_LAB
            agree = (c == upk(ck35["blocks"][name]["real"]["correct_packed"])).mean()
            if bi == 0:
                assert agree > 0.999, f"Real ablation agrees only {agree*100:.2f}% with Step 35 - stop and send me this output."
            E["real"] = {"correct_packed": np.packbits(c), "lpd_f16": lpd.astype(np.float16), "kl": float(kl.mean()),
                         "agree_step35": float(agree)}
            save_checkpoint(CKPT_40R, res40r)
        for j in NEW_PERMS_40R:
            if f"null{j}" in E: continue
            p, lpd, kl = run40r(name, blk, perm40r(bi, name, j))
            E[f"null{j}"] = {"correct_packed": np.packbits(p == SUB40_LAB), "lpd_f16": lpd.astype(np.float16), "kl": float(kl.mean())}
            save_checkpoint(CKPT_40R, res40r)
        base = B40_CORRECT.astype(float)
        rc = upk(E["real"]["correct_packed"]).astype(float)
        nl = [upk(ck35["blocks"][name]["null"]["correct_packed"]).astype(float)] + \
             [upk(E[f"null{j}"]["correct_packed"]).astype(float) for j in NEW_PERMS_40R]
        nd = [(x - base).mean() * 100 for x in nl]
        d, d_ci = mean_ci((rc - np.mean(nl, axis=0)) * 100.0)
        print(f"{name:5s} ({(time.time()-tb)/60:.1f} min) agree Step 35 real {E['real']['agree_step35']*100:.2f}% | "
              f"real {(rc - base).mean()*100:+.2f} | nulls {' '.join(f'{x:+.2f}' for x in nd)} (SD {np.std(nd, ddof=1):.2f}) | "
              f"real − mean null {d:+.2f} [{d_ci[0]:+.2f}, {d_ci[1]:+.2f}]")
finally:
    clear_all_hooks(); set_ws("default")
print("\nSTEP 40R-A COMPLETE")

### Step 40R-B — Summary with the Step 35 means (CPU only) ###

In [ ]:
# ============================================================
# STEP 40R-B — SUMMARY: STEP 35 MEANS, 4 NULL PERMUTATIONS, FIRST 10K
# ============================================================
from scipy.stats import spearmanr, norm
res40r = load_checkpoint(CKPT_40R); ck35 = load_checkpoint("step35_highres_384")
upk = lambda q: np.unpackbits(q)[:N40].astype(bool)
base = upk(load_checkpoint(CKPT_40)["baseline"]["correct_packed"]).astype(float)
names_ = BLOCKS_40
real = {n: upk(res40r["blocks"][n]["real"]["correct_packed"]).astype(float) for n in names_}
nul = {n: [upk(ck35["blocks"][n]["null"]["correct_packed"]).astype(float)] +
          [upk(res40r["blocks"][n][f"null{j}"]["correct_packed"]).astype(float) for j in NEW_PERMS_40R] for n in names_}
P = 1 + len(NEW_PERMS_40R)

rows, pv = [], []
for n in names_:
    r_pp = (real[n] - base).mean() * 100; n_pp = [(x - base).mean() * 100 for x in nul[n]]
    dd = (real[n] - np.mean(nul[n], axis=0)) * 100
    m, ci = mean_ci(dd); se = dd.std(ddof=1) / np.sqrt(len(dd)); pv.append(2 * norm.sf(abs(m) / se) if se > 0 else 1.0)
    pp = [(real[n] - x).mean() * 100 for x in nul[n]]
    lp_r = res40r["blocks"][n]["real"]["lpd_f16"].astype(np.float64)
    lp_n = np.mean([res40r["blocks"][n][f"null{j}"]["lpd_f16"].astype(np.float64) for j in NEW_PERMS_40R], axis=0)
    l, l_ci = mean_ci(lp_r - lp_n)
    rows.append((n, r_pp, n_pp, m, ci, pp, l, l_ci))
order = np.argsort(pv); holm = np.empty(len(pv)); run = 0
for r_, i in enumerate(order):
    run = max(run, min(1.0, (len(pv) - r_) * pv[i])); holm[i] = run

print(f"384 px, global windows, Step 35 means, first {N40:,} images | baseline {base.mean()*100:.2f}% | "
      f"{P} nulls (perm 0 = Step 35 null)\n")
print(f"{'block':5s} {'real':>7s} {'null mean':>9s} {'null SD':>7s} {'real−null [95% CI]':>24s} {'Holm p':>8s} "
      f"{'same sign':>9s} {'log-prob real−null (3 new perms)':>34s}")
for (n, r_pp, n_pp, m, ci, pp, l, l_ci), h in zip(rows, holm):
    print(f"{n:5s} {r_pp:+7.2f} {np.mean(n_pp):+9.2f} {np.std(n_pp, ddof=1):7.2f} {m:+8.2f} [{ci[0]:+.2f}, {ci[1]:+.2f}] "
          f"{h:8.4f} {sum(np.sign(x) == np.sign(m) for x in pp):>6d}/{P}   {l:+.4f} [{l_ci[0]:+.4f}, {l_ci[1]:+.4f}]")
rw = [r[0] for r, h in zip(rows, holm) if h < 0.05 and r[3] < 0]
nw = [r[0] for r, h in zip(rows, holm) if h < 0.05 and r[3] > 0]
print(f"\nHolm-significant, real worse: {rw}\nHolm-significant, null worse: {nw}")
print("Step 35 (one permutation, 50K): real worse L2B0 L2B1 L2B2 L2B4 L2B5 L2B6 L2B7 L3B2 L3B3; null worse L3B0 L3B1")

rho = spearmanr([abs(r[1]) for r in rows], [abs(np.mean(r[2])) for r in rows])
print(f"Spearman |real| vs |mean null|: {rho.correlation:+.2f} (p {rho.pvalue:.3f})   [Step 35: 0.50]")

def pooled(null_sets):
    per_img = np.mean([(np.mean(null_sets[n], axis=0) - real[n]) * 100 for n in names_], axis=0)
    rng = np.random.RandomState(42); bm = np.array([per_img[rng.randint(0, N40, N40)].mean() for _ in range(2000)])
    return per_img.mean(), np.percentile(bm, 2.5), np.percentile(bm, 97.5)
p_all = pooled(nul)
p_35 = pooled({n: [nul[n][0]] for n in names_})
p_new = pooled({n: nul[n][1:] for n in names_})
pj = [np.mean([(nul[n][j] - real[n]).mean() * 100 for n in names_]) for j in range(P)]
print(f"Pooled content (real worse = +), all 4 nulls:     {p_all[0]:+.3f} [{p_all[1]:+.3f}, {p_all[2]:+.3f}]")
print(f"Pooled content, Step 35 null only (same 10K):      {p_35[0]:+.3f} [{p_35[1]:+.3f}, {p_35[2]:+.3f}]   [Step 35 on 50K: +0.252]")
print(f"Pooled content, 3 new permutations:                {p_new[0]:+.3f} [{p_new[1]:+.3f}, {p_new[2]:+.3f}]")
print(f"Per permutation: {' '.join(f'{x:+.3f}' for x in pj)}")

res40r["summary"] = {"holm_p": dict(zip(names_, holm.tolist())), "real_worse": rw, "null_worse": nw,
                     "spearman": [float(rho.correlation), float(rho.pvalue)],
                     "pooled_all": [float(x) for x in p_all], "pooled_step35_null": [float(x) for x in p_35],
                     "pooled_new_perms": [float(x) for x in p_new], "pooled_per_perm": [float(x) for x in pj]}
save_checkpoint(CKPT_40R, res40r)
print("\nSTEP 40R-B COMPLETE")

## Step 41 — Pairwise interaction against a shuffled-null baseline ##
Step 36 found super-additive damage for adjacent within-type pairs. This step asks whether that super-additivity
is specific to the content of the blocks or also appears when the same two blocks receive size-matched shuffled
perturbations. Pairs: L2B4+L2B5, L2B5+L2B6, L2B4+L2B6, L2B6+L2B7 (the largest Step 36 interactions), 50,000 images.
Real pairs are re-run; real single-block ablations come from Step 9. Null permutations: 0 = Step 16 (single-block
nulls from its checkpoint), 1–2 = Step 27 seed scheme. Interaction per image = (pair − a − b + base) × 100.
Saves `step41_pairwise_null` after every pass.

### Step 41A — Real pairs, single-block nulls and pair nulls ###

In [ ]:
# ============================================================
# STEP 41A — PAIRWISE INTERACTION: REAL vs SHUFFLED NULL (resumable)
# ============================================================
assert "eval_pass_fast" in globals() and "block_means" in globals(), "Run the setup cells (Step 8, 24B, Step 30 setup) first."
if "set_ws" in globals(): set_ws("default")                 # back to 224 windows after Step 40
clear_all_hooks(); model.eval()
assert (eval_pass_fast(n=2000) == baseline_preds_np[:2000]).all(), "Model is not in its 224 baseline state."

CKPT_41 = "step41_pairwise_null"
PAIRS_41 = [("L2B4", "L2B5"), ("L2B5", "L2B6"), ("L2B4", "L2B6"), ("L2B6", "L2B7")]
N_PERM_41 = 3                                              # perm 0 = Step 16, perms 1..2 = Step 27 seed scheme
STEP36_REF = {"L2B4+L2B5": -2.164, "L2B5+L2B6": -0.680, "L2B4+L2B6": -0.670, "L2B6+L2B7": -0.490}
_blk = {n: blk for n, _, blk in mixer_blocks}
_bidx = {n: i for i, (n, _, _) in enumerate(mixer_blocks)}

torch.manual_seed(42)
_P0_41 = {n: torch.randperm(block_means[n].shape[-1]) for n, _, _ in mixer_blocks}   # = Step 16 permutations
def perm41(name, j):
    if j == 0: return _P0_41[name]
    return torch.randperm(block_means[name].shape[-1], generator=torch.Generator().manual_seed(1000 + 100 * _bidx[name] + j))
def _nh41(name, perm):
    mv = block_means[name]
    def f(module, inp, out):
        is_t = isinstance(out, tuple); t = out[0] if is_t else out
        delta = mv.to(device=t.device, dtype=t.dtype).expand_as(t) - t
        rep = t + delta[..., perm.to(t.device)]
        return (rep,) + out[1:] if is_t else rep
    return f
def run41(spec):
    """spec: list of (block_name, perm or None). None = real mean ablation."""
    clear_all_hooks()
    hs = [_blk[n].mixer.register_forward_hook(mean_ablation_hook(block_means[n]) if p is None else _nh41(n, p))
          for n, p in spec]
    try: return eval_pass_fast()
    finally:
        for h in hs: h.remove()

try:
    res41 = load_checkpoint(CKPT_41); print("Resuming; done:", len(res41), "passes")
except Exception:
    res41 = {}

blocks_41 = sorted({b for p in PAIRS_41 for b in p}, key=lambda n: _bidx[n])
t_all = time.time()
for a, b in PAIRS_41:                                       # 1) real pairs
    k = f"real_pair|{a}+{b}"
    if k not in res41:
        t0 = time.time(); c = run41([(a, None), (b, None)]) == eval_labels_np
        res41[k] = np.packbits(c); save_checkpoint(CKPT_41, res41)
        print(f"{k:24s} Δ {(c.mean() - baseline_correct.mean())*100:+.3f} pp ({(time.time()-t0)/60:.1f} min)")
for j in range(1, N_PERM_41):                               # 2) single-block nulls, perms 1..
    for n in blocks_41:
        k = f"null_single|{n}|{j}"
        if k not in res41:
            t0 = time.time(); c = run41([(n, perm41(n, j))]) == eval_labels_np
            res41[k] = np.packbits(c); save_checkpoint(CKPT_41, res41)
            print(f"{k:24s} Δ {(c.mean() - baseline_correct.mean())*100:+.3f} pp ({(time.time()-t0)/60:.1f} min)")
for j in range(N_PERM_41):                                  # 3) pair nulls
    for a, b in PAIRS_41:
        k = f"null_pair|{a}+{b}|{j}"
        if k not in res41:
            t0 = time.time(); c = run41([(a, perm41(a, j)), (b, perm41(b, j))]) == eval_labels_np
            res41[k] = np.packbits(c); save_checkpoint(CKPT_41, res41)
            print(f"{k:24s} Δ {(c.mean() - baseline_correct.mean())*100:+.3f} pp ({(time.time()-t0)/60:.1f} min)")
clear_all_hooks()
print(f"\nAll passes done ({(time.time()-t_all)/60:.1f} min this session)")
print("\nSTEP 41A COMPLETE")

### Step 41B — Summary: real vs null interaction per pair (CPU only) ###

In [ ]:
# ============================================================
# STEP 41B — PAIRWISE INTERACTION SUMMARY: REAL vs NULL
# ============================================================
from scipy.stats import norm
res41 = load_checkpoint(CKPT_41)
base = baseline_correct.astype(float)
_sp9 = load_checkpoint("step9_ablated_preds")
_n16 = {r["block"]: r for r in load_checkpoint("shuffled_null_50k")["results"]}
real_single = {n: (np.asarray(_sp9[n]).astype(np.int64) == eval_labels_np).astype(float) for n in blocks_41}
def null_single(n, j):
    return unpack(_n16[n]["correct_packed"]).astype(float) if j == 0 else unpack(res41[f"null_single|{n}|{j}"]).astype(float)

out, pv = {}, []
print(f"{'pair':10s} {'real Δ':>8s} {'real int [95% CI]':>24s} {'Step36':>7s} | {'null int per perm':>24s} {'mean':>7s} | "
      f"{'real − null int [95% CI]':>26s} | ratio pair/(a+b) real, null")
for a, b in PAIRS_41:
    key = f"{a}+{b}"
    rp = unpack(res41[f"real_pair|{key}"]).astype(float)
    r_int = (rp - real_single[a] - real_single[b] + base) * 100
    n_int = [(unpack(res41[f"null_pair|{key}|{j}"]).astype(float) - null_single(a, j) - null_single(b, j) + base) * 100
             for j in range(N_PERM_41)]
    n_mean = np.mean(n_int, axis=0)
    rm, rci = mean_ci(r_int); dm, dci = mean_ci(r_int - n_mean)
    se = (r_int - n_mean).std(ddof=1) / np.sqrt(len(base)); pv.append(2 * norm.sf(abs(dm) / se) if se > 0 else 1.0)
    d_pair = (rp.mean() - base.mean()) * 100
    d_sum = ((real_single[a] - base).mean() + (real_single[b] - base).mean()) * 100
    np_pair = np.mean([(unpack(res41[f"null_pair|{key}|{j}"]).astype(float) - base).mean() * 100 for j in range(N_PERM_41)])
    np_sum = np.mean([((null_single(a, j) - base).mean() + (null_single(b, j) - base).mean()) * 100 for j in range(N_PERM_41)])
    out[key] = {"real_pair_pp": d_pair, "real_interaction": rm, "real_interaction_ci": rci,
                "null_interaction_per_perm": [float(x.mean()) for x in n_int], "null_interaction_mean": float(n_mean.mean()),
                "real_minus_null_interaction": dm, "real_minus_null_ci": dci,
                "ratio_real": d_pair / d_sum if d_sum else float("nan"), "ratio_null": np_pair / np_sum if np_sum else float("nan")}
    o = out[key]
    print(f"{key:10s} {d_pair:+8.3f} {rm:+8.3f} [{rci[0]:+.3f}, {rci[1]:+.3f}] {STEP36_REF[key]:+7.3f} | "
          f"{' '.join(f'{x:+.3f}' for x in o['null_interaction_per_perm']):>24s} {o['null_interaction_mean']:+7.3f} | "
          f"{dm:+8.3f} [{dci[0]:+.3f}, {dci[1]:+.3f}] | {o['ratio_real']:.2f}×, {o['ratio_null']:.2f}×")
order = np.argsort(pv); holm = np.empty(len(pv)); run = 0
for r_, i in enumerate(order):
    run = max(run, min(1.0, (len(pv) - r_) * pv[i])); holm[i] = run
for (a, b), h in zip(PAIRS_41, holm):
    out[f"{a}+{b}"]["holm_p_real_vs_null"] = float(h)
print("\nHolm p (real interaction vs null interaction):", {f"{a}+{b}": round(float(h), 4) for (a, b), h in zip(PAIRS_41, holm)})
print("Negative interaction = super-additive damage (pair worse than the sum of single effects).")
res41["summary"] = out
save_checkpoint(CKPT_41, res41)
print("\nSTEP 41B COMPLETE")

## Step 42 — MambaVision-B: additional null permutations for the key blocks ##
Step 37 used one shuffled-null permutation per block. This step adds three permutations (Step 27 seed scheme
1000 + 100·block + j) for the four blocks whose real-vs-null contrast was significant or borderline
(L3B3, L2B7, L2B6, L3B0), on all 50,000 images, and judges each contrast against the mean of four nulls
(the Step 37 permutation plus three new ones). Needs Steps 37A and 37B in the session. Saves
`step42_b_multiperm` after every pass.

### Step 42A — Three extra null permutations per block ###

In [ ]:
# ============================================================
# STEP 42A — MAMBAVISION-B: EXTRA NULL PERMUTATIONS (resumable)
# ============================================================
assert "b_eval" in globals() and "b_block_means" in globals(), "Run Step 37A and Step 37B first."
CKPT_42 = "step42_b_multiperm"
BLOCKS_42 = ["L3B3", "L2B7", "L2B6", "L3B0"]
EXTRA_PERMS = [1, 2, 3]
res37 = load_checkpoint(CKPT_37)
_b_blk = {n: blk for n, _, blk in b_mixer_blocks}
_b_idx = {n: i for i, (n, _, _) in enumerate(b_mixer_blocks)}

def b_perm_j(name, j):
    return torch.randperm(b_block_means[name].shape[-1],
                          generator=torch.Generator().manual_seed(1000 + 100 * _b_idx[name] + j))
def _b_null_hook_p(name, perm):
    mv, g = b_block_means[name], b_gamma[name]
    def f(module, inp, out):
        is_t = isinstance(out, tuple); t = out[0] if is_t else out
        delta = mv.to(dtype=t.dtype).expand_as(t) - t
        if g is None:
            rep = t + delta[..., perm.to(t.device)]
        else:
            gg = g.to(device=t.device, dtype=t.dtype)
            rep = t + (gg * delta)[..., perm.to(t.device)] / gg
        return (rep,) + out[1:] if is_t else rep
    return f

try:
    res42 = load_checkpoint(CKPT_42); print("Resuming; done:", list(res42))
except Exception:
    res42 = {}

for name in BLOCKS_42:
    for j in EXTRA_PERMS:
        k = f"{name}|null{j}"
        if k in res42: continue
        t0 = time.time(); b_clear_hooks()
        h = _b_blk[name].mixer.register_forward_hook(_b_null_hook_p(name, b_perm_j(name, j)))
        try: p = b_eval()
        finally: h.remove()
        st, c = b_paired(p)
        res42[k] = {**st, "correct_packed": pack(c)}
        save_checkpoint(CKPT_42, res42)
        print(f"{k:12s} null Δ {st['delta_pp']:+.3f} pp ({(time.time()-t0)/60:.1f} min)   "
              f"[Step 37 null {res37['E2E5'][name]['null']['delta_pp']:+.3f}, real {res37['E2E5'][name]['real']['delta_pp']:+.3f}]")
b_clear_hooks()
print("\nSTEP 42A COMPLETE")

### Step 42B — Summary: real vs mean of four nulls (CPU only) ###

In [ ]:
# ============================================================
# STEP 42B — MAMBAVISION-B MULTI-PERMUTATION SUMMARY
# ============================================================
from scipy.stats import norm
res42 = load_checkpoint(CKPT_42); res37 = load_checkpoint(CKPT_37)
base = B_BASE_CORRECT.astype(float)
rows, pv = [], []
for name in BLOCKS_42:
    E = res37["E2E5"][name]
    rc = unpack(E["real"]["correct_packed"]).astype(float)
    nulls = [unpack(E["null"]["correct_packed"]).astype(float)] + \
            [unpack(res42[f"{name}|null{j}"]["correct_packed"]).astype(float) for j in EXTRA_PERMS]
    n_pp = [(x - base).mean() * 100 for x in nulls]
    dd = (rc - np.mean(nulls, axis=0)) * 100
    m, ci = mean_ci(dd); se = dd.std(ddof=1) / np.sqrt(len(dd))
    pv.append(2 * norm.sf(abs(m) / se) if se > 0 else 1.0)
    per_perm = [(rc - x).mean() * 100 for x in nulls]
    rows.append((name, (rc - base).mean() * 100, n_pp, m, ci, per_perm, E["real_minus_null_pp"], E["real_minus_null_ci"]))
order = np.argsort(pv); holm = np.empty(len(pv)); run = 0
for r_, i in enumerate(order):
    run = max(run, min(1.0, (len(pv) - r_) * pv[i])); holm[i] = run

print(f"MambaVision-B, 50,000 images | 4 null permutations per block (Step 37 + 3 new)\n")
print(f"{'block':5s} {'real':>7s} {'nulls (perm 0..3)':>30s} {'SD':>5s} | {'real − mean null [95% CI]':>27s} {'Holm(4)':>8s} "
      f"{'same sign':>9s} | Step 37 (1 perm)")
out = {}
for (n, r_pp, n_pp, m, ci, pp, m37, ci37), h in zip(rows, holm):
    same = sum(np.sign(x) == np.sign(m) for x in pp)
    print(f"{n:5s} {r_pp:+7.3f} {' '.join(f'{x:+.3f}' for x in n_pp):>30s} {np.std(n_pp, ddof=1):5.3f} | "
          f"{m:+8.3f} [{ci[0]:+.3f}, {ci[1]:+.3f}] {h:8.4f} {same:>6d}/4 | {m37:+.3f} [{ci37[0]:+.3f}, {ci37[1]:+.3f}]")
    out[n] = {"real_pp": r_pp, "null_pp": n_pp, "null_sd": float(np.std(n_pp, ddof=1)), "real_minus_mean_null": m,
              "ci": ci, "holm_p_4": float(h), "per_perm": pp}
res42["summary"] = out
save_checkpoint(CKPT_42, res42)
print("\nSTEP 42B COMPLETE")

## Export — all checkpoints (Step 0 to latest) ##
Downloads every file in the Hugging Face checkpoint repository and packs it into one zip: the original .pkl
checkpoints, one JSON with all checkpoints (large per-image arrays summarised, packed per-image correctness kept),
an index of checkpoint names, sizes and keys, and the figures and final results folders.

In [ ]:
# ============================================================
# EXPORT — ALL CHECKPOINTS (STEP 0 TO LATEST) → ONE ZIP
# ============================================================
import os, io, json, base64, pickle, zipfile, shutil, csv, time
import numpy as np
assert "api" in globals() and "HF_REPO_ID" in globals(), "Run the Hugging Face cell first."

OUT_DIR = "/kaggle/working/export_all"
ZIP_PATH = "/kaggle/working/mambavision_all_results.zip"
shutil.rmtree(OUT_DIR, ignore_errors=True); os.makedirs(OUT_DIR, exist_ok=True)

files = sorted(api.list_repo_files(repo_id=HF_REPO_ID, repo_type="dataset"))
files = [f for f in files if not f.startswith(".") and f != "README.md"]
print(f"{len(files)} files in {HF_REPO_ID}")

try:
    import torch
except Exception:
    torch = None

def to_jsonable(obj, key=""):
    if torch is not None and isinstance(obj, torch.Tensor):
        obj = obj.detach().cpu().numpy()
    if isinstance(obj, np.ndarray):
        if obj.dtype == np.uint8 and key.endswith("packed"):
            return {"__packbits_b64__": base64.b64encode(obj.tobytes()).decode(), "len_bytes": int(obj.size)}
        if obj.size <= 2000:
            return obj.astype(np.float64).tolist() if obj.dtype.kind == "f" else obj.tolist()
        a = obj.astype(np.float64)
        return {"__array_summary__": True, "shape": list(obj.shape), "dtype": str(obj.dtype),
                "mean": float(np.nanmean(a)), "std": float(np.nanstd(a)),
                "min": float(np.nanmin(a)), "max": float(np.nanmax(a))}
    if isinstance(obj, (np.integer,)): return int(obj)
    if isinstance(obj, (np.floating,)): return float(obj)
    if isinstance(obj, (np.bool_,)): return bool(obj)
    if isinstance(obj, dict): return {str(k): to_jsonable(v, str(k)) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)): return [to_jsonable(v, key) for v in obj]
    if isinstance(obj, set): return sorted(to_jsonable(v, key) for v in obj)
    if obj is None or isinstance(obj, (str, int, float, bool)): return obj
    return str(obj)

all_data, index_rows, t0 = {}, [], time.time()
for f in files:
    local = api.hf_hub_download(repo_id=HF_REPO_ID, repo_type="dataset", filename=f)
    sub = "pkl" if f.endswith(".pkl") else os.path.dirname(f) or "other"
    dest = os.path.join(OUT_DIR, sub if f.endswith(".pkl") else "", os.path.basename(f) if f.endswith(".pkl") else f)
    os.makedirs(os.path.dirname(dest), exist_ok=True); shutil.copy(local, dest)
    size_kb = os.path.getsize(local) / 1024
    keys = ""
    if f.endswith(".pkl"):
        name = os.path.basename(f)[:-4]
        try:
            with open(local, "rb") as fh:
                obj = pickle.load(fh)
            all_data[name] = to_jsonable(obj)
            keys = ", ".join(list(obj.keys())[:12]) if isinstance(obj, dict) else type(obj).__name__
        except Exception as e:
            all_data[name] = {"__load_error__": repr(e)}; keys = f"LOAD ERROR: {e!r}"
    index_rows.append([f, f"{size_kb:.1f}", keys])

with open(os.path.join(OUT_DIR, "report_data_all.json"), "w") as fh:
    json.dump(all_data, fh)
with open(os.path.join(OUT_DIR, "index.csv"), "w", newline="") as fh:
    w = csv.writer(fh); w.writerow(["file", "size_kb", "top_level_keys"]); w.writerows(index_rows)

if os.path.exists(ZIP_PATH): os.remove(ZIP_PATH)
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, fs in os.walk(OUT_DIR):
        for fn in fs:
            full = os.path.join(root, fn); z.write(full, os.path.relpath(full, OUT_DIR))

print(f"\n{'file':55s} {'KB':>9s}  keys")
for f, kb, k in index_rows:
    print(f"{f:55s} {kb:>9s}  {k[:80]}")
n_pkl = sum(f.endswith(".pkl") for f in files)
n_err = sum(1 for v in all_data.values() if isinstance(v, dict) and "__load_error__" in v)
print(f"\n{n_pkl} checkpoints ({n_err} could not be unpickled — their .pkl files are still in the zip)")
print(f"Zip: {ZIP_PATH}  ({os.path.getsize(ZIP_PATH)/1e6:.1f} MB)  in {(time.time()-t0)/60:.1f} min")
print("\nEXPORT COMPLETE")

## Step 43 — Second null, temperature-free metric and calibration (50K) ##
Answers three review points on the full 50K probe: whether the shuffled-channel null is itself off-distribution
(a sign-flip null that keeps every channel in place), whether the accuracy bound holds in a temperature-free metric
(true-class rank), and how the joint freeze changes confidence (entropy and ECE). After a session restart, run first:
HF checkpoint cell, Steps 0–5b, prediction helpers, mixer_blocks, Step 8, Step 11b, Cell B, Step 24B, Step 30 setup,
Step 13A and Step 13B.

### Step 43A — Setup and channel-RMS spread of the mixer output ###
Defines the shuffled permutations (perm 0 = Step 16, perms 1–2 = Step 27) and the sign-flip vectors. On the 256
reference images, reports per block how much of the mixer output and of the ablation perturbation sits in a few
channels (max/median channel RMS, energy share of the top 1% of channels). About 1 minute.

In [ ]:
# ============================================================
# STEP 43A — SETUP + CHANNEL-RMS SPREAD OF THE MIXER OUTPUT
# ============================================================
import time, numpy as np, torch, torch.nn.functional as F

assert "get_cached_batch" in globals() and "eval_pass_fast" in globals(), "Run Step 24B and the Step 30 setup cell first."
assert "block_means_cpu" in globals() and "baseline_log_probs" in globals(), "Run Step 8 and Step 11b first."
assert "install_freeze_patch" in globals() and "_freeze_refs" in globals(), "Run Step 13A and Step 13B first."
assert len(mixer_blocks) == 12
_LAB_GPU = torch.as_tensor(LAB_CACHE, device=device)
model.eval(); clear_all_hooks()

# Same permutations as Step 16 / 26 / 27 (perm 0 = torch.manual_seed(42), in mixer_blocks order)
torch.manual_seed(42)
channel_perms = {name: torch.randperm(block_means_cpu[name].shape[-1]) for name, _, _ in mixer_blocks}

N_SHUF, N_SIGN = 3, 3          # shuffled-null perms 0..2 (= Step 27 perms 0..2) and 3 sign-flip seeds

def shuffle_perms(bi, name):
    C = block_means_cpu[name].shape[-1]
    ps = [channel_perms[name]]
    for j in range(1, N_SHUF):
        g = torch.Generator().manual_seed(1000 + 100 * bi + j)      # identical to Step 27
        ps.append(torch.randperm(C, generator=g))
    return ps

def sign_vectors(bi, name):
    C = block_means_cpu[name].shape[-1]
    out = []
    for j in range(N_SIGN):
        g = torch.Generator().manual_seed(2000 + 100 * bi + j)
        out.append((torch.randint(0, 2, (C,), generator=g) * 2 - 1).float())
    return out

# ---- channel-RMS spread on the 256 reference images (cache positions 0..255 = Step 8 images) ----
CKPT_43A = "step43a_channel_rms"
_rms_acc = {}
def _rms_hook(name):
    def f(module, inp, out):
        t = (out[0] if isinstance(out, tuple) else out).float()                 # (B, L, C)
        d = block_means_cpu[name].to(t.device).float().expand_as(t) - t         # ablation perturbation
        a = _rms_acc.setdefault(name, {"x2": 0.0, "d2": 0.0, "n": 0})
        a["x2"] = a["x2"] + (t ** 2).sum((0, 1)).double().cpu()
        a["d2"] = a["d2"] + (d ** 2).sum((0, 1)).double().cpu()
        a["n"] += t.shape[0] * t.shape[1]
    return f
hs = [blk.mixer.register_forward_hook(_rms_hook(n)) for n, _, blk in mixer_blocks]
try:
    with torch.no_grad():
        for s in range(0, 256, 64):
            model(get_cached_batch(s, s + 64))
finally:
    for h in hs: h.remove()

res43a = {}
print(f"{'Block':6s}{'type':6s}{'out RMS max/med':>17s}{'top-1% share':>14s}{'Δ RMS max/med':>15s}{'Δ top-1% share':>16s}")
for name, mtype, _ in mixer_blocks:
    a = _rms_acc[name]
    rx = (a["x2"] / a["n"]).sqrt().numpy(); rd = (a["d2"] / a["n"]).sqrt().numpy()
    def share(r):   # fraction of total energy in the top 1% of channels
        e = np.sort(r ** 2)[::-1]; k = max(1, int(round(0.01 * len(e)))); return float(e[:k].sum() / e.sum())
    res43a[name] = {"type": mtype, "out_rms": rx.tolist(), "delta_rms": rd.tolist(),
                    "out_max_over_median": float(rx.max() / np.median(rx)), "out_top1pct_share": share(rx),
                    "delta_max_over_median": float(rd.max() / np.median(rd)), "delta_top1pct_share": share(rd)}
    r = res43a[name]
    print(f"{name:6s}{'M' if mtype=='MambaVisionMixer' else 'A':6s}{r['out_max_over_median']:17.2f}{r['out_top1pct_share']*100:13.2f}%"
          f"{r['delta_max_over_median']:15.2f}{r['delta_top1pct_share']*100:15.2f}%")
save_checkpoint(CKPT_43A, res43a)
print("\nSTEP 43A COMPLETE")

### Step 43B — Real ablation vs three shuffled and three sign-flip nulls, per block ###
Seven copies per batch after a shared prefix: real, three shuffled nulls and three sign-flip nulls (each channel of
the perturbation multiplied by a fixed random ±1). Stores per image: correctness, true-class log-probability,
true-class rank, max probability and entropy. One checkpoint per block, resumable; stops if the shuffled nulls do
not reproduce Step 27. About 10–15 min per block, 2.5–3 h in total.

In [ ]:
# ============================================================
# STEP 43B — REAL vs SHUFFLED vs SIGN-FLIP NULL, PER-IMAGE METRICS (50K)
# ============================================================
K_COPIES = 1 + N_SHUF + N_SIGN          # copy 0 = real, 1..3 = shuffled, 4..6 = sign-flip
LABELS = ["real"] + [f"shuf{j}" for j in range(N_SHUF)] + [f"sign{j}" for j in range(N_SIGN)]
_f43 = {"name": None, "perms": None, "signs": None}

def _f43_pre(module, args):
    x = args[0]
    return (x.repeat(K_COPIES, *([1] * (x.dim() - 1))),) + tuple(args[1:])

def _f43_mixer(module, inp, out):
    is_tuple = isinstance(out, tuple)
    t = out[0] if is_tuple else out
    B = t.shape[0] // K_COPIES
    mean_val = block_means_cpu[_f43["name"]].to(device=t.device, dtype=t.dtype)
    pieces = []
    for j in range(K_COPIES):
        tj = t[j * B:(j + 1) * B]
        d = mean_val.expand_as(tj) - tj
        if j == 0:
            pieces.append(tj + d)                                                   # real mean ablation
        elif j <= N_SHUF:
            pieces.append(tj + d[..., _f43["perms"][j - 1].to(t.device)])           # shuffled-channel null
        else:
            pieces.append(tj + d * _f43["signs"][j - 1 - N_SHUF].to(t.device, t.dtype))   # sign-flip null
    rep = torch.cat(pieces, 0)
    return (rep,) + out[1:] if is_tuple else rep

def eval_fork43(name, blk, perms, signs):
    clear_all_hooks()
    _f43.update({"name": name, "perms": perms, "signs": signs})
    h1 = blk.register_forward_pre_hook(_f43_pre)
    h2 = blk.mixer.register_forward_hook(_f43_mixer)
    k = K_COPIES
    corr = np.zeros((k, N_FULL), bool); lpt = np.zeros((k, N_FULL), np.float32)
    rank = np.zeros((k, N_FULL), np.int16); pmax = np.zeros((k, N_FULL), np.float16); ent = np.zeros((k, N_FULL), np.float16)
    try:
        with torch.no_grad():
            for s in range(0, N_FULL, BATCH_SIZE):
                e = min(s + BATCH_SIZE, N_FULL); B = e - s
                out = model(get_cached_batch(s, e))
                logits = (out["logits"] if isinstance(out, dict) else out.logits).float().view(k, B, -1)
                lp = F.log_softmax(logits, -1)
                y = _LAB_GPU[s:e].view(1, B, 1).expand(k, B, 1)
                lt = logits.gather(2, y)                                             # true-class logit
                corr[:, s:e] = (logits.argmax(-1) == y.squeeze(2)).cpu().numpy()
                lpt[:, s:e] = lp.gather(2, y).squeeze(2).cpu().numpy()
                rank[:, s:e] = (logits > lt).sum(-1).cpu().numpy().astype(np.int16)  # 0 = top-1
                p = lp.exp()
                pmax[:, s:e] = p.max(-1).values.cpu().numpy().astype(np.float16)
                ent[:, s:e] = (-(p * lp).sum(-1)).cpu().numpy().astype(np.float16)
    finally:
        h1.remove(); h2.remove()
    return corr, lpt, rank, pmax, ent

res27_chk = load_checkpoint("step27_multi_perm_null")
real10 = {r["block"]: r for r in load_checkpoint("step10")["ablation_results"]}
done = set(list_checkpoints())
for bi, (name, mtype, blk) in enumerate(mixer_blocks):
    ck = f"step43b_{name}"
    if ck in done:
        print(f"{name}: already done"); continue
    t0 = time.time()
    corr, lpt, rank, pmax, ent = eval_fork43(name, blk, shuffle_perms(bi, name), sign_vectors(bi, name))
    d_pp = (corr.mean(1) - baseline_correct.mean()) * 100
    agree = [float((corr[1 + j] == unpack(res27_chk[name]["correct_packed"][j])).mean()) for j in range(N_SHUF)]
    save_checkpoint(ck, {"block": name, "type": mtype, "labels": LABELS, "delta_pp": d_pp.tolist(),
                         "correct_packed": [pack(c) for c in corr], "lp_true": lpt, "rank": rank,
                         "pmax": pmax, "entropy": ent, "agree_shuf_vs_step27": agree})
    print(f"{name:5s} {(time.time()-t0)/60:4.1f} min | real {d_pp[0]:+.3f} (Step 10: {real10[name]['delta_pp']:+.3f}) | "
          f"shuf {np.round(d_pp[1:1+N_SHUF],3)} | sign {np.round(d_pp[1+N_SHUF:],3)} | "
          f"agree vs Step 27: {min(agree)*100:.3f}%")
    assert min(agree) > 0.999, "Shuffled null does not reproduce Step 27 - stop and send me this output."
print("\nSTEP 43B COMPLETE")

### Step 43C — Calibration of the joint freeze (Δ, B, C in all six Mamba mixers) ###
Accuracy, NLL, mean confidence, entropy, ECE (15 bins) and confidence on correct and wrong images for the baseline
(saved Step 6 log-probs) and for the joint freeze in mean and position mode. Per-image agreement with Step 25 is
checked. About 10 minutes.

In [ ]:
# ============================================================
# STEP 43C — CALIBRATION OF THE JOINT FREEZE
# ============================================================
CKPT_43C = "step43c_freeze_calibration"
def ece(conf, correct, n_bins=15):
    conf = np.asarray(conf, np.float64); correct = np.asarray(correct, np.float64)
    bins = np.minimum((conf * n_bins).astype(int), n_bins - 1); e = 0.0
    for b in range(n_bins):
        m = bins == b
        if m.any(): e += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(e)

def calib_from_logits_pass():
    preds, lpt, pmax, ent = [], [], [], []
    with torch.no_grad():
        for s in range(0, N_FULL, BATCH_SIZE):
            e = min(s + BATCH_SIZE, N_FULL)
            out = model(get_cached_batch(s, e))
            lg = (out["logits"] if isinstance(out, dict) else out.logits).float()
            lp = F.log_softmax(lg, -1); p = lp.exp()
            preds.append(lg.argmax(-1).cpu()); lpt.append(lp.gather(1, _LAB_GPU[s:e].view(-1, 1)).squeeze(1).cpu())
            pmax.append(p.max(-1).values.cpu()); ent.append((-(p * lp).sum(-1)).cpu())
    return [torch.cat(x).numpy() for x in (preds, lpt, pmax, ent)]

def summarize(preds, lpt, pmax, ent):
    c = preds == eval_labels_np
    return {"acc": float(c.mean() * 100), "nll": float(-lpt.mean()), "mean_conf": float(pmax.mean()),
            "entropy": float(ent.mean()), "ece": ece(pmax, c),
            "conf_correct": float(pmax[c].mean()), "conf_wrong": float(pmax[~c].mean())}

try:
    res43c = load_checkpoint(CKPT_43C)
except Exception:
    res43c = {}
if "baseline" not in res43c:
    bl = baseline_log_probs.float()
    res43c["baseline"] = summarize(baseline_preds_np, bl.gather(1, eval_labels.view(-1, 1)).squeeze(1).numpy(),
                                   bl.exp().max(-1).values.numpy(), (-(bl.exp() * bl).sum(-1)).numpy())
    save_checkpoint(CKPT_43C, res43c)
step25 = load_checkpoint("step25_joint_freeze_modes")
for mode in ["mean", "position"]:
    key = f"all|{mode}"
    if key in res43c: continue
    hs = install_freeze_patch()
    try:
        _freeze_config.update({"active": True, "block_names": set(mamba_block_names), "tensor": "all", "mode": mode})
        out = calib_from_logits_pass()
    finally:
        remove_freeze_patch(hs)
    res43c[key] = summarize(*out)
    res43c[key]["agree_step25"] = float(((out[0] == eval_labels_np) == unpack(step25[key]["correct_packed"])).mean())
    save_checkpoint(CKPT_43C, res43c)

print(f"{'condition':16s}{'acc %':>8s}{'NLL':>8s}{'conf':>8s}{'entropy':>9s}{'ECE %':>8s}{'conf|right':>11s}{'conf|wrong':>11s}")
for k in ["baseline", "all|mean", "all|position"]:
    r = res43c[k]
    print(f"{k:16s}{r['acc']:8.2f}{r['nll']:8.4f}{r['mean_conf']:8.4f}{r['entropy']:9.4f}{r['ece']*100:8.2f}{r['conf_correct']:11.4f}{r['conf_wrong']:11.4f}"
          + (f"   agree w/ Step 25: {r['agree_step25']*100:.3f}%" if "agree_step25" in r else ""))
print("\nSTEP 43C COMPLETE")

### Step 43D — Summary: content effect against both nulls (CPU only) ###
Content effect per block in accuracy (pp), true-class log-probability (nats) and log(1 + rank), against the mean of
three shuffled nulls and the mean of three sign-flip nulls, with paired bootstrap CIs, Holm correction over 12 blocks,
pooled bounds (95% and 90%) and Spearman correlations. Saved as step43d_summary.

In [ ]:
# ============================================================
# STEP 43D — SUMMARY: CONTENT EFFECT IN ACCURACY, LOG-PROB AND RANK
# ============================================================
from scipy.stats import spearmanr
CKPT_43D = "step43d_summary"
NB = 2000
_rng_idx = np.random.RandomState(42).randint(0, N_FULL, (NB, N_FULL)).astype(np.int32)   # shared resamples

def boot(x):
    x = np.asarray(x, np.float64)
    bm = np.array([x[_rng_idx[i]].mean() for i in range(NB)])
    return float(x.mean()), np.percentile(bm, [2.5, 97.5]).tolist(), np.percentile(bm, [5, 95]).tolist(), bm

def holm(ps):
    ps = np.asarray(ps); o = np.argsort(ps); m = len(ps); adj = np.empty(m); run = 0.0
    for r, i in enumerate(o):
        run = max(run, (m - r) * ps[i]); adj[i] = min(1.0, run)
    return adj

base_c = baseline_correct.astype(np.float64)
lr = lambda r: np.log1p(np.asarray(r, np.float64))       # temperature-free: log(1 + rank of true class)

rows, pooled = [], {f"{nl}|{m}": [] for nl in ("shuf", "sign") for m in ("acc", "lp", "rank")}
for name, mtype, _ in mixer_blocks:
    d = load_checkpoint(f"step43b_{name}")
    c = np.stack([unpack(p) for p in d["correct_packed"]]).astype(np.float64)
    lpt = d["lp_true"].astype(np.float64); rk = lr(d["rank"])
    sl = {"shuf": slice(1, 1 + N_SHUF), "sign": slice(1 + N_SHUF, K_COPIES)}
    row = {"block": name, "type": "M" if mtype == "MambaVisionMixer" else "A",
           "real_pp": float((c[0].mean() - base_c.mean()) * 100)}
    for nl, s_ in sl.items():
        row[f"{nl}_pp"] = float((c[s_].mean() - base_c.mean()) * 100)
        row[f"{nl}_pp_each"] = ((c[s_].mean(1) - base_c.mean()) * 100).tolist()
        # content effect, positive = real ablation worse than the null
        per = {"acc": (c[s_].mean(0) - c[0]) * 100,                      # pp
               "lp":  (lpt[s_].mean(0) - lpt[0]),                        # nats: null log p(y) − real log p(y)
               "rank": (rk[0] - rk[s_].mean(0))}                        # log(1+rank): real − null
        for m, x in per.items():
            mu, ci, ci90, bm = boot(x)
            p = float(2 * min((bm <= 0).mean(), (bm >= 0).mean())); p = max(p, 1.0 / NB)
            row[f"{nl}|{m}"] = {"C": mu, "ci": ci, "p": p}
            pooled[f"{nl}|{m}"].append(x)
    rows.append(row)
for key in pooled:
    adj = holm([r[key]["p"] for r in rows])
    for r, a in zip(rows, adj): r[key]["p_holm"] = float(a)
summ = {"rows": rows, "pooled": {}}
for key, xs in pooled.items():
    mu, ci, ci90, _ = boot(np.mean(xs, 0))
    summ["pooled"][key] = {"C": mu, "ci": ci, "ci90": ci90}
for nl in ("shuf", "sign"):
    summ[f"spearman_real_{nl}"] = float(spearmanr([abs(r["real_pp"]) for r in rows], [abs(r[f"{nl}_pp"]) for r in rows])[0])
summ["spearman_shuf_sign"] = float(spearmanr([r["shuf_pp"] for r in rows], [r["sign_pp"] for r in rows])[0])
summ["rms"] = {n: {k: v for k, v in r.items() if "over" in k or "share" in k} for n, r in load_checkpoint("step43a_channel_rms").items()}
summ["calibration"] = load_checkpoint("step43c_freeze_calibration")
save_checkpoint(CKPT_43D, summ)

f = lambda v: f"{v['C']:+.3f} [{v['ci'][0]:+.3f},{v['ci'][1]:+.3f}]{'*' if v['p_holm'] < 0.05 else ' '}"
print("Δ accuracy (pp): real | shuffled null (mean of 3) | sign-flip null (mean of 3)")
for r in rows:
    print(f"{r['block']:5s} {r['type']}  real {r['real_pp']:+.3f} | shuf {r['shuf_pp']:+.3f} | sign {r['sign_pp']:+.3f}")
for m, unit in [("acc", "pp"), ("lp", "nats"), ("rank", "log(1+rank)")]:
    print(f"\nContent effect C in {m} ({unit}); positive = real worse; * = Holm p<0.05")
    print(f"{'Block':6s}{'vs shuffled null':>30s}{'vs sign-flip null':>32s}")
    for r in rows:
        print(f"{r['block']:6s}{f(r[f'shuf|{m}']):>30s}{f(r[f'sign|{m}']):>32s}")
    for nl in ("shuf", "sign"):
        v = summ["pooled"][f"{nl}|{m}"]
        print(f"  pooled vs {nl}: {v['C']:+.4f}  95% [{v['ci'][0]:+.4f},{v['ci'][1]:+.4f}]  90% [{v['ci90'][0]:+.4f},{v['ci90'][1]:+.4f}]")
print(f"\nSpearman |real| vs |null|: shuffled {summ['spearman_real_shuf']:.2f}, sign-flip {summ['spearman_real_sign']:.2f}; "
      f"shuffled vs sign-flip null: {summ['spearman_shuf_sign']:.2f}")
print("\nSTEP 43D COMPLETE")

### STEP 43E — PRINT 43A / 43C RESULTS AND THE SCALE OF THE RANK METRIC (CPU only)

In [ ]:
# ============================================================
# STEP 43E — PRINT 43A / 43C RESULTS AND THE SCALE OF THE RANK METRIC (CPU only)
# ============================================================
import numpy as np, torch
assert "load_checkpoint" in globals(), "Run the Hugging Face cell first."
BLOCKS = ["L2B0", "L2B1", "L2B2", "L2B3", "L2B4", "L2B5", "L2B6", "L2B7", "L3B0", "L3B1", "L3B2", "L3B3"]
N = 50000

# ---- 43A: channel-RMS spread ----
a = load_checkpoint("step43a_channel_rms")
print("43A  channel-RMS spread (256 reference images)")
print(f"{'Block':6s}{'type':5s}{'out max/med':>12s}{'out top1%':>11s}{'Δ max/med':>11s}{'Δ top1%':>10s}")
for b in BLOCKS:
    r = a[b]
    print(f"{b:6s}{'M' if r['type']=='MambaVisionMixer' else 'A':5s}{r['out_max_over_median']:12.2f}{r['out_top1pct_share']*100:10.2f}%"
          f"{r['delta_max_over_median']:11.2f}{r['delta_top1pct_share']*100:9.2f}%")

# ---- 43C: calibration of the joint freeze ----
c = load_checkpoint("step43c_freeze_calibration")
print("\n43C  calibration of the joint freeze")
print(f"{'condition':14s}{'acc %':>8s}{'NLL':>8s}{'conf':>8s}{'entropy':>9s}{'ECE %':>8s}{'conf|right':>11s}{'conf|wrong':>11s}{'agree25 %':>10s}")
for k in ["baseline", "all|mean", "all|position"]:
    r = c[k]
    print(f"{k:14s}{r['acc']:8.2f}{r['nll']:8.4f}{r['mean_conf']:8.4f}{r['entropy']:9.4f}{r['ece']*100:8.2f}"
          f"{r['conf_correct']:11.4f}{r['conf_wrong']:11.4f}{r.get('agree_step25', float('nan'))*100:10.3f}")

# ---- scale of the rank metric: damage in log(1+rank) for real and nulls ----
_b = load_checkpoint("baseline_logits_probs_new")
blp = torch.as_tensor(_b["baseline_log_probs"]).float()
y = torch.as_tensor(_b["eval_labels"]).long().view(-1, 1)
base_rank = (blp > blp.gather(1, y)).sum(-1).numpy()
base_lr = np.log1p(base_rank.astype(np.float64))
print(f"\nBaseline: mean log(1+rank) {base_lr.mean():.4f} | true class in top-5: {(base_rank < 5).mean()*100:.2f}%")
print("\nRank damage = mean log(1+rank) after intervention − baseline (positive = true class pushed down)")
print(f"{'Block':6s}{'real':>9s}{'shuf':>9s}{'sign':>9s}{'real top5 Δpp':>15s}{'shuf top5':>11s}{'sign top5':>11s}")
for b in BLOCKS:
    d = load_checkpoint(f"step43b_{b}")
    lr = np.log1p(d["rank"].astype(np.float64))
    t5 = (d["rank"] < 5).mean(1) * 100 - (base_rank < 5).mean() * 100
    dm = lr.mean(1) - base_lr.mean()
    print(f"{b:6s}{dm[0]:+9.4f}{dm[1:4].mean():+9.4f}{dm[4:7].mean():+9.4f}{t5[0]:+15.3f}{t5[1:4].mean():+11.3f}{t5[4:7].mean():+11.3f}")
print("\nSTEP 43E COMPLETE")

## Step 44 — Swin-T: hierarchical attention-only control ##
MambaVision differs from DeiT-Small in two ways: it contains Mamba mixers, and it is hierarchical (four stages with falling resolution and windowed attention) where DeiT is flat. Swin-T is hierarchical like MambaVision but has no Mamba mixers, so it separates the two explanations. Same protocol as the DeiT control (Step 34): for each of the 12 Swin blocks, real mean ablation of the attention output and one size-matched shuffled null, on all 50,000 images, with per-image correctness, true-class log-probability and true-class rank.

### Step 44A — Fork runner (definitions only) ###
Each batch is repeated after the ablated block's input, and the copies receive different interventions on the block's
output: unchanged ("none", used only in the safety check), the real mean ablation, or the shuffled null. The shared
prefix up to the block is computed once. Also defines the plain evaluation, the paired bootstrap and the Holm
correction. No GPU work.

In [ ]:
# ============================================================
# STEP 44A — FORK RUNNER (DEFINITIONS ONLY)
# ============================================================
import time, numpy as np, torch, torch.nn.functional as F
from scipy.stats import spearmanr
assert "LAB_CACHE" in globals() and "save_checkpoint" in globals() and "pack" in globals(), \
    "Run the HF cell, Cell B, Step 24B and the Step 30 setup cell first."
_LAB_GPU = torch.as_tensor(LAB_CACHE, device=device)
SW_BS = 128                                                      # images per batch for Swin-T

def _out_t(out):
    return out[0] if isinstance(out, tuple) else out

def fork_eval(model_call, get_batch, n, block_module, out_module, mean, kinds, bs=SW_BS):
    """kinds: list of (label, 'none' | 'real' | 'shuf', permutation or None). Returns per-image arrays (K, n)."""
    K = len(kinds)
    def pre(module, args, kwargs):
        if len(args) > 0:
            x = args[0]
            return (x.repeat(K, *([1] * (x.dim() - 1))),) + tuple(args[1:]), kwargs
        kwargs = dict(kwargs); x = kwargs["hidden_states"]
        kwargs["hidden_states"] = x.repeat(K, *([1] * (x.dim() - 1)))
        return args, kwargs
    def post(module, inp, out):
        is_t = isinstance(out, tuple); t = out[0] if is_t else out
        R = t.shape[0] // K
        mv = mean.to(device=t.device, dtype=t.dtype)
        pieces = []
        for j, (lab, kind, p) in enumerate(kinds):
            tj = t[j * R:(j + 1) * R]
            if kind == "none":
                pieces.append(tj)
            elif kind == "real":
                pieces.append(mv.expand_as(tj).clone())                       # real mean ablation
            else:
                d = mv.expand_as(tj) - tj
                pieces.append(tj + d[..., p.to(t.device)])                    # shuffled-channel null
        rep = torch.cat(pieces, 0)
        return (rep,) + tuple(out[1:]) if is_t else rep
    h1 = block_module.register_forward_pre_hook(pre, with_kwargs=True)
    h2 = out_module.register_forward_hook(post)
    corr = np.zeros((K, n), bool); lpt = np.zeros((K, n), np.float32); rank = np.zeros((K, n), np.int16)
    try:
        with torch.no_grad():
            for s in range(0, n, bs):
                e = min(s + bs, n); B = e - s
                logits = model_call(get_batch(s, e)).float().view(K, B, -1)
                y = _LAB_GPU[s:e].view(1, B, 1).expand(K, B, 1)
                corr[:, s:e] = (logits.argmax(-1) == y.squeeze(2)).cpu().numpy()
                lpt[:, s:e] = F.log_softmax(logits, -1).gather(2, y).squeeze(2).cpu().numpy()
                rank[:, s:e] = (logits > logits.gather(2, y)).sum(-1).cpu().numpy().astype(np.int16)
    finally:
        h1.remove(); h2.remove()
    return corr, lpt, rank

def plain_eval(model_call, get_batch, n, bs=SW_BS):
    corr = np.zeros(n, bool); lpt = np.zeros(n, np.float32); rank = np.zeros(n, np.int16)
    with torch.no_grad():
        for s in range(0, n, bs):
            e = min(s + bs, n)
            logits = model_call(get_batch(s, e)).float(); y = _LAB_GPU[s:e].view(-1, 1)
            corr[s:e] = (logits.argmax(-1) == y.squeeze(1)).cpu().numpy()
            lpt[s:e] = F.log_softmax(logits, -1).gather(1, y).squeeze(1).cpu().numpy()
            rank[s:e] = (logits > logits.gather(1, y)).sum(-1).cpu().numpy().astype(np.int16)
    return corr, lpt, rank

NB_BOOT = 2000
def _boot(x, idx):
    x = np.asarray(x, np.float64); bm = np.array([x[i].mean() for i in idx])
    return float(x.mean()), np.percentile(bm, [2.5, 97.5]).tolist(), np.percentile(bm, [5, 95]).tolist(), bm

def _holm(ps):
    ps = np.asarray(ps); o = np.argsort(ps); m = len(ps); adj = np.empty(m); run = 0.0
    for r, i in enumerate(o):
        run = max(run, (m - r) * ps[i]); adj[i] = min(1.0, run)
    return adj
print("STEP 44A READY (functions defined)")

### Step 44B — Swin-T: load, image cache, blocks, reference means, baseline, safety check ###
Loads microsoft/swin-tiny-patch4-window7-224 and uses the same preprocessed images as DeiT-Small (bicubic resize to 248,
centre crop 224, ImageNet mean and std); the cache is rebuilt if Step 34A was not run in this session (about 10 min).
The hooked outputs are the attention outputs (including the output projection) of the 12 Swin blocks (depths 2-2-6-2).
Reference means: per channel over all tokens of the first 256 images. Safety check on 256 images and a shifted-window
block: the fork copies must reproduce the baseline and a plain (non-fork) real ablation and null exactly; otherwise the
cell stops before any long run.

In [ ]:
# ============================================================
# STEP 44B — SWIN-T: LOAD, CACHE, BLOCKS, REFERENCE MEANS, BASELINE, SAFETY CHECK
# ============================================================
import os
from transformers import AutoModelForImageClassification
from torchvision import transforms as T
from torch.utils.data import Dataset as _TorchDataset, DataLoader
assert "fork_eval" in globals(), "Run Step 44A first."

SW_ID, PREFIX_SW = "microsoft/swin-tiny-patch4-window7-224", "step44_swin"
sw_model = AutoModelForImageClassification.from_pretrained(SW_ID).to(device).eval()
sw_call = lambda x: sw_model(pixel_values=x).logits
print(f"Swin-T parameters: {sum(p.numel() for p in sw_model.parameters()):,}")

# ---- same image cache as DeiT (Step 34A); rebuilt only if not present in this session ----
if "d_get_batch" not in globals():
    D_PRE = T.Compose([T.Resize(248, interpolation=T.InterpolationMode.BICUBIC), T.CenterCrop(224)])
    D_MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
    D_STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
    D_CACHE = os.path.join(CACHE_DIR, f"mv_cache_DEIT_images_u8_{N_FULL}.npy")
    class _DCacheSource(_TorchDataset):
        def __len__(self): return N_FULL
        def __getitem__(self, pos):
            ex = dataset[eval_indices[pos]]
            return np.asarray(D_PRE(ex["image"].convert("RGB")), dtype=np.uint8).copy(), ex["label"]
    if not os.path.exists(D_CACHE + ".done"):
        t0 = time.time()
        mm = np.lib.format.open_memmap(D_CACHE, mode="w+", dtype=np.uint8, shape=(N_FULL, 224, 224, 3)); pos = 0
        for x, _ in DataLoader(_DCacheSource(), batch_size=256, num_workers=os.cpu_count() or 2):
            mm[pos:pos + len(x)] = x.numpy(); pos += len(x)
        mm.flush(); del mm; open(D_CACHE + ".done", "w").write("ok")
        print(f"Image cache built in {(time.time()-t0)/60:.1f} min")
    D_IMG_CACHE = np.load(D_CACHE, mmap_mode="r")
    def d_get_batch(start, end):
        x = torch.from_numpy(np.array(D_IMG_CACHE[start:end])).to(device, non_blocking=True)
        return x.permute(0, 3, 1, 2).float().div(255).sub(D_MEAN).div(D_STD)

# ---- the 12 Swin blocks; hooked output = attention output of each block ----
sw_blocks = []
for si, stage in enumerate(sw_model.swin.encoder.layers):
    for bj, blk in enumerate(stage.blocks):
        sw_blocks.append((f"S{si}B{bj}", blk, blk.attention))
assert len(sw_blocks) == 12, f"expected 12 Swin blocks, found {len(sw_blocks)}"
print("Swin blocks:", [b[0] for b in sw_blocks])

# ---- reference means: running per-channel sums over all tokens of the first 256 images ----
_acc = {}
def _mean_hook(n):
    def f(m, i, out):
        t = _out_t(out).detach().float().reshape(-1, _out_t(out).shape[-1])
        a = _acc.setdefault(n, [0.0, 0]); a[0] = a[0] + t.sum(0); a[1] += t.shape[0]
    return f
hs = [o.register_forward_hook(_mean_hook(n)) for n, _, o in sw_blocks]
try:
    with torch.no_grad():
        for s in range(0, 256, 64):
            sw_call(d_get_batch(s, s + 64))
finally:
    for h in hs: h.remove()
sw_means = {n: (a[0] / a[1]).view(1, 1, -1) for n, a in _acc.items()}
del _acc
print("Reference means ready:", len(sw_means), "blocks; channels per stage:",
      [sw_means[f"S{s}B0"].shape[-1] for s in range(4)])

# ---- baseline (saved once, reused after a restart) ----
if f"{PREFIX_SW}_baseline" not in set(list_checkpoints()):
    t0 = time.time()
    c, l, r = plain_eval(sw_call, d_get_batch, N_FULL)
    save_checkpoint(f"{PREFIX_SW}_baseline", {"correct_packed": pack(c), "lp_true": l, "rank": r, "n": N_FULL})
    print(f"Baseline pass: {(time.time()-t0)/60:.1f} min")
SW_BASE = load_checkpoint(f"{PREFIX_SW}_baseline")
SW_BASE_C = np.unpackbits(SW_BASE["correct_packed"])[:N_FULL].astype(bool)
SW_BASE_ACC = SW_BASE_C.mean()
print(f"Swin-T baseline: {SW_BASE_ACC*100:.3f}%   (published ≈ 81.2%)")
assert SW_BASE_ACC > 0.78, "Baseline too low - preprocessing or labels are wrong. Stop and send me this output."

# ---- safety check (about 1 min): fork vs plain hooks on 256 images, shifted-window block S1B1 ----
def _plain_hook(mean, perm=None):
    def f(m, i, out):
        is_t = isinstance(out, tuple); t = out[0] if is_t else out
        mv = mean.to(device=t.device, dtype=t.dtype).expand_as(t)
        rep = mv.clone() if perm is None else t + (mv - t)[..., perm.to(t.device)]
        return (rep,) + tuple(out[1:]) if is_t else rep
    return f
_bi = 3; _name, _blk, _outm = sw_blocks[_bi]
_perm = torch.randperm(sw_means[_name].shape[-1], generator=torch.Generator().manual_seed(42 + _bi))
cf, _, _ = fork_eval(sw_call, d_get_batch, 256, _blk, _outm, sw_means[_name],
                     [("none", "none", None), ("real", "real", None), ("shuf0", "shuf", _perm)])
ref = {}
for lab, perm in [("real", None), ("shuf0", _perm)]:
    h = _outm.register_forward_hook(_plain_hook(sw_means[_name], perm))
    try: ref[lab], _, _ = plain_eval(sw_call, d_get_batch, 256)
    finally: h.remove()
chk = {"none vs baseline": (cf[0] == SW_BASE_C[:256]).mean(),
       "real vs plain hook": (cf[1] == ref["real"]).mean(),
       "null vs plain hook": (cf[2] == ref["shuf0"]).mean()}
print("Safety check (256 images, block", _name + "):", {k: f"{v*100:.2f}%" for k, v in chk.items()})
assert min(chk.values()) >= 255 / 256, "Fork does not reproduce the plain evaluation - stop and send me this output."
print("\nSTEP 44B COMPLETE — safe to run Step 44C")

### Step 44C — Swin-T: real ablation vs shuffled null, per block (50K) ###
For each block, one pass with two copies of every batch after the block's input: the real mean ablation and the
shuffled null (seed 42 + block index, as for DeiT-Small and MambaVision-S). One checkpoint per block
(step44_swin_<block>), so the cell resumes after a restart. About 4 min per block, under 1 h in total.

In [ ]:
# ============================================================
# STEP 44C — SWIN-T: REAL ABLATION vs SHUFFLED NULL, PER BLOCK (50K)
# ============================================================
assert "sw_means" in globals() and "SW_BASE_ACC" in globals(), "Run Step 44B first."
done = set(list_checkpoints())
t_all = time.time()
for bi, (name, blk, outm) in enumerate(sw_blocks):
    ck = f"{PREFIX_SW}_{name}"
    if ck in done:
        print(f"{name}: already done"); continue
    t0 = time.time()
    perm = torch.randperm(sw_means[name].shape[-1], generator=torch.Generator().manual_seed(42 + bi))
    kinds = [("real", "real", None), ("shuf0", "shuf", perm)]
    corr, lpt, rank = fork_eval(sw_call, d_get_batch, N_FULL, blk, outm, sw_means[name], kinds)
    d_pp = (corr.mean(1) - SW_BASE_ACC) * 100
    save_checkpoint(ck, {"block": name, "type": "SwinAttention", "labels": [k[0] for k in kinds],
                         "delta_pp": d_pp.tolist(), "correct_packed": [pack(x) for x in corr],
                         "lp_true": lpt, "rank": rank})
    print(f"{name:5s} {(time.time()-t0)/60:4.1f} min | real {d_pp[0]:+.3f} pp | null {d_pp[1]:+.3f} pp | "
          f"real − null {d_pp[0]-d_pp[1]:+.3f}")
print(f"\nSTEP 44C COMPLETE ({(time.time()-t_all)/60:.1f} min this session)")

### Step 44D — Swin-T summary: content effect in three metrics (CPU only) ###
Content effect C per block (positive = real ablation worse than the null) in accuracy (pp), true-class log-probability
(nats) and log(1 + rank), with paired bootstrap 95% CIs (2,000 resamples) and Holm correction over the 12 blocks;
pooled C with 95% and 90% intervals and the ±0.10 pp equivalence test; Spearman correlation of |real| and |null|.
Compare with MambaVision-T (ρ = 0.83, pooled C within ±0.10 pp) and DeiT-Small (ρ = 0.57, |C| up to 8.1 pp).
Saved as step44_swin_summary. Send the full output of this cell.

In [ ]:
# ============================================================
# STEP 44D — SWIN-T SUMMARY: CONTENT EFFECT IN ACCURACY, LOG-PROB AND RANK
# ============================================================
assert all(f"{PREFIX_SW}_{n}" in set(list_checkpoints()) for n, _, _ in sw_blocks), "Step 44C is not finished yet."
n = N_FULL
idx = np.random.RandomState(42).randint(0, n, (NB_BOOT, n)).astype(np.int32)
base_c = SW_BASE_C.astype(np.float64)
rows, pooled = [], {"acc": [], "lp": [], "rank": []}
for name, _, _ in sw_blocks:
    d = load_checkpoint(f"{PREFIX_SW}_{name}")
    c = np.stack([np.unpackbits(p)[:n] for p in d["correct_packed"]]).astype(np.float64)
    lp = d["lp_true"].astype(np.float64); rk = np.log1p(d["rank"].astype(np.float64))
    row = {"block": name, "real_pp": float((c[0].mean() - base_c.mean()) * 100),
           "null_pp": float((c[1].mean() - base_c.mean()) * 100)}
    per = {"acc": (c[1] - c[0]) * 100, "lp": lp[1] - lp[0], "rank": rk[0] - rk[1]}
    for m, x in per.items():
        mu, ci, ci90, bm = _boot(x, idx)
        p = max(float(2 * min((bm <= 0).mean(), (bm >= 0).mean())), 1.0 / NB_BOOT)
        row[m] = {"C": mu, "ci": ci, "p": p}
        pooled[m].append(x)
    rows.append(row)
for m in pooled:
    for r, a in zip(rows, _holm([r[m]["p"] for r in rows])): r[m]["p_holm"] = float(a)
summ = {"rows": rows, "pooled": {}, "n": n, "baseline_acc": float(SW_BASE_ACC)}
for m, xs in pooled.items():
    mu, ci, ci90, _ = _boot(np.mean(xs, 0), idx)
    summ["pooled"][m] = {"C": mu, "ci": ci, "ci90": ci90}
summ["spearman_real_null"] = float(spearmanr([abs(r["real_pp"]) for r in rows], [abs(r["null_pp"]) for r in rows])[0])
save_checkpoint(f"{PREFIX_SW}_summary", summ)

f = lambda v: f"{v['C']:+.3f} [{v['ci'][0]:+.3f},{v['ci'][1]:+.3f}]{'*' if v['p_holm'] < 0.05 else ' '}"
print(f"Swin-T baseline {SW_BASE_ACC*100:.3f}%   (* = Holm p < 0.05; C > 0 = real ablation worse)\n")
print(f"{'Block':6s}{'real (pp)':>10s}{'null (pp)':>10s}{'C acc (pp)':>30s}{'C log-prob (nats)':>30s}{'C rank':>30s}")
for r in rows:
    print(f"{r['block']:6s}{r['real_pp']:+10.3f}{r['null_pp']:+10.3f}{f(r['acc']):>30s}{f(r['lp']):>30s}{f(r['rank']):>30s}")
print()
for m, unit in [("acc", "pp"), ("lp", "nats"), ("rank", "log(1+rank)")]:
    v = summ["pooled"][m]
    print(f"pooled C {m:4s} ({unit}): {v['C']:+.4f}  95% [{v['ci'][0]:+.4f},{v['ci'][1]:+.4f}]  "
          f"90% [{v['ci90'][0]:+.4f},{v['ci90'][1]:+.4f}]" +
          (f"  within ±0.10 pp: {-0.10 < v['ci90'][0] and v['ci90'][1] < 0.10}" if m == "acc" else ""))
n_rw = sum(r["acc"]["p_holm"] < 0.05 and r["acc"]["C"] > 0 for r in rows)
n_nw = sum(r["acc"]["p_holm"] < 0.05 and r["acc"]["C"] < 0 for r in rows)
print(f"\nHolm-significant blocks (accuracy): real worse {n_rw}, null worse {n_nw}; "
      f"max |C| = {max(abs(r['acc']['C']) for r in rows):.2f} pp")
print(f"Spearman |real| vs |null|: {summ['spearman_real_null']:.2f}   [MambaVision-T 0.83 | DeiT-Small 0.57]")
print("\nSTEP 44D COMPLETE")

## Step 45 — DeiT-Small: ten null permutations per layer ##
Brings the DeiT-Small control to the standard of MambaVision-T: for each of the 12 attention layers, the real mean
ablation and ten size-matched shuffled nulls on all 50,000 images (perm 0 = Step 34 seed 42 + layer; perms 1-9 =
1000 + 100·layer + j), with per-image correctness, true-class log-probability and true-class rank.

### Step 45A — Shared setup and fast fork runner (definitions only) ###
Self-contained: labels in the fixed order (checked against the dataset), image caches, reference means, the fork
evaluation (all copies share the forward pass up to the ablated block), a 512-image pre-check, a resumable per-block
runner with reproduction checks and ETA, and the summary (paired bootstrap, Holm, pooled 95%/90% intervals, ±0.10 pp
equivalence test, Spearman). Reused by Steps 46-50. No GPU work.

In [ ]:
# ============================================================
# STEP 45A — SHARED SETUP + FAST FORK RUNNER (DEFINITIONS, NO GPU WORK)
# ============================================================
import os, shutil, time, numpy as np, torch, torch.nn.functional as F
from scipy.stats import spearmanr
assert "load_checkpoint" in globals() and "dataset" in globals() and "GLOBAL_RANDOM_INDICES" in globals(), \
    "Run the HF checkpoint cell, Step 0, Step 2, Step 5 and Step 5b first."
device = torch.device("cuda")
torch.backends.cudnn.benchmark = True
N_FULL = 50000
eval_indices = GLOBAL_RANDOM_INDICES[:N_FULL]

# ---- labels in the fixed order (from the saved Step 6 checkpoint; checked against the dataset) ----
if "LAB" not in globals():
    _b = load_checkpoint("baseline_logits_probs_new")
    LAB = np.asarray(torch.as_tensor(_b["eval_labels"]).numpy()).astype(np.int64); del _b
_chk = np.random.RandomState(0).choice(N_FULL, 100, replace=False)
assert all(int(dataset[eval_indices[i]]["label"]) == int(LAB[i]) for i in _chk), "Label order mismatch - stop."
_LAB_GPU = torch.as_tensor(LAB, device=device)

def pk(c):  return np.packbits(np.asarray(c, dtype=bool))
def upk(p, n=N_FULL): return np.unpackbits(np.asarray(p, dtype=np.uint8))[:n].astype(bool)

# ---- folder for image caches ----
if "CACHE_DIR" not in globals() or CACHE_DIR is None:
    CACHE_DIR = None
    for d in ["/kaggle/temp", "/tmp", "/kaggle/working"]:
        try:
            os.makedirs(d, exist_ok=True)
            if shutil.disk_usage(d).free / 1e9 > 9 and CACHE_DIR is None: CACHE_DIR = d
        except Exception:
            pass
    assert CACHE_DIR is not None, "Need about 9 GB of free disk for the image cache."
print("Cache folder:", CACHE_DIR)

def make_cache(tag, pre_transform, mean, std):
    """uint8 cache of the 50K images (fixed order) after pre_transform; returns get_batch(start, end) -> normalised GPU batch."""
    from torch.utils.data import Dataset as _TD, DataLoader
    path = os.path.join(CACHE_DIR, f"mv_cache_{tag}_images_u8_{N_FULL}.npy")
    if not os.path.exists(path + ".done"):
        class _Src(_TD):
            def __len__(self): return N_FULL
            def __getitem__(self, pos):
                return np.asarray(pre_transform(dataset[eval_indices[pos]]["image"].convert("RGB")), dtype=np.uint8).copy()
        t0 = time.time()
        mm = np.lib.format.open_memmap(path, mode="w+", dtype=np.uint8, shape=(N_FULL, 224, 224, 3)); pos = 0
        for x in DataLoader(_Src(), batch_size=256, num_workers=os.cpu_count() or 2):
            mm[pos:pos + len(x)] = x.numpy(); pos += len(x)
        mm.flush(); del mm; open(path + ".done", "w").write("ok")
        print(f"cache {tag} built in {(time.time()-t0)/60:.1f} min")
    arr = np.load(path, mmap_mode="r")
    m = torch.tensor(mean, device=device).view(1, 3, 1, 1); s = torch.tensor(std, device=device).view(1, 3, 1, 1)
    def get_batch(start, end):
        x = torch.from_numpy(np.ascontiguousarray(arr[start:end])).to(device, non_blocking=True)
        return x.permute(0, 3, 1, 2).float().div(255).sub(m).div(s)
    return get_batch

def _ot(out): return out[0] if isinstance(out, tuple) else out

def ref_means(model_call, get_batch, outs, n_ref=256, bs=64):
    """Per-channel mean of each hooked output over all tokens of the first n_ref images (running sums)."""
    acc = {}
    def hk(n):
        def f(m, i, out):
            t = _ot(out).detach().float().reshape(-1, _ot(out).shape[-1])
            a = acc.setdefault(n, [0.0, 0]); a[0] = a[0] + t.sum(0); a[1] += t.shape[0]
        return f
    hs = [o.register_forward_hook(hk(n)) for n, o in outs]
    try:
        with torch.no_grad():
            for s in range(0, n_ref, bs): model_call(get_batch(s, min(s + bs, n_ref)))
    finally:
        for h in hs: h.remove()
    return {n: (a[0] / a[1]).view(1, 1, -1) for n, a in acc.items()}

def fork_eval(model_call, get_batch, n, block_module, out_module, mean, kinds, bs, gamma=None):
    """kinds: (label, 'none'|'real'|'shuf'|'sign', perm/sign/None). One pass; copies share the prefix up to the block."""
    K = len(kinds)
    def pre(module, args, kwargs):
        if len(args) > 0:
            x = args[0]; return (x.repeat(K, *([1] * (x.dim() - 1))),) + tuple(args[1:]), kwargs
        kwargs = dict(kwargs); x = kwargs["hidden_states"]
        kwargs["hidden_states"] = x.repeat(K, *([1] * (x.dim() - 1))); return args, kwargs
    def post(module, inp, out):
        is_t = isinstance(out, tuple); t = out[0] if is_t else out
        R = t.shape[0] // K; mv = mean.to(device=t.device, dtype=t.dtype)
        g = None if gamma is None else gamma.to(device=t.device, dtype=t.dtype)
        pieces = []
        for j, (lab, kind, p) in enumerate(kinds):
            tj = t[j * R:(j + 1) * R]
            if kind == "none":   pieces.append(tj)
            elif kind == "real": pieces.append(mv.expand_as(tj).clone())
            elif kind == "shuf":
                d = mv.expand_as(tj) - tj; pp = p.to(t.device)
                pieces.append(tj + (d[..., pp] if g is None else (g * d)[..., pp] / g))
            elif kind == "sign":
                pieces.append(tj + (mv.expand_as(tj) - tj) * p.to(device=t.device, dtype=t.dtype))
        rep = torch.cat(pieces, 0)
        return (rep,) + tuple(out[1:]) if is_t else rep
    h1 = block_module.register_forward_pre_hook(pre, with_kwargs=True)
    h2 = out_module.register_forward_hook(post)
    corr = np.zeros((K, n), bool); lpt = np.zeros((K, n), np.float32); rank = np.zeros((K, n), np.int16)
    try:
        with torch.no_grad():
            for s in range(0, n, bs):
                e = min(s + bs, n); B = e - s
                logits = model_call(get_batch(s, e)).float().view(K, B, -1)
                y = _LAB_GPU[s:e].view(1, B, 1).expand(K, B, 1)
                corr[:, s:e] = (logits.argmax(-1) == y.squeeze(2)).cpu().numpy()
                lpt[:, s:e] = F.log_softmax(logits, -1).gather(2, y).squeeze(2).cpu().numpy()
                rank[:, s:e] = (logits > logits.gather(2, y)).sum(-1).cpu().numpy().astype(np.int16)
    finally:
        h1.remove(); h2.remove()
    return corr, lpt, rank

def plain_eval(model_call, get_batch, n, bs=128):
    corr = np.zeros(n, bool); lpt = np.zeros(n, np.float32); rank = np.zeros(n, np.int16)
    with torch.no_grad():
        for s in range(0, n, bs):
            e = min(s + bs, n); logits = model_call(get_batch(s, e)).float(); y = _LAB_GPU[s:e].view(-1, 1)
            corr[s:e] = (logits.argmax(-1) == y.squeeze(1)).cpu().numpy()
            lpt[s:e] = F.log_softmax(logits, -1).gather(1, y).squeeze(1).cpu().numpy()
            rank[s:e] = (logits > logits.gather(1, y)).sum(-1).cpu().numpy().astype(np.int16)
    return corr, lpt, rank

def get_baseline(prefix, model_call, get_batch, n=N_FULL):
    ck = f"{prefix}_baseline"
    if ck not in set(list_checkpoints()):
        t0 = time.time(); c, l, r = plain_eval(model_call, get_batch, n)
        save_checkpoint(ck, {"correct_packed": pk(c), "lp_true": l, "rank": r, "n": n})
        print(f"baseline pass {(time.time()-t0)/60:.1f} min")
    b = load_checkpoint(ck); return upk(b["correct_packed"], n)

def pre_check(model_call, get_batch, blk, outm, mean, kinds, refs, n=512, bs=64, gamma=None):
    """Fork on the first n images must reproduce stored per-image results (refs: label -> packed array or bool array)."""
    c, _, _ = fork_eval(model_call, get_batch, n, blk, outm, mean, kinds, bs, gamma)
    labs = [k[0] for k in kinds]; res = {}
    for lab, ref in refs.items():
        r = ref[:n] if (isinstance(ref, np.ndarray) and ref.dtype == bool) else upk(ref, N_FULL)[:n]
        res[lab] = float((c[labs.index(lab)] == r).mean())
    print("pre-check:", {k: f"{v*100:.2f}%" for k, v in res.items()})
    assert min(res.values()) >= 1 - 2 / n, "Fork does not reproduce the earlier run - stop and send me this output."

def run_blocks(prefix, model_call, get_batch, n, blocks, means, kinds_fn, base_c, bs, k_chunk, checks=None):
    """blocks: (name, block_module, out_module, gamma or None). One checkpoint per block, resumable, with ETA."""
    done = set(list_checkpoints()); todo = [b for b in blocks if f"{prefix}_{b[0]}" not in done]
    print(f"{prefix}: {len(blocks)-len(todo)} done, {len(todo)} to run"); times = []
    for name, blk, outm, gamma in blocks:
        if f"{prefix}_{name}" in done: continue
        bi = [b[0] for b in blocks].index(name); t0 = time.time()
        kinds = kinds_fn(bi, name, means[name].shape[-1])
        parts = [fork_eval(model_call, get_batch, n, blk, outm, means[name], kinds[s:s + k_chunk], bs, gamma)
                 for s in range(0, len(kinds), k_chunk)]
        corr = np.concatenate([p[0] for p in parts]); lpt = np.concatenate([p[1] for p in parts]); rank = np.concatenate([p[2] for p in parts])
        labs = [k[0] for k in kinds]; d_pp = (corr.mean(1) - base_c.mean()) * 100
        msg = ""
        if checks is not None:
            for lab, ref in checks(name).items():
                if ref is None: continue
                agr = float((corr[labs.index(lab)] == upk(ref, n)).mean()); msg += f" | {lab} agree {agr*100:.3f}%"
                assert agr > 0.999, f"{name} {lab} does not reproduce the earlier run - stop and send me this output."
        save_checkpoint(f"{prefix}_{name}", {"block": name, "labels": labs, "delta_pp": d_pp.tolist(),
                        "correct_packed": [pk(x) for x in corr], "lp_true": lpt, "rank": rank, "n": n})
        times.append(time.time() - t0); left = len(todo) - len(times)
        sh = [d for l, d in zip(labs, d_pp) if l.startswith("shuf")]
        print(f"{name:6s} {times[-1]/60:4.1f} min | real {d_pp[0]:+.3f} | null mean {np.mean(sh):+.3f} SD {np.std(sh, ddof=1) if len(sh) > 1 else 0:.3f}"
              f"{msg} | ETA {np.mean(times)*left/60:.0f} min")
    print(f"{prefix}: all blocks done")

NB_BOOT = 2000
def _bm(x, idx):
    x = np.asarray(x, np.float32); out = np.empty(len(idx))
    for s in range(0, len(idx), 200): out[s:s + 200] = x[idx[s:s + 200]].mean(1)
    return out
def _holm(ps):
    ps = np.asarray(ps); o = np.argsort(ps); m = len(ps); adj = np.empty(m); run = 0.0
    for r, i in enumerate(o):
        run = max(run, (m - r) * ps[i]); adj[i] = min(1.0, run)
    return adj

def summarize(prefix, names, base_c, n=N_FULL, margin=0.10, fam="shuf"):
    """Content effect vs the mean of all `fam` copies, in accuracy (pp), log-prob (nats) and log(1+rank)."""
    idx = np.random.RandomState(42).randint(0, n, (NB_BOOT, n)).astype(np.int32)
    bc = base_c.astype(np.float64); rows, pooled = [], {"acc": [], "lp": [], "rank": []}
    for name in names:
        d = load_checkpoint(f"{prefix}_{name}"); L = d["labels"]
        c = np.stack([upk(p, n) for p in d["correct_packed"]]).astype(np.float64)
        lp = d["lp_true"].astype(np.float64); rk = np.log1p(d["rank"].astype(np.float64))
        ii = [i for i, l in enumerate(L) if l.startswith(fam)]
        each = (c[ii].mean(1) - bc.mean()) * 100
        row = {"block": name, "real_pp": float((c[0].mean() - bc.mean()) * 100), "null_pp": float(each.mean()),
               "null_sd": float(each.std(ddof=1)) if len(ii) > 1 else 0.0, "null_each": each.tolist(), "n_perm": len(ii)}
        per = {"acc": (c[ii].mean(0) - c[0]) * 100, "lp": lp[ii].mean(0) - lp[0], "rank": rk[0] - rk[ii].mean(0)}
        for m, x in per.items():
            bm = _bm(x, idx); p = max(float(2 * min((bm <= 0).mean(), (bm >= 0).mean())), 1.0 / NB_BOOT)
            row[m] = {"C": float(x.mean()), "ci": np.percentile(bm, [2.5, 97.5]).tolist(), "p": p}
            pooled[m].append(x)
        rows.append(row)
    for m in pooled:
        for r, a in zip(rows, _holm([r[m]["p"] for r in rows])): r[m]["p_holm"] = float(a)
    summ = {"rows": rows, "pooled": {}, "n": n, "fam": fam}
    for m, xs in pooled.items():
        x = np.mean(xs, 0); bm = _bm(x, idx)
        summ["pooled"][m] = {"C": float(x.mean()), "ci": np.percentile(bm, [2.5, 97.5]).tolist(), "ci90": np.percentile(bm, [5, 95]).tolist()}
    summ["spearman"] = float(spearmanr([abs(r["real_pp"]) for r in rows], [abs(r["null_pp"]) for r in rows])[0])
    save_checkpoint(f"{prefix}_summary", summ)
    f = lambda v: f"{v['C']:+.3f} [{v['ci'][0]:+.3f},{v['ci'][1]:+.3f}]{'*' if v['p_holm'] < 0.05 else ' '}"
    print(f"{prefix}: {rows[0]['n_perm']} null permutations per block (* = Holm p < 0.05; C > 0 = real ablation worse)\n")
    print(f"{'Block':6s}{'real':>8s}{'null mean':>10s}{'null SD':>8s}{'C acc (pp)':>28s}{'C log-prob':>28s}{'C rank':>28s}")
    for r in rows:
        print(f"{r['block']:6s}{r['real_pp']:+8.3f}{r['null_pp']:+10.3f}{r['null_sd']:8.3f}{f(r['acc']):>28s}{f(r['lp']):>28s}{f(r['rank']):>28s}")
    print()
    for m, unit in [("acc", "pp"), ("lp", "nats"), ("rank", "log(1+rank)")]:
        v = summ["pooled"][m]
        print(f"pooled C {m:4s} ({unit}): {v['C']:+.4f}  95% [{v['ci'][0]:+.4f},{v['ci'][1]:+.4f}]  90% [{v['ci90'][0]:+.4f},{v['ci90'][1]:+.4f}]"
              + (f"  within ±{margin} pp: {-margin < v['ci90'][0] and v['ci90'][1] < margin}" if m == "acc" else ""))
    rw = sum(r["acc"]["p_holm"] < 0.05 and r["acc"]["C"] > 0 for r in rows); nw = sum(r["acc"]["p_holm"] < 0.05 and r["acc"]["C"] < 0 for r in rows)
    print(f"\nHolm-significant (accuracy): real worse {rw}, null worse {nw}; max |C| {max(abs(r['acc']['C']) for r in rows):.2f} pp; "
          f"null SD per block {min(r['null_sd'] for r in rows):.3f}-{max(r['null_sd'] for r in rows):.3f} pp")
    print(f"Spearman |real| vs |null mean|: {summ['spearman']:.2f}")
    return summ

def kinds_shuf(bi, C, n_shuf=10, seed0=None, real=True):
    """real + shuf0..shuf{n-1}; perm 0 = seed0 (the model's original seed), perm j >= 1 = 1000 + 100*bi + j."""
    ks = [("real", "real", None)] if real else []
    for j in range(n_shuf):
        g = torch.Generator().manual_seed(seed0 if j == 0 else 1000 + 100 * bi + j)
        ks.append((f"shuf{j}", "shuf", torch.randperm(C, generator=g)))
    return ks
print("STEP 45A READY")

### Step 45B — DeiT-Small: load, image cache, reference means, baseline, pre-check ###
Loads facebook/deit-small-patch16-224 with the Step 34 preprocessing (bicubic 248, centre crop 224; same cache as
Swin-T), computes the per-channel reference means of each attention output on the first 256 images, and saves the
baseline with per-image log-probability and rank. Checks: the baseline matches Step 34 (≥ 99.9% of images) and, on the
first 512 images of layer 0, the real ablation and perm 0 reproduce Step 34B. About 3 min (+10 min if the cache is new).

In [ ]:
# ============================================================
# STEP 45B — DEIT-SMALL: LOAD, CACHE, REFERENCE MEANS, BASELINE, PRE-CHECK
# ============================================================
from transformers import AutoModelForImageClassification
from torchvision import transforms as T
assert "fork_eval" in globals(), "Run Step 45A first."
DEIT_ID, PREFIX_D = "facebook/deit-small-patch16-224", "step45_deit"
d_model = AutoModelForImageClassification.from_pretrained(DEIT_ID).to(device).eval()
_d_base = getattr(d_model, "vit", None) or getattr(d_model, "deit")
d_layers = list(_d_base.encoder.layer); assert len(d_layers) == 12
d_call = lambda x: d_model(pixel_values=x).logits
d_get = make_cache("DEIT", T.Compose([T.Resize(248, interpolation=T.InterpolationMode.BICUBIC), T.CenterCrop(224)]),
                   [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])        # same images as Step 34 and Swin-T
d_blocks = [(f"L{i}", layer, layer.attention, None) for i, layer in enumerate(d_layers)]
d_means = ref_means(d_call, d_get, [(b[0], b[2]) for b in d_blocks])

res34 = load_checkpoint("step34_deit_protocol")
D_BASE_C = get_baseline(PREFIX_D, d_call, d_get)
agree_base = (D_BASE_C == upk(res34["baseline"]["correct_packed"])).mean()
print(f"DeiT-Small baseline {D_BASE_C.mean()*100:.3f}% | agreement with Step 34: {agree_base*100:.3f}%")
assert agree_base > 0.999, "Baseline differs from Step 34 - stop and send me this output."

# pre-check on 512 images, layer 0: real and perm 0 must reproduce Step 34B
pre_check(d_call, d_get, d_layers[0], d_layers[0].attention, d_means["L0"], kinds_shuf(0, d_means["L0"].shape[-1], 1, seed0=42),
          {"real": res34["layers"]["L0"]["real"]["correct_packed"], "shuf0": res34["layers"]["L0"]["null"]["correct_packed"]})
print("\nSTEP 45B COMPLETE — safe to run Step 45C")

### Step 45C — DeiT-Small: real ablation + ten shuffled nulls per layer (50K) ###
For each layer, one pass with 11 copies of every batch after the layer's input (real, shuf0-shuf9; 96 images per
batch). After each layer the real ablation and perm 0 are compared with Step 34B on all 50,000 images (≥ 99.9%).
One checkpoint per layer (step45_deit_L<i>); resumes after a restart. About 6-8 min per layer, ETA printed.

In [ ]:
# ============================================================
# STEP 45C — DEIT-SMALL: REAL + TEN SHUFFLED NULLS PER LAYER (50K)
# ============================================================
assert "d_means" in globals() and "D_BASE_C" in globals(), "Run Step 45B first."
run_blocks(PREFIX_D, d_call, d_get, N_FULL, d_blocks, d_means,
           kinds_fn=lambda bi, name, C: kinds_shuf(bi, C, 10, seed0=42 + bi), base_c=D_BASE_C, bs=96, k_chunk=11,
           checks=lambda name: {"real": res34["layers"][name]["real"]["correct_packed"],
                                "shuf0": res34["layers"][name]["null"]["correct_packed"]})
print("\nSTEP 45C COMPLETE")

### Step 45D — DeiT-Small summary (CPU only) ###
Content effect per layer against the mean of the ten nulls, in accuracy (pp), true-class log-probability (nats) and
log(1 + rank), with paired bootstrap 95% CIs and Holm correction; pooled C with 95% and 90% intervals; the null's own
SD per layer; Spearman correlation; and the one-permutation C of Step 34 next to the ten-permutation C.
Saved as step45_deit_summary. Send the full output of this cell.

In [ ]:
# ============================================================
# STEP 45D — DEIT-SMALL SUMMARY (CPU ONLY)
# ============================================================
summ_d = summarize(PREFIX_D, [b[0] for b in d_blocks], D_BASE_C)
old = {k: res34["layers"][k]["real_minus_null_pp"] for k in res34["layers"]}
print("\nOne-permutation C (Step 34) vs ten-permutation C (accuracy, pp):")
print("  " + " | ".join(f"{r['block']} {-old[r['block']]:+.2f}→{r['acc']['C']:+.2f}" for r in summ_d["rows"]))
print("\nSTEP 45D COMPLETE — send me the full output.")

## Step 46 — Swin-T: ten null permutations per block, on both GPUs ##
Brings the Swin-T control to ten shuffled nulls per block (perm 0 = Step 44, seed 42 + block; perms 1-9 = 1000 +
100·block + j), with per-image correctness, log-probability and rank. The Kaggle session has two Tesla T4 GPUs; each
holds its own copy of the model, and blocks are taken from a shared queue (most expensive first), so both GPUs work in
parallel.

### Step 46A — Two-GPU block runner (definitions only) ###
One worker thread per GPU. Each worker takes the next block from the queue, runs the real ablation and the shuffled
nulls as copies of each batch after the block's input, checks real and perm 0 against the earlier run, and saves one
checkpoint per block (resumable). Uploads and printing are serialised with a lock; any error stops both workers.
Requires Step 45A in the same session. No GPU work.

In [ ]:
# ============================================================
# STEP 46A — TWO-GPU BLOCK RUNNER (DEFINITIONS ONLY)
# ============================================================
import threading, queue
assert "fork_eval" in globals() and "kinds_shuf" in globals(), "Run Step 45A first."
N_GPU = torch.cuda.device_count()
print("GPUs:", N_GPU, [torch.cuda.get_device_name(i) for i in range(N_GPU)])
DEVS = [torch.device(f"cuda:{i}") for i in range(N_GPU)]
LAB_DEV = {d: torch.as_tensor(LAB, device=d) for d in DEVS}

def cache_getter(tag, mean, std, dev):
    """get_batch on device `dev` for an existing cache built by make_cache(tag, ...)."""
    path = os.path.join(CACHE_DIR, f"mv_cache_{tag}_images_u8_{N_FULL}.npy")
    assert os.path.exists(path + ".done"), f"cache {tag} missing - run make_cache first"
    arr = np.load(path, mmap_mode="r")
    m = torch.tensor(mean, device=dev).view(1, 3, 1, 1); s = torch.tensor(std, device=dev).view(1, 3, 1, 1)
    def get_batch(start, end):
        x = torch.from_numpy(np.array(arr[start:end])).to(dev, non_blocking=True)
        return x.permute(0, 3, 1, 2).float().div(255).sub(m).div(s)
    return get_batch

def fork_eval_dev(model_call, get_batch, n, blk, outm, mean, kinds, bs, dev, gamma=None):
    """Same as fork_eval (Step 45A) but on device `dev` with that device's labels."""
    K = len(kinds); lab = LAB_DEV[dev]; mean_d = mean.to(dev)
    g_d = None if gamma is None else gamma.to(dev)
    def pre(module, args, kwargs):
        if len(args) > 0:
            x = args[0]; return (x.repeat(K, *([1] * (x.dim() - 1))),) + tuple(args[1:]), kwargs
        kwargs = dict(kwargs); x = kwargs["hidden_states"]
        kwargs["hidden_states"] = x.repeat(K, *([1] * (x.dim() - 1))); return args, kwargs
    def post(module, inp, out):
        is_t = isinstance(out, tuple); t = out[0] if is_t else out
        R = t.shape[0] // K; mv = mean_d.to(dtype=t.dtype); pieces = []
        for j, (l, kind, p) in enumerate(kinds):
            tj = t[j * R:(j + 1) * R]
            if kind == "none":   pieces.append(tj)
            elif kind == "real": pieces.append(mv.expand_as(tj).clone())
            elif kind == "shuf":
                d = mv.expand_as(tj) - tj; pp = p.to(t.device)
                pieces.append(tj + (d[..., pp] if g_d is None else (g_d * d)[..., pp] / g_d))
            elif kind == "sign":
                pieces.append(tj + (mv.expand_as(tj) - tj) * p.to(device=t.device, dtype=t.dtype))
        rep = torch.cat(pieces, 0)
        return (rep,) + tuple(out[1:]) if is_t else rep
    h1 = blk.register_forward_pre_hook(pre, with_kwargs=True); h2 = outm.register_forward_hook(post)
    corr = np.zeros((K, n), bool); lpt = np.zeros((K, n), np.float32); rank = np.zeros((K, n), np.int16)
    try:
        with torch.no_grad(), torch.cuda.device(dev):
            for s in range(0, n, bs):
                e = min(s + bs, n); B = e - s
                logits = model_call(get_batch(s, e)).float().view(K, B, -1)
                y = lab[s:e].view(1, B, 1).expand(K, B, 1)
                corr[:, s:e] = (logits.argmax(-1) == y.squeeze(2)).cpu().numpy()
                lpt[:, s:e] = F.log_softmax(logits, -1).gather(2, y).squeeze(2).cpu().numpy()
                rank[:, s:e] = (logits > logits.gather(2, y)).sum(-1).cpu().numpy().astype(np.int16)
    finally:
        h1.remove(); h2.remove()
    return corr, lpt, rank

def run_blocks_2gpu(prefix, setups, names, means, kinds_fn, base_c, bs, k_chunk, checks=None, n=N_FULL):
    """setups: one dict per GPU: {'dev', 'call', 'get', 'blocks': {name: (blk, outm, gamma)}}. Queue order = names order."""
    done = set(list_checkpoints()); todo = [nm for nm in names if f"{prefix}_{nm}" not in done]
    print(f"{prefix}: {len(names)-len(todo)} done, {len(todo)} to run on {len(setups)} GPU(s)")
    q = queue.Queue(); [q.put(nm) for nm in todo]
    lock = threading.Lock(); errors = []; t_start = time.time(); n_done = [0]
    def worker(S):
        torch.cuda.set_device(S["dev"])
        while not errors:
            try: name = q.get_nowait()
            except queue.Empty: return
            try:
                t0 = time.time(); bi = names.index(name); blk, outm, gamma = S["blocks"][name]
                kinds = kinds_fn(bi, name, means[name].shape[-1])
                parts = [fork_eval_dev(S["call"], S["get"], n, blk, outm, means[name], kinds[s:s + k_chunk], bs, S["dev"], gamma)
                         for s in range(0, len(kinds), k_chunk)]
                corr = np.concatenate([p[0] for p in parts]); lpt = np.concatenate([p[1] for p in parts])
                rank = np.concatenate([p[2] for p in parts]); labs = [k[0] for k in kinds]
                d_pp = (corr.mean(1) - base_c.mean()) * 100; msg = ""
                if checks is not None:
                    for lab_, ref in checks(name).items():
                        if ref is None: continue
                        agr = float((corr[labs.index(lab_)] == upk(ref, n)).mean()); msg += f" | {lab_} agree {agr*100:.3f}%"
                        if agr <= 0.999: raise AssertionError(f"{name} {lab_} agreement {agr*100:.3f}% - stop and send me this output.")
                with lock:
                    save_checkpoint(f"{prefix}_{name}", {"block": name, "labels": labs, "delta_pp": d_pp.tolist(),
                                    "correct_packed": [pk(x) for x in corr], "lp_true": lpt, "rank": rank, "n": n})
                    n_done[0] += 1; sh = [d for l, d in zip(labs, d_pp) if l.startswith("shuf")]
                    el = (time.time() - t_start) / 60
                    print(f"[{S['dev']}] {name:6s} {(time.time()-t0)/60:4.1f} min | real {d_pp[0]:+.3f} | null mean {np.mean(sh):+.3f} "
                          f"SD {np.std(sh, ddof=1) if len(sh) > 1 else 0:.3f}{msg} | {n_done[0]}/{len(todo)} done, {el:.0f} min elapsed")
            except Exception as ex:
                with lock: errors.append(f"[{S['dev']}] {name}: {ex!r}")
                return
    ts = [threading.Thread(target=worker, args=(S,)) for S in setups]
    [t.start() for t in ts]; [t.join() for t in ts]
    if errors: raise RuntimeError("Stopped: " + " | ".join(errors))
    print(f"{prefix}: all blocks done in {(time.time()-t_start)/60:.1f} min")
print("STEP 46A READY")

### Step 46B — Swin-T on both GPUs: load, reference means, pre-check ###
Loads one copy of Swin-T per GPU, reuses the DEIT image cache (same images as Steps 44 and 45) and the Step 44
baseline, and recomputes the reference means on the first 256 images. Pre-check on the first 512 images of block S1B1
(shifted windows), separately on each GPU: the unmodified copy must match the Step 44 baseline, and the real ablation and
perm 0 must match Step 44C. About 2 min.

In [ ]:
# ============================================================
# STEP 46B — SWIN-T ON BOTH GPUs: LOAD, REFERENCE MEANS, PRE-CHECK
# ============================================================
from transformers import AutoModelForImageClassification
from torchvision import transforms as T
assert "run_blocks_2gpu" in globals(), "Run Step 46A first."
SW_ID, PREFIX_SW10 = "microsoft/swin-tiny-patch4-window7-224", "step46_swin"
IMN = ([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
if "d_get" not in globals():                      # builds the DEIT cache only if this is a new session
    d_get = make_cache("DEIT", T.Compose([T.Resize(248, interpolation=T.InterpolationMode.BICUBIC), T.CenterCrop(224)]), *IMN)

SW_SETUPS = []
for dev in DEVS:
    m = AutoModelForImageClassification.from_pretrained(SW_ID).to(dev).eval()
    blocks = {f"S{si}B{bj}": (b, b.attention, None) for si, st in enumerate(m.swin.encoder.layers) for bj, b in enumerate(st.blocks)}
    SW_SETUPS.append({"dev": dev, "model": m, "call": (lambda mm: (lambda x: mm(pixel_values=x).logits))(m),
                      "get": cache_getter("DEIT", *IMN, dev), "blocks": blocks})
SW_NAMES = list(SW_SETUPS[0]["blocks"].keys()); assert len(SW_NAMES) == 12
S0 = SW_SETUPS[0]
with torch.cuda.device(S0["dev"]):
    sw_means10 = ref_means(S0["call"], S0["get"], [(nm, S0["blocks"][nm][1]) for nm in SW_NAMES])

SW_BASE_C = upk(load_checkpoint("step44_swin_baseline")["correct_packed"])
print(f"Swin-T baseline (Step 44): {SW_BASE_C.mean()*100:.3f}%")
ref44 = {nm: load_checkpoint(f"step44_swin_{nm}") for nm in SW_NAMES}
for S in SW_SETUPS:                               # pre-check on each GPU, block S1B1 (index 3)
    nm = "S1B1"; blk, outm, _ = S["blocks"][nm]
    ks = [("none", "none", None)] + kinds_shuf(3, sw_means10[nm].shape[-1], 1, seed0=42 + 3)
    c, _, _ = fork_eval_dev(S["call"], S["get"], 512, blk, outm, sw_means10[nm], ks, 64, S["dev"])
    res = {"none vs baseline": (c[0] == SW_BASE_C[:512]).mean(),
           "real vs Step 44": (c[1] == upk(ref44[nm]["correct_packed"][0])[:512]).mean(),
           "shuf0 vs Step 44": (c[2] == upk(ref44[nm]["correct_packed"][1])[:512]).mean()}
    print(f"pre-check {S['dev']}:", {k: f"{v*100:.2f}%" for k, v in res.items()})
    assert min(res.values()) >= 1 - 2 / 512, "Pre-check failed - stop and send me this output."
print("\nSTEP 46B COMPLETE — safe to run Step 46C")

### Step 46C — Swin-T: real + ten shuffled nulls per block, both GPUs (50K) ###
Eleven copies per batch (real, shuf0-shuf9; 48 images per batch), blocks shared between the two GPUs through a queue,
most expensive blocks first. After each block, real and perm 0 are compared with Step 44C on all 50,000 images
(≥ 99.9%). One checkpoint per block (step46_swin_<block>); resumes after a restart. About 2 h.

In [ ]:
# ============================================================
# STEP 46C — SWIN-T: REAL + TEN SHUFFLED NULLS PER BLOCK, BOTH GPUs (50K)
# ============================================================
assert "SW_SETUPS" in globals() and "sw_means10" in globals(), "Run Step 46B first."
run_blocks_2gpu(PREFIX_SW10, SW_SETUPS, SW_NAMES, sw_means10,
                kinds_fn=lambda bi, name, C: kinds_shuf(bi, C, 10, seed0=42 + bi),
                base_c=SW_BASE_C, bs=48, k_chunk=11,
                checks=lambda name: {"real": ref44[name]["correct_packed"][0], "shuf0": ref44[name]["correct_packed"][1]})
print("\nSTEP 46C COMPLETE")

### Step 46D — Swin-T summary, ten permutations (CPU only) ###
Content effect per block against the mean of the ten nulls in accuracy, log-probability and log(1 + rank), with
paired bootstrap CIs, Holm correction, pooled 95%/90% intervals, the null's own SD per block and the Spearman
correlation, plus the one-permutation C of Step 44 next to the ten-permutation C. Saved as step46_swin_summary.
Send the full output of this cell.

In [ ]:
# ============================================================
# STEP 46D — SWIN-T SUMMARY, TEN PERMUTATIONS (CPU ONLY)
# ============================================================
summ_sw10 = summarize(PREFIX_SW10, SW_NAMES, SW_BASE_C)
old44 = {r["block"]: r["acc"]["C"] for r in load_checkpoint("step44_swin_summary")["rows"]}
print("\nOne-permutation C (Step 44) vs ten-permutation C (accuracy, pp):")
print("  " + " | ".join(f"{r['block']} {old44[r['block']]:+.2f}→{r['acc']['C']:+.2f}" for r in summ_sw10["rows"]))
print("\nSTEP 46D COMPLETE — send me the full output.")

## Step 47 — MambaVision-T: resample null with per-image results, on both GPUs ##
Resample ablation replaces a block's output by the output of the same block on another image, so the replacement is a
real activation of the block, but for the wrong image (an in-distribution alternative to the shuffled null). Step 15C
measured it only as one accuracy change per block. Here each block runs once with four copies of every batch: the real
mean ablation and three resample seeds (777 + block, 1777 + block, 2777 + block; seed 0 = Step 15C), with per-image
correctness, true-class log-probability and true-class rank.

### Step 47A — Both GPUs: model copies, means, resample bank, pre-check ###
One copy of MambaVision-T per GPU with the Step 24B image cache. Reference means from Step 8 (compared with the saved
checkpoint). The resample bank is rebuilt exactly as in Step 15A: 256 evaluation images drawn with seed 777, sorted,
with the output of every mixer. Baseline and real-ablation predictions come from Steps 6 and 9. Pre-check on each GPU
(512 images, block L2B0): the unmodified copy must match the baseline and the real ablation must match Step 9.
About 3 min.

In [ ]:
# ============================================================
# STEP 47A — MAMBAVISION-T ON BOTH GPUs: MODELS, CACHE, MEANS, RESAMPLE BANK, PRE-CHECK
# ============================================================
import copy, re
assert "run_blocks_2gpu" in globals(), "Run Step 45A and Step 46A first."
assert "model" in globals() and "block_means" in globals() and "mixer_blocks" in globals(), \
    "Run Step 3, Step 4, the mixer_blocks cell and Step 8 first."
assert "CACHE_IMG" in globals(), "Run Step 24B first (MambaVision-T image cache)."
PREFIX_RS = "step47_t_resample"
model.eval()

# ---- reference means (Step 8), checked against the saved checkpoint ----
block_means_cpu = {k: v.detach().float().cpu() for k, v in block_means.items()}
_s8 = load_checkpoint("step8")["block_means"]
print(f"Step 8 means vs checkpoint: max |diff| = "
      f"{max((block_means_cpu[k] - torch.as_tensor(_s8[k]).float()).abs().max().item() for k in block_means_cpu):.2e}")

# ---- one model copy per GPU; blocks by name on each copy ----
def _mv_blocks(m):
    out = {}
    for nm, mod in m.named_modules():
        if hasattr(mod, "mixer"):
            g = re.search(r"levels\.(\d+)\.blocks\.(\d+)$", nm)
            if g: out[f"L{g.group(1)}B{g.group(2)}"] = (mod, mod.mixer, None)
    return out
T_NAMES = [n for n, _, _ in mixer_blocks]; assert len(T_NAMES) == 12
_arr_T = np.load(CACHE_IMG, mmap_mode="r")
def _t_getter(dev):
    m = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
    s = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)
    def get_batch(start, end):
        x = torch.from_numpy(np.array(_arr_T[start:end])).to(dev, non_blocking=True)
        return x.permute(0, 3, 1, 2).float().div(255).sub(m).div(s)
    return get_batch
_mv_logits = lambda o: o["logits"] if isinstance(o, dict) else (o.logits if hasattr(o, "logits") else o)
T_SETUPS = []
for dev in DEVS:
    m = model if dev == DEVS[0] and next(model.parameters()).device == dev else copy.deepcopy(model).to(dev).eval()
    T_SETUPS.append({"dev": dev, "model": m, "call": (lambda mm: (lambda x: _mv_logits(mm(x))))(m),
                     "get": _t_getter(dev), "blocks": _mv_blocks(m)})
    assert list(T_SETUPS[-1]["blocks"].keys()) == T_NAMES

# ---- resample bank exactly as Step 15A: 256 images drawn with seed 777, sorted, outputs of each mixer ----
_bank_idx = sorted(np.random.RandomState(777).choice(eval_indices, size=256, replace=False).tolist())
_pos = {di: p for p, di in enumerate(eval_indices)}
_bank_pos = [_pos[i] for i in _bank_idx]
S0 = T_SETUPS[0]; _cap = {n: [] for n in T_NAMES}
_hs = [S0["blocks"][n][1].register_forward_hook((lambda nn: (lambda mod, i, o: _cap[nn].append(_ot(o).detach().cpu())))(n)) for n in T_NAMES]
try:
    with torch.no_grad(), torch.cuda.device(S0["dev"]):
        for s in range(0, 256, 64):
            S0["call"](torch.cat([S0["get"](p, p + 1) for p in _bank_pos[s:s + 64]]))
finally:
    for h in _hs: h.remove()
BANK = {n: torch.cat(v, 0) for n, v in _cap.items()}; del _cap
print("bank:", {n: tuple(BANK[n].shape) for n in T_NAMES[:2]}, "...")

# ---- baseline (Step 6) and real-ablation predictions (Step 9) for the checks ----
_b = load_checkpoint("baseline_logits_probs_new")
T_BASE_C = (np.asarray(torch.as_tensor(_b["baseline_preds"]).numpy()) == LAB); del _b
_s9 = load_checkpoint("step9_ablated_preds")
T_REAL_C = {k: (np.asarray(v).astype(np.int64) == LAB) for k, v in _s9.items()}; del _s9
RS15 = {r["block"]: r["delta_pp"] for r in load_checkpoint("resampling_50k_v2")["results"]}
print(f"MambaVision-T baseline {T_BASE_C.mean()*100:.3f}%")

# ---- fork with resample copies (replaces fork_eval_dev of Step 46A; same behaviour for other kinds) ----
_fork_prev = fork_eval_dev
def fork_eval_dev(model_call, get_batch, n, blk, outm, mean, kinds, bs, dev, gamma=None):
    if not any(k[1] == "res" for k in kinds):
        return _fork_prev(model_call, get_batch, n, blk, outm, mean, kinds, bs, dev, gamma)
    K = len(kinds); lab = LAB_DEV[dev]; mean_d = mean.to(dev)
    gens = {l: torch.Generator().manual_seed(int(p[0])) for l, k, p in kinds if k == "res"}
    banks = {l: p[1].to(dev) for l, k, p in kinds if k == "res"}
    def pre(module, args, kwargs):
        if len(args) > 0:
            x = args[0]; return (x.repeat(K, *([1] * (x.dim() - 1))),) + tuple(args[1:]), kwargs
        kwargs = dict(kwargs); x = kwargs["hidden_states"]
        kwargs["hidden_states"] = x.repeat(K, *([1] * (x.dim() - 1))); return args, kwargs
    def post(module, inp, out):
        is_t = isinstance(out, tuple); t = out[0] if is_t else out
        R = t.shape[0] // K; pieces = []
        for j, (l, kind, p) in enumerate(kinds):
            tj = t[j * R:(j + 1) * R]
            if kind == "none":   pieces.append(tj)
            elif kind == "real": pieces.append(mean_d.to(dtype=t.dtype).expand_as(tj).clone())
            elif kind == "res":                                    # identical draw to Step 15B/15C
                idx = torch.randint(0, banks[l].shape[0], (R,), generator=gens[l])
                pieces.append(banks[l][idx.to(dev)].to(dtype=t.dtype))
        rep = torch.cat(pieces, 0)
        return (rep,) + tuple(out[1:]) if is_t else rep
    h1 = blk.register_forward_pre_hook(pre, with_kwargs=True); h2 = outm.register_forward_hook(post)
    corr = np.zeros((K, n), bool); lpt = np.zeros((K, n), np.float32); rank = np.zeros((K, n), np.int16)
    try:
        with torch.no_grad(), torch.cuda.device(dev):
            for s in range(0, n, bs):
                e = min(s + bs, n); B = e - s
                logits = model_call(get_batch(s, e)).float().view(K, B, -1)
                y = lab[s:e].view(1, B, 1).expand(K, B, 1)
                corr[:, s:e] = (logits.argmax(-1) == y.squeeze(2)).cpu().numpy()
                lpt[:, s:e] = F.log_softmax(logits, -1).gather(2, y).squeeze(2).cpu().numpy()
                rank[:, s:e] = (logits > logits.gather(2, y)).sum(-1).cpu().numpy().astype(np.int16)
    finally:
        h1.remove(); h2.remove()
    return corr, lpt, rank

def kinds_res(bi, name, C, n_res=3):
    return [("real", "real", None)] + [(f"res{j}", "res", (777 + bi + 1000 * j, BANK[name])) for j in range(n_res)]

# ---- pre-check on each GPU (512 images, block L2B0): unmodified = baseline, real = Step 9 ----
for S in T_SETUPS:
    blk, outm, _ = S["blocks"]["L2B0"]
    c, _, _ = _fork_prev(S["call"], S["get"], 512, blk, outm, block_means_cpu["L2B0"],
                         [("none", "none", None), ("real", "real", None)], 64, S["dev"])
    res = {"none vs baseline": (c[0] == T_BASE_C[:512]).mean(), "real vs Step 9": (c[1] == T_REAL_C["L2B0"][:512]).mean()}
    print(f"pre-check {S['dev']}:", {k: f"{v*100:.2f}%" for k, v in res.items()})
    assert min(res.values()) >= 1 - 2 / 512, "Pre-check failed - stop and send me this output."
print("\nSTEP 47A COMPLETE — safe to run Step 47B")

### Step 47B — Real ablation + three resample seeds per block, both GPUs (50K) ###
Four copies per batch (real, res0-res2; 64 images per batch, the batch size of Step 15C, so res0 draws exactly the
same replacement images as Step 15C). After each block the real ablation is compared with Step 9 on all 50,000 images
(≥ 99.9%). One checkpoint per block (step47_t_resample_<block>), resumable. About 50 min.

In [ ]:
# ============================================================
# STEP 47B — MAMBAVISION-T: REAL + THREE RESAMPLE SEEDS PER BLOCK, BOTH GPUs (50K)
# ============================================================
assert "T_SETUPS" in globals() and "BANK" in globals(), "Run Step 47A first."
run_blocks_2gpu(PREFIX_RS, T_SETUPS, T_NAMES, block_means_cpu,
                kinds_fn=lambda bi, name, C: kinds_res(bi, name, C, 3),
                base_c=T_BASE_C, bs=64, k_chunk=4,
                checks=lambda name: {"real": pk(T_REAL_C[name])})
print("\nSTEP 47B COMPLETE")

### Step 47C — Resample summary (CPU only) ###
Content effect per block against the mean of the three resample seeds, in accuracy, log-probability and log(1 + rank),
with paired bootstrap CIs, Holm correction, pooled 95%/90% intervals and Spearman correlation. Check: the res0 accuracy
change must equal Step 15C to within 0.02 pp. Saved as step47_t_resample_summary. Send the full output of this cell.

In [ ]:
# ============================================================
# STEP 47C — MAMBAVISION-T RESAMPLE SUMMARY (CPU ONLY)
# ============================================================
summ_rs = summarize(PREFIX_RS, T_NAMES, T_BASE_C, fam="res")
print("\nres0 accuracy change vs Step 15C (must agree to within 0.02 pp):")
dev15 = []
for nm in T_NAMES:
    d = load_checkpoint(f"{PREFIX_RS}_{nm}")["delta_pp"][1]
    dev15.append(abs(d - RS15[nm])); print(f"  {nm}: {d:+.3f} vs {RS15[nm]:+.3f}")
print(f"max |difference| = {max(dev15):.3f} pp")
print("\nSTEP 47C COMPLETE — send me the full output.")

## Step 48 — MambaVision-T under ImageNet-C corruptions: ten null permutations, on both GPUs ##
Contrast, JPEG compression, Gaussian noise and defocus blur at severity 3 on the first 10,000 images (same formulas and
seeds as Step 39A), clean-image reference means (Step 8). For each condition and each of the 12 blocks: the real mean
ablation and ten shuffled nulls (perm 0 = Step 16/39; perms 1-9 = 1000 + 100·block + j), with per-image correctness,
log-probability and rank. Uses the two MambaVision-T copies loaded in Step 47A.

### Step 48A — Corruption functions, permutations, pre-check ###
Defines the ImageNet-C corruptions exactly as in Step 39A, a GPU batch getter for an in-memory image array, and the
permutations (perm 0 = Step 16). Pre-check on each GPU (contrast, 512 images, block L2B0): the unmodified model, the real
ablation and perm 0 must reproduce Step 39B. About 1 min.

In [ ]:
# ============================================================
# STEP 48A — CORRUPTIONS: FUNCTIONS, MODELS ON BOTH GPUs, PRE-CHECK (DEFINITIONS + 1 MIN CHECK)
# ============================================================
import cv2, io
from PIL import Image
assert "T_SETUPS" in globals() and "block_means_cpu" in globals() and "CACHE_IMG" in globals(), \
    "Run Step 47A first (MambaVision-T on both GPUs)."
N_SUB, SEVERITY = 10_000, 3
CORR = ["contrast", "jpeg_compression", "gaussian_noise", "defocus_blur"]
CLEAN_SUB = np.ascontiguousarray(np.load(CACHE_IMG, mmap_mode="r")[:N_SUB])
res39 = load_checkpoint("step39_imagenet_c")

# ---- ImageNet-C severity 3, identical to Step 39A ----
def _disk(radius, alias_blur):
    if radius <= 8: L, ksize = np.arange(-8, 9), (3, 3)
    else:           L, ksize = np.arange(-radius, radius + 1), (5, 5)
    X, Y = np.meshgrid(L, L)
    k = np.array((X ** 2 + Y ** 2) <= radius ** 2, dtype=np.float32); k /= k.sum()
    return cv2.GaussianBlur(k, ksize=ksize, sigmaX=alias_blur)

def make_corrupted(name, sev=SEVERITY, chunk=500):
    out = np.empty_like(CLEAN_SUB)
    if name == "gaussian_noise":
        c = [.08, .12, .18, .26, .38][sev - 1]; rng = np.random.RandomState(39)
        for s in range(0, N_SUB, chunk):
            x = CLEAN_SUB[s:s + chunk].astype(np.float32) / 255.
            out[s:s + chunk] = np.uint8(np.clip(x + rng.normal(size=x.shape, scale=c).astype(np.float32), 0, 1) * 255)
    elif name == "contrast":
        c = [.4, .3, .2, .1, .05][sev - 1]
        for s in range(0, N_SUB, chunk):
            x = CLEAN_SUB[s:s + chunk].astype(np.float32) / 255.
            m = x.mean(axis=(1, 2, 3), keepdims=True)
            out[s:s + chunk] = np.uint8(np.clip((x - m) * c + m, 0, 1) * 255)
    elif name == "defocus_blur":
        r, a = [(3, .1), (4, .5), (6, .5), (8, .5), (10, .5)][sev - 1]; k = _disk(r, a)
        for i in range(N_SUB):
            x = CLEAN_SUB[i].astype(np.float32) / 255.
            y = np.stack([cv2.filter2D(x[:, :, d], -1, k) for d in range(3)], -1)
            out[i] = np.uint8(np.clip(y, 0, 1) * 255)
    elif name == "jpeg_compression":
        q = [25, 18, 15, 10, 7][sev - 1]
        for i in range(N_SUB):
            buf = io.BytesIO(); Image.fromarray(CLEAN_SUB[i]).save(buf, "JPEG", quality=q)
            out[i] = np.asarray(Image.open(buf).convert("RGB"), dtype=np.uint8)
    return out

def arr_getter(arr, dev):
    m = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
    s = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)
    def get_batch(a, b):
        x = torch.from_numpy(arr[a:b]).to(dev, non_blocking=True)
        return x.permute(0, 3, 1, 2).float().div(255).sub(m).div(s)
    return get_batch

# ---- perm 0 = Step 16 permutations (torch.manual_seed(42), block order); perms 1-9 = 1000 + 100*block + j ----
torch.manual_seed(42)
CH_PERMS = {n: torch.randperm(block_means_cpu[n].shape[-1]) for n in T_NAMES}
def kinds_c(bi, name, C):
    ks = [("real", "real", None), ("shuf0", "shuf", CH_PERMS[name])]
    for j in range(1, 10):
        ks.append((f"shuf{j}", "shuf", torch.randperm(C, generator=torch.Generator().manual_seed(1000 + 100 * bi + j))))
    return ks

# ---- pre-check on each GPU with the first condition (512 images, block L2B0) ----
_arr0 = make_corrupted(CORR[0])
for S in T_SETUPS:
    blk, outm, _ = S["blocks"]["L2B0"]
    c, _, _ = fork_eval_dev(S["call"], arr_getter(_arr0, S["dev"]), 512, blk, outm, block_means_cpu["L2B0"],
                            [("none", "none", None)] + kinds_c(0, "L2B0", block_means_cpu["L2B0"].shape[-1])[:2], 64, S["dev"])
    R = res39[CORR[0]]
    r = {"none vs Step 39 baseline": (c[0] == upk(R["baseline"]["correct_packed"], N_SUB)[:512]).mean(),
         "real vs Step 39": (c[1] == upk(R["blocks"]["L2B0"]["real_correct_packed"], N_SUB)[:512]).mean(),
         "shuf0 vs Step 39": (c[2] == upk(R["blocks"]["L2B0"]["null_correct_packed"], N_SUB)[:512]).mean()}
    print(f"pre-check {S['dev']} ({CORR[0]}):", {k: f"{v*100:.2f}%" for k, v in r.items()})
    assert min(r.values()) >= 1 - 2 / 512, "Pre-check failed - stop and send me this output."
del _arr0
print("\nSTEP 48A COMPLETE — safe to run Step 48B")

### Step 48B — Four corruptions: real + ten shuffled nulls per block, both GPUs (10K) ###
For each condition: corrupted images (1-3 min), baseline (checked against Step 39), then 11 copies per batch (real,
shuf0-shuf9; 48 images per batch) for the 12 blocks, shared between the two GPUs. After each block the real ablation
and perm 0 are compared with Step 39B (≥ 99.9%). One checkpoint per condition and block (step48_<condition>_<block>),
resumable. About 20 min per condition, 1.3-1.5 h in total.

In [ ]:
# ============================================================
# STEP 48B — FOUR CORRUPTIONS: REAL + TEN SHUFFLED NULLS PER BLOCK, BOTH GPUs (10K)
# ============================================================
assert "kinds_c" in globals(), "Run Step 48A first."
t_all = time.time()
for cname in CORR:
    pref = f"step48_{cname}"
    if all(f"{pref}_{n}" in set(list_checkpoints()) for n in T_NAMES):
        print(f"{cname}: already done"); continue
    t0 = time.time(); arr = make_corrupted(cname); R = res39[cname]
    print(f"\n=== {cname}: images ready in {(time.time()-t0)/60:.1f} min ===")
    with torch.cuda.device(T_SETUPS[0]["dev"]):
        base_c = get_baseline(pref, T_SETUPS[0]["call"], arr_getter(arr, T_SETUPS[0]["dev"]), n=N_SUB)
    ag = (base_c == upk(R["baseline"]["correct_packed"], N_SUB)).mean()
    print(f"baseline {base_c.mean()*100:.2f}% | agreement with Step 39: {ag*100:.3f}%")
    assert ag > 0.999, "Corrupted baseline differs from Step 39 - stop and send me this output."
    setups = [dict(S, get=arr_getter(arr, S["dev"])) for S in T_SETUPS]
    run_blocks_2gpu(pref, setups, T_NAMES, block_means_cpu, kinds_fn=kinds_c, base_c=base_c, bs=48, k_chunk=11, n=N_SUB,
                    checks=lambda name, R=R: {"real": R["blocks"][name]["real_correct_packed"],
                                              "shuf0": R["blocks"][name]["null_correct_packed"]})
    del arr
    print(f"{cname} done in {(time.time()-t0)/60:.1f} min")
print(f"\nSTEP 48B COMPLETE ({(time.time()-t_all)/60:.1f} min)")

### Step 48C — Corruptions summary, ten permutations (CPU only) ###
For each condition: content effect per block against the mean of the ten nulls in accuracy, log-probability and
log(1 + rank), with paired bootstrap CIs, Holm correction over 12 blocks, pooled 95%/90% intervals, the null's own SD
and Spearman correlation; then an overview comparing the Holm-significant blocks with the one-permutation result of
Step 39. Saved as step48_<condition>_summary. Send the full output of this cell.

In [ ]:
# ============================================================
# STEP 48C — CORRUPTIONS SUMMARY, TEN PERMUTATIONS (CPU ONLY)
# ============================================================
summ48 = {}
for cname in CORR:
    pref = f"step48_{cname}"
    base_c = upk(load_checkpoint(f"{pref}_baseline")["correct_packed"], N_SUB)
    print("\n" + "=" * 100 + f"\n{cname.upper()}  (severity 3, first 10,000 images, clean-image reference means)\n" + "=" * 100)
    summ48[cname] = summarize(pref, T_NAMES, base_c, n=N_SUB)
print("\nOverview (accuracy):")
for cname, sm in summ48.items():
    rw = [r["block"] for r in sm["rows"] if r["acc"]["p_holm"] < 0.05 and r["acc"]["C"] > 0]
    nw = [r["block"] for r in sm["rows"] if r["acc"]["p_holm"] < 0.05 and r["acc"]["C"] < 0]
    old = res39["summary"][cname]
    print(f"  {cname:17s} pooled C {sm['pooled']['acc']['C']:+.3f} [{sm['pooled']['acc']['ci'][0]:+.3f},{sm['pooled']['acc']['ci'][1]:+.3f}] "
          f"| real worse {rw} | null worse {nw} | 1 perm (Step 39): real worse {old['real_worse_holm']}, null worse {old['null_worse_holm']}")
print("\nSTEP 48C COMPLETE — send me the full output.")

## Step 49 — Freezing grid with log-probability and rank (54 configurations, both GPUs) ##
Adds the log-probability column to the full single-block freezing grid of Step 13 (6 Mamba blocks × 9 configurations
of tensor and mode), as requested in the review of report v5. Step 13 stored only accuracy. Here each block is run once
with nine copies of every batch, copy j receiving freeze configuration j inside the selective scan, so the 54
configurations need 6 passes instead of 54. Per image: correctness, true-class log-probability and rank.

### Step 49A — Thread-safe multi-configuration scan patch and checks ###
Replaces the selective-scan function by a version that freezes Δ, B and/or C (Step 13A references, Step 13B modes)
separately for each copy of the batch, only inside the block a GPU thread is working on (thread-local state, so the two
GPUs do not interfere). Checks on each GPU (512 images, block L2B0): with no configuration the model equals the baseline,
and a forked configuration equals the same freeze applied without the fork. Needs Steps 13A and 13B. About 3 min.

In [ ]:
# ============================================================
# STEP 49A — FREEZING GRID: THREAD-SAFE MULTI-CONFIG SCAN PATCH, CHECKS (≈3 MIN)
# ============================================================
import threading
assert "T_SETUPS" in globals() and "T_BASE_C" in globals(), "Run Step 47A first (MambaVision-T on both GPUs)."
assert "_freeze_refs" in globals() and "_apply_mode" in globals() and "_original_selective_scan_fn" in globals(), \
    "Run Step 13A and Step 13B first."
PREFIX_G = "step49_grid"
G13 = load_checkpoint("step13_freezing_50k")["results"]
GRID_BLOCKS = list(dict.fromkeys(r["block"] for r in G13))                  # 6 Mamba blocks, Step 13 order
GRID_CFGS = {b: [(r["tensor"], r["mode"]) for r in G13 if r["block"] == b] for b in GRID_BLOCKS}
G13_BY = {(r["block"], r["tensor"], r["mode"]): r for r in G13}
assert len(GRID_BLOCKS) == 6 and all(len(v) == 9 for v in GRID_CFGS.values())

# baseline true-class log-prob and rank (Step 6) for log-prob drop and rank damage
_b = load_checkpoint("baseline_logits_probs_new")
_blp = torch.as_tensor(_b["baseline_log_probs"]).float(); _y = torch.as_tensor(LAB).view(-1, 1)
T_BASE_LP = _blp.gather(1, _y).squeeze(1).numpy()
T_BASE_RANK = (_blp > _blp.gather(1, _y)).sum(-1).numpy(); del _b, _blp

# thread-local state: each GPU thread freezes its own block, one config per copy of the batch
_tl = threading.local()
def _grid_scan(x, dt, A, B, C, D, z=None, delta_bias=None, delta_softplus=None, return_last_state=None):
    name, cfgs = getattr(_tl, "name", None), getattr(_tl, "cfgs", None)
    if name is not None and cfgs is not None and name == getattr(_tl, "target", None):
        K = len(cfgs); R = dt.shape[0] // K; ref = _freeze_refs[name]; ds, bs_, cs = [], [], []
        for j, (tt, mode) in enumerate(cfgs):
            d_, b_, c_ = dt[j * R:(j + 1) * R], B[j * R:(j + 1) * R], C[j * R:(j + 1) * R]
            if tt in ("delta", "all"): d_ = _apply_mode(d_, ref["dt"], mode)
            if tt in ("B", "all"):     b_ = _apply_mode(b_, ref["B"], mode)
            if tt in ("C", "all"):     c_ = _apply_mode(c_, ref["C"], mode)
            ds.append(d_); bs_.append(b_); cs.append(c_)
        dt, B, C = torch.cat(ds), torch.cat(bs_), torch.cat(cs)
    return _original_selective_scan_fn(x, dt, A, B, C, D, z=z, delta_bias=delta_bias,
                                       delta_softplus=delta_softplus, return_last_state=return_last_state)

def grid_eval(S, name, cfgs, n, bs):
    """One pass: the batch is repeated len(cfgs) times at the block input; copy j gets freeze config j inside the scan."""
    K = len(cfgs); blk, mixer, _ = S["blocks"][name]; lab = LAB_DEV[S["dev"]]
    def pre(module, args, kwargs):
        if len(args) > 0:
            x = args[0]; return (x.repeat(K, *([1] * (x.dim() - 1))),) + tuple(args[1:]), kwargs
        kwargs = dict(kwargs); x = kwargs["hidden_states"]
        kwargs["hidden_states"] = x.repeat(K, *([1] * (x.dim() - 1))); return args, kwargs
    hs = [blk.register_forward_pre_hook(pre, with_kwargs=True)]
    for nm, (_, mx, _) in S["blocks"].items():
        if nm in GRID_BLOCKS:
            hs.append(mx.register_forward_pre_hook((lambda q: (lambda m, i: setattr(_tl, "name", q)))(nm)))
            hs.append(mx.register_forward_hook(lambda m, i, o: setattr(_tl, "name", None)))
    _tl.target, _tl.cfgs = name, cfgs
    corr = np.zeros((K, n), bool); lpt = np.zeros((K, n), np.float32); rank = np.zeros((K, n), np.int16)
    try:
        with torch.no_grad(), torch.cuda.device(S["dev"]):
            for s in range(0, n, bs):
                e = min(s + bs, n); Bn = e - s
                logits = S["call"](S["get"](s, e)).float().view(K, Bn, -1)
                y = lab[s:e].view(1, Bn, 1).expand(K, Bn, 1)
                corr[:, s:e] = (logits.argmax(-1) == y.squeeze(2)).cpu().numpy()
                lpt[:, s:e] = F.log_softmax(logits, -1).gather(2, y).squeeze(2).cpu().numpy()
                rank[:, s:e] = (logits > logits.gather(2, y)).sum(-1).cpu().numpy().astype(np.int16)
    finally:
        _tl.target, _tl.cfgs, _tl.name = None, None, None
        for h in hs: h.remove()
    return corr, lpt, rank

target_module.selective_scan_fn = _grid_scan            # inactive unless a thread sets a target block

# ---- checks on each GPU (512 images, block L2B0): no config = baseline; 9 configs vs a plain Step 13B freeze ----
for S in T_SETUPS:
    torch.cuda.set_device(S["dev"])
    c0, _, _ = grid_eval(S, "L2B0", [("none", "mean")], 512, 64)
    assert (c0[0] == T_BASE_C[:512]).all(), f"{S['dev']}: patched model differs from the baseline - stop."
    c9, _, _ = grid_eval(S, "L2B0", GRID_CFGS["L2B0"], 512, 64)
    tt, mode = GRID_CFGS["L2B0"][4]                                          # one config, run without the fork
    _tl.target, _tl.cfgs = "L2B0", [(tt, mode)]
    hs = [mx.register_forward_pre_hook((lambda q: (lambda m, i: setattr(_tl, "name", q)))(nm)) for nm, (_, mx, _) in S["blocks"].items() if nm in GRID_BLOCKS]
    hs += [mx.register_forward_hook(lambda m, i, o: setattr(_tl, "name", None)) for nm, (_, mx, _) in S["blocks"].items() if nm in GRID_BLOCKS]
    try:
        with torch.no_grad():
            ref = np.concatenate([(S["call"](S["get"](s, s + 64)).float().argmax(-1) == LAB_DEV[S["dev"]][s:s + 64]).cpu().numpy() for s in range(0, 512, 64)])
    finally:
        _tl.target, _tl.cfgs, _tl.name = None, None, None
        for h in hs: h.remove()
    ag = (c9[4] == ref).mean()
    print(f"check {S['dev']}: baseline identical | fork config 5 ({tt}, {mode}) vs plain freeze: {ag*100:.2f}%")
    assert ag >= 1 - 2 / 512, "Fork freeze differs from the plain freeze - stop and send me this output."
print("\nSTEP 49A COMPLETE — safe to run Step 49B")

### Step 49B — Freezing grid: 6 blocks × 9 configurations, both GPUs (50K) ###
One pass per Mamba block with nine copies of every batch (48 images per batch), blocks shared between the two GPUs.
After each block the accuracy change of every configuration must equal Step 13 to within 0.02 pp. One checkpoint per
block (step49_grid_<block>), resumable. About 35 min.

In [ ]:
# ============================================================
# STEP 49B — FREEZING GRID: 6 BLOCKS × 9 CONFIGS, BOTH GPUs (50K)
# ============================================================
assert "grid_eval" in globals(), "Run Step 49A first."
done = set(list_checkpoints()); todo = [b for b in GRID_BLOCKS if f"{PREFIX_G}_{b}" not in done]
print(f"{len(GRID_BLOCKS)-len(todo)} done, {len(todo)} to run")
q = queue.Queue(); [q.put(b) for b in todo]; lock = threading.Lock(); errors = []; t_all = time.time()
def _worker(S):
    torch.cuda.set_device(S["dev"])
    while not errors:
        try: name = q.get_nowait()
        except queue.Empty: return
        try:
            t0 = time.time(); cfgs = GRID_CFGS[name]
            corr, lpt, rank = grid_eval(S, name, cfgs, N_FULL, 48)
            d_pp = (corr.mean(1) - T_BASE_C.mean()) * 100
            diffs = [abs(d - G13_BY[(name, tt, md)]["delta_pp"]) for d, (tt, md) in zip(d_pp, cfgs)]
            if max(diffs) > 0.02: raise AssertionError(f"{name}: differs from Step 13 by {max(diffs):.3f} pp")
            with lock:
                save_checkpoint(f"{PREFIX_G}_{name}", {"block": name, "configs": cfgs, "delta_pp": d_pp.tolist(),
                                "correct_packed": [pk(x) for x in corr], "lp_true": lpt, "rank": rank})
                print(f"[{S['dev']}] {name} {(time.time()-t0)/60:4.1f} min | max |Δ − Step 13| = {max(diffs):.4f} pp | "
                      f"{(time.time()-t_all)/60:.0f} min elapsed")
        except Exception as ex:
            with lock: errors.append(f"[{S['dev']}] {name}: {ex!r}")
            return
_ts = [threading.Thread(target=_worker, args=(S,)) for S in T_SETUPS]
[t.start() for t in _ts]; [t.join() for t in _ts]
if errors: raise RuntimeError("Stopped: " + " | ".join(errors))
print(f"\nSTEP 49B COMPLETE ({(time.time()-t_all)/60:.1f} min)")

### Step 49C — Freezing grid summary: accuracy, log-probability drop and rank (CPU only) ###
For all 54 configurations: accuracy change, true-class log-probability drop relative to the Step 6 baseline with a
paired bootstrap 95% CI, and rank damage (mean increase of log(1 + rank)). Restores the original selective-scan function.
Saved as step49_grid_summary. Send the full output of this cell.

In [ ]:
# ============================================================
# STEP 49C — FREEZING GRID SUMMARY: ACCURACY, LOG-PROB DROP, RANK (CPU ONLY)
# ============================================================
target_module.selective_scan_fn = _original_selective_scan_fn          # restore the original scan
idx = np.random.RandomState(42).randint(0, N_FULL, (NB_BOOT, N_FULL)).astype(np.int32)
base_lr = np.log1p(T_BASE_RANK.astype(np.float64)); rows = []
print(f"{'block':6s}{'tensor':7s}{'mode':9s}{'Δ acc (pp)':>12s}{'log-prob drop (nats) [95% CI]':>34s}{'rank damage':>13s}")
for name in GRID_BLOCKS:
    d = load_checkpoint(f"{PREFIX_G}_{name}")
    for j, (tt, md) in enumerate(d["configs"]):
        lpd = T_BASE_LP - d["lp_true"][j].astype(np.float64)
        bm = _bm(lpd, idx); lo, hi = np.percentile(bm, [2.5, 97.5])
        rd = float((np.log1p(d["rank"][j].astype(np.float64)) - base_lr).mean())
        rows.append({"block": name, "tensor": tt, "mode": md, "delta_pp": d["delta_pp"][j],
                     "logprob_drop": float(lpd.mean()), "logprob_ci": [float(lo), float(hi)], "rank_damage": rd})
        print(f"{name:6s}{tt:7s}{md:9s}{d['delta_pp'][j]:+12.3f}{lpd.mean():+14.4f} [{lo:+.4f},{hi:+.4f}]{rd:+13.4f}")
save_checkpoint(f"{PREFIX_G}_summary", {"rows": rows})
print("\nSTEP 49C COMPLETE — send me the full output.")

## Step 50 — MambaVision-S: ten null permutations per block, on both GPUs ##
Brings MambaVision-S to the standard of MambaVision-T: for each of its 12 mixer blocks (7 Mamba, 5 attention), the real
mean ablation and ten shuffled nulls on all 50,000 images (perm 0 = Step 29D, seed 42 + block; perms 1-9 = 1000 +
100·block + j), with per-image correctness, log-probability and rank. One model copy per GPU; blocks from a shared queue.

### Step 50A — MambaVision-S on both GPUs: load, cache, means, baseline, pre-check ###
Loads nvidia/MambaVision-S-1K with its own evaluation transform (reusing the MambaVision-T image cache when the
transform is identical), computes the reference means exactly as Step 29B, saves the baseline with per-image
log-probability and rank, and checks it against Step 29. Pre-check on each GPU (512 images, block L2B0): the real
ablation and perm 0 must reproduce Step 29D. About 5 min (+3 min if a new cache is needed).

In [ ]:
# ============================================================
# STEP 50A — MAMBAVISION-S ON BOTH GPUs: LOAD, CACHE, MEANS, BASELINE, PRE-CHECK
# ============================================================
import copy, re
from transformers import AutoModelForImageClassification
from timm.data.transforms_factory import create_transform
from torchvision import transforms as T
assert "run_blocks_2gpu" in globals(), "Run Step 45A and Step 46A first."
PREFIX_S10 = "step50_s"
S_MODEL_ID = "nvidia/MambaVision-S-1K"

s_m0 = AutoModelForImageClassification.from_pretrained(S_MODEL_ID, trust_remote_code=True, low_cpu_mem_usage=False).to(DEVS[0]).eval()
s_tf = create_transform(input_size=(3, 224, 224), is_training=False, mean=s_m0.config.mean, std=s_m0.config.std,
                        crop_mode=s_m0.config.crop_mode, crop_pct=s_m0.config.crop_pct)
_tf = list(s_tf.transforms); _k = next(i for i, t in enumerate(_tf) if "ToTensor" in type(t).__name__)
S_PRE = T.Compose(_tf[:_k]); S_MS = (list(_tf[_k + 1].mean), list(_tf[_k + 1].std))

# image cache: reuse the MambaVision-T cache when the transform is identical, otherwise build the S cache
_same_as_T = ("TIMM_TRANSFORM" in globals() and repr(s_tf) == repr(TIMM_TRANSFORM) and "CACHE_IMG" in globals())
if _same_as_T:
    _arr_S = np.load(CACHE_IMG, mmap_mode="r")
    def _s_getter(dev):
        m = torch.tensor(S_MS[0], device=dev).view(1, 3, 1, 1); s = torch.tensor(S_MS[1], device=dev).view(1, 3, 1, 1)
        def get_batch(a, b):
            x = torch.from_numpy(np.array(_arr_S[a:b])).to(dev, non_blocking=True)
            return x.permute(0, 3, 1, 2).float().div(255).sub(m).div(s)
        return get_batch
    print("S transform identical to T: reusing the T image cache")
else:
    _deit = os.path.join(CACHE_DIR, f"mv_cache_DEIT_images_u8_{N_FULL}.npy")      # DeiT/Swin are finished
    if shutil.disk_usage(CACHE_DIR).free / 1e9 < 9 and os.path.exists(_deit):
        os.remove(_deit); os.remove(_deit + ".done"); print("removed the DEIT cache to free disk space")
    make_cache("S", S_PRE, *S_MS)
    _s_getter = lambda dev: cache_getter("S", *S_MS, dev)
    print("S cache ready")

def _mv_blocks(m):
    out = {}
    for nm, mod in m.named_modules():
        if hasattr(mod, "mixer"):
            g = re.search(r"levels\.(\d+)\.blocks\.(\d+)$", nm)
            if g: out[f"L{g.group(1)}B{g.group(2)}"] = (mod, mod.mixer, None)
    return out
_mv_logits = lambda o: o["logits"] if isinstance(o, dict) else (o.logits if hasattr(o, "logits") else o)
S_SETUPS = []
for dev in DEVS:
    m = s_m0 if dev == DEVS[0] else copy.deepcopy(s_m0).to(dev).eval()
    S_SETUPS.append({"dev": dev, "model": m, "call": (lambda mm: (lambda x: _mv_logits(mm(x))))(m),
                     "get": _s_getter(dev), "blocks": _mv_blocks(m)})
S_NAMES = list(S_SETUPS[0]["blocks"].keys()); assert len(S_NAMES) == 12, S_NAMES
print("S blocks:", S_NAMES)

# reference means exactly as Step 29B: concatenate the 256 reference outputs, mean over images and tokens
S0 = S_SETUPS[0]; _cap = {n: [] for n in S_NAMES}
_hs = [S0["blocks"][n][1].register_forward_hook((lambda nn: (lambda mod, i, o: _cap[nn].append(_ot(o).detach())))(n)) for n in S_NAMES]
try:
    with torch.no_grad(), torch.cuda.device(S0["dev"]):
        for s in range(0, 256, 64): S0["call"](S0["get"](s, s + 64))
finally:
    for h in _hs: h.remove()
s_means10 = {n: torch.cat(v).mean(dim=(0, 1), keepdim=True).cpu() for n, v in _cap.items()}; del _cap
torch.cuda.empty_cache()

# baseline (saved once) and the Step 29 per-block results for the checks
res29 = load_checkpoint("step29_mambavision_s")
with torch.cuda.device(S0["dev"]):
    S_BASE_C = get_baseline(PREFIX_S10, S0["call"], S0["get"])
_acc29 = float(res29["baseline"]["accuracy"]); _acc29 = _acc29 / 100 if _acc29 > 1 else _acc29
print(f"MambaVision-S baseline {S_BASE_C.mean()*100:.3f}% (Step 29: {_acc29*100:.3f}%)")
assert abs(S_BASE_C.mean() - _acc29) < 1e-4, "Baseline differs from Step 29 - stop and send me this output."

# pre-check on each GPU: 512 images, block L2B0, real and perm 0 must reproduce Step 29D
for S in S_SETUPS:
    blk, outm, _ = S["blocks"]["L2B0"]
    ks = kinds_shuf(0, s_means10["L2B0"].shape[-1], 1, seed0=42)
    c, _, _ = fork_eval_dev(S["call"], S["get"], 512, blk, outm, s_means10["L2B0"], ks, 64, S["dev"])
    r = {"real vs Step 29": (c[0] == upk(res29["E2E5"]["L2B0"]["real"]["correct_packed"])[:512]).mean(),
         "shuf0 vs Step 29": (c[1] == upk(res29["E2E5"]["L2B0"]["null"]["correct_packed"])[:512]).mean()}
    print(f"pre-check {S['dev']}:", {k: f"{v*100:.2f}%" for k, v in r.items()})
    assert min(r.values()) >= 1 - 2 / 512, "Pre-check failed - stop and send me this output."
print("\nSTEP 50A COMPLETE — safe to run Step 50B")

### Step 50B — MambaVision-S: real + ten shuffled nulls per block, both GPUs (50K) ###
Eleven copies per batch (real, shuf0-shuf9; 48 images per batch), blocks shared between the two GPUs. After each block
the real ablation and perm 0 are compared with Step 29D on all 50,000 images (≥ 99.9%). One checkpoint per block
(step50_s_<block>), resumable. About 2.5-3 h.

In [ ]:
# ============================================================
# STEP 50B — MAMBAVISION-S: REAL + TEN SHUFFLED NULLS PER BLOCK, BOTH GPUs (50K)
# ============================================================
assert "S_SETUPS" in globals() and "s_means10" in globals(), "Run Step 50A first."
run_blocks_2gpu(PREFIX_S10, S_SETUPS, S_NAMES, s_means10,
                kinds_fn=lambda bi, name, C: kinds_shuf(bi, C, 10, seed0=42 + bi),
                base_c=S_BASE_C, bs=48, k_chunk=11,
                checks=lambda name: {"real": res29["E2E5"][name]["real"]["correct_packed"],
                                     "shuf0": res29["E2E5"][name]["null"]["correct_packed"]})
print("\nSTEP 50B COMPLETE")

### Step 50C — MambaVision-S summary, ten permutations (CPU only) ###
Content effect per block against the mean of the ten nulls in accuracy, log-probability and log(1 + rank), with paired
bootstrap CIs, Holm correction, pooled 95%/90% intervals and the ±0.10 pp equivalence test, the null's own SD per block,
Spearman correlation, and the one-permutation C of Step 29 next to the ten-permutation C. Saved as step50_s_summary.
Send the full output of this cell.

In [ ]:
# ============================================================
# STEP 50C — MAMBAVISION-S SUMMARY, TEN PERMUTATIONS (CPU ONLY)
# ============================================================
summ_s10 = summarize(PREFIX_S10, S_NAMES, S_BASE_C)
print("\nOne-permutation C (Step 29) vs ten-permutation C (accuracy, pp):")
print("  " + " | ".join(f"{r['block']} {-res29['E2E5'][r['block']]['real_minus_null_pp']:+.2f}→{r['acc']['C']:+.2f}" for r in summ_s10["rows"]))
print("\nSTEP 50C COMPLETE — send me the full output.")

## Step 51 — MambaVision-B: ten null permutations per block, on both GPUs ##
Brings MambaVision-B to the standard of the other models: for each of its 15 mixer blocks (8 Mamba, 7 attention), the
real mean ablation and ten shuffled nulls on all 50,000 images (perm 0 = Step 37D, seed 42 + block; perms 1-9 = 1000 +
100·block + j, so perms 1-3 equal Step 42A where it was run), with per-image correctness, log-probability and rank.
As in Step 37D, the null is size-matched in the residual stream: the layer-scale-weighted perturbation γ1⊙Δ is permuted
and divided by γ1. One model copy per GPU; blocks from a shared queue.

### Step 51A — MambaVision-B on both GPUs: load, cache, means, γ1, baseline, pre-check ###
Loads nvidia/MambaVision-B-1K with its own evaluation transform and image cache (finished DEIT and S caches are removed
first if disk space is short), reads the layer scale γ1 of every block, computes the reference means exactly as
Step 37B, saves the baseline and checks it against Step 37. Pre-check on each GPU (512 images) for one Mamba block and
one attention block: the real ablation and perm 0 must reproduce Step 37D. About 10 min (cache ~3-4 min).

In [ ]:
# ============================================================
# STEP 51A — MAMBAVISION-B ON BOTH GPUs: LOAD, CACHE, MEANS, γ1, BASELINE, PRE-CHECK
# ============================================================
import copy, re
from transformers import AutoModelForImageClassification
from timm.data.transforms_factory import create_transform
from torchvision import transforms as T
assert "run_blocks_2gpu" in globals(), "Run Step 45A and Step 46A first."
PREFIX_B10 = "step51_b"
B_MODEL_ID = "nvidia/MambaVision-B-1K"

b_m0 = AutoModelForImageClassification.from_pretrained(B_MODEL_ID, trust_remote_code=True, low_cpu_mem_usage=False).to(DEVS[0]).eval()
b_tf = create_transform(input_size=(3, 224, 224), is_training=False, mean=b_m0.config.mean, std=b_m0.config.std,
                        crop_mode=b_m0.config.crop_mode, crop_pct=b_m0.config.crop_pct)
_tf = list(b_tf.transforms); _k = next(i for i, t in enumerate(_tf) if "ToTensor" in type(t).__name__)
B_PRE = T.Compose(_tf[:_k]); B_MS = (list(_tf[_k + 1].mean), list(_tf[_k + 1].std))

# image cache (same file name as Step 37A); free disk by removing finished caches if needed
for _tag in ["DEIT", "S"]:
    _p = os.path.join(CACHE_DIR, f"mv_cache_{_tag}_images_u8_{N_FULL}.npy")
    if shutil.disk_usage(CACHE_DIR).free / 1e9 < 9 and os.path.exists(_p):
        os.remove(_p); os.remove(_p + ".done"); print(f"removed the {_tag} cache to free disk space")
make_cache("B", B_PRE, *B_MS)
print("B cache ready")

def _mv_blocks_g(m):
    """blocks by name with their layer scale γ1 (None if the block has none), as in Step 37A."""
    out = {}
    for nm, mod in m.named_modules():
        if hasattr(mod, "mixer"):
            g = re.search(r"levels\.(\d+)\.blocks\.(\d+)$", nm)
            if g:
                gam = mod.gamma_1.detach().float() if isinstance(getattr(mod, "gamma_1", None), torch.Tensor) else None
                out[f"L{g.group(1)}B{g.group(2)}"] = (mod, mod.mixer, gam)
    return out
_mv_logits = lambda o: o["logits"] if isinstance(o, dict) else (o.logits if hasattr(o, "logits") else o)
B_SETUPS = []
for dev in DEVS:
    m = b_m0 if dev == DEVS[0] else copy.deepcopy(b_m0).to(dev).eval()
    B_SETUPS.append({"dev": dev, "model": m, "call": (lambda mm: (lambda x: _mv_logits(mm(x))))(m),
                     "get": cache_getter("B", *B_MS, dev), "blocks": _mv_blocks_g(m)})
B_NAMES = list(B_SETUPS[0]["blocks"].keys()); assert len(B_NAMES) == 15, B_NAMES
print("B blocks:", B_NAMES, "| blocks with γ1:", sum(B_SETUPS[0]["blocks"][n][2] is not None for n in B_NAMES))

# reference means exactly as Step 37B: concatenate the 256 reference outputs, mean over images and tokens
S0 = B_SETUPS[0]; _cap = {n: [] for n in B_NAMES}
_hs = [S0["blocks"][n][1].register_forward_hook((lambda nn: (lambda mod, i, o: _cap[nn].append(_ot(o).detach())))(n)) for n in B_NAMES]
try:
    with torch.no_grad(), torch.cuda.device(S0["dev"]):
        for s in range(0, 256, 32): S0["call"](S0["get"](s, s + 32))
finally:
    for h in _hs: h.remove()
b_means10 = {n: torch.cat(v).mean(dim=(0, 1), keepdim=True).cpu() for n, v in _cap.items()}; del _cap
torch.cuda.empty_cache()

# baseline (saved once) and the Step 37 per-block results for the checks
res37 = load_checkpoint("step37_mambavision_b")
with torch.cuda.device(S0["dev"]):
    B_BASE_C = get_baseline(PREFIX_B10, S0["call"], S0["get"])
_acc37 = res37.get("baseline", {}).get("accuracy")
if _acc37 is not None:
    _acc37 = float(_acc37); _acc37 = _acc37 / 100 if _acc37 > 1 else _acc37
    print(f"MambaVision-B baseline {B_BASE_C.mean()*100:.3f}% (Step 37: {_acc37*100:.3f}%)")
    assert abs(B_BASE_C.mean() - _acc37) < 1e-4, "Baseline differs from Step 37 - stop and send me this output."
else:
    print(f"MambaVision-B baseline {B_BASE_C.mean()*100:.3f}% (published 84.2%)")

# pre-check on each GPU: 512 images, a Mamba block (L2B0) and an attention block with γ1; real and perm 0 vs Step 37D
_att = next(n for n in B_NAMES if res37["E2E5"][n]["type"] != "MambaVisionMixer")
for S in B_SETUPS:
    for nm in ["L2B0", _att]:
        blk, outm, gam = S["blocks"][nm]; bi = B_NAMES.index(nm)
        ks = kinds_shuf(bi, b_means10[nm].shape[-1], 1, seed0=42 + bi)
        c, _, _ = fork_eval_dev(S["call"], S["get"], 512, blk, outm, b_means10[nm], ks, 32, S["dev"], gam)
        r = {"real": (c[0] == upk(res37["E2E5"][nm]["real"]["correct_packed"])[:512]).mean(),
             "shuf0": (c[1] == upk(res37["E2E5"][nm]["null"]["correct_packed"])[:512]).mean()}
        print(f"pre-check {S['dev']} {nm}:", {k: f"{v*100:.2f}%" for k, v in r.items()})
        assert min(r.values()) >= 1 - 2 / 512, "Pre-check failed - stop and send me this output."
print("\nSTEP 51A COMPLETE — safe to run Step 51B")

### Step 51B — MambaVision-B: real + ten shuffled nulls per block, both GPUs (50K) ###
Eleven copies per batch (real, shuf0-shuf9; 32 images per batch), blocks shared between the two GPUs. After each block
the real ablation and perm 0 are compared with Step 37D on all 50,000 images (≥ 99.9%). One checkpoint per block
(step51_b_<block>), resumable. About 5.5-6.5 h.

In [ ]:
# ============================================================
# STEP 51B — MAMBAVISION-B: REAL + TEN SHUFFLED NULLS PER BLOCK, BOTH GPUs (50K)
# ============================================================
assert "B_SETUPS" in globals() and "b_means10" in globals(), "Run Step 51A first."
run_blocks_2gpu(PREFIX_B10, B_SETUPS, B_NAMES, b_means10,
                kinds_fn=lambda bi, name, C: kinds_shuf(bi, C, 10, seed0=42 + bi),
                base_c=B_BASE_C, bs=32, k_chunk=11,
                checks=lambda name: {"real": res37["E2E5"][name]["real"]["correct_packed"],
                                     "shuf0": res37["E2E5"][name]["null"]["correct_packed"]})
print("\nSTEP 51B COMPLETE")

### Step 51C — MambaVision-B summary, ten permutations (CPU only) ###
Content effect per block against the mean of the ten nulls in accuracy, log-probability and log(1 + rank), with paired
bootstrap CIs, Holm correction over 15 blocks, pooled 95%/90% intervals and the ±0.10 pp equivalence test, the null's
own SD per block, Spearman correlation, and the one-permutation C of Step 37 next to the ten-permutation C.
Saved as step51_b_summary. Send the full output of this cell.

In [ ]:
# ============================================================
# STEP 51C — MAMBAVISION-B SUMMARY, TEN PERMUTATIONS (CPU ONLY)
# ============================================================
summ_b10 = summarize(PREFIX_B10, B_NAMES, B_BASE_C)
print("\nOne-permutation C (Step 37) vs ten-permutation C (accuracy, pp):")
print("  " + " | ".join(f"{r['block']} {-res37['E2E5'][r['block']]['real_minus_null_pp']:+.2f}→{r['acc']['C']:+.2f}" for r in summ_b10["rows"]))
print("\nSTEP 51C COMPLETE — send me the full output.")

## Step 52 — Content effect relative to damage (all five models, CPU only)

Professor's revision point 5: content effects must be compared *relative to the size of the ablation damage*, because MambaVision's damage is much smaller than Swin's or DeiT's.

For every block we compute damage d = baseline − real ablation and content effect C = mean of 10 shuffled nulls − real (accuracy, pp), plus the same in log(1+rank).
Per model we report:
- mean damage, mean |C|, median |C|/|d|
- **content share S** = sqrt(Σ(C_b² − noise_b) / Σ d_b²), noise-corrected, with a 95% bootstrap CI (2,000 image resamples) and a one-sided 90% upper bound (for a relative-margin equivalence test)
- Swin-T separately for stages 2–3

Input: saved checkpoints of Steps 6, 9, 27, 43B (T), 48 (S), 51 (B), 46 (Swin), 45 (DeiT). No GPU.

In [ ]:
# ============================================================
# STEP 52 — CONTENT EFFECTS RELATIVE TO DAMAGE, ALL FIVE MODELS (CPU ONLY, NO GPU)
# ============================================================
import time, re, numpy as np, torch
assert "load_checkpoint" in globals(), "Run the HF checkpoint cell first."
t_start = time.time()
N = 50000; NB = 2000
upk = lambda p: np.unpackbits(np.asarray(p, dtype=np.uint8))[:N].astype(np.float32)
lr = lambda r: np.log1p(np.asarray(r, dtype=np.float32))

# ---------- MambaVision-T: baseline (Step 6), real (Step 9), ten nulls (Step 27); rank from Step 43B ----------
_b = load_checkpoint("baseline_logits_probs_new")
LAB = np.asarray(torch.as_tensor(_b["eval_labels"]).numpy())
_blp = torch.as_tensor(_b["baseline_log_probs"]).float()
T_BASE_C = (np.asarray(torch.as_tensor(_b["baseline_preds"]).numpy()) == LAB).astype(np.float32)
T_BASE_LR = lr((_blp > _blp.gather(1, torch.as_tensor(LAB).view(-1, 1))).sum(-1).numpy()); del _b, _blp
s27 = load_checkpoint("step27_multi_perm_null"); s9 = load_checkpoint("step9_ablated_preds")

def model_T():
    out = {}
    for name in s27:
        real_c = (np.asarray(s9[name]).astype(np.int64) == LAB).astype(np.float32)
        null_c = np.mean([upk(p) for p in s27[name]["correct_packed"]], 0)
        d43 = load_checkpoint(f"step43b_{name}"); L = d43["labels"]
        rk = lr(d43["rank"]); ii = [i for i, l in enumerate(L) if l.startswith("shuf")]
        out[name] = {"d": (T_BASE_C - real_c) * 100, "C": (null_c - real_c) * 100,
                     "dr": rk[0] - T_BASE_LR, "Cr": rk[0] - rk[ii].mean(0), "stage": int(name[1])}
    return out

ALL_CK = set(list_checkpoints())
def first_ck(*cands):
    for c in cands:
        if c in ALL_CK: return c
    raise AssertionError(f"None of these checkpoints exists: {cands} - stop and send me this output.")

def model_ckpt(prefix, base_cands, names):
    b = load_checkpoint(first_ck(*base_cands)); bc = upk(b["correct_packed"])
    blr = lr(b["rank"]) if "rank" in b else None; out = {}
    for name in names:
        d = load_checkpoint(f"{prefix}_{name}"); L = d["labels"]
        c = np.stack([upk(p) for p in d["correct_packed"]]); rk = lr(d["rank"])
        ii = [i for i, l in enumerate(L) if l.startswith("shuf")]
        assert L[0] == "real" and len(ii) == 10, f"{prefix}_{name}: expected real + 10 shuffled copies, got {L}"
        out[name] = {"d": (bc - c[0]) * 100, "C": (c[ii].mean(0) - c[0]) * 100,
                     "dr": (rk[0] - blr) if blr is not None else None, "Cr": rk[0] - rk[ii].mean(0),
                     "stage": int(name[1]) if name[0] in "LS" and name[1].isdigit() else -1}
    return out

def find_prefix(cands):
    """first prefix that has saved per-block checkpoints; returns (prefix, block names in natural order)"""
    for p in cands:
        names = [c[len(p) + 1:] for c in ALL_CK if c.startswith(p + "_") and "_" not in c[len(p) + 1:]
                 and c[len(p) + 1:] not in ("baseline", "summary")]
        if names:
            names.sort(key=lambda n: [int(x) for x in re.findall(r"\d+", n)])
            print(f"{p}: {len(names)} blocks -> {names}")
            return p, names
    raise AssertionError(f"No per-block checkpoints for any of {cands} - stop and send me this output.")

def load_model(cands, extra_base=()):
    p, names = find_prefix(cands)
    return model_ckpt(p, [f"{p}_baseline", *extra_base], names)

MODELS = {
    "MambaVision-T": model_T(),
    "MambaVision-S": load_model(["step50_s", "step48_s"]),
    "MambaVision-B": load_model(["step51_b"]),
    "Swin-T":        load_model(["step46_swin"], ["step44_swin_baseline"]),
    "DeiT-Small":    load_model(["step45_deit"]),
}
for k, v in MODELS.items(): print(f"{k}: {len(v)} blocks loaded")

# ---------- statistics ----------
idx = np.random.RandomState(42).randint(0, N, (NB, N)).astype(np.int32)
def bmeans(x):
    out = np.empty(NB, np.float64)
    for s in range(0, NB, 250): out[s:s + 250] = x[idx[s:s + 250]].mean(1)
    return out

def analyse(blocks, kC="C", kd="d"):
    """Per block: mean damage d_b, content effect C_b, ratio |C_b|/|d_b|.
    Model level: noise-corrected RMS content share  S = sqrt(sum_b max(C_b^2 - v_b, 0) / sum_b d_b^2),
    v_b = sampling variance of C_b; bootstrap over images (2,000 resamples; replicate uses C*^2 - 2 v_b)."""
    Cb = np.array([b[kC].mean() for b in blocks]); db = np.array([b[kd].mean() for b in blocks])
    vb = np.array([b[kC].var() / N for b in blocks])
    Cs = np.stack([bmeans(b[kC]) for b in blocks], 1); ds = np.stack([bmeans(b[kd]) for b in blocks], 1)
    S = np.sqrt(max(np.sum(Cb ** 2 - vb), 0.0) / np.sum(db ** 2))
    S_bs = np.sqrt(np.maximum(np.sum(Cs ** 2 - 2 * vb, 1), 0.0) / np.sum(ds ** 2, 1))
    ratio = np.abs(Cb) / np.maximum(np.abs(db), 1e-9)
    noise_floor = np.sqrt(np.sum(vb) / np.sum(db ** 2))          # RMS share expected from sampling noise alone
    return {"mean_damage": float(np.mean(np.abs(db))), "mean_absC": float(np.mean(np.abs(Cb))),
            "median_ratio": float(np.median(ratio)), "ratio_per_block": ratio.tolist(),
            "S": float(S), "S_ci": np.percentile(S_bs, [2.5, 97.5]).tolist(), "S_upper90": float(np.percentile(S_bs, 95)),
            "noise_floor_uncorrected": float(noise_floor), "n_blocks": len(blocks)}

res = {}
print(f"\n{'model':22s}{'metric':6s}{'mean dmg':>10s}{'mean|C|':>9s}{'med |C|/|d|':>12s}{'content share S [95% CI]':>28s}{'S upper 90%':>12s}")
for mname, M in MODELS.items():
    groups = {"all": list(M.values())}
    if mname == "Swin-T": groups["stages 2-3"] = [b for b in M.values() if b["stage"] in (2, 3)]
    for g, blocks in groups.items():
        for metric, (kC, kd) in [("acc", ("C", "d")), ("rank", ("Cr", "dr"))]:
            if blocks[0][kd] is None: continue
            r = analyse(blocks, kC, kd); res[f"{mname}|{g}|{metric}"] = r
            lab = mname + ("" if g == "all" else f" ({g})")
            print(f"{lab:22s}{metric:6s}{r['mean_damage']:10.4f}{r['mean_absC']:9.4f}{r['median_ratio']:12.2f}"
                  f"{r['S']:10.3f} [{r['S_ci'][0]:.3f}, {r['S_ci'][1]:.3f}]{r['S_upper90']:12.3f}")
save_checkpoint("step52_relative_content", res)

print("\nPer-block |C|/|damage| (accuracy):")
for mname, M in MODELS.items():
    r = res[f"{mname}|all|acc"]
    print(f"  {mname:14s} " + " ".join(f"{n}:{x:.2f}" for n, x in zip(M.keys(), r["ratio_per_block"])))
print(f"\nSTEP 52 COMPLETE in {(time.time()-t_start)/60:.1f} min — send me the full output.")

# Step 53 — Group Nulls That Keep the Alignment Between Blocks (Review Point M1)

**Question.** Removing all six Mamba or all six attention mixers of MambaVision-T costs far more than the sum of single-block effects, while the earlier group null (independent shuffled perturbation in each block) stays much closer to additive. But an independent permutation per block also destroys the alignment between the perturbations of different blocks, which the real group ablation keeps. Aligned perturbations add up in the residual stream, so the gap could reflect alignment rather than content.

**Test.** For each group (Mamba: L2B0–L2B3, L3B0, L3B1; attention: L2B4–L2B7, L3B2, L3B3), on all 50,000 images:
- real group ablation (must reproduce Step 26),
- 10 **independent** nulls (own permutation per block),
- 10 **shared** nulls (one permutation per stage for all blocks of the group, which keeps their alignment),
- relative displacement of the residual stream after the last block (L3B3).

**Reading.** If the real ablation stays far worse than the shared null, compensation is content-specific; if the shared null approaches the real ablation, the compounding comes from alignment.

### Step 53A — Group nulls with shared permutations: definitions and pre-check

Review point M1: the earlier group null used an independent permutation in each block, so it destroys any alignment between the perturbations of different blocks. The real group ablation keeps that alignment. Here we add a **shared null** (one permutation per stage, applied to every block of the group in that stage), which keeps the cross-block alignment but removes the content. We also measure the displacement of the residual stream after the last block (L3B3).

Per group (6 Mamba / 6 attention mixers of MambaVision-T): real, 10 independent nulls, 10 shared nulls. Pre-check on 512 images on each GPU: the unmodified copy must equal the baseline.

In [ ]:
# ============================================================
# STEP 53A — GROUP NULLS WITH SHARED PERMUTATIONS: DEFINITIONS + PRE-CHECK (≈2 MIN)
# ============================================================
import threading, queue
assert "T_SETUPS" in globals() and "block_means_cpu" in globals() and "T_BASE_C" in globals(), "Run Step 47A first."
PREFIX_G53 = "step53_group"
G26 = load_checkpoint("step26_group_nulls")                       # earlier group results (real + one null)
GROUPS = {g: sorted(G26[g]["blocks"], key=T_NAMES.index) for g in ["mamba", "attn"]}
assert all(b in T_NAMES for g in GROUPS.values() for b in g), GROUPS
print("groups:", GROUPS)
STAGE = lambda name: int(name[1])
CH = {n: block_means_cpu[n].shape[-1] for n in T_NAMES}
print("channels per block:", CH)

def kinds_group(group):
    """none, real, 10 independent nulls (own permutation per block), 10 shared nulls (one permutation per stage,
    applied to every block of that stage, so the alignment between the blocks' perturbations is kept)."""
    ks = [("none", "none", None), ("real", "real", None)]
    for j in range(10):
        ks.append((f"ind{j}", "shuf", {b: torch.randperm(CH[b], generator=torch.Generator().manual_seed(4000 + 100 * T_NAMES.index(b) + j))
                                       for b in GROUPS[group]}))
    for j in range(10):
        sp = {}
        for b in GROUPS[group]:
            st = STAGE(b)
            if st not in sp: sp[st] = torch.randperm(CH[b], generator=torch.Generator().manual_seed(5000 + 100 * st + j))
        ks.append((f"shd{j}", "shuf", {b: sp[STAGE(b)] for b in GROUPS[group]}))
    return ks

def group_eval(S, group, kinds, n, bs):
    """One pass: the batch is repeated K times at the first block of the group; copy j gets kind j in every block of the
    group. Also returns the displacement of the residual stream after the last block (L3B3) relative to the 'none' copy."""
    K = len(kinds); names = GROUPS[group]; dev = S["dev"]; lab = LAB_DEV[dev]
    assert kinds[0][1] == "none"
    first_blk = S["blocks"][names[0]][0]
    def pre(module, args, kwargs):
        if len(args) > 0:
            x = args[0]; return (x.repeat(K, *([1] * (x.dim() - 1))),) + tuple(args[1:]), kwargs
        kwargs = dict(kwargs); x = kwargs["hidden_states"]
        kwargs["hidden_states"] = x.repeat(K, *([1] * (x.dim() - 1))); return args, kwargs
    def make_post(b):
        mv = block_means_cpu[b].to(dev)
        def post(module, inp, out):
            is_t = isinstance(out, tuple); t = out[0] if is_t else out
            R = t.shape[0] // K; m = mv.to(dtype=t.dtype); pieces = []
            for j, (l, kind, p) in enumerate(kinds):
                tj = t[j * R:(j + 1) * R]
                if kind == "none":   pieces.append(tj)
                elif kind == "real": pieces.append(m.expand_as(tj).clone())
                else:
                    d = m.expand_as(tj) - tj; pieces.append(tj + d[..., p[b].to(dev)])
            rep = torch.cat(pieces, 0)
            return (rep,) + tuple(out[1:]) if is_t else rep
        return post
    disp = np.zeros((K, 2))                                         # [sum ||h_j - h_none||^2, sum ||h_none||^2]
    def last_hook(module, inp, out):
        t = (out[0] if isinstance(out, tuple) else out).float(); R = t.shape[0] // K
        t = t.reshape(K, R, -1); t0 = t[0]
        disp[:, 0] += ((t - t0.unsqueeze(0)) ** 2).sum(dim=(1, 2)).cpu().numpy(); disp[:, 1] += float((t0 ** 2).sum())
    hs = [first_blk.register_forward_pre_hook(pre, with_kwargs=True)]
    hs += [S["blocks"][b][1].register_forward_hook(make_post(b)) for b in names]
    hs.append(S["blocks"]["L3B3"][0].register_forward_hook(last_hook))
    corr = np.zeros((K, n), bool); lpt = np.zeros((K, n), np.float32); rank = np.zeros((K, n), np.int16)
    try:
        with torch.no_grad(), torch.cuda.device(dev):
            for s in range(0, n, bs):
                e = min(s + bs, n); B = e - s
                logits = S["call"](S["get"](s, e)).float().view(K, B, -1)
                y = lab[s:e].view(1, B, 1).expand(K, B, 1)
                corr[:, s:e] = (logits.argmax(-1) == y.squeeze(2)).cpu().numpy()
                lpt[:, s:e] = F.log_softmax(logits, -1).gather(2, y).squeeze(2).cpu().numpy()
                rank[:, s:e] = (logits > logits.gather(2, y)).sum(-1).cpu().numpy().astype(np.int16)
    finally:
        for h in hs: h.remove()
    return corr, lpt, rank, np.sqrt(disp[:, 0] / disp[:, 1])

# ---- jobs: each group in two chunks of 12 copies (each chunk has its own 'none' copy) ----
JOBS = {}
for g in GROUPS:
    ks = kinds_group(g)                                            # none, real, ind0-9, shd0-9
    JOBS[(g, 0)] = ks[:12]                                         # none, real, ind0-9
    JOBS[(g, 1)] = [ks[0]] + ks[12:]                               # none, shd0-9

# ---- pre-check on each GPU, 512 images: 'none' = baseline; independent and shared copies differ from each other ----
for S in T_SETUPS:
    torch.cuda.set_device(S["dev"])
    c, _, _, dsp = group_eval(S, "attn", [JOBS[("attn", 0)][0], JOBS[("attn", 0)][1], JOBS[("attn", 0)][2], JOBS[("attn", 1)][1]], 512, 64)
    ag = (c[0] == T_BASE_C[:512]).mean()
    print(f"pre-check {S['dev']}: none vs baseline {ag*100:.2f}% | acc none/real/ind0/shd0 = "
          f"{', '.join(f'{x*100:.1f}' for x in c.mean(1))} | displacement real/ind0/shd0 = {dsp[1]:.3f}/{dsp[2]:.3f}/{dsp[3]:.3f}")
    assert ag >= 1 - 2 / 512, "Pre-check failed - stop and send me this output."
    assert dsp[0] < 1e-6 and dsp[1] > 0, "Displacement check failed - stop and send me this output."
print("\nSTEP 53A COMPLETE — safe to run Step 53B")

### Step 53B — Group ablation: real + 10 independent + 10 shared nulls, both GPUs (50,000 images)

Four jobs (2 groups × 2 chunks of 12 copies), split over both T4 GPUs; each job is saved as its own checkpoint, so the step resumes after a disconnect. Safety checks: the unmodified copy must equal the baseline (≥ 99.9%), and the real group ablation must reproduce Step 26 within 0.02 pp. Expected time ≈ 35–45 min.

In [ ]:
# ============================================================
# STEP 53B — GROUP ABLATION, REAL + 10 INDEPENDENT + 10 SHARED NULLS, BOTH GPUs (50K, ≈35-45 MIN)
# ============================================================
assert "group_eval" in globals() and "JOBS" in globals(), "Run Step 53A first."
REAL26 = {g: G26[g]["real_delta_pp"] for g in GROUPS}
done = set(list_checkpoints()); todo = [k for k in JOBS if f"{PREFIX_G53}_{k[0]}_c{k[1]}" not in done]
print(f"{len(JOBS)-len(todo)} done, {len(todo)} to run")
q = queue.Queue(); [q.put(k) for k in sorted(todo, key=lambda k: k[0] != "mamba")]   # mamba jobs (longer) first
lock = threading.Lock(); errors = []; t_all = time.time()
def _worker(S):
    torch.cuda.set_device(S["dev"])
    while not errors:
        try: key = q.get_nowait()
        except queue.Empty: return
        try:
            t0 = time.time(); g, ch = key; ks = JOBS[key]
            corr, lpt, rank, dsp = group_eval(S, g, ks, N_FULL, 32)
            d_pp = (corr.mean(1) - T_BASE_C.mean()) * 100
            ag = (corr[0] == T_BASE_C).mean()
            if ag < 0.999: raise AssertionError(f"{key}: 'none' copy agrees with baseline on only {ag*100:.2f}%")
            if ch == 0 and abs(d_pp[1] - REAL26[g]) > 0.02:
                raise AssertionError(f"{key}: real group {d_pp[1]:+.3f} pp differs from Step 26 ({REAL26[g]:+.3f} pp)")
            with lock:
                save_checkpoint(f"{PREFIX_G53}_{g}_c{ch}", {"group": g, "labels": [k[0] for k in ks], "delta_pp": d_pp.tolist(),
                                "displacement": dsp.tolist(), "correct_packed": [pk(x) for x in corr], "lp_true": lpt, "rank": rank})
                nul = d_pp[2:] if ch == 0 else d_pp[1:]
                print(f"[{S['dev']}] {g} chunk {ch}: {(time.time()-t0)/60:4.1f} min | "
                      + (f"real {d_pp[1]:+.2f} (Step 26 {REAL26[g]:+.2f}) | " if ch == 0 else "")
                      + f"{'independent' if ch == 0 else 'shared'} nulls {nul.mean():+.2f} ± {nul.std(ddof=1):.2f} pp | "
                      f"{(time.time()-t_all)/60:.0f} min elapsed")
        except Exception as ex:
            with lock: errors.append(f"[{S['dev']}] {key}: {ex!r}")
            return
_ts = [threading.Thread(target=_worker, args=(S,)) for S in T_SETUPS]
[t.start() for t in _ts]; [t.join() for t in _ts]
if errors: raise RuntimeError("Stopped: " + " | ".join(errors))
print(f"\nSTEP 53B COMPLETE ({(time.time()-t_all)/60:.1f} min)")

### Step 53C — Summary: real vs independent vs shared group nulls (CPU only)

Accuracy change, ratio to the sum of single-block effects (Step 9 real, Step 27 ten-permutation nulls), residual-stream displacement after L3B3, and paired bootstrap CIs (2,000 resamples) for real vs each null and for shared vs independent null.

In [ ]:
# ============================================================
# STEP 53C — SUMMARY: REAL vs INDEPENDENT vs SHARED GROUP NULLS (CPU ONLY)
# ============================================================
idx = np.random.RandomState(42).randint(0, N_FULL, (NB_BOOT, N_FULL)).astype(np.int32)
_s27 = load_checkpoint("step27_multi_perm_null")
single_real = {b: (T_REAL_C[b].mean() - T_BASE_C.mean()) * 100 for b in T_NAMES}
single_null = {b: (np.mean([upk(p).mean() for p in _s27[b]["correct_packed"]]) - T_BASE_C.mean()) * 100 for b in T_NAMES}
out53 = {}
for g, names in GROUPS.items():
    a = load_checkpoint(f"{PREFIX_G53}_{g}_c0"); b = load_checkpoint(f"{PREFIX_G53}_{g}_c1")
    ca = np.stack([upk(p) for p in a["correct_packed"]]).astype(np.float64)
    cb = np.stack([upk(p) for p in b["correct_packed"]]).astype(np.float64)
    real, ind, shd = ca[1], ca[2:].mean(0), cb[1:].mean(0)
    sr, sn = sum(single_real[x] for x in names), sum(single_null[x] for x in names)
    r = {"real_pp": a["delta_pp"][1], "ind_pp": float(np.mean(a["delta_pp"][2:])), "ind_sd": float(np.std(a["delta_pp"][2:], ddof=1)),
         "shd_pp": float(np.mean(b["delta_pp"][1:])), "shd_sd": float(np.std(b["delta_pp"][1:], ddof=1)),
         "sum_single_real": sr, "sum_single_null": sn,
         "disp_real": a["displacement"][1], "disp_ind": float(np.mean(a["displacement"][2:])), "disp_shd": float(np.mean(b["displacement"][1:]))}
    for nm, x in [("C_ind", (ind - real) * 100), ("C_shd", (shd - real) * 100), ("shd_minus_ind", (shd - ind) * 100)]:
        bm = _bm(x, idx); r[nm] = [float(x.mean())] + np.percentile(bm, [2.5, 97.5]).tolist()
    out53[g] = r
    print(f"\n=== {g} group ({len(names)} blocks: {', '.join(names)}) ===")
    print(f"accuracy change (pp): real {r['real_pp']:+.2f} | independent null {r['ind_pp']:+.2f} ± {r['ind_sd']:.2f} | shared null {r['shd_pp']:+.2f} ± {r['shd_sd']:.2f}")
    print(f"ratio to sum of single effects: real {r['real_pp']/sr:.2f}x | independent {r['ind_pp']/sn:.2f}x | shared {r['shd_pp']/sn:.2f}x")
    print(f"residual displacement after L3B3 (relative): real {r['disp_real']:.3f} | independent {r['disp_ind']:.3f} | shared {r['disp_shd']:.3f}")
    for nm in ["C_ind", "C_shd", "shd_minus_ind"]:
        print(f"  {nm:14s} {r[nm][0]:+.3f} pp [{r[nm][1]:+.3f}, {r[nm][2]:+.3f}]")
save_checkpoint(f"{PREFIX_G53}_summary", out53)
print("\nSTEP 53C COMPLETE — send me the full output.")

# Step 54 — Pair Nulls With a Shared Permutation, and the Noise Floor of R

**Question.** The pairwise experiment (Step 41) found that the four strongest attention pairs compound far more under real ablation than under size-matched nulls. Those pair nulls used an independent permutation in each block (three permutations), so they have the same alignment issue as the group nulls of Step 53.

**Test.** For each pair (L2B4+L2B5, L2B5+L2B6, L2B4+L2B6, L2B6+L2B7), on all 50,000 images:
- real pair ablation (must reproduce Step 41),
- 10 **shared** nulls (the same permutation for both blocks, which keeps their alignment),
- interaction = pair effect − the two single effects (real: Step 9; null: Step 27, ten permutations), compared per image with a paired bootstrap and Holm correction over the four pairs.

The last cell also prints the **noise floor** of the content share R from Step 52 (review point M3).

### Step 54A — Pair nulls with a shared permutation: definitions and pre-check

Same alignment check as Step 53, now for the four attention pairs of the pairwise null experiment (Step 41). The earlier pair nulls used an independent permutation in each block (three permutations); here both blocks of a pair get the **same** permutation (ten permutations), which keeps the alignment between their perturbations. Requires Step 53A (`group_eval`).

In [ ]:
# ============================================================
# STEP 54A — PAIR NULLS WITH A SHARED PERMUTATION: DEFINITIONS + PRE-CHECK (≈1 MIN)
# ============================================================
assert "group_eval" in globals() and "GROUPS" in globals(), "Run Step 53A first (it defines group_eval)."
PREFIX_P54 = "step54_pair"
S41 = load_checkpoint("step41_pairwise_null")["summary"]           # real pairs and 3-permutation independent pair nulls
PAIRS = list(S41.keys()); print("pairs:", PAIRS)
for p in PAIRS:
    GROUPS[p] = sorted(p.split("+"), key=T_NAMES.index)
    assert all(b in T_NAMES for b in GROUPS[p]) and len(set(CH[b] for b in GROUPS[p])) == 1, p

def kinds_pair(p):
    """none, real, 10 shared nulls: the same permutation for both blocks of the pair (keeps their alignment)."""
    ks = [("none", "none", None), ("real", "real", None)]
    for j in range(10):
        perm = torch.randperm(CH[GROUPS[p][0]], generator=torch.Generator().manual_seed(6000 + j))
        ks.append((f"shd{j}", "shuf", {b: perm for b in GROUPS[p]}))
    return ks
PJOBS = {p: kinds_pair(p) for p in PAIRS}

for S in T_SETUPS:
    torch.cuda.set_device(S["dev"])
    c, _, _, dsp = group_eval(S, PAIRS[0], PJOBS[PAIRS[0]][:3], 512, 64)
    ag = (c[0] == T_BASE_C[:512]).mean()
    print(f"pre-check {S['dev']}: none vs baseline {ag*100:.2f}% | displacement real/shd0 = {dsp[1]:.3f}/{dsp[2]:.3f}")
    assert ag >= 1 - 2 / 512 and dsp[0] < 1e-6, "Pre-check failed - stop and send me this output."
print("\nSTEP 54A COMPLETE — safe to run Step 54B")

### Step 54B — Four pairs: real + 10 shared nulls, both GPUs (50,000 images)

One job per pair (12 copies: unmodified, real, ten shared nulls), split over both T4 GPUs and saved per pair, so the step resumes after a disconnect. Safety checks: the unmodified copy must equal the baseline (≥ 99.9%), and the real pair ablation must reproduce Step 41 within 0.02 pp. Expected time ≈ 25–35 min.

In [ ]:
# ============================================================
# STEP 54B — FOUR PAIRS: REAL + 10 SHARED NULLS, BOTH GPUs (50K, ≈25-35 MIN)
# ============================================================
assert "PJOBS" in globals(), "Run Step 54A first."
done = set(list_checkpoints()); todo = [p for p in PAIRS if f"{PREFIX_P54}_{p}" not in done]
print(f"{len(PAIRS)-len(todo)} done, {len(todo)} to run")
q = queue.Queue(); [q.put(p) for p in todo]; lock = threading.Lock(); errors = []; t_all = time.time()
def _worker(S):
    torch.cuda.set_device(S["dev"])
    while not errors:
        try: p = q.get_nowait()
        except queue.Empty: return
        try:
            t0 = time.time(); ks = PJOBS[p]
            corr, lpt, rank, dsp = group_eval(S, p, ks, N_FULL, 32)
            d_pp = (corr.mean(1) - T_BASE_C.mean()) * 100
            ag = (corr[0] == T_BASE_C).mean()
            if ag < 0.999: raise AssertionError(f"{p}: 'none' copy agrees with baseline on only {ag*100:.2f}%")
            if abs(d_pp[1] - S41[p]["real_pair_pp"]) > 0.02:
                raise AssertionError(f"{p}: real pair {d_pp[1]:+.3f} pp differs from Step 41 ({S41[p]['real_pair_pp']:+.3f} pp)")
            with lock:
                save_checkpoint(f"{PREFIX_P54}_{p}", {"pair": p, "labels": [k[0] for k in ks], "delta_pp": d_pp.tolist(),
                                "displacement": dsp.tolist(), "correct_packed": [pk(x) for x in corr], "lp_true": lpt, "rank": rank})
                print(f"[{S['dev']}] {p}: {(time.time()-t0)/60:4.1f} min | real {d_pp[1]:+.2f} (Step 41 {S41[p]['real_pair_pp']:+.2f}) | "
                      f"shared nulls {d_pp[2:].mean():+.2f} ± {d_pp[2:].std(ddof=1):.2f} pp | {(time.time()-t_all)/60:.0f} min elapsed")
        except Exception as ex:
            with lock: errors.append(f"[{S['dev']}] {p}: {ex!r}")
            return
_ts = [threading.Thread(target=_worker, args=(S,)) for S in T_SETUPS]
[t.start() for t in _ts]; [t.join() for t in _ts]
if errors: raise RuntimeError("Stopped: " + " | ".join(errors))
print(f"\nSTEP 54B COMPLETE ({(time.time()-t_all)/60:.1f} min)")

### Step 54C — Pair summary and noise floor of the content share (CPU only)

Interaction = pair effect minus the two single effects (real: Step 9; null: Step 27, ten permutations). We compare the real interaction with the shared-null interaction per image (paired bootstrap, Holm over four pairs), list the earlier independent pair-null interactions (Step 41) for comparison, and print the noise floor of the content share R from Step 52 (review point M3).

In [ ]:
# ============================================================
# STEP 54C — PAIR SUMMARY + NOISE FLOOR OF THE CONTENT SHARE (CPU ONLY)
# ============================================================
idx = np.random.RandomState(42).randint(0, N_FULL, (NB_BOOT, N_FULL)).astype(np.int32)
_s27 = load_checkpoint("step27_multi_perm_null")
BASE = T_BASE_C.astype(np.float64)
NULL1 = {b: np.mean([upk(p) for p in _s27[b]["correct_packed"]], 0) for b in T_NAMES}   # single-block null, 10 perms
REAL1 = {b: T_REAL_C[b].astype(np.float64) for b in T_NAMES}
out54 = {}
print(f"{'pair':11s}{'real pair':>10s}{'real inter.':>12s}{'shared null pair':>18s}{'shared inter.':>14s}{'ratio real/shared':>19s}"
      f"{'real − shared interaction [95% CI]':>36s}")
for p in PAIRS:
    d = load_checkpoint(f"{PREFIX_P54}_{p}"); a, b = GROUPS[p]
    c = np.stack([upk(x) for x in d["correct_packed"]]).astype(np.float64)
    real_pair, shd_pair = c[1], c[2:].mean(0)
    i_real = (real_pair - BASE) - (REAL1[a] - BASE) - (REAL1[b] - BASE)
    i_shd = (shd_pair - BASE) - (NULL1[a] - BASE) - (NULL1[b] - BASE)
    diff = (i_real - i_shd) * 100; bm = _bm(diff, idx)
    sr = ((REAL1[a] - BASE).mean() + (REAL1[b] - BASE).mean()) * 100; sn = ((NULL1[a] - BASE).mean() + (NULL1[b] - BASE).mean()) * 100
    r = {"real_pair_pp": d["delta_pp"][1], "real_interaction": float(i_real.mean() * 100),
         "shd_pair_pp": float(np.mean(d["delta_pp"][2:])), "shd_sd": float(np.std(d["delta_pp"][2:], ddof=1)),
         "shd_interaction": float(i_shd.mean() * 100), "ratio_real": d["delta_pp"][1] / sr, "ratio_shd": float(np.mean(d["delta_pp"][2:])) / sn,
         "indep_interaction_step41": S41[p]["null_interaction_mean"],
         "real_minus_shd": [float(diff.mean())] + np.percentile(bm, [2.5, 97.5]).tolist(),
         "p": max(float(2 * min((bm <= 0).mean(), (bm >= 0).mean())), 1.0 / NB_BOOT),
         "disp_real": d["displacement"][1], "disp_shd": float(np.mean(d["displacement"][2:]))}
    out54[p] = r
    print(f"{p:11s}{r['real_pair_pp']:+10.2f}{r['real_interaction']:+12.2f}{r['shd_pair_pp']:+12.2f} ± {r['shd_sd']:.2f}{r['shd_interaction']:+14.2f}"
          f"{r['ratio_real']:11.2f} / {r['ratio_shd']:.2f}{r['real_minus_shd'][0]:+14.2f} [{r['real_minus_shd'][1]:+.2f}, {r['real_minus_shd'][2]:+.2f}]")
ps = _holm([out54[p]["p"] for p in PAIRS])
for p, h in zip(PAIRS, ps): out54[p]["p_holm"] = float(h)
print("\nHolm p (real vs shared interaction):", {p: f"{out54[p]['p_holm']:.1e}" for p in PAIRS})
print("independent pair-null interaction (Step 41, 3 perms):", {p: round(out54[p]["indep_interaction_step41"], 3) for p in PAIRS})
print("residual displacement real / shared:", {p: f"{out54[p]['disp_real']:.3f}/{out54[p]['disp_shd']:.3f}" for p in PAIRS})
save_checkpoint(f"{PREFIX_P54}_summary", out54)

# ---- noise floor of the content share R (from Step 52) ----
r52 = load_checkpoint("step52_relative_content")
print("\nContent share R and its noise floor (Step 52):")
for k, v in r52.items():
    print(f"  {k:32s} R {v['S']:.3f} [{v['S_ci'][0]:.3f}, {v['S_ci'][1]:.3f}] | noise floor {v['noise_floor_uncorrected']:.3f}")
print("\nSTEP 54C COMPLETE — send me the full output.")

# Step 55 — Group Nulls With Shared Permutations for MambaVision-S and -B

**Question.** Step 53 showed for MambaVision-T that whole groups of same-type blocks compound far more under real ablation than under size-matched nulls, even nulls that keep the alignment between the blocks' perturbations (shared permutation). For MambaVision-S and -B the group null so far used only one independent permutation per block, which destroys that alignment. Does the group-level result hold at all three model sizes?

**Test.** For each model and each group (all Mamba mixers / all attention mixers), on all 50,000 images:
- real group ablation (must reproduce Step 29 for S and Step 37 for B within 0.03 pp),
- 10 **independent** nulls (own permutation per block),
- 10 **shared** nulls (one permutation per stage for all blocks of the group),
- relative displacement of the residual stream after the last block.

## Step 55A — Definitions for any MambaVision size (no GPU work)

`make_ctx` reads the groups, block types and earlier real group effects from the Step 29 (S) or Step 37 (B) checkpoint and builds the 2 × 2 jobs per model (each group: chunk 0 = unmodified, real, ten independent nulls; chunk 1 = unmodified, ten shared nulls). `group_eval_m` is the Step 53 group fork for any model, with the γ1 rule of Step 51. `run_groups_m` runs the jobs on both GPUs with safety checks and saves each job separately.

In [ ]:
# ============================================================
# STEP 55A — GROUP NULLS FOR ANY MAMBAVISION SIZE: DEFINITIONS ONLY (NO GPU WORK)
# ============================================================
import threading, queue
assert "DEVS" in globals() and "LAB_DEV" in globals() and "_bm" in globals(), "Run Step 45A and Step 46A first."

def make_ctx(tag, setups, names, means, base_c, res_ckpt):
    """tag: 's' or 'b'. Groups and types from the Step 29/37 checkpoint; real group effects from its E3 entry."""
    E3 = res_ckpt["E3"]; ctx = {"tag": tag, "setups": setups, "names": names, "means": means, "base_c": base_c}
    ctx["groups"] = {g: sorted(E3[g]["blocks"], key=names.index) if "blocks" in E3[g] else None for g in ["mamba", "attn"]}
    types = {n: res_ckpt["E2E5"][n]["type"] for n in names}
    if ctx["groups"]["mamba"] is None: ctx["groups"]["mamba"] = [n for n in names if types[n] == "MambaVisionMixer"]
    if ctx["groups"]["attn"] is None:  ctx["groups"]["attn"] = [n for n in names if types[n] != "MambaVisionMixer"]
    assert all(types[n] == "MambaVisionMixer" for n in ctx["groups"]["mamba"]), ctx["groups"]
    assert all(types[n] != "MambaVisionMixer" for n in ctx["groups"]["attn"]), ctx["groups"]
    ctx["real_ref"] = {g: float(E3[g]["delta_pp"]) for g in ["mamba", "attn"]}
    ctx["ch"] = {n: means[n].shape[-1] for n in names}
    ctx["gamma"] = {n: setups[0]["blocks"][n][2] for n in names}           # γ1 per block (None if the block has none)
    jobs = {}
    for g, blocks in ctx["groups"].items():
        ks = [("none", "none", None), ("real", "real", None)]
        for j in range(10):
            ks.append((f"ind{j}", "shuf", {b: torch.randperm(ctx["ch"][b], generator=torch.Generator().manual_seed(7000 + 100 * names.index(b) + j))
                                           for b in blocks}))
        for j in range(10):
            sp = {}
            for b in blocks:
                st = int(b[1])
                if st not in sp: sp[st] = torch.randperm(ctx["ch"][b], generator=torch.Generator().manual_seed(8000 + 100 * st + j))
            ks.append((f"shd{j}", "shuf", {b: sp[int(b[1])] for b in blocks}))
        jobs[(g, 0)] = ks[:12]; jobs[(g, 1)] = [ks[0]] + ks[12:]
    ctx["jobs"] = jobs
    print(f"[{tag}] groups:", {g: v for g, v in ctx["groups"].items()}, "| real (earlier run):", ctx["real_ref"],
          "| blocks with γ1:", [n for n in names if ctx["gamma"][n] is not None])
    return ctx

def group_eval_m(ctx, S, group, kinds, n, bs):
    """As Step 53 (group_eval), for any model: shuffled nulls are applied in residual space when the block has γ1."""
    K = len(kinds); names = ctx["groups"][group]; dev = S["dev"]; lab = LAB_DEV[dev]
    assert kinds[0][1] == "none"
    def pre(module, args, kwargs):
        if len(args) > 0:
            x = args[0]; return (x.repeat(K, *([1] * (x.dim() - 1))),) + tuple(args[1:]), kwargs
        kwargs = dict(kwargs); x = kwargs["hidden_states"]
        kwargs["hidden_states"] = x.repeat(K, *([1] * (x.dim() - 1))); return args, kwargs
    def make_post(b):
        mv = ctx["means"][b].to(dev); gm = ctx["gamma"][b]; gm = None if gm is None else gm.to(dev)
        def post(module, inp, out):
            is_t = isinstance(out, tuple); t = out[0] if is_t else out
            R = t.shape[0] // K; m = mv.to(dtype=t.dtype); g = None if gm is None else gm.to(dtype=t.dtype); pieces = []
            for j, (l, kind, p) in enumerate(kinds):
                tj = t[j * R:(j + 1) * R]
                if kind == "none":   pieces.append(tj)
                elif kind == "real": pieces.append(m.expand_as(tj).clone())
                else:
                    d = m.expand_as(tj) - tj; pp = p[b].to(dev)
                    pieces.append(tj + (d[..., pp] if g is None else (g * d)[..., pp] / g))
            rep = torch.cat(pieces, 0)
            return (rep,) + tuple(out[1:]) if is_t else rep
        return post
    disp = np.zeros((K, 2))
    def last_hook(module, inp, out):
        t = (out[0] if isinstance(out, tuple) else out).float(); R = t.shape[0] // K
        t = t.reshape(K, R, -1); t0 = t[0]
        disp[:, 0] += ((t - t0.unsqueeze(0)) ** 2).sum(dim=(1, 2)).cpu().numpy(); disp[:, 1] += float((t0 ** 2).sum())
    hs = [S["blocks"][names[0]][0].register_forward_pre_hook(pre, with_kwargs=True)]
    hs += [S["blocks"][b][1].register_forward_hook(make_post(b)) for b in names]
    hs.append(S["blocks"][ctx["names"][-1]][0].register_forward_hook(last_hook))
    corr = np.zeros((K, n), bool); lpt = np.zeros((K, n), np.float32); rank = np.zeros((K, n), np.int16)
    try:
        with torch.no_grad(), torch.cuda.device(dev):
            for s in range(0, n, bs):
                e = min(s + bs, n); B = e - s
                logits = S["call"](S["get"](s, e)).float().view(K, B, -1)
                y = lab[s:e].view(1, B, 1).expand(K, B, 1)
                corr[:, s:e] = (logits.argmax(-1) == y.squeeze(2)).cpu().numpy()
                lpt[:, s:e] = F.log_softmax(logits, -1).gather(2, y).squeeze(2).cpu().numpy()
                rank[:, s:e] = (logits > logits.gather(2, y)).sum(-1).cpu().numpy().astype(np.int16)
    finally:
        for h in hs: h.remove()
    return corr, lpt, rank, np.sqrt(disp[:, 0] / disp[:, 1])

def pre_check_m(ctx):
    """512 images on each GPU, attention group: unmodified copy = baseline; displacement of the unmodified copy = 0."""
    for S in ctx["setups"]:
        torch.cuda.set_device(S["dev"])
        J0, J1 = ctx["jobs"][("attn", 0)], ctx["jobs"][("attn", 1)]
        c, _, _, dsp = group_eval_m(ctx, S, "attn", [J0[0], J0[1], J0[2], J1[1]], 512, 32)
        ag = (c[0] == ctx["base_c"][:512]).mean()
        print(f"[{ctx['tag']}] pre-check {S['dev']}: none vs baseline {ag*100:.2f}% | acc none/real/ind0/shd0 = "
              f"{', '.join(f'{x*100:.1f}' for x in c.mean(1))} | displacement real/ind0/shd0 = {dsp[1]:.3f}/{dsp[2]:.3f}/{dsp[3]:.3f}")
        assert ag >= 1 - 2 / 512 and dsp[0] < 1e-6, "Pre-check failed - stop and send me this output."

def run_groups_m(ctx, bs):
    pref = f"step55_{ctx['tag']}"; base = ctx["base_c"]
    done = set(list_checkpoints()); todo = [k for k in ctx["jobs"] if f"{pref}_{k[0]}_c{k[1]}" not in done]
    print(f"[{ctx['tag']}] {len(ctx['jobs'])-len(todo)} done, {len(todo)} to run")
    q = queue.Queue(); [q.put(k) for k in todo]; lock = threading.Lock(); errors = []; t_all = time.time()
    def _worker(S):
        torch.cuda.set_device(S["dev"])
        while not errors:
            try: key = q.get_nowait()
            except queue.Empty: return
            try:
                t0 = time.time(); g, ch = key; ks = ctx["jobs"][key]
                corr, lpt, rank, dsp = group_eval_m(ctx, S, g, ks, N_FULL, bs)
                d_pp = (corr.mean(1) - base.mean()) * 100
                ag = (corr[0] == base).mean()
                if ag < 0.999: raise AssertionError(f"{key}: 'none' copy agrees with baseline on only {ag*100:.2f}%")
                if ch == 0 and abs(d_pp[1] - ctx["real_ref"][g]) > 0.03:
                    raise AssertionError(f"{key}: real group {d_pp[1]:+.3f} pp differs from the earlier run ({ctx['real_ref'][g]:+.3f} pp)")
                with lock:
                    save_checkpoint(f"{pref}_{g}_c{ch}", {"group": g, "blocks": ctx["groups"][g], "labels": [k[0] for k in ks],
                                    "delta_pp": d_pp.tolist(), "displacement": dsp.tolist(),
                                    "correct_packed": [pk(x) for x in corr], "lp_true": lpt, "rank": rank})
                    nul = d_pp[2:] if ch == 0 else d_pp[1:]
                    print(f"[{S['dev']}] {ctx['tag']} {g} chunk {ch}: {(time.time()-t0)/60:4.1f} min | "
                          + (f"real {d_pp[1]:+.2f} (earlier {ctx['real_ref'][g]:+.2f}) | " if ch == 0 else "")
                          + f"{'independent' if ch == 0 else 'shared'} nulls {nul.mean():+.2f} ± {nul.std(ddof=1):.2f} pp | "
                          f"{(time.time()-t_all)/60:.0f} min elapsed")
            except Exception as ex:
                with lock: errors.append(f"[{S['dev']}] {key}: {ex!r}")
                return
    _ts = [threading.Thread(target=_worker, args=(S,)) for S in ctx["setups"]]
    [t.start() for t in _ts]; [t.join() for t in _ts]
    if errors: raise RuntimeError("Stopped: " + " | ".join(errors))
    print(f"[{ctx['tag']}] all group jobs done ({(time.time()-t_all)/60:.1f} min)")
print("STEP 55A COMPLETE — definitions ready")

## Step 55B — MambaVision-S: pre-check and group ablation (both GPUs, ≈ 1 hour)

Requires Step 50A (MambaVision-S on both GPUs). Pre-check on 512 images on each GPU, then four jobs: Mamba and attention group, each with real + ten independent nulls and ten shared nulls. Stops if the unmodified copy differs from the baseline or the real group differs from Step 29 by more than 0.03 pp.

In [ ]:
# ============================================================
# STEP 55B — MAMBAVISION-S: PRE-CHECK + GROUP ABLATION, REAL + 10 INDEPENDENT + 10 SHARED NULLS (≈1 HOUR)
# ============================================================
assert "S_SETUPS" in globals() and "s_means10" in globals() and "S_BASE_C" in globals(), "Run Step 50A (MambaVision-S setup) first."
CTX_S = make_ctx("s", S_SETUPS, S_NAMES, s_means10, S_BASE_C, load_checkpoint("step29_mambavision_s"))
pre_check_m(CTX_S)
run_groups_m(CTX_S, bs=32)
print("\nSTEP 55B COMPLETE")

## Step 55C — MambaVision-B: pre-check and group ablation (both GPUs, ≈ 2 hours)

Requires Step 51A (MambaVision-B on both GPUs). Same four jobs as for S; the shuffled nulls use the γ1 residual-space rule of Step 51. Stops if the unmodified copy differs from the baseline or the real group differs from Step 37 by more than 0.03 pp.

In [ ]:
# ============================================================
# STEP 55C — MAMBAVISION-B: PRE-CHECK + GROUP ABLATION, REAL + 10 INDEPENDENT + 10 SHARED NULLS (≈2 HOURS)
# ============================================================
assert "B_SETUPS" in globals() and "b_means10" in globals() and "B_BASE_C" in globals(), "Run Step 51A (MambaVision-B setup) first."
CTX_B = make_ctx("b", B_SETUPS, B_NAMES, b_means10, B_BASE_C, load_checkpoint("step37_mambavision_b"))
pre_check_m(CTX_B)
run_groups_m(CTX_B, bs=24)
print("\nSTEP 55C COMPLETE")

## Step 55D — Summary for MambaVision-S and -B (CPU only)

Accuracy change of the real group, the independent and the shared nulls; ratio to the sum of single-block effects (ten-permutation single nulls from Steps 50 and 51); residual displacement after the last block; paired bootstrap CIs (2,000 resamples) for real vs each null and shared vs independent. A model whose jobs are not all saved is skipped.

In [ ]:
# ============================================================
# STEP 55D — SUMMARY FOR MAMBAVISION-S AND -B (CPU ONLY)
# ============================================================
idx = np.random.RandomState(42).randint(0, N_FULL, (NB_BOOT, N_FULL)).astype(np.int32)
ALL_CK = set(list_checkpoints()); out55 = {}
for tag, model_name, singles_prefix in [("s", "MambaVision-S", ["step50_s", "step48_s"]), ("b", "MambaVision-B", ["step51_b"])]:
    pref = f"step55_{tag}"
    if not all(f"{pref}_{g}_c{c}" in ALL_CK for g in ["mamba", "attn"] for c in (0, 1)):
        print(f"{model_name}: not all group jobs are saved yet - skipped"); continue
    sp = next(p for p in singles_prefix if f"{p}_baseline" in ALL_CK)
    out55[tag] = {}
    print("\n" + "=" * 90 + f"\n{model_name}  (single-block effects from {sp}, ten permutations)\n" + "=" * 90)
    for g in ["mamba", "attn"]:
        a = load_checkpoint(f"{pref}_{g}_c0"); b = load_checkpoint(f"{pref}_{g}_c1"); names = a["blocks"]
        sr = sn = 0.0
        for nm in names:
            d = load_checkpoint(f"{sp}_{nm}"); L = d["labels"]; dp = np.asarray(d["delta_pp"])
            sr += dp[0]; sn += dp[[i for i, l in enumerate(L) if l.startswith("shuf")]].mean()
        ca = np.stack([upk(p) for p in a["correct_packed"]]).astype(np.float64)
        cb = np.stack([upk(p) for p in b["correct_packed"]]).astype(np.float64)
        real, ind, shd = ca[1], ca[2:].mean(0), cb[1:].mean(0)
        r = {"blocks": names, "real_pp": a["delta_pp"][1], "ind_pp": float(np.mean(a["delta_pp"][2:])), "ind_sd": float(np.std(a["delta_pp"][2:], ddof=1)),
             "shd_pp": float(np.mean(b["delta_pp"][1:])), "shd_sd": float(np.std(b["delta_pp"][1:], ddof=1)),
             "sum_single_real": float(sr), "sum_single_null": float(sn),
             "disp_real": a["displacement"][1], "disp_ind": float(np.mean(a["displacement"][2:])), "disp_shd": float(np.mean(b["displacement"][1:]))}
        for nm, x in [("C_ind", (ind - real) * 100), ("C_shd", (shd - real) * 100), ("shd_minus_ind", (shd - ind) * 100)]:
            bm = _bm(x, idx); r[nm] = [float(x.mean())] + np.percentile(bm, [2.5, 97.5]).tolist()
        out55[tag][g] = r
        print(f"\n--- {g} group ({len(names)} blocks: {', '.join(names)}) ---")
        print(f"accuracy change (pp): real {r['real_pp']:+.2f} | independent null {r['ind_pp']:+.2f} ± {r['ind_sd']:.2f} | shared null {r['shd_pp']:+.2f} ± {r['shd_sd']:.2f}")
        print(f"ratio to sum of single effects: real {r['real_pp']/sr:.2f}x | independent {r['ind_pp']/sn:.2f}x | shared {r['shd_pp']/sn:.2f}x")
        print(f"residual displacement after the last block (relative): real {r['disp_real']:.3f} | independent {r['disp_ind']:.3f} | shared {r['disp_shd']:.3f}")
        for nm in ["C_ind", "C_shd", "shd_minus_ind"]:
            print(f"  {nm:14s} {r[nm][0]:+.3f} pp [{r[nm][1]:+.3f}, {r[nm][2]:+.3f}]")
save_checkpoint("step55_group_summary", out55)
print("\nSTEP 55D COMPLETE — send me the full output.")